# One workflow for development comparisons

Use one configuration and one command to produce a verified comparison, tables, figures and a portable HTML report. The first supported experiment is the existing **fixed reference model versus compact U-Net**, on all seven chronological blocks from 2007–2020. Architecture, training windows, input lags, weights and epoch selection remain unchanged.

There are two explicit modes:

| Mode | Inputs | Work performed |
|---|---|---|
| `saved` — default | Official observed rainfall and the seven saved prediction maps | Check provenance, reproduce errors, generate diagnostics and a report; no training |
| `train` | Official training files, prepared SEAS5 and CFSv2 | Run the existing seven-block experiment, then the same checks and report |

These are previously consulted **development years**. Neither mode claims an independent holdout, promotes a model or issues a prospective forecast. Operational data collection and its incomplete-source gate remain separate.

The optional `experiment` field selects `hybrid_unet_v1` (the default comparison) or `fixed_blend_v1` (one prespecified 75% reference model + 25% U-Net blend, saved mode only). The latter uses the [blend example](blend.example.json), retains the original diagnostics, and adds paired candidate tables, maps and a report. It never searches weights. See the [fixed-blend protocol and results](../fixed_blend/README.md).

## Easiest route: Kaggle

Import [experiment_workflow.ipynb](experiment_workflow.ipynb). Its source bundle is self-contained: no private GitHub login, API key or new source download is required for saved-mode evaluation.

1. Attach the official competition data and the **saved output** of the seven-block U-Net notebook. Verify that the output contains `spatial_unet/results/H1/predictions.nc` through `C/predictions.nc`, not just the notebook source or report ZIP.
2. Leave `MODE = "saved"`. The configuration cell finds one official rainfall file and one complete seven-block output. If duplicates are attached, set `MANUAL_INPUTS` to the intended paths; it will not silently choose a copy.
3. Run the check cell, then the execution cell. No GPU is needed in saved mode.
4. Open or download `report.html`. Save the notebook version **with outputs** to retain the run package.

To reproduce the fixed combination, set `EXPERIMENT = "fixed_blend_v1"` and leave `MODE = "saved"`. It uses the same attached maps; neither a GPU nor new training is needed. Existing notebook code bundles are preserved: import the updated notebook into a fresh session/output location if an older bundle is already present.

For training, set `MODE = "train"`, attach the three prepared input datasets and enable a GPU. This runs all seven blocks and needs the [fixed training dependencies](../spatial_unet/requirements.txt). It can be substantially more expensive than saved-mode analysis. The notebook does not install packages, download data or start training during its check cell.

## Local use

Run commands from the repository root using Python 3.12 or later. For saved-mode analysis in a separate environment:

```bash
python -m pip install -r research/workflow/requirements_evaluation.txt
python -m unittest research.workflow.test_workflow research.diagnostics.test_diagnostics research.fixed_blend.test_blend -v
```

Copy [saved.example.json](saved.example.json) to your own configuration file and edit its paths. Keep `evidence: null` only when using the original archived maps: it selects the bundled seven-block evidence. For another complete run of the same protocol, set both `predictions` and `evidence` to that run's results folder. Do not mix its maps with the original run's hashes.

Paths may contain spaces. Relative paths resolve from the **configuration file's directory**, not the shell's working directory. `output_root` must be separate from the input datasets and source folders.

```bash
python -m research.workflow check --config "/path/to/my_config.json"
python -m research.workflow run --config "/path/to/my_config.json"
python -m research.workflow verify --run "/path/to/outputs/experiments/RUN_ID"
python -m research.workflow list --root "/path/to/outputs/experiments"
```

`check` validates paths, source protocol, hashes and dependencies without creating a run. `run` repeats preflight to catch intervening changes. `verify` checks the complete output package, including added or missing files. `list` displays run status and recorded scores; it does not choose a winner or assert that listed runs have just been reverified.

The [training example](train.example.json) uses explicit official, SEAS5 and CFSv2 directories and a `device` setting. Use the training environment rather than installing evaluation pins over it. CUDA availability and the existing exact baseline package versions are checked before fitting. A changed architecture or scientific protocol needs a separate reviewed adapter; unknown configuration keys are rejected.

## What each run contains

Each run gets a UTC timestamp and a fingerprint derived from configuration-relevant settings, code, input hashes and environment. Paths and output-directory names are not scientific identity. Repeated evaluations create separate directories, with the same fingerprint when the recorded scientific and software inputs match.

```text
RUN_ID/
  config.json              Resolved user settings
  identity.json            Protocol, source/code hashes, environment and summary
  state.json               Running, completed, failed or interrupted
  execution.log            Progress from this run
  code/                    Source snapshot captured before execution
  source_evidence/         Signature, block completion records and histories
  diagnostics/             Pooled metrics, error maps and four figures
  comparison/              Block/year comparisons against the reference model
  report.html              Portable report with embedded figures
  REPORT.md                Short result and report entry point
  manifest.json            Completed output inventory and hashes
  training/                Models and forecasts, only in train mode
  fixed_blend/             Prespecified blend maps/tables, only for fixed_blend_v1
```

The output snapshot preserves configuration and code but does not vendor Python packages or copy the large source datasets in saved mode. Reproduction still requires the same input snapshots. The HTML can be read by itself; full verification requires the entire run directory. Hashes check byte integrity and consistency, not independent authenticity or real historical publication dates.

Existing runs are never overwritten. A failed or interrupted run keeps its partial output and error stage, and `verify` refuses it. Correct the cause and launch a new run. If fitting completed but report generation failed, saved mode can evaluate the complete `training/` output, avoiding another fit. Incomplete training blocks are not silently resumed or treated as a seven-block comparison.

## Checks and interpretation

The workflow reuses the [archived-map diagnostic](../diagnostics/README.md): exact dates, units and grid; prediction and observation hashes; 2,016 monthly error records; pooled global metrics; intensity partitions; and original epoch selection. It adds strict configuration checks, a code/input fingerprint, source snapshots, lifecycle status, an output manifest and an HTML report. Evaluation dependencies are recorded separately from the environment that originally trained the models.

Regional, seasonal and intensity differences remain descriptive. Future observed rainfall cannot select an operational model. No blend is fitted from the report, and a small development-score improvement is not automatically an accepted model.

This implements the evaluation consolidation and brings forward part of the usability work: one configuration, early error messages, an explicit saved-versus-training choice, reproducible outputs and tests. New architectures, source migration, model tuning and a general-purpose experiment platform are outside this change.

## Validation of this integration

The CLI completed saved-mode evaluation on all 168 real months and reproduced the archived RMSEs: reference model **1.7533987173382521**, U-Net **1.764725739441376**. The completed run verified all 115 output artifacts. Thirteen configuration, integrity and diagnostic tests passed. The standalone notebook bootstrap was executed in an isolated local directory and its preflight verified all 73 configured input artifacts; notebook syntax and bundled-source consistency were also checked.

The HTML report was inspected in a browser. The full Kaggle notebook and a new GPU training run were not executed for this integration. Train mode delegates to the unchanged experiment implementation; the existing seven-block training result remains the measured evidence. See the [validation record](evidence/validation.json).


In [ ]:
from pathlib import Path
import hashlib, json, sys, runpy, shutil
BASE = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'outputs'
RUN_ROOT = BASE/'experiment_workflow'
CODE_ROOT = RUN_ROOT/'code'
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = {'research/common/__init__.py': '"""Audited input loading and fixed-reference model helpers."""\n', 'research/common/baseline.py': '"""Fit and predict the fixed reference model with longer input lags, without changing its architecture."""\nfrom pathlib import Path\nimport gc\nimport numpy as np\nimport xarray as xr\nfrom .inputs import require, sha256, write_json\n\n\ndef fit(data, cutoff, folder):\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    m = data.lib\n    train = data.rain.sel(time=slice(None, cutoff))\n    X, y, ids, dates, climate, atmosphere, sums, counts, seas, cfs = m.amostrar_arvores_multissistema(\n        train, data.atmosphere, data.seas, data.cfs, cutoff)\n    models = []\n    for name, features in [(\'arvores_seas5\', m.FEATURES_SEAS5), (\'arvores_multissistema\', m.FEATURES_MULTISSISTEMA)]:\n        dataset = m.lgb.Dataset(np.ascontiguousarray(X[:, :len(features)]), label=y, feature_name=features)\n        model = m.lgb.train(m.PARAMETROS, dataset, num_boost_round=m.ARVORES)\n        require(model.feature_name() == features, \'Baseline feature order changed.\')\n        model.save_model(str(folder / (name + \'.txt\')))\n        models.append(model)\n        del dataset\n    ridge = m.mos_fit(train, data.seas, data.cfs, cutoff)\n    np.savez_compressed(folder / \'ridge.npz\', **ridge)\n    np.savez_compressed(folder / \'clima_atmosfera_indices.npz\', **atmosphere)\n    for name, field in [(\'chuva\', climate), (\'seas5\', seas), (\'cfsv2\', cfs)]:\n        field.to_netcdf(folder / (\'clima_\' + name + \'.nc\'))\n    audit = dict(training_start=str(dates[0].date()), training_end=str(dates[-1].date()),\n        training_months=len(dates), examples=len(y), parameters=m.PARAMETROS,\n        tree_count=m.ARVORES, weights=[.375, .375, .25], residual_scales=[.9, .875],\n        sample_hash=__import__(\'hashlib\').sha256(ids.tobytes()).hexdigest(), inputs=data.audit)\n    del X, y, ids\n    gc.collect()\n    write_json(folder / \'fit.json\', audit)\n    return dict(models=models, ridge=ridge, climate=climate, atmosphere=atmosphere, seas=seas, cfs=cfs, audit=audit)\n\n\ndef predict(data, state, dates):\n    m = data.lib\n    components = []\n    for model, name, scale in zip(state[\'models\'], m.NOMES_MODELOS, [.9, .875]):\n        anomalies = m.prever_par(model, name, data.atmosphere, data.seas, data.cfs,\n            state[\'atmosphere\'], state[\'climate\'], state[\'seas\'], state[\'cfs\'], dates)\n        components.append(m.reconstruir_chuva(state[\'climate\'], anomalies, scale).values.astype(\'float64\'))\n    ridge = m.mos_predict(state[\'ridge\'], data.seas, data.cfs, dates)\n    result = .75 * (.5 * components[0] + .5 * components[1]) + .25 * ridge\n    return xr.DataArray(result, dims=(\'time\', \'lat\', \'lon\'),\n        coords=dict(time=dates, lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n', 'research/common/inputs.py': '"""Load the existing audited inputs; no downloads, imputation or new sources."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nLIBRARY = ROOT / \'research/lagged_sources/library.py\'\nREQUIRED = {\'numpy\': \'2.0.2\', \'pandas\': \'2.3.3\', \'xarray\': \'2025.12.0\', \'lightgbm\': \'4.6.0\'}\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False, default=str) + \'\\n\', encoding=\'utf-8\')\n\n\ndef library(path=LIBRARY):\n    # Separate module globals: callers cannot change another run\'s index table.\n    spec = importlib.util.spec_from_file_location(\'fixed_lag_library\', path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef find_unique(filename, supplied=None):\n    if supplied:\n        p = Path(supplied)\n        require((p / filename).is_file(), f\'Missing {filename} in {p}\')\n        return p.resolve()\n    candidates = sorted({p.parent.resolve() for p in Path(\'/kaggle/input\').rglob(filename)})\n    require(len(candidates) == 1, f\'Attach exactly one input containing {filename}, or set its directory explicitly. Found {len(candidates)}.\')\n    return candidates[0]\n\n\ndef preflight(official=None, seas5=None, cfsv2=None, final=False):\n    paths = dict(official=find_unique(\'treino_tp.nc\', official),\n                 seas5=find_unique(\'seas5_51_manifesto.json\', seas5),\n                 cfsv2=find_unique(\'cfsv2_manifesto.json\', cfsv2))\n    expected = json.loads((ROOT / \'research/lagged_sources/official_hashes.json\').read_text())\n    files = {}\n    for row in expected:\n        if not row[\'nome\'].startswith(\'treino_\'):\n            continue\n        p = paths[\'official\'] / row[\'nome\']\n        require(p.is_file(), f\'Missing official input: {p.name}\')\n        require(sha256(p) == row[\'sha256\'], f\'Official input hash mismatch: {p.name}\')\n        files[p.name] = row[\'sha256\']\n    for source, name in [(\'seas5\', \'seas5_51_manifesto.json\'), (\'cfsv2\', \'cfsv2_manifesto.json\')]:\n        manifest = paths[source] / name\n        files[name] = sha256(manifest)\n        m = json.loads(manifest.read_text(encoding=\'utf-8\'))\n        for phase in [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else []):\n            row = m[\'arquivos\'][phase]\n            p = paths[source] / row[\'arquivo\']\n            require(p.is_file() and sha256(p) == row[\'sha256\'], f\'Missing or altered {source}: {phase}\')\n            files[p.name] = row[\'sha256\']\n    versions = {n: importlib.metadata.version(n) for n in REQUIRED}\n    require(versions == REQUIRED, f\'The fixed baseline needs {REQUIRED}; found {versions}.\')\n    return paths, dict(files=files, versions=versions, final_training=final)\n\n\nclass Inputs:\n    def __init__(self, official=None, seas5=None, cfsv2=None, output=\'outputs\', final=False):\n        self.paths, self.audit = preflight(official, seas5, cfsv2, final)\n        self.lib = m = library()\n        m.PASTA = self.paths[\'official\']\n        m.PASTA_SEAS5_MANUAL = self.paths[\'seas5\']\n        m.PASTA_CFSV2_MANUAL = self.paths[\'cfsv2\']\n        m.PASTA_SEAS5, m.MANIFESTO_SEAS5 = m.localizar_seas5()\n        m.PASTA_CFSV2, m.MANIFESTO_CFSV2 = m.localizar_cfsv2()\n        m.SAIDA = Path(output)\n        m.SAIDA.mkdir(parents=True, exist_ok=True)\n        end = \'2022-12-01\' if final else \'2020-12-01\'\n        start = \'1993-01-01\' if final else \'1976-10-01\'\n        with xr.open_dataset(m.PASTA / \'treino_tp.nc\') as ds:\n            require(ds.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n            self.rain = ds.tp.sel(time=slice(start, end)).transpose(\'time\', \'lat\', \'lon\').load()\n        require(pd.DatetimeIndex(self.rain.time.values).equals(pd.date_range(start, end, freq=\'MS\')), \'Incomplete rainfall calendar.\')\n        require(np.array_equal(self.rain.lat, np.arange(-60, 15.25, .25)) and\n                np.array_equal(self.rain.lon, np.arange(-90, -24.75, .25)), \'Different target grid.\')\n        require(np.isfinite(self.rain.values).all() and (self.rain.values >= 0).all(), \'Invalid rainfall.\')\n        atm_start = pd.Timestamp(start) - pd.DateOffset(months=4)\n        atm_end = pd.Timestamp(end) - pd.DateOffset(months=4)\n        self.atmosphere, _ = m.carregar_atmosfera(self.rain, atm_start, atm_end)\n        index_path = ROOT / (\'competition/metadata/NOAA/indices_noaa.csv\' if final else \'research/lagged_sources/ocean_indices.csv\')\n        m.INDICES_OC = pd.read_csv(index_path, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n        m.conferir_indices(m.INDICES_OC)\n        self.audit[\'indices_sha256\'] = sha256(index_path)\n        self.audit[\'library_sha256\'] = sha256(LIBRARY)\n        phases = [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else [])\n        self.seas = xr.concat([m.carregar_seas5(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.cfs = xr.concat([m.carregar_cfsv2(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.audit.update(rainfall_read=[start, end], forecast_partitions=phases,\n                          test_partition_read=False, sst_used=False)\n\n\nclass Context:\n    """Training-only climatologies, with a bounded reference window for inner selection."""\n    def __init__(self, data, reference, training):\n        self.data = data\n        m = data.lib\n        self.reference = pd.DatetimeIndex(reference)\n        self.training = pd.DatetimeIndex(training)\n        require(24 <= len(self.training) <= 360 and len(self.reference) <= 360, \'Invalid training window.\')\n        require(self.training.isin(self.reference).all() and self.reference.is_monotonic_increasing and\n                self.reference.is_unique and self.training.is_unique, \'Invalid training/reference dates.\')\n        self.cutoff = self.reference[-1]\n        rain = data.rain.sel(time=self.reference).values\n        self.counts = np.array([(self.reference.month == month).sum() for month in range(1, 13)])\n        require((self.counts > 1).all(), \'At least two reference years per month are needed.\')\n        self.sums = np.stack([rain[self.reference.month == month].sum(0, dtype=np.float64) for month in range(1, 13)])\n        self.climate = xr.DataArray((self.sums / self.counts[:, None, None]).astype(\'float32\'),\n            dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n        origins = m.origem_mensal(self.reference, 4)\n        self.atm = m.ajustar_clima_atmosfera(data.atmosphere, origins)\n        origins = m.origem_mensal(self.reference, 3)\n        vals = m.INDICES_OC.loc[origins, m.INDICES_NOMES].to_numpy(dtype=np.float64)\n        self.atm[\'_clima_indices\'] = np.stack([vals[origins.month == month].mean(0) for month in range(1, 13)])\n        self.seasonal = []\n        for forecasts in [data.seas, data.cfs]:\n            v = forecasts.sel(time=self.training)\n            require(pd.DatetimeIndex(v.time_origem.values).equals(m.origem_mensal(self.training, 1)), \'Wrong seasonal initialization.\')\n            means = np.stack([v.values[self.training.month == month].mean(0, dtype=np.float64) for month in range(1, 13)]).astype(\'float32\')\n            require(np.isfinite(means).all(), \'Missing seasonal reference month.\')\n            self.seasonal.append(xr.DataArray(means, dims=self.climate.dims, coords=self.climate.coords))\n\n    def features(self, target, training=False):\n        d, m = self.data, self.data.lib\n        t = pd.Timestamp(target)\n        require(t in self.training if training else t > self.cutoff, \'Month crosses the training boundary.\')\n        x = np.column_stack([m.matriz_mes(d.atmosphere, self.atm, self.climate, t),\n            m.valores_seas5(d.seas, t), m.valores_anomalia_seas5(d.seas, self.seasonal[0], t),\n            m.valores_cfsv2(d.cfs, t), m.valores_anomalia_cfsv2(d.cfs, self.seasonal[1], t)]).astype(\'float32\')\n        if training:\n            # Exclude the entire target map, including neighbors seen by convolutions.\n            y = d.rain.sel(time=t).values\n            month = t.month - 1\n            x[:, 22] = ((self.sums[month] - y.astype(\'float64\')) / (self.counts[month] - 1)).ravel().astype(\'float32\')\n        require(np.isfinite(x).all(), \'Nonfinite features.\')\n        return x.T.reshape(31, d.rain.sizes[\'lat\'], d.rain.sizes[\'lon\'])\n\n    def save(self, folder):\n        folder = Path(folder)\n        folder.mkdir(parents=True, exist_ok=True)\n        self.climate.to_netcdf(folder / \'rainfall_climatology.nc\')\n        np.savez_compressed(folder / \'references.npz\', sums=self.sums, counts=self.counts,\n                            seas5=self.seasonal[0].values, cfsv2=self.seasonal[1].values, **self.atm)\n        write_json(folder / \'context.json\', dict(reference=self.reference.strftime(\'%Y-%m\').tolist(),\n                                                training=self.training.strftime(\'%Y-%m\').tolist(), features=self.data.lib.FEATURES_MULTISSISTEMA))\n', 'research/common/presentation.py': '"""Reader-facing model names, without changing saved experiment identifiers."""\n\nMODEL_LABELS = {\n    \'hybrid\': \'Reference model\',\n    \'hybrid_sst\': \'Reference model + SST\',\n    \'unet\': \'U-Net\',\n    \'blend\': \'Reference model + U-Net\',\n    \'blend_reference\': \'Reference model + U-Net\',\n    \'blend_sst\': \'Reference model + SST + U-Net\',\n    \'climatology\': \'Climatology\',\n    \'direct_unet\': \'Direct U-Net\',\n    \'residual_unet\': \'Residual U-Net correction\',\n    \'direct_blend\': \'Reference model + direct U-Net\',\n    \'unet_corrected\': \'Bias-corrected U-Net\',\n    \'blend_corrected\': \'Reference model + bias-corrected U-Net\',\n}\n\n\ndef model_label(value):\n    """Leave unknown names intact so unrelated table values are never relabelled."""\n    return MODEL_LABELS.get(value, value)\n\n\ndef display_frame(frame):\n    """Return a presentation copy; preserve the original data and numeric values."""\n    result = frame.copy()\n    for column in (\'model\', \'modelo\', \'Model\', \'reference\', \'comparator\'):\n        if column in result:\n            result[column] = result[column].map(model_label)\n    return result\n', 'research/common/synthetic_fixture.py': '"""Small synthetic fields for tests; never operational inputs."""\nfrom types import SimpleNamespace\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom .inputs import library\n\n\ndef fixture():\n    m = library()\n    dates = pd.date_range(\'1976-06-01\', \'2009-01-01\', freq=\'MS\')\n    rng = np.random.default_rng(71)\n    coords = dict(time=dates, lat=np.linspace(-60, 15, 9), lon=np.linspace(-90, -25, 13))\n    def field():\n        return xr.DataArray(rng.uniform(1, 5, (len(dates), 9, 13)).astype(\'float32\'), dims=(\'time\', \'lat\', \'lon\'), coords=coords)\n    rain, seas, cfs = field(), field(), field()\n    origins = m.origem_mensal(dates, 1)\n    seas = seas.assign_coords(time_origem=(\'time\', origins.values))\n    cfs = cfs.assign_coords(time_origem=(\'time\', origins.values), inicializacao_mais_recente=(\'time\', origins.values))\n    atmosphere = {name: field() for name in m.VARIAVEIS}\n    m.INDICES_OC = pd.DataFrame(rng.normal(size=(len(dates), 4)), index=dates, columns=m.INDICES_NOMES)\n    return SimpleNamespace(lib=m, rain=rain, seas=seas, cfs=cfs, atmosphere=atmosphere)\n', 'research/diagnostics/__init__.py': '"""Read-only diagnostics of archived development predictions."""\n', 'research/diagnostics/analyze_errors.py': '"""Audit saved maps and describe errors without fitting or selecting a model.\n\nRun from the repository root with --predictions, --observations and --output.\nThe observations must be the original treino_tp.nc, not the shifted tp_alvo.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\n\nimport numpy as np\nimport pandas as pd\nfrom research.common.presentation import model_label\nimport xarray as xr\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nROOT = Path(__file__).resolve().parents[2]\nEVIDENCE = ROOT / \'research/spatial_unet/evidence/seven_blocks\'\nBLOCKS = (\'H1\', \'H2\', \'H3\', \'H4\', \'A\', \'B\', \'C\')\nMODELS = (\'hybrid\', \'unet\', \'climatology\')\nBINS = np.array([0., 1., 3., 5., 10., np.inf])\nBIN_NAMES = (\'[0,1)\', \'[1,3)\', \'[3,5)\', \'[5,10)\', \'[10,infinity)\')\nCOLORS = dict(hybrid=\'#087e8b\', unet=\'#db654a\', climatology=\'#9299a1\')\nSUMS = [\'n\', \'sse\', \'error_sum\', \'absolute_error_sum\', \'observed_sum\', \'predicted_sum\']\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\', newline=\'\\n\')\n\n\ndef stats(observed, predicted):\n    observed, predicted = np.asarray(observed, dtype=\'float64\'), np.asarray(predicted, dtype=\'float64\')\n    require(observed.shape == predicted.shape, \'Different observation/prediction shapes.\')\n    require(np.isfinite(observed).all() and np.isfinite(predicted).all(), \'Nonfinite rainfall.\')\n    error = predicted - observed\n    return dict(n=error.size, sse=float(np.square(error).sum()), error_sum=float(error.sum()),\n                absolute_error_sum=float(np.abs(error).sum()), observed_sum=float(observed.sum()),\n                predicted_sum=float(predicted.sum()))\n\n\ndef pooled(frame, groups):\n    result = frame.groupby(groups, as_index=False, sort=False, observed=True)[SUMS].sum()\n    denominator = result.n.replace(0, np.nan)\n    result[\'rmse\'] = np.sqrt(result.sse / denominator)\n    result[\'mae\'] = result.absolute_error_sum / denominator\n    result[\'bias\'] = result.error_sum / denominator\n    result[\'observed_mean\'] = result.observed_sum / denominator\n    result[\'predicted_mean\'] = result.predicted_sum / denominator\n    return result\n\n\ndef intensity_masks(observed):\n    require(np.isfinite(observed).all() and (observed >= 0).all(), \'Invalid observed rainfall.\')\n    return [(observed >= low) & (observed < high) for low, high in zip(BINS[:-1], BINS[1:])]\n\n\ndef exact_alignment(observed, forecast, dates):\n    require(set(forecast.data_vars) == set(MODELS), \'Unexpected forecast variables.\')\n    require(pd.DatetimeIndex(forecast.time.values).equals(dates), \'Wrong or reordered forecast dates.\')\n    for name in MODELS:\n        require(forecast[name].dims == (\'time\', \'lat\', \'lon\'), \'Wrong forecast dimensions.\')\n        require(forecast[name].attrs.get(\'units\') == \'mm/day\', \'Wrong forecast units.\')\n    # Exact alignment rejects flipped, reordered or merely similar coordinates.\n    obs = observed.sel(time=dates).transpose(\'time\', \'lat\', \'lon\')\n    obs, forecast = xr.align(obs, forecast, join=\'exact\')\n    require(np.isfinite(obs.values).all() and (obs.values >= 0).all(), \'Invalid observations.\')\n    for name in MODELS:\n        require(np.isfinite(forecast[name]).all() and (forecast[name] >= 0).all(), \'Invalid predictions.\')\n    return obs, forecast\n\n\ndef verify_file(path, expected):\n    actual = digest(path)\n    require(actual == expected, f\'Hash mismatch: {Path(path).name}\')\n    return actual\n\n\ndef comparison(table, keys, total_n):\n    hybrid = table[table.model == \'hybrid\'].set_index(keys)\n    unet = table[table.model == \'unet\'].set_index(keys).reindex(hybrid.index)\n    require(np.array_equal(hybrid.n, unet.n), \'Unpaired diagnostic groups.\')\n    result = hybrid[[\'n\', \'rmse\', \'mae\', \'bias\']].rename(columns={c: \'hybrid_\' + c for c in [\'rmse\', \'mae\', \'bias\']})\n    for c in [\'rmse\', \'mae\', \'bias\']:\n        result[\'unet_\' + c] = unet[c]\n        result[\'delta_\' + c] = unet[c] - hybrid[c]\n    result[\'delta_sse\'] = unet.sse - hybrid.sse\n    result[\'contribution_to_global_delta_mse\'] = result.delta_sse / total_n\n    return result.reset_index()\n\n\ndef figure_setup():\n    plt.rcParams.update({\'font.size\': 10, \'axes.spines.top\': False, \'axes.spines.right\': False,\n                         \'figure.facecolor\': \'white\', \'savefig.facecolor\': \'white\'})\n\n\ndef plot_results(tables, maps, histories, output):\n    figure_setup()\n    monthly = tables[\'calendar_comparison\']\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)\n    for model in MODELS:\n        data = tables[\'calendar_month\'].query(\'model == @model\').sort_values(\'calendar_month\')\n        axes[0].plot(data.calendar_month, data.rmse, marker=\'o\', label=model_label(model), color=COLORS[model])\n    axes[0].set(xlabel=\'Target calendar month\', ylabel=\'Pooled RMSE (mm/day)\', xticks=range(1, 13))\n    axes[0].legend(frameon=False)\n    axes[1].bar(monthly.calendar_month, monthly.delta_rmse,\n                color=np.where(monthly.delta_rmse > 0, COLORS[\'unet\'], COLORS[\'hybrid\']))\n    axes[1].axhline(0, color=\'#555\', linewidth=.8)\n    axes[1].set(xlabel=\'Target calendar month\', ylabel=\'U-Net minus reference model RMSE (mm/day)\', xticks=range(1, 13))\n    fig.suptitle(\'Calendar-month errors | 2007–2020 development years\')\n    fig.savefig(output / \'calendar_month.png\', dpi=155); plt.close(fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)\n    for model in MODELS:\n        data = tables[\'intensity\'].query(\'model == @model\').set_index(\'intensity\').loc[list(BIN_NAMES)]\n        axes[0].plot(range(5), data.rmse, marker=\'o\', label=model_label(model), color=COLORS[model])\n        axes[1].plot(range(5), data.bias, marker=\'o\', color=COLORS[model])\n    for ax in axes:\n        ax.set_xticks(range(5), [\'0–<1\', \'1–<3\', \'3–<5\', \'5–<10\', \'≥10\'])\n        ax.set_xlabel(\'Observed monthly mean rainfall (mm/day)\')\n    axes[0].set_ylabel(\'Conditional RMSE (mm/day)\'); axes[0].legend(frameon=False)\n    axes[1].set_ylabel(\'Conditional bias: predicted − observed (mm/day)\')\n    axes[1].axhline(0, color=\'#555\', linewidth=.8)\n    fig.suptitle(\'Rainfall intensity | fixed descriptive bins, not daily extremes\')\n    fig.savefig(output / \'rainfall_intensity.png\', dpi=155); plt.close(fig)\n\n    fig, axes = plt.subplots(1, 3, figsize=(12, 6.7), constrained_layout=True)\n    delta = maps.unet_rmse - maps.hybrid_rmse\n    vmax = float(max(maps.hybrid_rmse.max(), maps.unet_rmse.max()))\n    for ax, name in zip(axes[:2], [\'hybrid\', \'unet\']):\n        artist = ax.pcolormesh(maps.lon, maps.lat, maps[name + \'_rmse\'], shading=\'auto\', cmap=\'viridis\', vmin=0, vmax=vmax)\n        ax.set_title(model_label(name) + \' RMSE\')\n    fig.colorbar(artist, ax=list(axes[:2]), label=\'RMSE (mm/day)\', orientation=\'horizontal\', shrink=.9)\n    bound = float(np.abs(delta).max())\n    artist = axes[2].pcolormesh(maps.lon, maps.lat, delta, shading=\'auto\', cmap=\'RdBu_r\', vmin=-bound, vmax=bound)\n    axes[2].set_title(\'U-Net − reference model RMSE\')\n    fig.colorbar(artist, ax=axes[2], label=\'Difference (mm/day)\', orientation=\'horizontal\')\n    for ax in axes:\n        ax.set(xlabel=\'Longitude (°)\', ylabel=\'Latitude (°)\', xlim=(-90, -25), ylim=(-60, 15), aspect=\'equal\')\n    fig.suptitle(\'Errors at each grid cell | all 168 months, full supplied domain\')\n    fig.savefig(output / \'spatial_errors.png\', dpi=155); plt.close(fig)\n\n    fig, axes = plt.subplots(4, 2, figsize=(12, 12), constrained_layout=True)\n    for ax, (block, inner, outer, selected) in zip(axes.flat, histories):\n        ax.plot(inner.epoch, np.sqrt(inner.training_mse), label=\'Inner training √MSE (online)\', color=\'#9299a1\')\n        ax.plot(inner.epoch, inner.inner_rmse, label=\'Inner validation RMSE\', color=COLORS[\'unet\'])\n        ax.plot(outer.epoch, np.sqrt(outer.training_mse), \'--\', label=\'Outer training √MSE (online)\', color=COLORS[\'hybrid\'])\n        ax.axvline(selected, color=\'#333\', linestyle=\':\', label=\'Selected epoch\')\n        ax.set(title=f\'{block} | selected epoch {selected}\', xlabel=\'Epoch\', ylabel=\'mm/day\')\n    axes.flat[-1].axis(\'off\')\n    handles, labels = axes.flat[0].get_legend_handles_labels()\n    axes.flat[-1].legend(handles, labels, loc=\'upper left\', frameon=False)\n    axes.flat[-1].text(.05, .12, \'Training losses use evolving weights and unclipped\\nanomalies. Validation uses a fixed end-of-epoch\\nmodel and clipped rainfall; gaps are not directly\\ncomparable generalization-error estimates.\', transform=axes.flat[-1].transAxes, fontsize=9)\n    fig.suptitle(\'Training histories | original epoch selection, no new fitting\')\n    fig.savefig(output / \'training_curves.png\', dpi=150); plt.close(fig)\n\n\ndef execute(predictions, observations, output, evidence=EVIDENCE):\n    predictions, observations, output, evidence = map(Path, (predictions, observations, output, evidence))\n    require(not output.exists(), \'Choose a new output directory; existing reports are preserved.\')\n    signature = read_json(evidence / \'signature.json\')\n    signature_hash = digest(evidence / \'signature.json\')\n    inputs = {\'observations\': verify_file(observations, signature[\'inputs\'][\'files\'][\'treino_tp.nc\']), \'signature\': signature_hash}\n    monthly_rows, intensity_rows, histories, training_rows = [], [], [], []\n    pixel_sums = None\n    for block_index, block in enumerate(BLOCKS):\n        completed = read_json(evidence / block / \'complete.json\')\n        require(completed[\'signature\'] == signature_hash, \'Run signature mismatch.\')\n        p = predictions / block / \'predictions.nc\'\n        inputs[block + \'/predictions.nc\'] = verify_file(p, completed[\'files\'][\'predictions.nc\'])\n        for name in [\'monthly_metrics.csv\', \'inner/training_history.json\', \'inner/training.json\', \'unet/training_history.json\', \'unet/training.json\']:\n            inputs[block + \'/\' + name] = verify_file(evidence / block / name, completed[\'files\'][name])\n        dates = pd.date_range(f\'{2007 + 2 * block_index}-01-01\', periods=24, freq=\'MS\')\n        archived = pd.read_csv(evidence / block / \'monthly_metrics.csv\')\n        with xr.open_dataset(observations) as source, xr.open_dataset(p) as ds:\n            require(source.tp.attrs.get(\'units\') == \'mm/day\', \'Wrong observation units.\')\n            observed, ds = exact_alignment(source.tp, ds.load(), dates)\n            observed = observed.load()\n            lat, lon = observed.lat.values, observed.lon.values\n            require(np.array_equal(lat, np.arange(-60, 15.25, .25)) and np.array_equal(lon, np.arange(-90, -24.75, .25)), \'Unexpected target grid.\')\n            regions = {\'full_domain\': np.ones(len(lat), bool), \'south_of_35S\': lat < -35,\n                       \'35S_to_15S\': (lat >= -35) & (lat < -15), \'north_of_15S\': lat >= -15}\n            original_names = dict(full_domain=\'dominio_inteiro\', south_of_35S=\'lat_menor_que_menos35\',\n                                  **{\'35S_to_15S\': \'lat_menos35_a_menos15\', \'north_of_15S\': \'lat_maior_igual_menos15\'})\n            if pixel_sums is None:\n                pixel_sums = {model: {metric: np.zeros((len(lat), len(lon))) for metric in [\'sse\', \'error\', \'absolute_error\']} for model in MODELS}\n            for i, date in enumerate(dates):\n                truth = observed.values[i].astype(\'float64\')\n                masks = intensity_masks(truth)\n                for model in MODELS:\n                    pred = ds[model].values[i].astype(\'float64\')\n                    error = pred - truth\n                    pixel_sums[model][\'sse\'] += error ** 2\n                    pixel_sums[model][\'error\'] += error\n                    pixel_sums[model][\'absolute_error\'] += np.abs(error)\n                    for region, mask in regions.items():\n                        row = stats(truth[mask], pred[mask])\n                        old = archived[(archived.modelo == model) & (archived.mes_alvo == str(date.date())) & (archived.regiao == original_names[region])]\n                        require(len(old) == 1, \'Missing or duplicated archived monthly score.\')\n                        old = old.iloc[0]\n                        for new_key, old_key in [(\'n\', \'n\'), (\'sse\', \'sse\'), (\'error_sum\', \'soma_erro\'), (\'absolute_error_sum\', \'soma_erro_absoluto\')]:\n                            require(np.isclose(row[new_key], old[old_key], rtol=1e-10, atol=1e-8), f\'Recomputed score differs: {block}/{model}/{date}/{region}/{new_key}\')\n                        monthly_rows.append(dict(block=block, date=str(date.date()), calendar_month=date.month, region=region, model=model, **row))\n                    for label, mask in zip(BIN_NAMES, masks):\n                        intensity_rows.append(dict(block=block, model=model, intensity=label, **stats(truth[mask], pred[mask])))\n        inner = pd.DataFrame(read_json(evidence / block / \'inner/training_history.json\'))\n        outer = pd.DataFrame(read_json(evidence / block / \'unet/training_history.json\'))\n        chosen = read_json(evidence / block / \'inner/training.json\')[\'selected_epochs\']\n        require(chosen == int(inner.loc[inner.inner_rmse.idxmin(), \'epoch\']) == len(outer), \'Epoch selection/history mismatch.\')\n        require(chosen == read_json(evidence / block / \'unet/training.json\')[\'selected_epochs\'], \'Outer epochs differ from inner selection.\')\n        histories.append((block, inner, outer, chosen))\n        training_rows.append(dict(block=block, selected_epochs=chosen, inner_epochs_run=len(inner),\n                                  first_inner_rmse=float(inner.inner_rmse.iloc[0]), best_inner_rmse=float(inner.inner_rmse.min()),\n                                  last_inner_rmse=float(inner.inner_rmse.iloc[-1]),\n                                  inner_training_mse_first=float(inner.training_mse.iloc[0]), inner_training_mse_last=float(inner.training_mse.iloc[-1])))\n        print(block, \'maps, dates, hashes, archived scores and epoch selection verified\', flush=True)\n    monthly = pd.DataFrame(monthly_rows)\n    domain = monthly.query("region == \'full_domain\'")\n    intensity_months = pd.DataFrame(intensity_rows)\n    tables = {\'global\': pooled(domain, [\'model\']), \'calendar_month\': pooled(domain, [\'model\', \'calendar_month\']),\n              \'region\': pooled(monthly, [\'model\', \'region\']), \'region_calendar_month\': pooled(monthly, [\'model\', \'region\', \'calendar_month\']),\n              \'intensity\': pooled(intensity_months, [\'model\', \'intensity\']), \'block_intensity\': pooled(intensity_months, [\'model\', \'block\', \'intensity\']),\n              \'training\': pd.DataFrame(training_rows)}\n    total_n = int(tables[\'global\'].query("model == \'reference model\'").n.iloc[0])\n    require(total_n == 168 * 301 * 261, \'Incomplete evaluation.\')\n    np.testing.assert_allclose(tables[\'intensity\'].groupby(\'model\')[SUMS].sum().sort_index(), tables[\'global\'].set_index(\'model\')[SUMS].sort_index(), rtol=1e-12, atol=1e-7)\n    archived_global = pd.read_csv(evidence / \'global.csv\').set_index(\'modelo\')\n    for model in MODELS:\n        now = tables[\'global\'].set_index(\'model\').loc[model]\n        require(np.isclose(now.rmse, archived_global.loc[model, \'rmse\'], rtol=0, atol=1e-12), \'Global score did not reproduce.\')\n    for name, keys in [(\'calendar\', [\'calendar_month\']), (\'region\', [\'region\']), (\'region_calendar\', [\'region\', \'calendar_month\']), (\'intensity\', [\'intensity\']), (\'block_intensity\', [\'block\', \'intensity\'])]:\n        base = {\'calendar\': \'calendar_month\', \'region_calendar\': \'region_calendar_month\'}.get(name, name)\n        tables[name + \'_comparison\'] = comparison(tables[base], keys, total_n)\n    variables = {}\n    for model in MODELS:\n        for name, values in [(\'rmse\', np.sqrt(pixel_sums[model][\'sse\'] / 168)), (\'bias\', pixel_sums[model][\'error\'] / 168), (\'mae\', pixel_sums[model][\'absolute_error\'] / 168)]:\n            variables[model + \'_\' + name] = ((\'lat\', \'lon\'), values, dict(units=\'mm/day\'))\n    maps = xr.Dataset(variables, coords=dict(lat=lat, lon=lon), attrs=dict(months=168, independent_holdout=\'false\', analysis=\'development diagnostics; no fitting\'))\n    better = maps.unet_rmse < maps.hybrid_rmse\n    global_values = tables[\'global\'].set_index(\'model\')\n    summary = dict(months=168, gridpoint_months=total_n, all_seven_prediction_hashes_verified=True,\n                   monthly_region_scores_reproduced=len(monthly), intensity_partition_verified=True,\n                   hybrid_rmse=float(global_values.loc[\'hybrid\', \'rmse\']), unet_rmse=float(global_values.loc[\'unet\', \'rmse\']),\n                   global_delta_mse=float((global_values.loc[\'unet\', \'sse\'] - global_values.loc[\'hybrid\', \'sse\']) / total_n),\n                   unet_lower_rmse_grid_fraction=float(better.mean()), unet_higher_rmse_grid_fraction=float((maps.unet_rmse > maps.hybrid_rmse).mean()),\n                   calendar_months_unet_improved=int((tables[\'calendar_comparison\'].delta_rmse < 0).sum()),\n                   independent_holdout=False, model_fitted=False, model_promoted=False, forecast_issued=False)\n    output.mkdir(parents=True)\n    for name, table in tables.items():\n        table.to_csv(output / (name + \'.csv\'), index=False, lineterminator=\'\\n\')\n    maps.to_netcdf(output / \'error_maps.nc\', engine=\'h5netcdf\')\n    save_json(output / \'summary.json\', summary)\n    plot_results(tables, maps, histories, output)\n    save_json(output / \'verification.json\', dict(inputs=inputs, code_sha256=digest(__file__),\n        python=platform.python_version(), versions={name: importlib.metadata.version(name) for name in [\'numpy\', \'pandas\', \'xarray\', \'matplotlib\', \'h5netcdf\']},\n        historical_run_environment=signature[\'inputs\'][\'versions\'], environment_scope=\'diagnostics only; original model was not rerun\',\n        bins_mm_day=[\'0\', \'1\', \'3\', \'5\', \'10\', \'infinity\'],\n        files={p.name: digest(p) for p in sorted(output.iterdir()) if p.is_file()}))\n    print(json.dumps(summary, indent=2))\n    return summary\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--predictions\', type=Path, required=True, help=\'Directory containing H1/.../C predictions.nc files\')\n    parser.add_argument(\'--observations\', type=Path, required=True, help=\'Original official treino_tp.nc\')\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'New report directory\')\n    args = parser.parse_args()\n    execute(args.predictions, args.observations, args.output)\n', 'research/diagnostics/test_diagnostics.py': '"""Small mathematical and alignment checks; no model fitting or downloads."""\nimport tempfile\nimport unittest\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom research.diagnostics.analyze_errors import (\n    MODELS, stats, pooled, intensity_masks, exact_alignment, verify_file,\n)\n\n\nclass DiagnosticChecks(unittest.TestCase):\n    def test_rmse_pools_squared_errors_and_sample_counts(self):\n        frame = pd.DataFrame([dict(model=\'x\', **stats([0], [4])),\n                              dict(model=\'x\', **stats([0, 0, 0], [0, 0, 0]))])\n        result = pooled(frame, [\'model\']).iloc[0]\n        self.assertEqual(result.rmse, 2.)\n        self.assertEqual(result.mae, 1.)\n        self.assertEqual(result.bias, 1.)\n\n    def test_intensity_edges_cover_each_observation_once(self):\n        observed = np.array([0., .9, 1., 2.9, 3., 4.9, 5., 9.9, 10., 90.])\n        masks = intensity_masks(observed)\n        np.testing.assert_array_equal(np.sum(masks, axis=0), np.ones(10))\n        self.assertEqual([int(m.sum()) for m in masks], [2] * 5)\n        with self.assertRaises(ValueError):\n            intensity_masks(np.array([-1.]))\n\n    def test_empty_bin_has_no_invented_zero_rmse(self):\n        frame = pd.DataFrame([dict(model=\'x\', **stats([], []))])\n        row = pooled(frame, [\'model\']).iloc[0]\n        self.assertEqual(row.n, 0)\n        self.assertTrue(np.isnan(row.rmse))\n\n    def test_alignment_rejects_time_shift_and_flipped_grid(self):\n        dates = pd.date_range(\'2007-01-01\', periods=2, freq=\'MS\')\n        obs = xr.DataArray(np.ones((2, 2, 2)), dims=(\'time\', \'lat\', \'lon\'),\n                           coords=dict(time=dates, lat=[-1., 0.], lon=[-40., -39.]), attrs=dict(units=\'mm/day\'))\n        forecast = xr.Dataset({model: obs.copy() for model in MODELS})\n        exact_alignment(obs, forecast, dates)\n        with self.assertRaises(ValueError):\n            exact_alignment(obs, forecast.isel(lat=slice(None, None, -1)), dates)\n        with self.assertRaises(ValueError):\n            exact_alignment(obs, forecast.assign_coords(time=dates + pd.Timedelta(days=1)), dates)\n\n    def test_changed_archive_is_rejected(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            file = Path(tmp) / \'artifact\'\n            file.write_text(\'changed\')\n            with self.assertRaisesRegex(ValueError, \'Hash mismatch\'):\n                verify_file(file, \'0\' * 64)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'research/fixed_blend/__init__.py': '"""One prespecified combination of archived monthly rainfall forecasts."""\n', 'research/fixed_blend/evaluate.py': '"""Evaluate one fixed blend after the workflow verifies the archived experiment."""\nfrom pathlib import Path\nimport json\n\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nfrom research.diagnostics.analyze_errors import (\n    BLOCKS, BIN_NAMES, MODELS as ORIGINAL_MODELS, SUMS, digest, exact_alignment,\n    intensity_masks, pooled, require, save_json, stats, verify_file,\n)\n\nPROTOCOL = Path(__file__).with_name(\'protocol.json\')\nMODELS = (\'hybrid\', \'unet\', \'blend\', \'climatology\')\n\n\ndef validate_protocol():\n    protocol = json.loads(PROTOCOL.read_text(encoding=\'utf-8\'))\n    require(protocol[\'id\'] == \'fixed_blend_v1\' and protocol[\'weights\'] == {\'hybrid\': .75, \'unet\': .25},\n            \'This adapter only supports the prespecified 75/25 blend.\')\n    require(protocol[\'weight_search\'] is False and protocol[\'conditional_switching\'] is False\n            and protocol[\'new_training\'] is False, \'Fixed-blend protocol changed.\')\n    require(protocol[\'blocks\'] == list(BLOCKS) and protocol[\'evaluation\'] == [\'2007-01\', \'2020-12\'],\n            \'Fixed-blend evaluation period changed.\')\n    return protocol\n\n\ndef fixed_blend(hybrid, unet):\n    hybrid, unet = np.asarray(hybrid, dtype=\'float64\'), np.asarray(unet, dtype=\'float64\')\n    require(hybrid.shape == unet.shape, \'Blend maps have different shapes.\')\n    require(np.isfinite(hybrid).all() and np.isfinite(unet).all(), \'Blend inputs contain nonfinite values.\')\n    require((hybrid >= 0).all() and (unet >= 0).all(), \'Blend inputs must be reconstructed nonnegative rainfall.\')\n    return .75 * hybrid + .25 * unet\n\n\ndef error_products(observed, hybrid, unet):\n    require(np.shape(observed) == np.shape(hybrid) == np.shape(unet), \'Unpaired error products.\')\n    h = np.asarray(hybrid, dtype=\'float64\') - np.asarray(observed, dtype=\'float64\')\n    u = np.asarray(unet, dtype=\'float64\') - np.asarray(observed, dtype=\'float64\')\n    return dict(n=h.size, hybrid_error_sum=float(h.sum()), unet_error_sum=float(u.sum()),\n                hybrid_sse=float(np.square(h).sum()), unet_sse=float(np.square(u).sum()),\n                error_product_sum=float((h * u).sum()), disagreement_sse=float(np.square(u - h).sum()))\n\n\ndef summarize_products(rows):\n    result = {key: sum(row[key] for row in rows) for key in rows[0]}\n    n = result[\'n\']\n    h2, u2, cross = result[\'hybrid_sse\'], result[\'unet_sse\'], result[\'error_product_sum\']\n    hvar = max(0., h2 - result[\'hybrid_error_sum\'] ** 2 / n)\n    uvar = max(0., u2 - result[\'unet_error_sum\'] ** 2 / n)\n    covariance_sum = cross - result[\'hybrid_error_sum\'] * result[\'unet_error_sum\'] / n\n    result[\'centered_error_correlation\'] = float(covariance_sum / np.sqrt(hvar * uvar)) if hvar * uvar > 0 else None\n    result[\'blend_sse_from_products\'] = .75 ** 2 * h2 + .25 ** 2 * u2 + 2 * .75 * .25 * cross\n    # This identity tests the fixed combination, without estimating an optimal weight.\n    result[\'blend_sse_from_disagreement\'] = .75 * h2 + .25 * u2 - .75 * .25 * result[\'disagreement_sse\']\n    return result\n\n\ndef paired(table, keys):\n    baseline = table[table.model == \'hybrid\'].set_index(keys)\n    candidate = table[table.model == \'blend\'].set_index(keys).reindex(baseline.index)\n    require(np.array_equal(candidate.n, baseline.n), \'Unpaired blend comparison.\')\n    result = baseline[[\'n\', \'rmse\', \'mae\', \'bias\']].rename(columns={k: \'hybrid_\' + k for k in [\'rmse\', \'mae\', \'bias\']})\n    for metric in [\'rmse\', \'mae\', \'bias\']:\n        result[\'blend_\' + metric] = candidate[metric]\n        result[\'delta_\' + metric] = candidate[metric] - baseline[metric]\n    result[\'delta_absolute_bias\'] = candidate.bias.abs() - baseline.bias.abs()\n    result[\'delta_sse\'] = candidate.sse - baseline.sse\n    return result.reset_index()\n\n\ndef execute(predictions, observations, output, diagnostics, evidence):\n    """Run only on maps already checked by the original diagnostic in this run."""\n    predictions, observations, output, diagnostics, evidence = map(Path, (predictions, observations, output, diagnostics, evidence))\n    require(not output.exists(), \'Choose a fresh fixed-blend output directory.\')\n    protocol = validate_protocol()\n    verified = json.loads((diagnostics / \'verification.json\').read_text(encoding=\'utf-8\'))\n    verify_file(observations, verified[\'inputs\'][\'observations\'])\n    output.mkdir(parents=True)\n    save_json(output / \'protocol.json\', protocol)\n    monthly, intensities, products = [], [], []\n    pixel_sums = None\n    for index, block in enumerate(BLOCKS):\n        path = predictions / block / \'predictions.nc\'\n        verify_file(path, verified[\'inputs\'][block + \'/predictions.nc\'])\n        dates = pd.date_range(f\'{2007 + 2 * index}-01-01\', periods=24, freq=\'MS\')\n        with xr.open_dataset(observations) as source, xr.open_dataset(path) as dataset:\n            require(source.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n            observed, dataset = exact_alignment(source.tp, dataset.load(), dates)\n            observed = observed.load()\n            lat, lon = observed.lat.values, observed.lon.values\n            blend = fixed_blend(dataset.hybrid.values, dataset.unet.values)\n            candidate = xr.Dataset({\'blend\': ((\'time\', \'lat\', \'lon\'), blend, {\'units\': \'mm/day\'})},\n                                   coords=dataset.coords,\n                                   attrs=dict(protocol=\'fixed_blend_v1\', hybrid_weight=.75, unet_weight=.25,\n                                              source_prediction_sha256=digest(path), independent_holdout=\'false\',\n                                              scope=\'historical development; not a new operational forecast\'))\n            target = output / \'predictions\' / block / \'predictions.nc\'\n            target.parent.mkdir(parents=True)\n            candidate.to_netcdf(target, engine=\'h5netcdf\')\n            with xr.open_dataset(target) as written:\n                np.testing.assert_array_equal(written.blend.values, blend)\n                require(written.blend.dtype == np.dtype(\'float64\'), \'Blend precision changed on disk.\')\n            regions = {\'full_domain\': np.ones(len(lat), bool), \'south_of_35S\': lat < -35,\n                       \'35S_to_15S\': (lat >= -35) & (lat < -15), \'north_of_15S\': lat >= -15}\n            if pixel_sums is None:\n                pixel_sums = {model: {key: np.zeros((len(lat), len(lon))) for key in [\'sse\', \'error\', \'absolute_error\']} for model in MODELS}\n            for i, date in enumerate(dates):\n                truth = observed.values[i].astype(\'float64\')\n                masks = intensity_masks(truth)\n                product = error_products(truth, dataset.hybrid.values[i], dataset.unet.values[i])\n                product.update(block=block, date=str(date.date()))\n                products.append(product)\n                for model in MODELS:\n                    pred = blend[i] if model == \'blend\' else dataset[model].values[i].astype(\'float64\')\n                    error = pred - truth\n                    for metric, value in [(\'sse\', error ** 2), (\'error\', error), (\'absolute_error\', np.abs(error))]:\n                        pixel_sums[model][metric] += value\n                    for region, mask in regions.items():\n                        monthly.append(dict(block=block, date=str(date.date()), year=date.year,\n                                            calendar_month=date.month, region=region, model=model,\n                                            **stats(truth[mask], pred[mask])))\n                    for label, mask in zip(BIN_NAMES, masks):\n                        intensities.append(dict(block=block, model=model, intensity=label, **stats(truth[mask], pred[mask])))\n        print(block, \'fixed 75/25 maps saved and read back; paired errors calculated\', flush=True)\n\n    monthly = pd.DataFrame(monthly)\n    domain = monthly[monthly.region == \'full_domain\']\n    intensity_rows = pd.DataFrame(intensities)\n    tables = {\'global\': pooled(domain, [\'model\']), \'blocks\': pooled(domain, [\'model\', \'block\']),\n              \'years\': pooled(domain, [\'model\', \'year\']), \'calendar\': pooled(domain, [\'model\', \'calendar_month\']),\n              \'region\': pooled(monthly, [\'model\', \'region\']), \'intensity\': pooled(intensity_rows, [\'model\', \'intensity\']),\n              \'block_intensity\': pooled(intensity_rows, [\'model\', \'block\', \'intensity\'])}\n    values = tables[\'global\'].set_index(\'model\')\n    require((values.n == 168 * 301 * 261).all(), \'Incomplete comparison.\')\n    original = pd.read_csv(diagnostics / \'global.csv\').set_index(\'model\')\n    np.testing.assert_allclose(values.loc[list(ORIGINAL_MODELS), SUMS], original.loc[list(ORIGINAL_MODELS), SUMS], rtol=1e-12, atol=1e-7)\n    np.testing.assert_allclose(tables[\'intensity\'].groupby(\'model\')[SUMS].sum().sort_index(), values[SUMS].sort_index(), rtol=1e-12, atol=1e-7)\n    for name, keys in [(\'blocks\', [\'block\']), (\'years\', [\'year\']), (\'calendar\', [\'calendar_month\']),\n                       (\'region\', [\'region\']), (\'intensity\', [\'intensity\']), (\'block_intensity\', [\'block\', \'intensity\'])]:\n        tables[name + \'_comparison\'] = paired(tables[name], keys)\n    numerical_products = [{key: value for key, value in row.items() if key not in {\'block\', \'date\'}} for row in products]\n    complementarity = summarize_products(numerical_products)\n    np.testing.assert_allclose([complementarity[\'blend_sse_from_products\'], complementarity[\'blend_sse_from_disagreement\']],\n                               values.loc[\'blend\', \'sse\'], rtol=1e-12, atol=1e-7)\n    maps = xr.Dataset({f\'{model}_{metric}\': ((\'lat\', \'lon\'), array, {\'units\': \'mm/day\'})\n                       for model in MODELS for metric, array in [\n                           (\'rmse\', np.sqrt(pixel_sums[model][\'sse\'] / 168)),\n                           (\'mae\', pixel_sums[model][\'absolute_error\'] / 168),\n                           (\'bias\', pixel_sums[model][\'error\'] / 168)]}, coords=dict(lat=lat, lon=lon),\n                      attrs=dict(months=168, protocol=\'fixed_blend_v1\', independent_holdout=\'false\'))\n    summary = dict(protocol=\'fixed_blend_v1\', months=168, gridpoint_months=int(values.loc[\'blend\', \'n\']),\n                   hybrid_rmse=float(values.loc[\'hybrid\', \'rmse\']), unet_rmse=float(values.loc[\'unet\', \'rmse\']),\n                   blend_rmse=float(values.loc[\'blend\', \'rmse\']),\n                   delta_rmse_vs_hybrid=float(values.loc[\'blend\', \'rmse\'] - values.loc[\'hybrid\', \'rmse\']),\n                   relative_rmse_reduction_percent=float(100 * (1 - values.loc[\'blend\', \'rmse\'] / values.loc[\'hybrid\', \'rmse\'])),\n                   delta_mae_vs_hybrid=float(values.loc[\'blend\', \'mae\'] - values.loc[\'hybrid\', \'mae\']),\n                   delta_absolute_bias_vs_hybrid=float(abs(values.loc[\'blend\', \'bias\']) - abs(values.loc[\'hybrid\', \'bias\'])),\n                   blend_lower_rmse_grid_fraction=float((maps.blend_rmse < maps.hybrid_rmse).mean()),\n                   input_hashes_verified=True, original_scores_reproduced=True, blend_maps_read_back_exactly=True,\n                   mse_identity_verified=True, intensity_partition_verified=True, independent_holdout=False,\n                   model_fitted=False, weight_search=False, model_promoted=False, forecast_issued=False)\n    for name in [\'blocks\', \'years\', \'calendar\']:\n        delta = tables[name + \'_comparison\'].delta_rmse\n        summary[name] = dict(improved=int((delta < 0).sum()), worsened=int((delta > 0).sum()), tied=int((delta == 0).sum()))\n    for name, table in tables.items():\n        table.to_csv(output / (name + \'.csv\'), index=False, lineterminator=\'\\n\')\n    monthly.to_csv(output / \'monthly_metrics.csv\', index=False, lineterminator=\'\\n\')\n    pd.DataFrame(products).to_csv(output / \'monthly_error_products.csv\', index=False, lineterminator=\'\\n\')\n    maps.to_netcdf(output / \'error_maps.nc\', engine=\'h5netcdf\')\n    save_json(output / \'complementarity.json\', complementarity)\n    save_json(output / \'summary.json\', summary)\n    from .report import plot_results\n    plot_results(tables, maps, output)\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n', 'research/fixed_blend/protocol.json': '{\n  "id": "fixed_blend_v1",\n  "hypothesis": "The compact U-Net adds complementary information to the fixed hybrid when given a small fixed contribution.",\n  "design": "Chosen after inspecting the original models\' development diagnostics, before evaluating this combination. Not an independent holdout.",\n  "evaluation": ["2007-01", "2020-12"],\n  "blocks": ["H1", "H2", "H3", "H4", "A", "B", "C"],\n  "formula": "0.75 * archived_hybrid + 0.25 * archived_unet",\n  "weights": {"hybrid": 0.75, "unet": 0.25},\n  "input_stage": "Final archived nonnegative rainfall maps in mm/day; not unbounded anomalies.",\n  "arithmetic": "float64, no additional clipping, rounding, normalization or bias correction",\n  "weight_search": false,\n  "conditional_switching": false,\n  "new_training": false,\n  "sources": "Unchanged spatial_unet_v1 information cutoffs and source snapshots",\n  "primary_metric": "RMSE pooled over all 168 months and the full 301 by 261 grid",\n  "secondary_metrics": ["MAE", "signed bias", "block", "year", "calendar month", "latitude band", "observed monthly intensity", "pixel"],\n  "interpretation": "Describe complementary errors and temporal/regional consistency. No significance claim or automatic promotion. No new forecasts are issued."\n}\n', 'research/fixed_blend/report.py': '"""Figures and a self-contained report for the prespecified fixed blend."""\nfrom html import escape\nfrom pathlib import Path\nimport base64\nimport json\n\nimport numpy as np\nimport pandas as pd\nfrom research.common.presentation import model_label\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nfrom research.diagnostics.analyze_errors import BLOCKS, BIN_NAMES, figure_setup\nfrom research.workflow.report import table_html\n\nCOLORS = dict(hybrid=\'#087e8b\', unet=\'#db654a\', blend=\'#6246a5\', climatology=\'#9299a1\')\n\n\ndef plot_results(tables, maps, output):\n    figure_setup()\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)\n    for ax, name, key, order in [(axes[0], \'blocks\', \'block\', BLOCKS),\n                                (axes[1], \'years\', \'year\', range(2007, 2021))]:\n        frame = tables[name + \'_comparison\'].set_index(key).loc[list(order)]\n        ax.bar(range(len(frame)), frame.delta_rmse, color=np.where(frame.delta_rmse < 0, COLORS[\'blend\'], COLORS[\'unet\']))\n        ax.axhline(0, color=\'#555\', linewidth=.8)\n        ax.set_xticks(range(len(frame)), frame.index, rotation=45 if name == \'years\' else 0)\n        ax.set(xlabel=key.title(), ylabel=\'Blend minus reference model RMSE (mm/day)\')\n    fig.suptitle(\'Fixed 75% reference model + 25% U-Net | negative differences favor the blend\')\n    fig.savefig(output / \'temporal_comparison.png\', dpi=150); plt.close(fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)\n    for model in [\'hybrid\', \'unet\', \'blend\']:\n        month = tables[\'calendar\'].query(\'model == @model\').sort_values(\'calendar_month\')\n        intensity = tables[\'intensity\'].query(\'model == @model\').set_index(\'intensity\').loc[list(BIN_NAMES)]\n        axes[0].plot(month.calendar_month, month.rmse, marker=\'o\', label=model_label(model), color=COLORS[model])\n        axes[1].plot(range(5), intensity.rmse, marker=\'o\', label=model_label(model), color=COLORS[model])\n    axes[0].set(xlabel=\'Target calendar month\', ylabel=\'RMSE (mm/day)\', xticks=range(1, 13))\n    axes[0].legend(frameon=False)\n    axes[1].set_xticks(range(5), [\'0–<1\', \'1–<3\', \'3–<5\', \'5–<10\', \'≥10\'])\n    axes[1].set(xlabel=\'Observed monthly mean rainfall (mm/day)\', ylabel=\'Conditional RMSE (mm/day)\')\n    fig.suptitle(\'Development diagnostics | observed intensity cannot select a forecast\')\n    fig.savefig(output / \'calendar_intensity.png\', dpi=150); plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)\n    difference = maps.blend_rmse - maps.hybrid_rmse\n    bound = max(float(np.abs(difference).max()), 1e-12)\n    artist = ax.pcolormesh(maps.lon, maps.lat, difference, cmap=\'RdBu_r\', vmin=-bound, vmax=bound, shading=\'auto\')\n    ax.set(xlabel=\'Longitude (°)\', ylabel=\'Latitude (°)\', xlim=(-90, -25), ylim=(-60, 15), aspect=\'equal\',\n           title=\'Blend − reference model RMSE | all 168 months\\nBlue favors the blend; full grid includes ocean\')\n    fig.colorbar(artist, ax=ax, label=\'Difference in RMSE (mm/day)\')\n    fig.savefig(output / \'spatial_comparison.png\', dpi=150); plt.close(fig)\n\n\ndef render(run, record):\n    run = Path(run)\n    folder = run / \'fixed_blend\'\n    summary = json.loads((folder / \'summary.json\').read_text())\n    scores = pd.read_csv(folder / \'global.csv\')\n    scores[\'delta_rmse_vs_hybrid\'] = scores.rmse - summary[\'hybrid_rmse\']\n    scores = scores[[\'model\', \'rmse\', \'mae\', \'bias\', \'delta_rmse_vs_hybrid\', \'n\']]\n    scores[\'model\'] = scores.model.replace({\'blend\': \'Fixed 75/25 blend\'})\n    delta = summary[\'delta_rmse_vs_hybrid\']\n    outcome = (\'The fixed blend reduces pooled development RMSE.\' if delta < 0 else\n               \'The fixed blend increases pooled development RMSE.\' if delta > 0 else \'Pooled RMSE is tied.\')\n    parts = [\'<h1>One fixed blend: reference model + U-Net</h1>\',\n             f\'<p class="lead">{outcome} Difference: <strong>{delta:+.6f} mm/day</strong>.</p>\',\n             \'<p>75% reference model + 25% U-Net · January 2007–December 2020 · Seven chronological blocks</p>\',\n             \'<p class="note">One weight fixed before this calculation, after inspecting the original model diagnostics. No weight search, retraining, new source or automatic promotion. These years are development data, not an independent holdout.</p>\',\n             \'<h2>Overall comparison</h2><p>Metrics pool errors across the full supplied grid, including ocean. Bias is prediction minus observation; a smaller absolute bias is closer to zero.</p>\',\n             table_html(scores),\n             \'<details><summary>Run identity</summary><dl>\' + \'\'.join(\n                 f\'<dt>{escape(k)}</dt><dd>{escape(str(v))}</dd>\' for k, v in {\n                     \'Run\': record[\'run_id\'], \'Started UTC\': record[\'started_at\'], \'Fingerprint\': record[\'fingerprint\'],\n                     \'Source signature\': record[\'source_signature_sha256\'], \'Formula\': \'0.75 × reference model + 0.25 × U-Net\',\n                 }.items()) + \'</dl></details>\']\n    for name, title, text in [\n        (\'temporal_comparison.png\', \'Temporal consistency\',\n         f"RMSE improves in {summary[\'blocks\'][\'improved\']}/7 blocks and {summary[\'years\'][\'improved\']}/14 years. A global mean can hide losses in specific periods."),\n        (\'calendar_intensity.png\', \'Season and rainfall intensity\',\n         \'Intensity groups use observed monthly means, not daily extremes. Outcomes are unknown at issue time; these groups are descriptive and cannot choose weights.\'),\n        (\'spatial_comparison.png\', \'Location\',\n         f"RMSE improves in {summary[\'blend_lower_rmse_grid_fraction\']:.2%} of grid cells. This is a cell count, not a geographic-area or land-only fraction."),\n    ]:\n        encoded = base64.b64encode((folder / name).read_bytes()).decode(\'ascii\')\n        parts += [f\'<h2>{title}</h2><p>{escape(text)}</p><img alt="{title}" src="data:image/png;base64,{encoded}">\']\n    for name, key, title in [(\'blocks\', \'block\', \'Blocks\'), (\'years\', \'year\', \'Years\'),\n                              (\'calendar\', \'calendar_month\', \'Calendar months\'), (\'region\', \'region\', \'Latitude bands\'),\n                              (\'intensity\', \'intensity\', \'Observed intensity\')]:\n        frame = pd.read_csv(folder / (name + \'_comparison.csv\'))\n        cols = [key, \'n\', \'hybrid_rmse\', \'blend_rmse\', \'delta_rmse\', \'delta_mae\', \'hybrid_bias\', \'blend_bias\', \'delta_absolute_bias\']\n        parts += [f\'<details><summary>{title}: paired numbers</summary>\' + table_html(frame[cols]) + \'</details>\']\n    products = json.loads((folder / \'complementarity.json\').read_text())\n    rho = products[\'centered_error_correlation\']\n    correlation = f\'{rho:.6f}\' if rho is not None else \'undefined (zero error variance)\'\n    parts += [\'<h2>Complementarity check</h2>\',\n              f\'<p>Pooled error correlation: {correlation}. The exact MSE identity was checked against the blended maps. No optimal weight was estimated. The correlation is descriptive; it mixes dates and locations and is not an independent-sample significance test.</p>\',\n              \'<h2>What was verified</h2><p>Original input hashes, dates, coordinates and units; original model scores; the fixed blend written and read back in float64; complete intensity partitions; the MSE identity; and the workflow artifact inventory. Archived rainfall maps were already nonnegative, so no additional clipping was applied.</p>\',\n              \'<p>The workflow keeps the original diagnostics under diagnostics/ and the new candidate results under fixed_blend/. Code, input signatures and configuration are recorded. The current forecasting model and source-availability gate are unchanged.</p>\']\n    css = \'body{font:16px/1.55 system-ui,sans-serif;color:#172b36;max-width:1100px;margin:40px auto;padding:0 24px;background:#f8fafb}h1,h2{line-height:1.2}h2{margin-top:38px}.lead{font-size:20px}.note{border-left:4px solid #6246a5;padding:12px 16px;background:#f0edf7}details{margin:18px 0}summary{cursor:pointer;font-weight:650}dd{overflow-wrap:anywhere;margin-bottom:12px}table{border-collapse:collapse;width:100%;display:block;overflow:auto;font-size:14px;background:white}th,td{padding:9px 12px;border-bottom:1px solid #dae2e6;text-align:right}th:first-child,td:first-child{text-align:left}img{max-width:100%;height:auto}\'\n    (run / \'report.html\').write_text(\'<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Fixed rainfall blend</title><style>\' + css + \'</style><body>\' + \'\\n\'.join(parts) + \'</body></html>\', encoding=\'utf-8\', newline=\'\\n\')\n    (run / \'REPORT.md\').write_text(\'# Fixed 75/25 blend\\n\\n\' + outcome + f\' Difference: {delta:+.6f} mm/day.\\n\\n\'\n                                  + \'Open report.html for tables and embedded figures. One fixed weight; no fitting or promotion. Previously consulted development years.\\n\', encoding=\'utf-8\', newline=\'\\n\')\n', 'research/fixed_blend/test_blend.py': '"""Check pairing, reconstruction stage and complementary-error accounting."""\nfrom pathlib import Path\nimport unittest\nimport numpy as np\nimport pandas as pd\n\nfrom research.fixed_blend.evaluate import fixed_blend, error_products, summarize_products, paired\nfrom research.workflow.config import normalize, WorkflowError\nfrom research.workflow.test_workflow import example\n\n\nclass FixedBlendChecks(unittest.TestCase):\n    def test_constant_reference_and_convex_bounds(self):\n        h = np.array([0, 1, 10], dtype=\'float32\')\n        np.testing.assert_array_equal(fixed_blend(h, h), h)\n        u = np.array([4, 0, 2], dtype=\'float32\')\n        blend = fixed_blend(h, u)\n        self.assertEqual(blend.dtype, np.dtype(\'float64\'))\n        self.assertTrue(((blend >= np.minimum(h, u)) & (blend <= np.maximum(h, u))).all())\n        with self.assertRaisesRegex(ValueError, \'nonnegative\'):\n            fixed_blend(h, [-1, 0, 2])\n        with self.assertRaisesRegex(ValueError, \'different shapes\'):\n            fixed_blend(h, [[4, 0, 2]])\n        with self.assertRaisesRegex(ValueError, \'nonfinite\'):\n            fixed_blend(h, [np.nan, 0, 2])\n\n    def test_mse_identity_and_complementarity(self):\n        y = np.full(4, 5.)\n        h = y + np.array([1., -1., 2., -2.])\n        u = y - 3 * (h - y)\n        # Nonnegative construction with exactly cancelling weighted errors.\n        y += 10; h += 10; u += 10\n        result = summarize_products([error_products(y[:2], h[:2], u[:2]), error_products(y[2:], h[2:], u[2:])])\n        np.testing.assert_array_equal(fixed_blend(h, u), y)\n        self.assertAlmostEqual(result[\'centered_error_correlation\'], -1)\n        self.assertAlmostEqual(result[\'blend_sse_from_products\'], 0)\n        self.assertAlmostEqual(result[\'blend_sse_from_disagreement\'], 0)\n\n    def test_pairing_and_signed_vs_absolute_bias(self):\n        frame = pd.DataFrame([dict(model=\'hybrid\', year=1, n=4, rmse=2., mae=1., bias=-.2, sse=16.),\n                              dict(model=\'blend\', year=1, n=4, rmse=1., mae=.8, bias=-.1, sse=4.)])\n        result = paired(frame, [\'year\']).iloc[0]\n        self.assertAlmostEqual(result.delta_bias, .1)\n        self.assertAlmostEqual(result.delta_absolute_bias, -.1)\n        frame.loc[1, \'n\'] = 3\n        with self.assertRaisesRegex(ValueError, \'Unpaired\'):\n            paired(frame, [\'year\'])\n\n    def test_config_rejects_training_and_weight_search(self):\n        config = example(Path(\'/example\'))\n        config[\'experiment\'] = \'fixed_blend_v1\'\n        self.assertEqual(normalize(config, Path.cwd())[\'experiment\'], \'fixed_blend_v1\')\n        config[\'weights\'] = [.5, .5]\n        with self.assertRaisesRegex(WorkflowError, \'Unknown configuration\'):\n            normalize(config, Path.cwd())\n        del config[\'weights\']\n        config[\'mode\'] = \'train\'\n        with self.assertRaisesRegex(WorkflowError, \'saved maps only\'):\n            normalize(config, Path.cwd())\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'research/lagged_sources/evidence/metricas_blocos.csv': 'modelo,bloco,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area,rmse,mae,vies,rmse_area\nclimatologia,A,1885464,6483793.032390847,247939.59247075464,2122255.6579590403,6038937.505138205,1619418.8409421304,1.854408671157335,1.125588002719246,0.1315005709314814,1.9310818134839693\nclimatologia,B,1885464,6485643.75706844,-150120.7514444294,2069800.4850491479,6102796.701143213,1619418.8409421304,1.8546733121631138,1.0977671729872052,-0.07962005715538954,1.9412651476568235\nclimatologia,C,1885464,6436085.131536505,262635.45612728415,2068391.4041558888,6110980.556575299,1619418.8409421304,1.8475736837464716,1.0970198339272925,0.13929486647704975,1.9425663306667187\nclimatologia,H1,1885464,6401491.782505364,-174106.14684297366,2030454.079381687,6064846.86348311,1619418.8409421304,1.8426017268670747,1.0768988850392727,-0.09234127347060121,1.9352199205750888\nclimatologia,H2,1885464,7433339.993815827,-154987.7861661935,2219894.617458496,7032419.844942207,1619418.8409421304,1.9855593655436024,1.1773731121137798,-0.08220140303192927,2.08388045581921\nclimatologia,H3,1885464,6360026.256373299,-147269.18968771095,2081298.099301327,5988413.4375352375,1619418.8409421304,1.83662432487597,1.1038652020411566,-0.07810766457896356,1.9229867607508768\nclimatologia,H4,1885464,5602144.191952371,46220.0596004611,1933057.2457860655,5254510.271905689,1619418.8409421304,1.7237251533320521,1.0252421927897142,0.0245138913288512,1.8013019900613414\ncontrole_defasado,A,1885464,5718423.53498111,23207.31116099255,1976427.6625193635,5295033.650443283,1619418.8409421304,1.7415222874502265,1.0482447092701657,0.01230854111295286,1.8082345720332853\ncontrole_defasado,B,1885464,6124671.955066858,-89840.5353237101,2009301.7122752154,5761746.876286968,1619418.8409421304,1.802321626064037,1.0656802316433596,-0.04764903245233539,1.8862423597027211\ncontrole_defasado,C,1885464,5713691.500375254,306106.6676142241,1943420.5199247042,5403561.990782675,1619418.8409421304,1.7408015775162955,1.0307385979921675,0.16235084181624473,1.8266715938581932\ncontrole_defasado,H1,1885464,5717386.989668707,-63237.31423828187,1954637.501365696,5386647.97874958,1619418.8409421304,1.7413644424984764,1.0366877868607918,-0.033539390960676985,1.8238104662436954\ncontrole_defasado,H2,1885464,6103053.668604041,-86302.90089028547,2042798.715182023,5709538.535129083,1619418.8409421304,1.7991379822396205,1.0834461518130407,-0.0457727651603454,1.8776771029787227\ncontrole_defasado,H3,1885464,5727186.065267311,-42184.875250228695,1992655.7199189912,5364761.4835922215,1619418.8409421304,1.7428560729125753,1.056851639659517,-0.022373736783215536,1.8201015318382443\ncontrole_defasado,H4,1885464,5472373.142781409,17137.38105349867,1902131.8049236808,5129050.457443741,1619418.8409421304,1.7036435332466897,1.0088401607899598,0.009089211490380443,1.779667591749282\nsst_defasada,A,1885464,5715477.845253004,65086.18084917702,1977404.2384530364,5290436.898130808,1619418.8409421304,1.7410736808460576,1.048762659193194,0.03451998067805963,1.8074495145862108\nsst_defasada,B,1885464,6123432.431848174,-95188.47479395218,2011196.380176176,5757629.408905434,1619418.8409421304,1.8021392381286345,1.0666851131478385,-0.05048543742757866,1.8855682647115124\nsst_defasada,C,1885464,5756161.930511232,316289.7261935604,1949143.450793509,5444593.505302643,1619418.8409421304,1.7472593739308742,1.0337738884399326,0.1677516654752148,1.8335938199767292\nsst_defasada,H1,1885464,5659258.808561306,-78454.42295129443,1945213.9619282382,5327502.105798034,1619418.8409421304,1.7324896764703681,1.0316897919706969,-0.04161014103228406,1.8137700285402012\nsst_defasada,H2,1885464,6087308.589751015,-43385.96637334982,2046189.1119687043,5693281.347570842,1619418.8409421304,1.796815713347927,1.0852443281699913,-0.02301076359630829,1.8750019735273016\nsst_defasada,H3,1885464,5754459.463448089,9217.205489752872,2004233.325897917,5394042.853104605,1619418.8409421304,1.7470009663710935,1.06299209419958,0.004888560847490523,1.8250619146150826\nsst_defasada,H4,1885464,5451203.129562975,53674.234981407244,1907135.917188048,5107821.079312822,1619418.8409421304,1.7003450461446072,1.0114942089523045,0.028467387858589314,1.7759807093276283\n', 'research/lagged_sources/library.py': 'from pathlib import Path\nfrom datetime import datetime, timezone\nfrom time import perf_counter\nimport base64, gzip, io, json, hashlib, gc, platform, importlib.metadata\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport lightgbm as lgb\nfrom sklearn.decomposition import PCA\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display, Markdown, Image, FileLink\nDATASET, SYSTEM, SCHEMA = (\'seasonal-monthly-single-levels\', \'51\', \'worcap_seas5_v1\')\nPASTA_SEAS5_MANUAL = PASTA_CFSV2_MANUAL = None\nPONTOS_POR_MES = 5000\nANOS_TREINO = 30\nARVORES = 300\nSEMENTE = 42\nCORTE_FINAL = \'2022-12-01\'\nFIM_DESENVOLVIMENTO = \'2020-12-01\'\nPARAMETROS = {\'objective\': \'regression\', \'metric\': \'rmse\', \'learning_rate\': 0.05, \'num_leaves\': 31, \'min_data_in_leaf\': 150, \'lambda_l2\': 10.0, \'feature_fraction\': 0.9, \'bagging_fraction\': 0.8, \'bagging_freq\': 1, \'num_threads\': 4, \'seed\': SEMENTE, \'deterministic\': True, \'force_col_wise\': True, \'verbosity\': -1}\nBLOCOS = [{\'nome\': \'H1\', \'corte\': \'2006-12-01\', \'inicio\': \'2007-01-01\'}, {\'nome\': \'H2\', \'corte\': \'2008-12-01\', \'inicio\': \'2009-01-01\'}, {\'nome\': \'H3\', \'corte\': \'2010-12-01\', \'inicio\': \'2011-01-01\'}, {\'nome\': \'H4\', \'corte\': \'2012-12-01\', \'inicio\': \'2013-01-01\'}, {\'nome\': \'A\', \'corte\': \'2014-12-01\', \'inicio\': \'2015-01-01\'}, {\'nome\': \'B\', \'corte\': \'2016-12-01\', \'inicio\': \'2017-01-01\'}, {\'nome\': \'C\', \'corte\': \'2018-12-01\', \'inicio\': \'2019-01-01\'}]\nVARIAVEIS = [\'cloud_cover\', \'geopotential_850\', \'rel_hum_850\', \'shum_850\', \'surface_pressure\', \'t2\', \'temperature_850\', \'u_850\', \'v_850\']\nINDICES_NOMES = [\'nino34\', \'nino12\', \'tna\', \'tsa\']\nFONTES_INDICES = {\'fonte\': \'NOAA Physical Sciences Laboratory\', \'captura_utc\': \'2026-09-21T23:00:40.933121+00:00\', \'unidade\': \'degC\', \'tipo\': \'monthly sea surface temperature anomaly indices\', \'interpretacao_temporal\': \'Retrospective series with observation month strictly T-1. These are not archived real-time vintages.\', \'inicio\': \'1976-12-01\', \'fim\': \'2024-11-01\', \'meses\': 576, \'colunas\': [\'time_origem\', \'nino34\', \'nino12\', \'tna\', \'tsa\'], \'fontes\': [{\'indice\': \'nino34\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data\', \'arquivo_bruto\': \'nino34.data\', \'sha256_bruto\': \'46e501b91f624ef85b525d73fbb427f9d04d30411ba6e5c7cff579b35fb4ff01\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA34\\n 5N-5S 170W-120W \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'nino12\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data\', \'arquivo_bruto\': \'nino12.data\', \'sha256_bruto\': \'b13ad5ad085cd13e70ec5d82662b509d522204e38a8eefd133f14be270e51928\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA12\\n 0N-10S 270W-280E \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'tna\', \'url\': \'https://psl.noaa.gov/data/correlation/tna.data\', \'arquivo_bruto\': \'tna.data\', \'sha256_bruto\': \'387c4125f377e57bcb683a849aefcd76da7d7f3cdcb2ac68ab086ceadf300e8b\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TNAa\\n Tropical Northern Atlantic Index\\n SST Anom 5.5N-23.5N; 15W-57.5W\\n  Using HadISST1.1: Climo 1991-2020\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}, {\'indice\': \'tsa\', \'url\': \'https://psl.noaa.gov/data/correlation/tsa.data\', \'arquivo_bruto\': \'tsa.data\', \'sha256_bruto\': \'700c0dffda0f19c1254ba482808b345eaa0538f5bf09fc3bc025f46a5fbe3a69\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TSAa\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}], \'sha256_csv\': \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\', \'transformacao_no_modelo\': \'Monthly recentering within each fold, using only the 360 training origins; no standardization or moving averages.\'}\nINDICES_CSV_GZIP_BASE64 = \'H4sIAAAAAAACCm1cy44tuY3cz7ecGuj9+JqBF4bRC7eBmf5/zElGBMkse1N90YpSSqL4CJKqv/7459//51//+8c//v7Pz59//PmvPuw/tX3++vNvn7/+72//Ve9eP7X9lPop/73G90ebn5/vP8vzc+wHsL+jABwDXBsywOwEcIZpv1xHIJqm6EDg//aDMfv3IGI8iB9fwYl/O2QSUrp9wQabAfslZOE7fX1/FMyBvVQCNhcyPz5Pw3oJOADg09PGlv2s+sblFLbPadMvW1dtQNTCA7v4PUMsWyeXUe1IK1eAM507nxik8kVgzH4bX8RxHBfLttUV++3asWoiJBdbXbctN0xHAMVSbPq9n3/a904hQFLpmz+/37AdX00hoeAg5ohl3EvIIgQSOzgJ+84+hGxBTlwe3MLeCTmEQBprxKkOrfYKgt/EinCAhEA0dor2m50nYzIkpBJS7cBGTOUInmt5jqP1tG5b63XRlOkAXMZxCdAMWNzxY52VAEoGJ4nLyL1NIgYRGxKW4vXCcdeW+zFlsw35DbuuK+VR6/0oTIXCaAmSyVeDpDDpmt9QFbtjhQt9joXjUpTz3ONGnX/WiXGpSW+P9cEFg6ZxgspjtJtbW2izVuCm64ZIqVLPxTrFJUHTVPwUhgCcondfPwRmwn4A3UVl6iMd8nHKASsrxa3caARMFyU/MGkFOS6btXWWWIDZtGecJgufhQ3AbepaglSDdxEQaOIiRKqBY6z49wYQEFcNDML00VRPQniauKu00XHzH0RzA/xTeXt7hdo/iEqJPL9ikHbdEONEK0US9qLH5d+dkK7LTdP8EdqMzoOQ5cIme/X7fTTHfO23PaKtMISTiBWW4EdWEj+nJnHD1cNO44xxw2pIB5/HTWyQ0SZE0oEVx2DrZkGAcOGUnoRzP9SFB6FzhV/lhT/mB4kIsxXCb3FLWsiGl6fFhs2sPBAJpybTiQVbvPBA3sKBf2rdHc8DkdLAh9bQ/SUEtWbN8OQG3pUAqs1tH+0Zerc1w05xzUreaWsrtF+3u5+FV9nayIUP/p5zk/WFw+EnIJX2aH2jIuztTusBVAL2p3MXAPgMDYCv+ek0IfuGQnWKpD33q32vfwQbpRLAGb73t/F2mDpUjXdsYj2b6BbCnbienaKoj7A74iaGI4vj08cHFR9fsRjhAVAQ6wHM5eZha4Idetafw/4xx5g/ceJOdapjT2a8h4LMgtMMXeM2qSAWUsnbzGQluxSkwhDjt9tyd/IgdLevuQHdfgFG6McpEbf1Yi6KCE0xe7oNjxbC3I/QDoYM1f3OGESMVxijIPjDEPRByHQ997XqxGCe9BnZrotwjT5L7vdBuOlKurHMSekz541QMPzEqkRILrADWIIhFudww3XS3TWd7YsIl8sJK2mxaO1ENCJwqDOpCbRgJsncOHc4qNIIcenW8E3TJD2JkGjuisgRUzV9R7K555kE7vHUMKIzhPOcfKWALyCFEAlnwwfOEIBFZw9E0lm41YMmMu3HpXMN0YLM+HdcbXaIQCE6IC4fmL7TCJfrmeFY4OJuuJ7CHbve0ABrO4+leBArxAOp4WRBBHAdV4gHC8GFOsUiKyIkHviLM3iCCmFWSGfU+MChzSJE0hk79rtHqM4K6cBgQ4GM/FathK6lriC8izECEccdnFzttgOpHGdsjJOgLzUsJ1BwfItH9qvnVTqFDJN+qYoE0N3fx/pAcy796AMQr69wF7jmk1E4AeSg3/XrfkEssEPbfct3l5V0CSdyOD6cCFdylCC5Z7uTv+Zlp4/jFEXnOb9ZYjDQyuFN/zy0C4TPk8MHw+PgE4zxfPv07vNxGL0quq9cGwRQH5tWH5OjYW5dx2/xqwnJyN7gt52+T2xf8aXdwuDu6zoJNH6BKCuYO8Or7TQMpCCYO311A0FS2JCI+7i8w9r9IWDKX077iUDRTpiARQDi9EvHLl7jnL2aQak0KyUU1il7NUZSedftpH2fl4jH3VfR8eZ64IS92SlVOmzzUnUSUYmA51jJmBZB5ECgAHeHO0MILsr+QFJaaW0Pn8XZn7XCOkY4CPMj0s7dOH0xm12IGPJ12XKPcFOJtx9zHzAxsGQwHjcMFMNVpbHSd/bLyg0x44SQ90CQNJZuMC74Ta79xERwQAsIdx3jdzTkCJn9nvJ+ogyEvPkEbgADvGe7Nzg8LVCPn+sQkVinq8tx73KDxSsbGLkEcww38fgV89t9L5pierrC+UzZnn+6weRLiRiXi9FH5DhwNTaJ5w/V7hZ3HDCEK7GxrjnoOpgQOX4yrQEg14ET3Yk+V04hHl+Ox0CYovDIncannAPZoJ1GjcSKckye07NQ6dYQCkOSuCNmH26NJFfxQ+v7NcXwDJOHc7IzRMycjPUEoQRfg51Mp5vLY8tbgyU2Z4lrvxZJgYzui0QG0rKst7o8UgIBpmVyF+7KbZoW7GBeAuSIx0c6PVb+hLyJLRJmZUT2/DZ35StpwfCcym3uyjHDiLDFEhm3uSs3T8cYD9ZFnyBPHM+Pne7FOQSQKLYnWjhdF6LyUgVjn89JbucsFf78BmOHzlB/riePbktMMeUd5npBPNOV8vMwckb8b0tJ4BkkEHRMu40k8EhFCYTym5Dm+VNPcpEi2Zl21xHoaOIuEHx3FTHbTdMUub/bo26yPYajOR4EjJQFwaG35rz49gixnignZ5MtyLuJvj+X7yQK1vWJ7RnSt0Fo+sTxaFiqijtqPvz20JDlZsuumUUBt0ciePkXLI9gCaXb3VzBpDM9O0VDb3dRoNSCK7OLYr07woWMsNrKKxLhfijZ/518SGLuJSU393LCdYeLo1g403L1qBOhbPCIiH2X8BHDBcI83g2TZNWDO37RD1CYtcNgDBcJ7iMYDOgD/MyI5HzyVVhp515eDIRRVRT17nibrZlS8JVnKrNlUTms94ig685gIOXj7KV80jjlerYbb7OyuJoz1CMX0KrC5DvDgWzkhulNmeK408UBgYU3avrCennK/mFeFs54RnI+ygegEpr/vKIoC9QbrQ4Rb46ebEXlHG6uaHVrZGUbD8rNFW6sKRrDG37G4yxMYsSDZMwQiaOv6bkiHKxF2jdT9OXBYPNg/a7fuS2rj9kPS6DdRNC7MQLbrPFGSDTT88LMCu2KEZubyPljRypV2ajh1CL2y3Qj5LePWBH4riSUEr7WjI/v9L7KhDMZ1sJJXCqwSsgy8P4XQt7hL3IfgBcuNlInnclLd264YjsVTLpbRhYhJhGapEWcD9uwChHJdMliIJY5jQhqiiUhLGUAQ30qx6erYqOlRxb4cpxU/bv2jpokzPA9HCdX/4ZYHfwtFVJukPU7niTxiQpF0wcu8txf9Ry0V9XZyN2ei/9+doCG1vAh2zPxw9LYTSVRHZDn4Z/ZVSxlDvueyMI/SfIJmqSc7j2eg/9aifb4UDOH+vbx2Grc1zDu/Akf/hwNCoiwMxxWJeQ8w7BuItk3VddRpSgKJywNdFNpfXuew3IWiEFSXf08H71Dnn5ofqmCEdkelUwejSfeUXVoXmCCopzI7+LCtCpY5f4jvSu11eWEOiZyTkXMpadCiCYZ1dk5i5eNCM/u2kYSIy1CyDqdbEuPLzWT853Yqm38ELGihvDzrjsLIYGclDc0RBfipCQzi/tMBxFwX1PcLQDczn1VQyJIOcrq3htSGSkfUc8nDj3EYraa1c+GNGgrpYRYYNmQemcGuhHyTt2P8HGOUNIEYTiCeRTbJxEjMiKy1mZhvqoPwPwPZRmRCEI84z7j0OFjvtEKIPvliNWOQ4MOiDuOVCBA0FkEuaGO7hAA/4aABnHPgcsx/sOxhT+fESDitmm5Uaxqv+f6RgVfSCq1I6zHLei0rYS4S09pFTF2QpJTl9bJmRAxws/5TlBpJmC+HOEarufYcC6115zvQvoFCEVa1aP70uJAUp0d5gfdTOW1zptcdkVCFVG8PhJOHSH+cD9VOhH15fZhL1ntJqK9GhzM1DGtYZcxldkZoQwnd9+QD4iWM0msMDIoAECpE89dtCSSKLDX4b0tYHW4p1Ffb9EYMdAdw/HlXucn9e1pm0HWZ8owmuNoApCDkN3ky6ddkoMgRZpodOMulYM3T44P0bdwBlEQM/qw10hF6ahFQUa0CcG2nGc8KPo9nodqvocg6IuE101PLwR073Fy9avgqgSMV+PGLR6N4t721EFnEW2f4QS1CleOG6SROT9NsjOttIYyuKXB8ZM60izPbxzBt3FzXhLhx0GIYuPigjC+FtKjqM3Pe6fWVXiBU1ocbqnTqzMPq68HO2/Lw1c1kmG85ZwofnWEdU/MvBYvpIH+YAWJmC+mJ7w9UN+YubXprtRnKsTKR7xuZKQvAeotDQulkjYBFMJSN6c3ZXKcUljbQwKqJ8Ylhd2l+eR4Gq8pl2miUL4ewzzGXRX9sjr5DM/I6K6PhND8IgcXZwXGhQhVCS4O22o3fDJOxPjISRxjWyyZagLl171WdcgEOb48zyoVYtJDG5ADT1RnvhHn1d8Ed1XDJCU2DsK/UhGuAxFtcpPlM0/FcyNRMB8RkF5SbSBacu8qXS6ynS8ikXGEgKDrasQCwufw5Jz2RkCPwF+tlyXuw3q57R9VKVc4ifW7RQ7anaKQFfmR641HiO40Q+RHFIBGaBdl8u5NrHTKHL7pQu8T/APDrg9PiRZmARECh7Ntv5Hs0trkG0zRUDQ4vrZoezeuNne42Mvx5nUYXrZ+wgUnzl2nJyzVjEjESDlHXiVvDARipmiyIg2cY6+dfIPNwXReBNg71AJByB0pGBDkpNC3MlztKTLaUaktdqn7iHZeTuKcA4yCvObGbdlBOuaJUis9+iFEhAF9L6uH48WWz7/RjhKWH+7uBO1A8FGX16YhvRO0A46qS5AIHFIB3dRiFrcoiMDOuwdLlnZ/5C9y2/vw3Pf8yF0kYm4JSROtrQX27ryMlYpWzX3ueUWzSsaF8uRW9xKMqfoFT43uKrr5x3yK9moi443oHiYmZl5KpGzq9vDmvuxU1Emmx2CJmCM1ecRkjNwDkR6HKCge0EgCZmqGr9uDmDE4vjxR7A1Q153bjSrg/chzb4+eotMdRcIlSRcNX4+VPUvY07hn0pc71fKRR7kRxF4F7JAj9yYzdXwD6BZ8frsWz6Hbfs2t2uGOxWEZuaNW/wuvx2GmpqiE40PCMgfHRwoKpiXzJ5pEMDzzW4l+1QbUNL5+XjVZ5fJ9+p3a9mgBN+IiAs6rQcRy7+pRBEB2aUY7ed+6XbWEWVqTeQhGXY1HEEYJxu96HNsIcJOkHK3U0RZZk0EqHjQgRmsEtMhxKHy2A9N4f3n74RFwPwRIAdipNv2GCjCj71TPGIrb75oINuZdHtfYNaupk93OOFKgRWs8r+R/Kw4oBNxXU6OHYrS5NbFrNAHv+opIag1RlH+z7KYstYYwSnonxvdLttVEr2+PRAz7uxchLborPSzQiyQgfpUyUukFmteSVbrOANmXopXMaP2P/A0ffwCxEk+/15OnUKBg2eZDTrw+wfVvmVJIaKwycPym3zf178NzUbW5adKTk09q8QOg5vcx1iY09LIEgHcDz0pEzqboqYDRghh5FFh7evBRcl1JAUrtKZYanohAxss/MlLgXsk03UjXRLNBUOxAhju7mlg2e9FGZLKF2K8p+npnh2rqZGcN2g2+b/XGqywlqRhRc6uRguIxB+IQUV/vK2akwoxf1NTIrmTI66nHFzJeQlEEQBrTiXg/w2FXzHTeXXNFfEdfLmKlqlletK/NV6c6EDM9ydwjPxPD+ErPyFQTcuubquH9o146JvUuASfpWOLWvoubSAhSM13+twbrRvpzRM6QJykNOc4TCjtPOC7avbUGVSee8Zl5RoSqzqFqMG9w2ZWeJUwCXkmo/AoIe5i5xOQdRnEE03t3LADwnuT0+2rEfQpcNfo4jafU6dW9rxNuSh+3MCPTy3tWPizHGWPXB1jdQ5Xt/DJDM6p7VqQ7YouQwfTq3tzAqJSh32Z9zpaHvJhHh3V5dW8+PPI0j7A2h/nble9xPeLmcPfKZlXzgIhEXd43BXbnqTHc/vW+/fYDkVrl8PK8ldp8I/5Kte4241aUDDg5eY8S3RSLqanQbZa9TXGEyu9H78GM2J9d1ADUVwGheEUftzdVuKe7e4swEIGm8nb30mM6v1TcVhtTajIHoIexq6ofJQ2N2rZZYouge4Ql+1fiaSq3CYO8c95JycepvFaNF+VwWGOJi2H/+/fzv728lUqf9waQ9e6J0AKiXtTctUEQnMFlcIZ7rtY8dtshgzNZ0PMbY1OcVCrSixTlAAVor2zS6Km5dROSE05VtDO5rvyefEWdVbSSkPl6qTpWvIQq+tDKHbCoKe7mBvH8biQ8oQCNgEiHe0spwo5KwE13Qp2vadyf+A/PJJtJ6RrX64wSz5Pz/P4447EFyGOo6v4dv7m91n3Wdq2JB+RipJ5G0DhdwklXjylTAobnmr0+uZXxr8GoVzzTLJ7xr8Go14pq33Lmd38lwxEqXbcbQam9ZU/jjePXC1w/YtbFDVdwakSB20vkMBvxdhyvbpqzpsP11dffRqjRGXYeCbToOl8jWk1MYpPj7eV0u/Tecs0tOs57JPunH2ALZj2uVA3BtYbfamAmEfN3ApzQ1Uh3bGlsK6/80g8dlxk2B5xXXJ5Yp47gvijfmU75OEPu/fghf4GP4SEEoVPXhNTRbHPLZewwj8ySPjajJXZdorbF6KYR4XMMf2hFQtOJeKeYmM6rXkFriWLjRuG9y1KM0RLFHjJyWedaotgl/vqIOeOpnaS/qOBcxjzi0Tfe7YJ1e7lta6/39fc22k0NcgYIj31oOXW9OEN9dwWg8lkV0LRMsbenVHASdi8SwcajMQsq0T43CHB6fbyMbtzSslItsWs0sCDixJ/TaESM1ytKxvVVRr619IKpEaHwuhKwXss8Hnp07WMnz6sXi56ZbKndHPVA+2W0LQjgWfH4CypMvgAQKY8bT4dOWkN0eizvTbIrgaub+DV6IpB48tpsS/R6H/8zL4iwLgHt9ZdTqqcCLEnQMrvuclnDi8et59f6DXRzNYUfLVrMx8Yzc7aSDf22HurPJ3S/H6bcLX5q3alDe152fwNgXhdf20nMYk48RNTxdX/Et57Jiz8TNC/QujcQzOez1j50blq5P89n5yGLTLtzODKvFN2xP2Vikw/PvG42QXNj93JYrQf2evLo6QQOdThtqCotWJ90pdyDOF9ljfgO0+LKlmizbFJl0bRxfOU/ZIS/wXKVem1Bm/nswFOz/v2TX12hAhdnl55+1+pmeXlPQRu/Exnmm5a3RrSR8hjR8rBoA/4ffzwna7xKAAA=\'\nINDICES_CSV_SHA256 = \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\'\nSCHEMA_CFSV2 = \'worcap_cfsv2_pentad_media_auditada_v1\'\nPOLITICA_CFSV2 = \'ensemble_completo_ou_excecao_201908_m17_auditada_v2\'\nFONTE_CFSV2 = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\nFEATURE_SEAS5 = \'seas5_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_SEAS5_ANOM = \'seas5_tp_anomalia_mensal_treino_fold\'\nFEATURE_CFSV2 = \'cfsv2_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_CFSV2_ANOM = \'cfsv2_tp_anomalia_mensal_treino_fold\'\nNOMES_MODELOS = [\'arvores_seas5\', \'arvores_multissistema\']\nFASES_CFSV2 = dict(desenvolvimento=(\'1982-02-01\', \'2020-12-01\', 467), somente_ajuste_final=(\'2021-01-01\', \'2022-12-01\', 24), teste=(\'2023-01-01\', \'2024-12-01\', 24))\nMOS_ALPHA = 0.1\nMOS_FRACAO = 0.25\n\ndef exigir(condicao, mensagem):\n    if not bool(condicao):\n        raise ValueError(mensagem)\n\ndef sha256(arquivo):\n    digest = hashlib.sha256()\n    with Path(arquivo).open(\'rb\') as stream:\n        for bloco in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(bloco)\n    return digest.hexdigest()\n\ndef salvar_json(nome, conteudo):\n    caminho = SAIDA / nome\n    caminho.write_text(json.dumps(conteudo, indent=2, ensure_ascii=False, default=str, allow_nan=False), encoding=\'utf-8\')\n    return caminho\n\ndef localizar_dados(pasta_manual=None):\n    obrigatorios = (\'treino_tp.nc\', \'treino_tp_alvo.nc\', \'teste_features.nc\', \'sample_submission.csv\')\n    if pasta_manual is not None:\n        candidatas = [Path(pasta_manual)]\n    elif Path(\'/kaggle/input\').is_dir():\n        candidatas = sorted({p.parent for p in Path(\'/kaggle/input\').rglob(obrigatorios[0])})\n    else:\n        candidatas = [Path.cwd() / \'data\' / \'raw\', Path.cwd().parent / \'data\' / \'raw\']\n    validas = sorted({p.resolve() for p in candidatas if all(((p / nome).is_file() for nome in obrigatorios))})\n    exigir(len(validas) == 1, f\'Exactly one folder containing the four official files is required. Attach the competition data or set PASTA_DADOS_MANUAL. Complete folders found: {validas}\')\n    return validas[0]\n\ndef conferir_campo(campo, referencia, datas):\n    exigir(set(campo.dims) == {\'time\', \'lat\', \'lon\'}, \'Unexpected atmospheric dimensions.\')\n    for eixo in (\'lat\', \'lon\'):\n        exigir(np.array_equal(campo[eixo].values, referencia[eixo].values), f\'Misaligned atmospheric grid: {eixo}.\')\n    exigir(pd.DatetimeIndex(campo.time.values).equals(pd.DatetimeIndex(datas)), \'Misaligned atmospheric dates.\')\n\ndef ajustar_clima_atmosfera(campos, origens):\n    origens = pd.DatetimeIndex(origens)\n    resultado = {}\n    for nome in VARIAVEIS:\n        campo = campos[nome]\n        indices = pd.DatetimeIndex(campo.time.values).get_indexer(origens)\n        exigir((indices >= 0).all(), \'Missing atmospheric fitting months.\')\n        mapas = []\n        for mes in range(1, 13):\n            ids = indices[origens.month == mes]\n            exigir(len(ids) > 0, \'Calendar month with no atmospheric history.\')\n            valores = campo.values[ids]\n            exigir(np.isfinite(valores).all(), f\'{nome}: nonfinite training values.\')\n            mapas.append(valores.mean(axis=0, dtype=np.float64).astype(np.float32))\n        resultado[nome] = np.stack(mapas)\n    return resultado\n\ndef reconstruir_chuva(clima_tp, anomalias, peso):\n    peso = float(peso)\n    exigir(np.isfinite(peso) and peso >= 0, \'The weight must be finite and nonnegative.\')\n    meses = pd.DatetimeIndex(anomalias.time.values).month.to_numpy() - 1\n    valores = np.maximum(0, clima_tp.values[meses] + peso * anomalias.values)\n    exigir(np.isfinite(valores).all(), \'Rainfall reconstruction produced invalid values.\')\n    return xr.DataArray(valores.astype(np.float32), dims=anomalias.dims, coords=anomalias.coords, name=\'tp_mm_day\', attrs={\'units\': \'mm/day\'})\n\ndef auditar_chuva_desenv():\n    with xr.open_dataset(PASTA / \'treino_tp.nc\') as ds, xr.open_dataset(PASTA / \'treino_tp_alvo.nc\') as da:\n        bruto, alvo = (ds.tp, da.tp_alvo)\n        esperado = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n        exigir(set(bruto.dims) == set(alvo.dims) == {\'time\', \'lat\', \'lon\'}, \'Unexpected precipitation dimensions.\')\n        bruto, alvo = (bruto.transpose(\'time\', \'lat\', \'lon\'), alvo.transpose(\'time\', \'lat\', \'lon\'))\n        exigir(bruto.shape == alvo.shape == (996, 301, 261), \'Unexpected grid or calendar.\')\n        exigir(pd.DatetimeIndex(bruto.time.values).equals(esperado), \'Official calendar mismatch.\')\n        exigir(bruto.attrs.get(\'units\') == alvo.attrs.get(\'units\') == \'mm/day\', \'Rainfall must use mm/day; no conversion is applied.\')\n        for eixo in [\'time\', \'lat\', \'lon\']:\n            exigir(np.array_equal(bruto[eixo].values, alvo[eixo].values), f\'Misaligned target at {eixo}.\')\n        for eixo in [\'lat\', \'lon\']:\n            exigir(np.all(np.diff(bruto[eixo].values) == 0.25), f\'Invalid grid: {eixo}.\')\n        chuva = bruto.sel(time=slice(None, FIM_DESENVOLVIMENTO)).load()\n        exigir(np.isfinite(chuva.values).all() and (chuva.values >= 0).all(), \'Invalid historical precipitation.\')\n        for i in range(0, chuva.sizes[\'time\'] - 1, 24):\n            j = min(i + 24, chuva.sizes[\'time\'] - 1)\n            exigir(np.array_equal(alvo.isel(time=slice(i, j)).values, chuva.values[i + 1:j + 1]), \'tp_alvo[M] does not match tp[M+1].\')\n    return chuva\n\ndef carregar_atmosfera(referencia, inicio, fim):\n    resultado, metadados = ({}, [])\n    esperado_completo = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n    esperado_recorte = pd.date_range(inicio, fim, freq=\'MS\')\n    for nome in VARIAVEIS:\n        with xr.open_dataset(PASTA / f\'treino_{nome}.nc\') as ds:\n            bruto = ds[nome].transpose(\'time\', \'lat\', \'lon\')\n            conferir_campo(bruto, referencia, esperado_completo)\n            campo = bruto.sel(time=slice(inicio, fim)).astype(\'float32\').load()\n            conferir_campo(campo, referencia, esperado_recorte)\n            exigir(np.isfinite(campo.values).all(), f\'{nome}: nonfinite atmospheric data.\')\n            resultado[nome] = campo\n            metadados.append(dict(variavel=nome, atributos=dict(bruto.attrs), inicio=str(inicio), fim=str(fim), minimo=float(campo.min()), maximo=float(campo.max())))\n        print(\'Loaded:\', nome, campo.shape, flush=True)\n    return (resultado, metadados)\n\ndef carregar_indices_incorporados():\n    dados = gzip.decompress(base64.b64decode(INDICES_CSV_GZIP_BASE64))\n    exigir(hashlib.sha256(dados).hexdigest() == INDICES_CSV_SHA256, \'Embedded NOAA snapshot hash mismatch.\')\n    tabela = pd.read_csv(io.BytesIO(dados), parse_dates=[\'time_origem\'])\n    exigir(tabela.columns.tolist() == [\'time_origem\'] + INDICES_NOMES, \'Ocean index columns differ from the reference.\')\n    tabela = tabela.set_index(\'time_origem\')\n    conferir_indices(tabela)\n    esperado = pd.date_range(FONTES_INDICES[\'inicio\'], FONTES_INDICES[\'fim\'], freq=\'MS\')\n    exigir(tabela.index.equals(esperado), \'The NOAA snapshot has missing or altered dates.\')\n    return (tabela, dados)\n\ndef conferir_indices(tabela):\n    exigir(tabela.columns.tolist() == INDICES_NOMES, \'The order of the four ocean indices changed.\')\n    datas = pd.DatetimeIndex(tabela.index)\n    exigir(len(datas) > 0 and datas.is_unique and datas.is_monotonic_increasing, \'NOAA dates are empty, duplicated or unordered.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'NOAA dates must be consecutive first-of-month timestamps.\')\n    exigir(np.isfinite(tabela.to_numpy(dtype=np.float64)).all(), \'Missing or infinite NOAA index. Do not interpolate or fill from future months.\')\n\ndef auditar_dataset(ds, esperado=None):\n    exigir(ds.attrs.get(\'schema\') == SCHEMA, \'Unknown SEAS5 artifact version.\')\n    exigir(ds.attrs.get(\'dataset\') == DATASET and str(ds.attrs.get(\'system\')) == SYSTEM, \'Invalid SEAS5 provenance.\')\n    exigir(ds.attrs.get(\'product_type\') == \'monthly_mean\' and int(ds.attrs.get(\'leadtime_month\')) == 2, \'Incorrect product or lead time.\')\n    exigir(ds.seas5_tp_media.attrs.get(\'units\') == \'mm/day\', \'SEAS5 must use mm/day after conversion.\')\n    exigir(ds.seas5_tp_media.dims == (\'time\', \'lat\', \'lon\'), \'Unexpected SEAS5 dimensions.\')\n    datas = pd.DatetimeIndex(ds.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing, \'Duplicate or unordered targets.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'Missing SEAS5 target months.\')\n    if esperado is not None:\n        exigir(datas.equals(pd.DatetimeIndex(esperado)), \'SEAS5 dates do not match the request.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals((datas.to_period(\'M\') - 1).to_timestamp()), \'SEAS5 must be initialized exactly in T-1.\')\n    exigir(np.isin(ds.n_membros.values, [25, 51]).all(), \'Incomplete ensemble member set.\')\n    for eixo in [\'lat\', \'lon\']:\n        v = ds[eixo].values\n        exigir(len(v) > 1 and np.all(np.diff(v) == 1), f\'Incorrect native one-degree grid: {eixo}.\')\n    exigir(ds.lat.min() <= -60 and ds.lat.max() >= 15 and (ds.lon.min() <= -90) and (ds.lon.max() >= -25), \'The input domain does not cover the official grid. Do not extrapolate.\')\n    exigir(np.isfinite(ds.seas5_tp_media.values).all() and (ds.seas5_tp_media.values >= 0).all(), \'Invalid SEAS5 values.\')\n\ndef localizar_seas5():\n    if PASTA_SEAS5_MANUAL:\n        candidatos = [Path(PASTA_SEAS5_MANUAL) / \'seas5_51_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_SEAS5_dados\'), Path.cwd() / \'outputs/worcap_SEAS5_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'seas5_51_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Attach the prepared SEAS5 dataset or set PASTA_SEAS5_MANUAL. Exactly one SEAS5 manifest is required.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    exigir(m.get(\'schema\') == SCHEMA and m.get(\'system\') == SYSTEM and (m.get(\'dataset\') == DATASET), \'Incompatible SEAS5 manifest.\')\n    exigir(m.get(\'leadtime_month\') == 2 and m.get(\'paramId\') == 172228 and (m.get(\'product_type\') == \'monthly_mean\') and (m.get(\'variable\') == \'total_precipitation\'), \'SEAS5 product does not match the experiment.\')\n    return (caminho.parent, m)\n\ndef carregar_seas5(uso, referencia):\n    item = MANIFESTO_SEAS5[\'arquivos\'][uso]\n    caminho = PASTA_SEAS5 / item[\'arquivo\']\n    exigir(caminho.is_file() and sha256(caminho) == item[\'sha256\'], f\'Missing or altered SEAS5: {uso}.\')\n    with xr.open_dataset(caminho) as ds:\n        bruto = ds.load()\n    auditar_dataset(bruto, pd.date_range(item[\'inicio\'], item[\'fim\'], freq=\'MS\'))\n    exigir(bruto.attrs.get(\'uso\') == uso, \'SEAS5 file partition mismatch.\')\n    grade = bruto.seas5_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Invalid SEAS5 interpolation.\')\n    for coord in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[coord].values, referencia[coord].values), \'Misaligned SEAS5 grid.\')\n    grade.attrs.update(units=\'mm/day\', system=SYSTEM, leadtime_month=2)\n    return grade\n\ndef valores_seas5(seas, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.DatetimeIndex(seas.time.values).is_unique, \'Duplicate SEAS5 targets.\')\n    i = pd.DatetimeIndex(seas.time.values).get_indexer([destino])[0]\n    exigir(i >= 0, f\'Missing SEAS5 prediction for {destino.date()}.\')\n    origem = pd.Timestamp(seas.time_origem.values[i])\n    exigir(origem == (destino.to_period(\'M\') - 1).to_timestamp() and origem < destino, \'SEAS5 initialization is incorrect or beyond the cutoff.\')\n    v = seas.values[i].reshape(-1)\n    if pontos is not None:\n        v = v[pontos]\n    exigir(np.isfinite(v).all(), \'Nonfinite SEAS5 feature.\')\n    return v\n\ndef amostrar_m5_documentado(chuva, atmosfera, seas, corte):\n    destinos = calendario_pareado(corte)\n    origens = (destinos.to_period(\'M\') - 1).to_timestamp()\n    ref_origens, ref_alvos = calendario_pares(atmosfera[VARIAVEIS[0]].time.values, corte, 30)\n    clima, somas, contagens = estatisticas_climatologia(chuva, corte, 30)\n    exigir(np.all(contagens == 30), \'Rainfall climatology must contain exactly 30 years.\')\n    ca = ajustar_clima_atmosfera(atmosfera, ref_origens)\n    ca[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, ref_origens, corte)\n    exigir(set(destinos).issubset(set(ref_alvos)), \'Example outside the reference window.\')\n    ngrade = chuva.sizes[\'lat\'] * chuva.sizes[\'lon\']\n    n = min(PONTOS_POR_MES, ngrade)\n    X = np.empty((len(destinos) * n, 28), dtype=np.float32)\n    y = np.empty(len(X), dtype=np.float32)\n    ids = np.empty((len(destinos), n), dtype=np.int32)\n    for i, t in enumerate(destinos):\n        pontos = np.random.default_rng(np.random.SeedSequence([SEMENTE, t.year, t.month])).choice(ngrade, size=n, replace=False)\n        ids[i] = pontos\n        basicas = matriz_mes(atmosfera, ca, clima, t, pontos)\n        obs = chuva.sel(time=t).values.reshape(-1)[pontos]\n        basicas[:, 22] = ((somas[t.month - 1].reshape(-1)[pontos] - obs.astype(np.float64)) / 29).astype(np.float32)\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, :27] = basicas\n        X[sl, 27] = valores_seas5(seas, t, pontos)\n        y[sl] = obs - basicas[:, 22]\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Nonfinite training data.\')\n    exigir((origens < destinos).all() and destinos.max() <= pd.Timestamp(corte), \'Temporal cutoff violated.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens)\n\ndef valores_anomalia_seas5(seas, clima_seas, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(seas[eixo].values, clima_seas[eixo].values), \'Misaligned SEAS5 climatology.\')\n    bruto = valores_seas5(seas, destino, pontos)\n    referencia = clima_seas.sel(month=pd.Timestamp(destino).month).values.reshape(-1)\n    if pontos is not None:\n        referencia = referencia[pontos]\n    anomalia = (bruto - referencia).astype(np.float32)\n    exigir(np.isfinite(anomalia).all(), \'Nonfinite SEAS5 anomaly.\')\n    return anomalia\n\ndef amostrar_arvores_seas5(chuva, atmosfera, seas, corte):\n    x5, y, ids, destinos, clima, ca, somas, contagens = amostrar_m5_documentado(chuva, atmosfera, seas, corte)\n    clima_seas = ajustar_clima_seas5(seas, corte)\n    X = np.empty((len(x5), 29), dtype=np.float32)\n    X[:, :28] = x5\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        X[i * n:(i + 1) * n, 28] = valores_anomalia_seas5(seas, clima_seas, t, ids[i])\n    exigir(np.array_equal(X[:, :28], x5), \'The 28 base SEAS5 features changed.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Nonfinite training data.\')\n    del x5\n    return (X, y, ids, destinos, clima, ca, somas, contagens, clima_seas)\n\ndef validar_manifesto_cfsv2(m):\n    exigir(m.get(\'schema\') == SCHEMA_CFSV2 and m.get(\'modelo\') == \'NCEP-CFSv2\' and (m.get(\'fonte\') == FONTE_CFSV2), \'Incompatible CFSv2 manifest.\')\n    exigir(m.get(\'lead_iri\') == 1.5 and m.get(\'unidade\') == \'mm/day\' and (m.get(\'chuva_observada_utilizada\') is False), \'Incorrect CFSv2 product or temporal alignment.\')\n    exigir(m.get(\'meses\') == 515 and m.get(\'inicio_alvos\') == \'1982-02-01\' and (m.get(\'fim_alvos\') == \'2024-12-01\'), \'CFSv2 coverage mismatch.\')\n    exigir(m.get(\'politica_membros\') == POLITICA_CFSV2 and m.get(\'excecoes_permitidas\') == {\'2019-08-01\': [17]}, \'CFSv2 ensemble policy mismatch.\')\n    aplicadas = m.get(\'excecoes_aplicadas\', [])\n    exigir(isinstance(aplicadas, list) and len(aplicadas) <= 1, \'Unspecified CFSv2 exceptions.\')\n    if aplicadas:\n        r = aplicadas[0]\n        exigir(r.get(\'time_origem\') == \'2019-08-01\' and r.get(\'time_alvo\') == \'2019-09-01\' and (r.get(\'membros\') == 23) and (r.get(\'membros_esperados\') == 24) and (r.get(\'membros_indisponiveis_ids\') == [17]), \'Incorrect CFSv2 exception.\')\n        a = r.get(\'auditoria_excecao\', {})\n        exigir(a.get(\'membros_indisponiveis_ids\') == [17] and np.isfinite(a.get(\'maximo_delta_media_mm_day\', np.nan)) and (0 <= a[\'maximo_delta_media_mm_day\'] <= 1e-05), \'Exception missing the individual-member mean check.\')\n    for fase, (inicio, fim, n) in FASES_CFSV2.items():\n        item = m.get(\'arquivos\', {}).get(fase, {})\n        exigir(item.get(\'arquivo\') == f\'cfsv2_{fase}.nc\' and item.get(\'inicio_alvos\') == inicio and (item.get(\'fim_alvos\') == fim) and (item.get(\'meses\') == n), \'Incorrect CFSv2 partition.\')\n\ndef localizar_cfsv2():\n    if PASTA_CFSV2_MANUAL:\n        candidatos = [Path(PASTA_CFSV2_MANUAL) / \'cfsv2_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_CFSv2_dados\'), Path.cwd() / \'outputs/worcap_CFSv2_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'cfsv2_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Attach the prepared CFSv2 output or set PASTA_CFSV2_MANUAL. Exactly one CFSv2 manifest is required.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    validar_manifesto_cfsv2(m)\n    return (caminho.parent, m)\n\ndef auditar_cfsv2(ds, uso, m):\n    inicio, fim, n = FASES_CFSV2[uso]\n    datas = pd.date_range(inicio, fim, freq=\'MS\')\n    origens = (datas.to_period(\'M\') - 1).to_timestamp()\n    exigir(ds.attrs.get(\'schema\') == SCHEMA_CFSV2 and ds.attrs.get(\'modelo\') == \'NCEP-CFSv2\' and (ds.attrs.get(\'fonte\') == FONTE_CFSV2) and (ds.attrs.get(\'lead_iri\') == 1.5), \'CFSv2 NetCDF provenance mismatch.\')\n    exigir(ds.attrs.get(\'politica_membros\') == POLITICA_CFSV2, \'CFSv2 NetCDF is missing the corrected ensemble policy.\')\n    exigir(ds.cfsv2_tp_media.dims == (\'time\', \'lat\', \'lon\') and ds.cfsv2_tp_media.attrs.get(\'units\') == \'mm/day\', \'Invalid CFSv2 dimensions or units.\')\n    exigir(pd.DatetimeIndex(ds.time.values).equals(datas), \'Missing, extra or unordered CFSv2 dates.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals(origens), \'CFSv2 nominal issue month is not T-1.\')\n    exigir(np.array_equal(ds.lat.values, np.arange(-61, 17)) and np.array_equal(ds.lon.values, np.arange(-91, -23)), \'CFSv2 native grid mismatch.\')\n    for var in [\'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\', \'time_origem\']:\n        exigir(ds[var].dims == (\'time\',), f\'Invalid CFSv2 coordinate: {var}.\')\n    dmin = pd.DatetimeIndex(ds.inicializacao_mais_antiga.values)\n    dmax = pd.DatetimeIndex(ds.inicializacao_mais_recente.values)\n    exigir(not dmin.hasnans and (not dmax.hasnans) and (dmin <= dmax).all() and (dmax < datas).all() and (dmin >= origens - pd.Timedelta(days=40)).all() and (dmax <= origens + pd.Timedelta(days=7)).all(), \'CFSv2 initializations are outside the allowed window.\')\n    esperados = np.where(origens.month == 11, 28, 24)\n    if m[\'excecoes_aplicadas\'] and pd.Timestamp(\'2019-09-01\') in datas:\n        esperados[datas == pd.Timestamp(\'2019-09-01\')] = 23\n    exigir(np.array_equal(ds.n_membros.values, esperados), \'CFSv2 member counts differ from the manifest.\')\n    exigir(np.array_equal(ds.fase_fonte.values, (origens >= pd.Timestamp(\'2011-04-01\')).astype(np.int8)), \'Incorrect CFSv2 hindcast/real-time transition.\')\n    v = ds.cfsv2_tp_media.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), \'Invalid CFSv2 predicted rainfall.\')\n\ndef carregar_cfsv2(uso, referencia):\n    item = MANIFESTO_CFSV2[\'arquivos\'][uso]\n    path = PASTA_CFSV2 / item[\'arquivo\']\n    exigir(path.is_file() and sha256(path) == item[\'sha256\'], f\'Missing or altered CFSv2: {uso}.\')\n    with xr.open_dataset(path) as d:\n        bruto = d.load()\n    auditar_cfsv2(bruto, uso, MANIFESTO_CFSV2)\n    grade = bruto.cfsv2_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Invalid CFSv2 interpolation; do not extrapolate.\')\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[eixo].values, referencia[eixo].values), \'Misaligned CFSv2 grid.\')\n    for coord in [\'time_origem\', \'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\']:\n        grade = grade.assign_coords({coord: (\'time\', bruto[coord].values)})\n    grade.attrs.update(units=\'mm/day\', lead_iri=1.5)\n    salvar_json(f\'auditoria_CFSv2_{uso}.json\', dict(arquivo=str(path), sha256=item[\'sha256\'], meses=bruto.sizes[\'time\'], minimo=float(grade.min()), maximo=float(grade.max()), membros_usados=np.unique(bruto.n_membros.values).tolist(), interpolacao=\'bilinear without extrapolation\'))\n    return grade\n\ndef ajustar_clima_previsao(previsoes, corte, nome):\n    treino = calendario_pareado(corte)\n    datas = pd.DatetimeIndex(previsoes.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing and treino.isin(datas).all(), f\'Incomplete or unordered dates: {nome}.\')\n    sel = previsoes.sel(time=treino)\n    exigir(pd.DatetimeIndex(sel.time_origem.values).equals((treino.to_period(\'M\') - 1).to_timestamp()), f\'Incorrect origin in climatology {nome}.\')\n    v = sel.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), f\'Invalid values in climatology {nome}.\')\n    n = np.array([(treino.month == m).sum() for m in range(1, 13)], dtype=np.int16)\n    exigir((n > 0).all() and n.max() <= 30, \'Forecast climatology exceeds 30 years or has a missing calendar month.\')\n    medias = np.stack([v[treino.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)]).astype(np.float32)\n    return xr.DataArray(medias, dims=(\'month\', \'lat\', \'lon\'), coords={\'month\': np.arange(1, 13), \'lat\': previsoes.lat, \'lon\': previsoes.lon, \'n_meses\': (\'month\', n)}, name=f\'climatologia_{nome}\', attrs=dict(units=\'mm/day\', inicio=str(treino[0].date()), corte=str(pd.Timestamp(corte).date()), meses_treino=len(treino), anos_min=int(n.min()), anos_max=int(n.max()), referencia=\'training-month predictors only; one ensemble mean per year and month\'))\n\ndef ajustar_clima_seas5(seas, corte):\n    return ajustar_clima_previsao(seas, corte, \'SEAS5\')\n\ndef valores_cfsv2(cfs, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.Timestamp(cfs.inicializacao_mais_recente.sel(time=destino).values) < destino, \'A real CFSv2 initialization extends into the target month.\')\n    return valores_seas5(cfs, destino, pontos)\n\ndef valores_anomalia_cfsv2(cfs, clima_cfs, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(cfs[eixo], clima_cfs[eixo]), \'Misaligned CFSv2 climatology.\')\n    bruto = valores_cfsv2(cfs, destino, pontos)\n    ref = clima_cfs.sel(month=pd.Timestamp(destino).month).values.ravel()\n    if pontos is not None:\n        ref = ref[pontos]\n    anom = (bruto - ref).astype(np.float32)\n    exigir(np.isfinite(anom).all(), \'Invalid CFSv2 anomaly.\')\n    return anom\n\n# Both tree components receive exactly the same sampled pixels and residual targets.\ndef amostrar_arvores_multissistema(chuva, atmosfera, seas, cfs, corte):\n    x6, y, ids, destinos, clima, ca, somas, contagens, cs = amostrar_arvores_seas5(chuva, atmosfera, seas, corte)\n    cc = ajustar_clima_previsao(cfs, corte, \'CFSv2\')\n    X = np.empty((len(x6), 31), dtype=np.float32)\n    X[:, :29] = x6\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, 29] = valores_cfsv2(cfs, t, ids[i])\n        X[sl, 30] = valores_anomalia_cfsv2(cfs, cc, t, ids[i])\n    exigir(np.array_equal(X[:, :29], x6), \'The 29 control features changed.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Invalid training matrix.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens, cs, cc)\n\ndef prever_par(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, destinos):\n    destinos = pd.DatetimeIndex(destinos)\n    exigir(nome in NOMES_MODELOS, \'Unknown model.\')\n    exigir((destinos > pd.Timestamp(cs.attrs[\'corte\'])).all() and cs.attrs[\'corte\'] == cc.attrs[\'corte\'], \'Inference must follow the training cutoff.\')\n    out = np.empty((len(destinos), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    for i, t in enumerate(destinos):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        exigir(x.shape[1] == (29 if nome == \'arvores_seas5\' else 31) and np.isfinite(x).all(), \'Invalid inference.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    exigir(np.isfinite(out).all(), \'Invalid predictions.\')\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords={\'time\': destinos, \'lat\': clima.lat, \'lon\': clima.lon}, attrs={\'units\': \'mm/day\'})\n\n# Fit a local two-predictor ridge with training-only monthly means and scales.\ndef mos_fit(chuva, seas, cfs, corte):\n    dates = calendario_pareado(corte)\n    r = chuva.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    s = seas.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    c = cfs.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    for pre in [seas, cfs]:\n        assert pd.DatetimeIndex(pre.sel(time=dates).time_origem.values).equals((dates.to_period(\'M\') - 1).to_timestamp())\n    assert (pd.DatetimeIndex(cfs.sel(time=dates).inicializacao_mais_recente.values) < dates).all()\n    baseline = estatisticas_climatologia(chuva, corte, 30)[0]\n    shape = baseline.shape[1:]\n    ng = r.shape[1]\n    coef = np.zeros((2, ng), np.float64)\n    mu = np.zeros((2, 12, ng), np.float64)\n    for lo in range(0, ng, 4096):\n        hi = min(lo + 4096, ng)\n        y = r[:, lo:hi].astype(np.float64)\n        x = s[:, lo:hi].astype(np.float64)\n        z = c[:, lo:hi].astype(np.float64)\n        assert np.isfinite(y).all() and np.isfinite(x).all() and np.isfinite(z).all()\n        for m in range(1, 13):\n            ix = dates.month == m\n            mu[0, m - 1, lo:hi] = x[ix].mean(0)\n            mu[1, m - 1, lo:hi] = z[ix].mean(0)\n            x[ix] -= mu[0, m - 1, lo:hi]\n            z[ix] -= mu[1, m - 1, lo:hi]\n            y[ix] -= y[ix].mean(0)\n        sx = np.sqrt(np.mean(x * x, axis=0))\n        sz = np.sqrt(np.mean(z * z, axis=0))\n        sx = np.where(sx > 1e-12, sx, 1.0)\n        sz = np.where(sz > 1e-12, sz, 1.0)\n        x /= sx\n        z /= sz\n        a = np.mean(x * x, axis=0) + MOS_ALPHA\n        b = np.mean(x * z, axis=0)\n        d = np.mean(z * z, axis=0) + MOS_ALPHA\n        u = np.mean(x * y, axis=0)\n        v = np.mean(z * y, axis=0)\n        det = a * d - b * b\n        assert (det > 0).all()\n        coef[0, lo:hi] = (d * u - b * v) / det / sx\n        coef[1, lo:hi] = (a * v - b * u) / det / sz\n    assert np.isfinite(coef).all()\n    return dict(coef=coef.reshape(2, *shape), mu=mu.reshape(2, 12, *shape), baseline=baseline.values, lat=baseline.lat.values, lon=baseline.lon.values, corte=str(pd.Timestamp(corte).date()), inicio=str(dates[0].date()), meses=len(dates))\n\ndef mos_predict(fit, seas, cfs, dates):\n    dates = pd.DatetimeIndex(dates)\n    assert (dates > pd.Timestamp(fit[\'corte\'])).all()\n    for pre in [seas, cfs]:\n        assert np.array_equal(pre.lat, fit[\'lat\']) and np.array_equal(pre.lon, fit[\'lon\'])\n    out = []\n    for t in dates:\n        s = valores_seas5(seas, t).reshape(fit[\'baseline\'].shape[1:]).astype(np.float64)\n        c = valores_cfsv2(cfs, t).reshape(s.shape).astype(np.float64)\n        anom = fit[\'coef\'][0] * (s - fit[\'mu\'][0, t.month - 1]) + fit[\'coef\'][1] * (c - fit[\'mu\'][1, t.month - 1])\n        out.append(np.maximum(0, fit[\'baseline\'][t.month - 1].astype(np.float64) + anom))\n    return np.stack(out)\n\ndef matriz_mes(campos, clima_atmos, clima_tp, destino, pontos=None):\n    """Build features at the specified source month. The full training rainfall climatology is used here; the sampler replaces its training column with the leave-one-out value."""\n    destino = pd.Timestamp(destino)\n    origem = origem_mensal([destino], LAG_ATMOSFERA)[0]\n    nlat = clima_tp.sizes[\'lat\']\n    nlon = clima_tp.sizes[\'lon\']\n    pontos = np.arange(nlat * nlon) if pontos is None else np.asarray(pontos)\n    exigir(pontos.ndim == 1 and np.issubdtype(pontos.dtype, np.integer), \'Invalid indices.\')\n    exigir(((pontos >= 0) & (pontos < nlat * nlon)).all(), \'Point outside the grid.\')\n    X = np.empty((len(pontos), len(FEATURES)), dtype=np.float32)\n    for j, nome in enumerate(VARIAVEIS):\n        campo = campos[nome]\n        i = pd.DatetimeIndex(campo.time.values).get_indexer([origem])[0]\n        exigir(i >= 0, f\'Origin {origem.date()} missing in {nome}.\')\n        valores = campo.values[i].reshape(-1)[pontos]\n        clima_origem = clima_atmos[nome][origem.month - 1].reshape(-1)[pontos]\n        X[:, j] = valores\n        X[:, j + len(VARIAVEIS)] = valores - clima_origem\n    X[:, 18] = clima_tp.lat.values[pontos // nlon]\n    X[:, 19] = clima_tp.lon.values[pontos % nlon]\n    X[:, 20] = np.sin(2 * np.pi * destino.month / 12)\n    X[:, 21] = np.cos(2 * np.pi * destino.month / 12)\n    X[:, 22] = clima_tp.values[destino.month - 1].reshape(-1)[pontos]\n    exigir(origem < destino, \'Temporal violation: input origin is not earlier than its target.\')\n    valores_oceano = valores_indices_mes(INDICES_OC, clima_atmos[\'_clima_indices\'], destino)\n    X[:, 23:] = valores_oceano[None, :]\n    exigir(np.isfinite(X).all(), \'Features contain missing or infinite values.\')\n    return X\nSST_COMPONENTES = 8\nSST_NOME = \'ersstv5_4graus_198201_202411.nc\'\nSST_HASH = \'908034f6418833302ea432f25850ad2982d1114b747b7b298e26a350eaf2891d\'\n\ndef sst_auditar_arquivo(path):\n    with xr.open_dataset(path) as ds:\n        exigir(\'sst\' in ds and ds.sst.dims == (\'time\', \'lat\', \'lon\'), \'Incorrect SST variable or dimensions.\')\n        exigir(ds.sst.attrs.get(\'units\') == \'degC\', \'SST must use degrees Celsius.\')\n        exigir(\'ERSSTv5\' in str(ds.attrs.get(\'title\', \'\')) and str(ds.attrs.get(\'product_version\', \'\')) == \'Version 5\', \'Product differs from ERSSTv5.\')\n        t = pd.DatetimeIndex(ds.time.values)\n        exigir(t.equals(pd.date_range(\'1982-01-01\', \'2024-11-01\', freq=\'MS\')), \'Incomplete or different SST calendar.\')\n        exigir(np.array_equal(ds.lat.values, np.arange(60, -61, -4)) and np.array_equal(ds.lon.values, np.arange(0, 360, 4)), \'SST must use the 60S-60N subset on a four-degree grid.\')\n        sst = ds.sst.astype(\'float32\').load()\n        atributos = {k: str(v) for k, v in ds.attrs.items()}\n    v = sst.values\n    validos = v[np.isfinite(v)]\n    exigir(validos.size > 0 and (not np.isinf(v).any()) and (validos.min() >= -1.801) and (validos.max() <= 45), "SST values outside the product\'s physical range.")\n    return (sst, dict(arquivo=Path(path).name, sha256=sha256(path), bytes=Path(path).stat().st_size, inicio_origens=str(t[0].date()), fim_origens=str(t[-1].date()), meses=len(t), grade=[31, 90], minimo=float(validos.min()), maximo=float(validos.max()), atributos=atributos, referencia=\'https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html\', origem=\'NOAA PSL / ERSSTv5; retrospective snapshot, without verified historical operational vintages\'))\n\n# Fit the ocean mask, monthly anomalies and PCA using training SST only.\ndef sst_ajustar_pca(sst, alvos_treino):\n    alvos_treino = pd.DatetimeIndex(alvos_treino)\n    exigir(alvos_treino.is_unique and alvos_treino.equals(pd.date_range(alvos_treino.min(), alvos_treino.max(), freq=\'MS\')), \'Invalid PCA fitting calendar.\')\n    origens = origem_mensal(alvos_treino, LAG_SST)\n    treino = sst.sel(time=origens).transpose(\'time\', \'lat\', \'lon\')\n    bruto = treino.values.reshape(len(origens), -1).astype(np.float64)\n    mascara = np.isfinite(bruto).all(axis=0)\n    exigir(mascara.any(), \'No complete training ocean grid cells.\')\n    mensal = np.full((12, bruto.shape[1]), np.nan, dtype=np.float64)\n    contagens = np.array([(origens.month == m).sum() for m in range(1, 13)])\n    exigir((contagens >= 2).all(), \'Insufficient monthly SST climatology.\')\n    for m in range(1, 13):\n        mensal[m - 1, mascara] = bruto[origens.month == m][:, mascara].mean(axis=0)\n    anom = bruto[:, mascara] - mensal[origens.month - 1][:, mascara]\n    ativos = np.flatnonzero(mascara)\n    mascara[ativos[np.std(anom, axis=0) <= 1e-08]] = False\n    latitudes = np.repeat(sst.lat.values, sst.sizes[\'lon\'])\n    pesos = np.sqrt(np.cos(np.deg2rad(latitudes[mascara]))).astype(np.float64)\n    exigir(int(mascara.sum()) > SST_COMPONENTES and len(origens) > SST_COMPONENTES, \'Insufficient dimensions for PCA.\')\n    matriz = (bruto[:, mascara] - mensal[origens.month - 1][:, mascara]) * pesos\n    exigir(np.isfinite(matriz).all(), \'Invalid PCA matrix.\')\n    with threadpool_limits(limits=1):\n        pca = PCA(n_components=SST_COMPONENTES, svd_solver=\'full\').fit(matriz)\n    return dict(mascara=mascara, clima_mensal=mensal[:, mascara], pesos_area=pesos, centro=pca.mean_, componentes=pca.components_, variancia_explicada=pca.explained_variance_ratio_, contagens_mensais=contagens, origens_treino=origens.values, lat=sst.lat.values.copy(), lon=sst.lon.values.copy())\n\ndef sst_transformar(sst, estado, alvos):\n    alvos = pd.DatetimeIndex(alvos)\n    origens = origem_mensal(alvos, LAG_SST)\n    exigir(np.array_equal(sst.lat, estado[\'lat\']) and np.array_equal(sst.lon, estado[\'lon\']), \'SST grid changed.\')\n    v = sst.sel(time=origens).values.reshape(len(origens), -1).astype(np.float64)[:, estado[\'mascara\']]\n    exigir(np.isfinite(v).all(), \'Missing SST in the training-selected ocean mask; do not impute from the future.\')\n    x = (v - estado[\'clima_mensal\'][origens.month - 1]) * estado[\'pesos_area\'] - estado[\'centro\']\n    with threadpool_limits(limits=1):\n        scores = x @ estado[\'componentes\'].T\n    exigir(scores.shape == (len(alvos), SST_COMPONENTES) and np.isfinite(scores).all(), \'Invalid PCA scores.\')\n    return scores.astype(np.float32)\n\ndef pesquisa_prever_sst(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, datas, scores):\n    datas = pd.DatetimeIndex(datas)\n    exigir((datas > pd.Timestamp(cs.attrs[\'corte\'])).all(), \'Inference overlaps training.\')\n    exigir(scores.shape == (len(datas), 8), \'Misaligned SST scores.\')\n    out = np.empty((len(datas), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    expected = (FEATURES_SEAS5 if nome == \'arvores_seas5\' else FEATURES_MULTISSISTEMA) + SST_FEATURES\n    exigir(modelo.feature_name() == expected, \'Incorrect SST feature order.\')\n    for i, t in enumerate(datas):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        x = np.column_stack([x, np.broadcast_to(scores[i], (len(x), 8))]).astype(np.float32)\n        exigir(x.shape[1] == len(expected) and np.isfinite(x).all(), \'Invalid SST features.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords=dict(time=datas, lat=clima.lat, lon=clima.lon), attrs=dict(units=\'mm/day\'))\n\ndef pesquisa_diagnosticos(obs, previsoes, bloco):\n    """Complete fields, without pixel subsampling or hiding errors."""\n    linhas = []\n    lat = obs.lat.values\n    regioes = {\'dominio_inteiro\': np.ones(lat.size, dtype=bool), \'lat_menor_que_menos35\': lat < -35, \'lat_menos35_a_menos15\': (lat >= -35) & (lat < -15), \'lat_maior_igual_menos15\': lat >= -15}\n    for modelo, pred in previsoes.items():\n        obs, pred = xr.align(obs, pred, join=\'exact\')\n        for i, t in enumerate(pd.DatetimeIndex(obs.time.values)):\n            erro = pred.values[i].astype(np.float64) - obs.values[i].astype(np.float64)\n            exigir(np.isfinite(erro).all(), \'Nonfinite diagnostic error.\')\n            for regiao, mask in regioes.items():\n                if not mask.any():\n                    continue\n                e = erro[mask]\n                w = np.broadcast_to(np.cos(np.deg2rad(lat[mask]))[:, None], e.shape)\n                linhas.append(dict(modelo=modelo, bloco=bloco, mes_alvo=str(t.date()), ano=t.year, mes=t.month, regiao=regiao, n=e.size, sse=float(np.square(e).sum()), soma_erro=float(e.sum()), soma_erro_absoluto=float(np.abs(e).sum()), sse_area=float((w * np.square(e)).sum()), peso_area=float(w.sum())))\n    return pd.DataFrame(linhas)\n\ndef pesquisa_resumir(mensais, grupos):\n    cols = [\'n\', \'sse\', \'soma_erro\', \'soma_erro_absoluto\', \'sse_area\', \'peso_area\']\n    r = mensais.groupby(grupos, as_index=False)[cols].sum()\n    r[\'rmse\'] = np.sqrt(r.sse / r.n)\n    r[\'mae\'] = r.soma_erro_absoluto / r.n\n    r[\'vies\'] = r.soma_erro / r.n\n    r[\'rmse_area\'] = np.sqrt(r.sse_area / r.peso_area)\n    return r\n\'Candidate lag policy; not evidence of historical publication dates or vintages.\'\nLAG_ATMOSFERA = 4\nLAG_INDICES = 3\nLAG_SST = 2\nLAG_CHUVA_TREINO = 4\nINICIO_COMUM = pd.Timestamp(\'1982-03-01\')\nFEATURES = [v + \'_lag4\' for v in VARIAVEIS] + [v + \'_lag4_anomalia\' for v in VARIAVEIS] + [\'latitude\', \'longitude\', \'mes_alvo_sin\', \'mes_alvo_cos\', \'clima_tp_alvo\'] + [n + \'_lag3_anomalia_fold\' for n in INDICES_NOMES]\nFEATURES_SEAS5 = FEATURES + [FEATURE_SEAS5, FEATURE_SEAS5_ANOM]\nFEATURES_MULTISSISTEMA = FEATURES_SEAS5 + [FEATURE_CFSV2, FEATURE_CFSV2_ANOM]\nSST_FEATURES = [f\'sst_lag2_pc_{i:02d}_treino_fold\' for i in range(1, 9)]\n\ndef origem_mensal(alvos, lag):\n    datas = pd.DatetimeIndex(alvos)\n    exigir(lag >= 1 and datas.equals(datas.to_period(\'M\').to_timestamp()), \'Invalid dates or lag.\')\n    return (datas.to_period(\'M\') - lag).to_timestamp()\n\ndef janela_referencia(corte, anos=30):\n    corte = pd.Timestamp(corte)\n    exigir(corte.day == 1, \'The cutoff must identify a month.\')\n    return pd.date_range(end=corte, periods=12 * anos, freq=\'MS\')\n\ndef calendario_pareado(corte):\n    datas = janela_referencia(corte)\n    datas = datas[datas >= INICIO_COMUM]\n    exigir(24 <= len(datas) <= 360, \'Invalid common window.\')\n    return datas\n\ndef calendario_pares(datas_disponiveis, corte, anos=30):\n    alvos = janela_referencia(corte, anos)\n    origens = origem_mensal(alvos, LAG_ATMOSFERA)\n    disponiveis = pd.DatetimeIndex(datas_disponiveis)\n    exigir(disponiveis.is_unique and origens.isin(disponiveis).all(), \'Missing atmospheric origins.\')\n    return (origens, alvos)\n\n# Monthly rainfall sums and counts support leave-one-out training references.\ndef estatisticas_climatologia(tp, corte, anos=30):\n    datas = janela_referencia(corte, anos)\n    treino = tp.sel(time=datas).transpose(\'time\', \'lat\', \'lon\')\n    exigir(np.isfinite(treino.values).all(), \'Incomplete training rainfall.\')\n    somas = np.stack([treino.values[datas.month == m].sum(axis=0, dtype=np.float64) for m in range(1, 13)])\n    contagens = np.array([(datas.month == m).sum() for m in range(1, 13)], dtype=np.int64)\n    exigir((contagens == anos).all(), \'Climatology requires every calendar month.\')\n    clima = xr.DataArray((somas / contagens[:, None, None]).astype(np.float32), dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=tp.lat, lon=tp.lon), attrs=dict(units=\'mm/day\', ajuste_inicio=str(datas[0].date()), ajuste_fim=str(datas[-1].date())))\n    return (clima, somas, contagens)\n\ndef ajustar_clima_indices(tabela, origens_atmosfera, corte):\n    alvos = janela_referencia(corte)\n    exigir(pd.DatetimeIndex(origens_atmosfera).equals(origem_mensal(alvos, LAG_ATMOSFERA)), \'Atmospheric reference outside the protocol.\')\n    origens = origem_mensal(alvos, LAG_INDICES)\n    valores = tabela.loc[origens, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Incomplete training indices.\')\n    return np.stack([valores[origens.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)])\n\ndef valores_indices_mes(tabela, clima_indices, destino):\n    origem = origem_mensal([destino], LAG_INDICES)[0]\n    valores = tabela.loc[origem, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Unavailable index; do not fill.\')\n    return (valores - clima_indices[origem.month - 1]).astype(np.float32)\n\ndef exigir_disponibilidade_real(registros, emitido_em, fontes_necessarias):\n    """Validate supplied availability metadata for a future production gate. Receipt records must identify the actual bytes. This check alone cannot authenticate a historical publication timestamp."""\n    emissao = pd.Timestamp(emitido_em)\n    exigir(emissao.tzinfo is not None, \'Issue time must include a timezone.\')\n    exigir(len(registros) == len(fontes_necessarias), \'Incorrect source count.\')\n    exigir({r.get(\'fonte\') for r in registros} == set(fontes_necessarias), \'Missing or duplicate source.\')\n    for r in registros:\n        for chave in [\'disponivel_em\', \'recebido_em\', \'url\', \'sha256\', \'versao\']:\n            exigir(bool(r.get(chave)), \'Unknown availability: \' + r.get(\'fonte\', \'?\') + \'/\' + chave)\n        recebido, publicado = (pd.Timestamp(r[\'recebido_em\']), pd.Timestamp(r[\'disponivel_em\']))\n        exigir(recebido.tzinfo is not None and publicado.tzinfo is not None, \'Dates must include a timezone.\')\n        exigir(publicado <= recebido <= emissao, \'A source arrived after issue time or its chronology is invalid.\')\n        exigir(len(r[\'sha256\']) == 64 and set(r[\'sha256\']) <= set(\'0123456789abcdef\'), \'Invalid SHA-256.\')\n    return True\n\ndef calendario_emissoes(datas, corte):\n    datas = pd.DatetimeIndex(datas)\n    limite = origem_mensal([datas[0]], LAG_CHUVA_TREINO)[0]\n    exigir(pd.Timestamp(corte) <= limite, \'Training rainfall is too recent for the first issue date.\')\n    return pd.DataFrame(dict(time_alvo=datas, limite_emissao=datas - pd.Timedelta(nanoseconds=1), atmosfera=origem_mensal(datas, LAG_ATMOSFERA), indices=origem_mensal(datas, LAG_INDICES), sst=origem_mensal(datas, LAG_SST), inicializacao_sazonal=origem_mensal(datas, 1), ultimo_alvo_treino=pd.Timestamp(corte), disponibilidade_historica_comprovada=False))\n\ndef auditar_invariancia_defasada(chuva, atmosfera, seas, cfs, sst, corte, datas, clima, ca, cs, cc, pca):\n    """Mutate small copies of real data; values beyond each cutoff must not affect T."""\n    primeiro = pd.Timestamp(datas[0])\n    pontos = np.array([0, 1, chuva.sizes[\'lon\']], dtype=np.int32)\n    ch = chuva.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n    at = {k: v.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True) for k, v in atmosfera.items()}\n    alvos_ref = janela_referencia(corte)\n    orig_at = origem_mensal(alvos_ref, LAG_ATMOSFERA)\n    c0, _, _ = estatisticas_climatologia(ch, corte)\n    a0 = ajustar_clima_atmosfera(at, orig_at)\n    a0[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, orig_at, corte)\n    x0 = matriz_mes(at, a0, c0, primeiro)\n    ch.values[pd.DatetimeIndex(ch.time.values) > pd.Timestamp(corte)] = 123456\n    c1, _, _ = estatisticas_climatologia(ch, corte)\n    exigir(np.array_equal(c0.values, c1.values), \'Post-cutoff rainfall changed the climatology.\')\n    for v in at.values():\n        v.values[pd.DatetimeIndex(v.time.values) > origem_mensal([primeiro], LAG_ATMOSFERA)[0]] = 7654321\n    a1 = ajustar_clima_atmosfera(at, orig_at)\n    a1[\'_clima_indices\'] = a0[\'_clima_indices\']\n    exigir(all((np.array_equal(a0[k], a1[k]) for k in a0)), \'Future data changed atmospheric climatology.\')\n    exigir(np.array_equal(x0, matriz_mes(at, a1, c1, primeiro)), \'Recent or future atmospheric data changed T.\')\n    ni = INDICES_OC.copy(deep=True)\n    ni.loc[ni.index > origem_mensal([primeiro], LAG_INDICES)[0], :] = 99999\n    ci = ajustar_clima_indices(ni, orig_at, corte)\n    exigir(np.array_equal(ci, a0[\'_clima_indices\']), \'Future indices changed the fit.\')\n    exigir(np.array_equal(valores_indices_mes(ni, ci, primeiro), x0[0, 23:]), \'A future index changed the feature.\')\n    for pre, nome in [(seas, \'SEAS5\'), (cfs, \'CFSv2\')]:\n        pequeno = pre.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n        antes = ajustar_clima_previsao(pequeno, corte, nome)\n        pequeno.values[pd.DatetimeIndex(pequeno.time.values) > pd.Timestamp(corte)] = 12345\n        exigir(np.array_equal(antes, ajustar_clima_previsao(pequeno, corte, nome)), \'Future data changed seasonal climatology.\')\n    copia = sst.copy(deep=True)\n    copia.values[pd.DatetimeIndex(copia.time.values) > origem_mensal([primeiro], LAG_SST)[0]] = 12345\n    novo = sst_ajustar_pca(copia, calendario_pareado(corte))\n    for k in [\'mascara\', \'clima_mensal\', \'centro\', \'componentes\', \'origens_treino\']:\n        exigir(np.array_equal(novo[k], pca[k]), \'Future SST changed PCA: \' + k)\n    exigir(np.array_equal(sst_transformar(copia, novo, [primeiro]), sst_transformar(sst, pca, [primeiro])), \'SST more recent than T-2 changed the prediction for T.\')\n    return dict(chuva_pos_corte_invariante=True, atmosfera_apos_Tmenos4_invariante=True, indices_apos_Tmenos3_invariantes=True, sst_apos_Tmenos2_invariante=True, climas_sazonais_somente_treino=True, publicacao_historica_comprovada=False)\n', 'research/lagged_sources/ocean_indices.csv': 'time_origem,nino34,nino12,tna,tsa\n1976-07-01,0.150000000,2.020000000,-0.640000000,-0.930000000\n1976-08-01,0.310000000,1.810000000,-0.270000000,-0.680000000\n1976-09-01,0.570000000,1.230000000,-0.240000000,-0.470000000\n1976-10-01,0.860000000,0.760000000,-0.250000000,-0.580000000\n1976-11-01,0.880000000,0.440000000,-0.270000000,-0.630000000\n1976-12-01,0.640000000,0.250000000,-0.600000000,-0.470000000\n1977-01-01,0.840000000,0.290000000,-0.400000000,-0.530000000\n1977-02-01,0.550000000,-0.140000000,-0.400000000,-0.270000000\n1977-03-01,0.400000000,-0.380000000,-0.270000000,-0.340000000\n1977-04-01,-0.250000000,-0.680000000,-0.250000000,-0.340000000\n1977-05-01,-0.030000000,-0.450000000,-0.280000000,-0.390000000\n1977-06-01,0.360000000,0.080000000,-0.200000000,-0.410000000\n1977-07-01,0.450000000,0.030000000,-0.420000000,-0.400000000\n1977-08-01,0.250000000,-0.520000000,-0.620000000,-0.190000000\n1977-09-01,0.440000000,-0.500000000,-0.610000000,-0.120000000\n1977-10-01,0.890000000,-0.190000000,-0.660000000,-0.310000000\n1977-11-01,1.030000000,-0.290000000,-0.570000000,-0.270000000\n1977-12-01,1.090000000,-0.560000000,-0.440000000,-0.340000000\n1978-01-01,0.720000000,-0.070000000,-0.130000000,-0.520000000\n1978-02-01,0.510000000,-0.350000000,0.020000000,-0.560000000\n1978-03-01,0.060000000,-0.770000000,0.090000000,-0.800000000\n1978-04-01,-0.370000000,-0.370000000,0.100000000,-0.960000000\n1978-05-01,-0.500000000,-0.540000000,-0.130000000,-0.990000000\n1978-06-01,-0.450000000,-0.860000000,-0.390000000,-0.780000000\n1978-07-01,-0.480000000,-0.400000000,-0.600000000,-0.330000000\n1978-08-01,-0.620000000,-0.640000000,-0.660000000,-0.400000000\n1978-09-01,-0.600000000,-0.300000000,-0.590000000,-0.500000000\n1978-10-01,-0.340000000,-0.430000000,-0.350000000,-0.610000000\n1978-11-01,-0.150000000,0.040000000,-0.300000000,-0.610000000\n1978-12-01,0.070000000,0.230000000,-0.480000000,-0.430000000\n1979-01-01,0.050000000,0.230000000,-0.080000000,-0.490000000\n1979-02-01,0.090000000,-0.580000000,0.100000000,-0.510000000\n1979-03-01,0.130000000,-0.250000000,-0.010000000,-0.650000000\n1979-04-01,0.170000000,0.510000000,0.140000000,-0.300000000\n1979-05-01,-0.090000000,0.270000000,0.150000000,-0.270000000\n1979-06-01,0.040000000,0.760000000,0.180000000,-0.210000000\n1979-07-01,-0.200000000,0.450000000,0.010000000,-0.120000000\n1979-08-01,0.210000000,0.500000000,-0.250000000,0.070000000\n1979-09-01,0.480000000,1.020000000,-0.270000000,0.140000000\n1979-10-01,0.320000000,0.670000000,-0.320000000,-0.170000000\n1979-11-01,0.350000000,0.120000000,-0.200000000,-0.120000000\n1979-12-01,0.690000000,-0.340000000,-0.030000000,-0.280000000\n1980-01-01,0.550000000,-0.100000000,0.150000000,-0.480000000\n1980-02-01,0.330000000,-0.250000000,0.230000000,-0.610000000\n1980-03-01,0.090000000,0.020000000,0.020000000,-0.610000000\n1980-04-01,0.200000000,-0.000000000,0.400000000,-0.420000000\n1980-05-01,0.130000000,0.250000000,0.500000000,-0.380000000\n1980-06-01,0.370000000,0.210000000,0.230000000,-0.390000000\n1980-07-01,0.020000000,-0.440000000,-0.080000000,-0.310000000\n1980-08-01,-0.230000000,-0.440000000,-0.040000000,-0.160000000\n1980-09-01,-0.120000000,-0.110000000,-0.170000000,-0.040000000\n1980-10-01,-0.110000000,-0.520000000,-0.380000000,-0.250000000\n1980-11-01,0.100000000,-0.680000000,-0.280000000,-0.430000000\n1980-12-01,0.360000000,-1.130000000,-0.310000000,-0.860000000\n1981-01-01,-0.460000000,-1.290000000,0.030000000,-0.610000000\n1981-02-01,-0.450000000,-0.830000000,-0.010000000,-0.730000000\n1981-03-01,-0.020000000,-0.190000000,0.450000000,-0.990000000\n1981-04-01,-0.170000000,-0.310000000,0.180000000,-0.830000000\n1981-05-01,-0.110000000,-0.220000000,0.160000000,-0.750000000\n1981-06-01,-0.150000000,-0.060000000,-0.060000000,-0.530000000\n1981-07-01,-0.430000000,-0.560000000,-0.120000000,-0.480000000\n1981-08-01,-0.180000000,-0.550000000,-0.240000000,-0.070000000\n1981-09-01,-0.070000000,-0.240000000,-0.230000000,0.050000000\n1981-10-01,-0.030000000,-0.520000000,-0.390000000,0.120000000\n1981-11-01,-0.270000000,-0.550000000,-0.180000000,0.080000000\n1981-12-01,0.070000000,-0.170000000,-0.020000000,-0.160000000\n1982-01-01,-0.040000000,-0.120000000,-0.060000000,-0.300000000\n1982-02-01,-0.130000000,-0.350000000,-0.240000000,-0.470000000\n1982-03-01,-0.020000000,-0.720000000,-0.230000000,-0.560000000\n1982-04-01,0.240000000,-0.510000000,-0.320000000,-0.660000000\n1982-05-01,0.650000000,0.080000000,-0.240000000,-0.710000000\n1982-06-01,0.920000000,0.270000000,-0.400000000,-0.760000000\n1982-07-01,0.640000000,0.600000000,-0.540000000,-0.700000000\n1982-08-01,0.930000000,1.090000000,-0.640000000,-0.770000000\n1982-09-01,1.390000000,2.010000000,-0.660000000,-0.460000000\n1982-10-01,2.000000000,2.280000000,-0.770000000,-0.330000000\n1982-11-01,2.070000000,3.080000000,-0.770000000,-0.460000000\n1982-12-01,2.330000000,3.200000000,-0.790000000,-0.430000000\n1983-01-01,2.430000000,2.730000000,-0.610000000,-0.010000000\n1983-02-01,2.220000000,2.030000000,-0.190000000,0.110000000\n1983-03-01,1.690000000,2.340000000,0.280000000,-0.480000000\n1983-04-01,1.120000000,3.360000000,0.290000000,-0.560000000\n1983-05-01,1.120000000,4.100000000,-0.030000000,-0.800000000\n1983-06-01,0.620000000,4.560000000,0.100000000,-0.760000000\n1983-07-01,-0.110000000,3.930000000,-0.050000000,-0.560000000\n1983-08-01,-0.130000000,3.110000000,-0.330000000,-0.280000000\n1983-09-01,-0.500000000,2.070000000,-0.560000000,-0.110000000\n1983-10-01,-1.030000000,1.020000000,-0.580000000,-0.310000000\n1983-11-01,-1.130000000,0.560000000,-0.260000000,-0.120000000\n1983-12-01,-0.950000000,0.220000000,-0.060000000,0.120000000\n1984-01-01,-0.800000000,-0.290000000,-0.300000000,0.350000000\n1984-02-01,-0.530000000,-0.770000000,-0.380000000,0.380000000\n1984-03-01,-0.480000000,-0.410000000,-0.250000000,0.440000000\n1984-04-01,-0.600000000,-0.250000000,-0.340000000,0.190000000\n1984-05-01,-0.640000000,-1.110000000,-0.360000000,-0.080000000\n1984-06-01,-0.900000000,-0.920000000,-0.610000000,0.140000000\n1984-07-01,-0.400000000,-0.540000000,-0.630000000,0.290000000\n1984-08-01,-0.400000000,-0.520000000,-0.620000000,0.570000000\n1984-09-01,-0.350000000,-0.360000000,-0.620000000,0.690000000\n1984-10-01,-0.830000000,-0.610000000,-0.920000000,0.360000000\n1984-11-01,-1.180000000,-0.180000000,-0.770000000,0.130000000\n1984-12-01,-1.530000000,-0.500000000,-0.790000000,-0.010000000\n1985-01-01,-0.890000000,-0.770000000,-0.680000000,-0.020000000\n1985-02-01,-0.910000000,-1.130000000,-0.550000000,0.250000000\n1985-03-01,-0.960000000,-0.590000000,-0.770000000,-0.210000000\n1985-04-01,-0.980000000,-1.380000000,-0.810000000,-0.160000000\n1985-05-01,-0.900000000,-1.410000000,-0.910000000,-0.100000000\n1985-06-01,-0.760000000,-1.250000000,-0.540000000,-0.270000000\n1985-07-01,-0.610000000,-1.040000000,-0.320000000,0.020000000\n1985-08-01,-0.490000000,-1.020000000,-0.420000000,-0.100000000\n1985-09-01,-0.570000000,-0.830000000,-0.540000000,-0.130000000\n1985-10-01,-0.640000000,-0.820000000,-0.570000000,0.050000000\n1985-11-01,-0.430000000,-0.920000000,-0.390000000,0.060000000\n1985-12-01,-0.400000000,-0.750000000,-0.540000000,0.030000000\n1986-01-01,-0.790000000,-0.590000000,-0.860000000,-0.140000000\n1986-02-01,-0.640000000,-0.350000000,-0.800000000,0.200000000\n1986-03-01,-0.510000000,-0.840000000,-0.760000000,0.040000000\n1986-04-01,-0.410000000,-0.860000000,-0.800000000,-0.030000000\n1986-05-01,-0.470000000,-0.820000000,-0.740000000,-0.080000000\n1986-06-01,-0.080000000,-0.900000000,-0.840000000,0.100000000\n1986-07-01,0.160000000,-0.140000000,-0.650000000,-0.060000000\n1986-08-01,0.240000000,0.170000000,-0.740000000,0.010000000\n1986-09-01,0.590000000,-0.130000000,-0.340000000,0.160000000\n1986-10-01,0.900000000,0.140000000,-0.630000000,-0.080000000\n1986-11-01,1.040000000,0.280000000,-0.910000000,-0.350000000\n1986-12-01,0.980000000,0.530000000,-0.980000000,-0.240000000\n1987-01-01,1.190000000,0.770000000,-0.570000000,-0.320000000\n1987-02-01,1.170000000,1.020000000,-0.350000000,-0.190000000\n1987-03-01,1.240000000,1.150000000,0.060000000,-0.180000000\n1987-04-01,0.890000000,1.510000000,0.170000000,-0.270000000\n1987-05-01,0.910000000,1.350000000,0.170000000,-0.060000000\n1987-06-01,1.240000000,1.060000000,0.370000000,0.110000000\n1987-07-01,1.340000000,1.190000000,0.210000000,0.250000000\n1987-08-01,1.480000000,0.910000000,0.220000000,0.240000000\n1987-09-01,1.530000000,1.310000000,0.020000000,0.170000000\n1987-10-01,1.360000000,1.500000000,0.020000000,0.190000000\n1987-11-01,1.290000000,0.900000000,0.050000000,0.450000000\n1987-12-01,1.050000000,0.890000000,0.220000000,0.130000000\n1988-01-01,0.690000000,0.090000000,-0.090000000,0.300000000\n1988-02-01,0.350000000,-0.270000000,-0.200000000,0.390000000\n1988-03-01,0.290000000,-0.520000000,0.120000000,0.430000000\n1988-04-01,-0.490000000,-0.900000000,0.220000000,0.280000000\n1988-05-01,-1.050000000,-1.080000000,0.080000000,0.370000000\n1988-06-01,-1.460000000,-1.960000000,0.100000000,0.420000000\n1988-07-01,-1.540000000,-1.540000000,-0.000000000,0.200000000\n1988-08-01,-1.440000000,-1.590000000,-0.310000000,0.300000000\n1988-09-01,-1.330000000,-1.430000000,-0.320000000,0.160000000\n1988-10-01,-2.090000000,-1.260000000,-0.630000000,0.150000000\n1988-11-01,-2.180000000,-0.650000000,-0.540000000,-0.050000000\n1988-12-01,-1.980000000,-0.970000000,-0.520000000,-0.070000000\n1989-01-01,-1.950000000,-0.280000000,-0.670000000,0.080000000\n1989-02-01,-1.370000000,0.050000000,-0.660000000,-0.040000000\n1989-03-01,-1.330000000,-0.450000000,-0.820000000,0.000000000\n1989-04-01,-1.110000000,-0.490000000,-1.040000000,-0.020000000\n1989-05-01,-0.800000000,-1.160000000,-0.860000000,-0.010000000\n1989-06-01,-0.640000000,-0.730000000,-0.420000000,0.000000000\n1989-07-01,-0.470000000,-0.440000000,-0.030000000,0.000000000\n1989-08-01,-0.580000000,-0.460000000,0.050000000,0.190000000\n1989-09-01,-0.380000000,-0.470000000,-0.320000000,0.060000000\n1989-10-01,-0.400000000,-0.290000000,-0.300000000,0.060000000\n1989-11-01,-0.300000000,-0.380000000,-0.350000000,-0.240000000\n1989-12-01,-0.130000000,-0.430000000,-0.070000000,-0.510000000\n1990-01-01,0.020000000,-0.330000000,-0.330000000,-0.680000000\n1990-02-01,0.360000000,-0.270000000,-0.280000000,-0.140000000\n1990-03-01,0.200000000,-0.410000000,-0.200000000,-0.030000000\n1990-04-01,0.260000000,-0.330000000,-0.100000000,0.040000000\n1990-05-01,0.320000000,-0.230000000,-0.070000000,-0.400000000\n1990-06-01,0.000000000,-0.050000000,-0.270000000,-0.230000000\n1990-07-01,0.150000000,-0.870000000,-0.220000000,-0.370000000\n1990-08-01,0.170000000,-0.610000000,-0.040000000,-0.330000000\n1990-09-01,0.120000000,-0.280000000,0.020000000,-0.220000000\n1990-10-01,0.130000000,-0.730000000,-0.010000000,-0.130000000\n1990-11-01,0.080000000,-0.550000000,0.020000000,-0.010000000\n1990-12-01,0.320000000,-0.440000000,-0.120000000,-0.040000000\n1991-01-01,0.510000000,-0.090000000,-0.390000000,-0.210000000\n1991-02-01,0.320000000,-0.100000000,-0.290000000,-0.200000000\n1991-03-01,0.100000000,0.000000000,-0.370000000,-0.210000000\n1991-04-01,0.210000000,-0.680000000,-0.540000000,-0.000000000\n1991-05-01,0.440000000,-0.010000000,-0.620000000,0.140000000\n1991-06-01,0.650000000,0.240000000,-0.670000000,0.130000000\n1991-07-01,0.620000000,0.600000000,-0.670000000,-0.200000000\n1991-08-01,0.430000000,0.000000000,-0.620000000,-0.520000000\n1991-09-01,0.220000000,0.160000000,-0.520000000,-0.510000000\n1991-10-01,0.910000000,0.420000000,-0.530000000,-0.590000000\n1991-11-01,1.140000000,0.640000000,-0.460000000,-0.520000000\n1991-12-01,1.620000000,0.450000000,-0.470000000,-0.340000000\n1992-01-01,1.610000000,0.360000000,-0.240000000,-0.610000000\n1992-02-01,1.620000000,0.440000000,-0.140000000,-0.530000000\n1992-03-01,1.500000000,0.920000000,-0.340000000,-0.740000000\n1992-04-01,1.410000000,1.720000000,-0.440000000,-0.880000000\n1992-05-01,1.270000000,1.830000000,-0.390000000,-1.010000000\n1992-06-01,0.530000000,0.710000000,-0.360000000,-1.110000000\n1992-07-01,0.270000000,-0.050000000,-0.290000000,-0.860000000\n1992-08-01,-0.150000000,-0.240000000,-0.560000000,-0.860000000\n1992-09-01,-0.140000000,-0.500000000,-0.580000000,-0.700000000\n1992-10-01,-0.350000000,-0.260000000,-0.480000000,-0.740000000\n1992-11-01,-0.140000000,-0.250000000,-0.610000000,-0.970000000\n1992-12-01,0.040000000,-0.310000000,-0.520000000,-0.630000000\n1993-01-01,0.280000000,0.030000000,-0.500000000,-0.590000000\n1993-02-01,0.420000000,0.200000000,-0.430000000,-0.390000000\n1993-03-01,0.470000000,0.530000000,-0.380000000,-0.440000000\n1993-04-01,0.920000000,0.830000000,-0.220000000,-0.110000000\n1993-05-01,0.930000000,1.130000000,-0.250000000,-0.320000000\n1993-06-01,0.640000000,0.880000000,-0.180000000,-0.340000000\n1993-07-01,0.330000000,0.320000000,-0.440000000,-0.210000000\n1993-08-01,0.160000000,0.210000000,-0.450000000,-0.160000000\n1993-09-01,0.260000000,0.120000000,-0.500000000,0.300000000\n1993-10-01,0.360000000,0.320000000,-0.560000000,0.470000000\n1993-11-01,0.280000000,-0.230000000,-0.650000000,0.570000000\n1993-12-01,0.190000000,-0.290000000,-0.700000000,0.240000000\n1994-01-01,0.040000000,-0.070000000,-0.830000000,-0.010000000\n1994-02-01,-0.160000000,-0.330000000,-0.780000000,-0.140000000\n1994-03-01,-0.060000000,-1.130000000,-0.760000000,-0.100000000\n1994-04-01,0.030000000,-1.220000000,-0.620000000,-0.130000000\n1994-05-01,0.140000000,-0.910000000,-0.700000000,-0.230000000\n1994-06-01,0.270000000,-0.590000000,-0.670000000,-0.490000000\n1994-07-01,0.160000000,-0.820000000,-0.670000000,-0.520000000\n1994-08-01,0.520000000,-0.900000000,-0.630000000,-0.370000000\n1994-09-01,0.400000000,-0.050000000,-0.620000000,-0.300000000\n1994-10-01,0.900000000,0.650000000,-0.500000000,-0.270000000\n1994-11-01,1.140000000,0.590000000,-0.340000000,-0.110000000\n1994-12-01,1.210000000,0.680000000,-0.420000000,-0.070000000\n1995-01-01,1.100000000,0.910000000,-0.300000000,0.070000000\n1995-02-01,0.870000000,0.240000000,-0.220000000,0.340000000\n1995-03-01,0.490000000,-0.570000000,-0.210000000,0.390000000\n1995-04-01,0.270000000,-0.990000000,-0.040000000,0.290000000\n1995-05-01,0.030000000,-1.010000000,0.100000000,0.240000000\n1995-06-01,0.080000000,-0.530000000,0.230000000,0.330000000\n1995-07-01,0.030000000,-0.250000000,0.310000000,0.090000000\n1995-08-01,-0.380000000,-0.520000000,0.250000000,-0.260000000\n1995-09-01,-0.570000000,-0.280000000,0.030000000,-0.190000000\n1995-10-01,-0.720000000,-0.410000000,-0.060000000,-0.270000000\n1995-11-01,-0.780000000,-0.260000000,0.200000000,-0.060000000\n1995-12-01,-0.720000000,-0.690000000,0.170000000,-0.260000000\n1996-01-01,-0.650000000,-0.610000000,0.490000000,-0.150000000\n1996-02-01,-0.660000000,-0.320000000,0.220000000,0.160000000\n1996-03-01,-0.480000000,-0.070000000,0.070000000,0.120000000\n1996-04-01,-0.340000000,-1.280000000,0.150000000,0.340000000\n1996-05-01,-0.400000000,-0.810000000,0.160000000,0.430000000\n1996-06-01,-0.120000000,-1.070000000,-0.060000000,0.560000000\n1996-07-01,-0.140000000,-0.980000000,-0.150000000,0.410000000\n1996-08-01,-0.300000000,-0.460000000,-0.360000000,0.150000000\n1996-09-01,-0.340000000,-0.530000000,-0.220000000,-0.010000000\n1996-10-01,-0.280000000,-0.800000000,-0.340000000,-0.100000000\n1996-11-01,-0.300000000,-0.790000000,-0.280000000,-0.060000000\n1996-12-01,-0.430000000,-1.030000000,-0.150000000,-0.330000000\n1997-01-01,-0.430000000,-0.560000000,0.060000000,-0.550000000\n1997-02-01,-0.240000000,-0.030000000,0.160000000,-0.600000000\n1997-03-01,-0.060000000,0.820000000,-0.000000000,-0.820000000\n1997-04-01,0.340000000,0.980000000,0.120000000,-0.810000000\n1997-05-01,0.870000000,2.160000000,0.230000000,-0.890000000\n1997-06-01,1.150000000,3.170000000,0.290000000,-0.980000000\n1997-07-01,1.600000000,3.630000000,0.060000000,-0.530000000\n1997-08-01,1.940000000,3.980000000,-0.170000000,-0.290000000\n1997-09-01,2.100000000,4.050000000,-0.120000000,0.060000000\n1997-10-01,2.290000000,4.000000000,0.160000000,0.470000000\n1997-11-01,2.420000000,4.120000000,0.270000000,0.600000000\n1997-12-01,2.300000000,4.270000000,0.150000000,0.760000000\n1998-01-01,2.420000000,3.550000000,0.190000000,0.380000000\n1998-02-01,2.080000000,2.880000000,0.590000000,0.470000000\n1998-03-01,1.490000000,2.880000000,0.590000000,0.340000000\n1998-04-01,0.900000000,3.000000000,0.430000000,0.230000000\n1998-05-01,0.680000000,3.010000000,0.390000000,0.200000000\n1998-06-01,-0.390000000,2.000000000,0.420000000,0.350000000\n1998-07-01,-0.730000000,1.310000000,0.290000000,0.590000000\n1998-08-01,-0.830000000,0.940000000,0.330000000,0.400000000\n1998-09-01,-0.820000000,0.310000000,0.130000000,0.260000000\n1998-10-01,-1.190000000,0.120000000,0.070000000,-0.010000000\n1998-11-01,-1.230000000,-0.210000000,0.120000000,0.140000000\n1998-12-01,-1.510000000,-0.140000000,0.000000000,-0.150000000\n1999-01-01,-1.530000000,-0.230000000,-0.350000000,-0.200000000\n1999-02-01,-1.410000000,0.050000000,-0.380000000,-0.220000000\n1999-03-01,-0.920000000,0.300000000,-0.380000000,-0.020000000\n1999-04-01,-0.810000000,-0.780000000,-0.280000000,0.140000000\n1999-05-01,-0.870000000,-0.430000000,-0.090000000,0.180000000\n1999-06-01,-0.950000000,-0.550000000,-0.100000000,0.180000000\n1999-07-01,-0.840000000,-0.650000000,-0.100000000,0.380000000\n1999-08-01,-0.980000000,-0.510000000,0.010000000,0.350000000\n1999-09-01,-0.840000000,-0.970000000,0.010000000,0.090000000\n1999-10-01,-1.030000000,-0.760000000,-0.180000000,0.050000000\n1999-11-01,-1.410000000,-1.110000000,-0.180000000,0.150000000\n1999-12-01,-1.540000000,-1.060000000,-0.020000000,0.100000000\n2000-01-01,-1.790000000,-0.890000000,-0.420000000,-0.120000000\n2000-02-01,-1.530000000,-0.470000000,-0.280000000,0.120000000\n2000-03-01,-1.260000000,-0.560000000,-0.090000000,0.050000000\n2000-04-01,-0.800000000,0.460000000,-0.170000000,0.060000000\n2000-05-01,-0.800000000,-0.290000000,-0.270000000,-0.050000000\n2000-06-01,-0.750000000,-0.650000000,-0.400000000,-0.110000000\n2000-07-01,-0.570000000,-0.570000000,-0.270000000,-0.100000000\n2000-08-01,-0.360000000,-0.590000000,-0.190000000,-0.000000000\n2000-09-01,-0.390000000,-0.340000000,-0.360000000,-0.130000000\n2000-10-01,-0.550000000,-0.490000000,-0.420000000,-0.120000000\n2000-11-01,-0.750000000,-0.700000000,-0.430000000,-0.110000000\n2000-12-01,-0.920000000,-0.550000000,-0.490000000,-0.270000000\n2001-01-01,-0.880000000,-0.760000000,-0.380000000,-0.170000000\n2001-02-01,-0.630000000,-0.330000000,-0.320000000,-0.100000000\n2001-03-01,-0.480000000,0.350000000,-0.010000000,-0.060000000\n2001-04-01,-0.300000000,0.420000000,-0.110000000,0.070000000\n2001-05-01,-0.300000000,-0.640000000,-0.140000000,0.120000000\n2001-06-01,-0.110000000,-0.860000000,-0.010000000,0.040000000\n2001-07-01,0.010000000,-0.700000000,0.020000000,-0.110000000\n2001-08-01,-0.070000000,-0.770000000,0.000000000,-0.060000000\n2001-09-01,-0.280000000,-1.200000000,0.130000000,-0.040000000\n2001-10-01,-0.260000000,-1.140000000,0.060000000,-0.030000000\n2001-11-01,-0.280000000,-0.940000000,0.310000000,-0.020000000\n2001-12-01,-0.460000000,-1.120000000,0.370000000,-0.250000000\n2002-01-01,-0.140000000,-0.940000000,0.530000000,-0.190000000\n2002-02-01,0.000000000,-0.300000000,0.350000000,-0.330000000\n2002-03-01,0.110000000,0.620000000,0.210000000,-0.060000000\n2002-04-01,0.140000000,0.330000000,-0.180000000,-0.000000000\n2002-05-01,0.210000000,0.180000000,-0.400000000,0.150000000\n2002-06-01,0.680000000,-0.400000000,-0.380000000,0.020000000\n2002-07-01,0.570000000,-0.440000000,-0.330000000,0.220000000\n2002-08-01,0.700000000,-0.260000000,-0.380000000,-0.190000000\n2002-09-01,0.820000000,0.040000000,-0.310000000,-0.230000000\n2002-10-01,1.160000000,0.570000000,-0.070000000,-0.090000000\n2002-11-01,1.410000000,0.780000000,-0.090000000,-0.250000000\n2002-12-01,1.410000000,0.690000000,-0.090000000,0.080000000\n2003-01-01,0.980000000,0.100000000,0.020000000,0.220000000\n2003-02-01,0.640000000,-0.210000000,-0.170000000,0.300000000\n2003-03-01,0.480000000,-0.490000000,-0.110000000,0.280000000\n2003-04-01,-0.030000000,-0.900000000,-0.150000000,0.110000000\n2003-05-01,-0.520000000,-1.350000000,-0.140000000,0.020000000\n2003-06-01,-0.190000000,-0.910000000,-0.020000000,-0.210000000\n2003-07-01,0.140000000,-0.580000000,0.090000000,0.140000000\n2003-08-01,0.050000000,0.080000000,0.260000000,0.300000000\n2003-09-01,0.150000000,-0.430000000,0.280000000,0.340000000\n2003-10-01,0.460000000,-0.070000000,0.380000000,0.310000000\n2003-11-01,0.390000000,0.290000000,0.170000000,0.360000000\n2003-12-01,0.320000000,0.310000000,0.200000000,0.300000000\n2004-01-01,0.260000000,-0.030000000,0.250000000,0.070000000\n2004-02-01,0.170000000,-0.200000000,0.340000000,-0.050000000\n2004-03-01,-0.100000000,-0.340000000,0.200000000,-0.360000000\n2004-04-01,0.060000000,-0.450000000,0.300000000,-0.570000000\n2004-05-01,0.100000000,-0.960000000,-0.030000000,-0.470000000\n2004-06-01,0.140000000,-0.690000000,0.020000000,-0.390000000\n2004-07-01,0.410000000,-0.700000000,0.180000000,-0.180000000\n2004-08-01,0.660000000,-0.440000000,0.350000000,0.020000000\n2004-09-01,0.670000000,-0.100000000,0.210000000,0.180000000\n2004-10-01,0.730000000,0.110000000,0.200000000,-0.080000000\n2004-11-01,0.620000000,0.390000000,0.320000000,0.060000000\n2004-12-01,0.710000000,0.270000000,0.240000000,0.170000000\n2005-01-01,0.560000000,0.030000000,0.220000000,0.280000000\n2005-02-01,0.260000000,-0.610000000,0.200000000,0.220000000\n2005-03-01,0.280000000,-1.150000000,0.530000000,0.260000000\n2005-04-01,0.280000000,-0.360000000,0.650000000,-0.020000000\n2005-05-01,0.300000000,0.220000000,0.810000000,-0.400000000\n2005-06-01,0.220000000,-0.150000000,0.670000000,-0.570000000\n2005-07-01,-0.010000000,-0.060000000,0.570000000,-0.570000000\n2005-08-01,-0.040000000,-0.300000000,0.410000000,-0.190000000\n2005-09-01,-0.080000000,-0.610000000,0.350000000,-0.230000000\n2005-10-01,-0.150000000,-1.050000000,0.120000000,-0.220000000\n2005-11-01,-0.440000000,-1.410000000,0.090000000,-0.260000000\n2005-12-01,-0.750000000,-1.190000000,0.260000000,-0.290000000\n2006-01-01,-0.980000000,-0.690000000,0.160000000,-0.300000000\n2006-02-01,-0.710000000,0.330000000,0.010000000,-0.040000000\n2006-03-01,-0.730000000,-0.270000000,0.020000000,0.060000000\n2006-04-01,-0.300000000,-0.890000000,0.260000000,-0.000000000\n2006-05-01,-0.110000000,-0.390000000,0.230000000,-0.110000000\n2006-06-01,0.090000000,-0.190000000,0.280000000,0.100000000\n2006-07-01,0.030000000,0.180000000,0.090000000,0.050000000\n2006-08-01,0.370000000,0.510000000,0.200000000,0.130000000\n2006-09-01,0.620000000,0.780000000,0.310000000,0.130000000\n2006-10-01,0.760000000,1.200000000,0.310000000,0.010000000\n2006-11-01,0.980000000,0.910000000,0.400000000,-0.100000000\n2006-12-01,1.100000000,0.620000000,0.180000000,0.050000000\n2007-01-01,0.590000000,0.570000000,0.210000000,-0.090000000\n2007-02-01,0.120000000,0.060000000,0.380000000,-0.060000000\n2007-03-01,-0.150000000,-0.650000000,0.170000000,-0.310000000\n2007-04-01,-0.160000000,-1.050000000,0.240000000,-0.070000000\n2007-05-01,-0.390000000,-1.470000000,0.010000000,-0.060000000\n2007-06-01,-0.160000000,-1.400000000,-0.080000000,0.120000000\n2007-07-01,-0.370000000,-0.940000000,-0.180000000,-0.020000000\n2007-08-01,-0.570000000,-1.360000000,-0.330000000,-0.040000000\n2007-09-01,-1.040000000,-1.340000000,-0.270000000,0.120000000\n2007-10-01,-1.410000000,-1.650000000,-0.190000000,-0.110000000\n2007-11-01,-1.580000000,-1.950000000,-0.070000000,-0.080000000\n2007-12-01,-1.610000000,-1.630000000,-0.210000000,-0.060000000\n2008-01-01,-1.790000000,-0.800000000,-0.100000000,-0.140000000\n2008-02-01,-1.700000000,-0.160000000,-0.000000000,0.090000000\n2008-03-01,-1.170000000,0.330000000,0.120000000,0.250000000\n2008-04-01,-0.890000000,0.500000000,-0.060000000,0.330000000\n2008-05-01,-0.640000000,-0.320000000,0.070000000,0.470000000\n2008-06-01,-0.440000000,0.260000000,0.150000000,0.390000000\n2008-07-01,-0.040000000,0.470000000,0.170000000,0.400000000\n2008-08-01,-0.040000000,0.670000000,0.120000000,0.310000000\n2008-09-01,-0.280000000,0.320000000,0.200000000,0.130000000\n2008-10-01,-0.300000000,-0.170000000,0.010000000,0.050000000\n2008-11-01,-0.370000000,-0.210000000,-0.040000000,0.130000000\n2008-12-01,-0.900000000,-0.330000000,-0.030000000,0.150000000\n2009-01-01,-1.000000000,-0.170000000,-0.170000000,0.110000000\n2009-02-01,-0.710000000,-0.310000000,-0.550000000,0.080000000\n2009-03-01,-0.720000000,-0.850000000,-0.490000000,0.200000000\n2009-04-01,-0.250000000,0.000000000,-0.470000000,0.210000000\n2009-05-01,0.170000000,0.170000000,-0.430000000,0.440000000\n2009-06-01,0.490000000,0.540000000,-0.190000000,0.280000000\n2009-07-01,0.690000000,0.700000000,0.170000000,0.140000000\n2009-08-01,0.620000000,0.660000000,0.070000000,0.040000000\n2009-09-01,0.680000000,0.220000000,-0.030000000,0.040000000\n2009-10-01,0.960000000,0.220000000,0.200000000,0.230000000\n2009-11-01,1.490000000,0.140000000,0.200000000,0.110000000\n2009-12-01,1.810000000,0.490000000,0.250000000,0.430000000\n2010-01-01,1.430000000,0.360000000,0.550000000,0.460000000\n2010-02-01,1.180000000,0.040000000,0.900000000,0.410000000\n2010-03-01,1.070000000,-0.040000000,1.120000000,0.540000000\n2010-04-01,0.560000000,0.510000000,1.250000000,0.370000000\n2010-05-01,-0.150000000,0.390000000,1.150000000,0.270000000\n2010-06-01,-0.620000000,0.140000000,0.870000000,0.540000000\n2010-07-01,-0.890000000,-1.050000000,0.740000000,0.350000000\n2010-08-01,-1.330000000,-1.230000000,0.630000000,0.290000000\n2010-09-01,-1.560000000,-1.290000000,0.370000000,0.080000000\n2010-10-01,-1.650000000,-1.530000000,0.350000000,0.210000000\n2010-11-01,-1.570000000,-1.390000000,0.390000000,0.220000000\n2010-12-01,-1.630000000,-1.030000000,0.670000000,0.150000000\n2011-01-01,-1.700000000,-0.610000000,0.730000000,0.120000000\n2011-02-01,-1.260000000,0.070000000,0.350000000,0.320000000\n2011-03-01,-0.980000000,-0.430000000,0.250000000,0.380000000\n2011-04-01,-0.740000000,-0.050000000,0.170000000,0.180000000\n2011-05-01,-0.530000000,0.250000000,0.200000000,-0.040000000\n2011-06-01,-0.250000000,0.360000000,0.350000000,-0.110000000\n2011-07-01,-0.230000000,0.270000000,0.150000000,0.020000000\n2011-08-01,-0.660000000,-0.200000000,0.150000000,0.000000000\n2011-09-01,-0.760000000,-1.010000000,-0.010000000,0.090000000\n2011-10-01,-0.930000000,-0.710000000,0.020000000,0.060000000\n2011-11-01,-1.090000000,-0.800000000,-0.100000000,-0.230000000\n2011-12-01,-1.050000000,-0.680000000,-0.060000000,-0.740000000\n2012-01-01,-0.930000000,-0.390000000,-0.090000000,-0.660000000\n2012-02-01,-0.610000000,0.400000000,-0.190000000,-0.780000000\n2012-03-01,-0.480000000,-0.020000000,-0.410000000,-0.430000000\n2012-04-01,-0.290000000,0.480000000,-0.330000000,-0.640000000\n2012-05-01,-0.180000000,0.430000000,-0.110000000,-0.380000000\n2012-06-01,0.140000000,0.990000000,0.070000000,-0.270000000\n2012-07-01,0.440000000,0.820000000,0.020000000,-0.290000000\n2012-08-01,0.660000000,0.270000000,0.190000000,-0.280000000\n2012-09-01,0.440000000,0.040000000,0.340000000,-0.130000000\n2012-10-01,0.230000000,-0.390000000,0.350000000,-0.190000000\n2012-11-01,0.330000000,-0.460000000,0.440000000,-0.160000000\n2012-12-01,-0.130000000,-0.610000000,0.260000000,-0.090000000\n2013-01-01,-0.420000000,-0.700000000,0.190000000,0.050000000\n2013-02-01,-0.400000000,-0.320000000,0.220000000,-0.020000000\n2013-03-01,-0.140000000,-0.380000000,0.550000000,-0.090000000\n2013-04-01,-0.080000000,-1.320000000,0.370000000,0.040000000\n2013-05-01,-0.280000000,-1.440000000,0.440000000,0.130000000\n2013-06-01,-0.330000000,-1.440000000,0.070000000,-0.030000000\n2013-07-01,-0.280000000,-1.360000000,0.060000000,-0.030000000\n2013-08-01,-0.290000000,-0.760000000,0.220000000,0.050000000\n2013-09-01,-0.090000000,-0.770000000,0.230000000,-0.120000000\n2013-10-01,-0.240000000,-0.160000000,0.230000000,-0.180000000\n2013-11-01,-0.020000000,-0.500000000,0.020000000,-0.260000000\n2013-12-01,-0.090000000,-0.250000000,-0.020000000,-0.160000000\n2014-01-01,-0.420000000,0.370000000,-0.120000000,-0.230000000\n2014-02-01,-0.450000000,-0.670000000,-0.250000000,-0.050000000\n2014-03-01,-0.070000000,-0.510000000,-0.500000000,-0.130000000\n2014-04-01,0.280000000,-0.250000000,-0.360000000,-0.080000000\n2014-05-01,0.450000000,0.740000000,-0.270000000,0.140000000\n2014-06-01,0.480000000,1.170000000,-0.290000000,0.150000000\n2014-07-01,0.130000000,1.040000000,-0.130000000,-0.090000000\n2014-08-01,0.140000000,0.670000000,-0.100000000,0.050000000\n2014-09-01,0.370000000,0.150000000,0.130000000,0.110000000\n2014-10-01,0.480000000,0.340000000,0.130000000,-0.060000000\n2014-11-01,0.890000000,0.570000000,0.040000000,-0.500000000\n2014-12-01,0.770000000,0.140000000,0.050000000,-0.380000000\n2015-01-01,0.590000000,-0.160000000,-0.080000000,0.100000000\n2015-02-01,0.570000000,-0.620000000,-0.060000000,0.150000000\n2015-03-01,0.480000000,-0.010000000,-0.270000000,-0.110000000\n2015-04-01,0.900000000,0.640000000,-0.400000000,0.050000000\n2015-05-01,1.040000000,1.590000000,-0.130000000,0.110000000\n2015-06-01,1.280000000,2.190000000,-0.310000000,0.070000000\n2015-07-01,1.560000000,2.320000000,-0.120000000,-0.190000000\n2015-08-01,1.870000000,2.080000000,0.090000000,-0.310000000\n2015-09-01,2.010000000,2.480000000,0.350000000,-0.190000000\n2015-10-01,2.210000000,2.380000000,0.510000000,0.060000000\n2015-11-01,2.570000000,2.210000000,0.180000000,0.190000000\n2015-12-01,2.560000000,2.090000000,0.130000000,0.440000000\n2016-01-01,2.560000000,1.820000000,0.250000000,0.470000000\n2016-02-01,2.110000000,1.090000000,0.280000000,0.320000000\n2016-03-01,1.600000000,1.100000000,0.240000000,0.250000000\n2016-04-01,1.050000000,0.290000000,0.170000000,0.080000000\n2016-05-01,0.450000000,0.450000000,0.180000000,0.010000000\n2016-06-01,0.060000000,0.520000000,0.150000000,0.370000000\n2016-07-01,-0.250000000,-0.060000000,0.100000000,0.370000000\n2016-08-01,-0.480000000,0.190000000,0.150000000,0.330000000\n2016-09-01,-0.460000000,0.250000000,0.010000000,0.110000000\n2016-10-01,-0.750000000,-0.030000000,0.120000000,0.240000000\n2016-11-01,-0.630000000,-0.080000000,0.150000000,0.100000000\n2016-12-01,-0.510000000,0.020000000,0.230000000,0.410000000\n2017-01-01,-0.340000000,0.330000000,0.240000000,0.250000000\n2017-02-01,-0.010000000,0.930000000,0.060000000,-0.180000000\n2017-03-01,-0.090000000,1.560000000,-0.030000000,-0.060000000\n2017-04-01,0.220000000,0.490000000,0.310000000,0.090000000\n2017-05-01,0.300000000,0.250000000,0.380000000,0.120000000\n2017-06-01,0.220000000,0.200000000,0.350000000,0.260000000\n2017-07-01,0.220000000,-0.460000000,0.320000000,0.100000000\n2017-08-01,-0.180000000,-0.760000000,0.330000000,0.390000000\n2017-09-01,-0.560000000,-1.280000000,0.150000000,0.120000000\n2017-10-01,-0.520000000,-1.320000000,0.030000000,0.190000000\n2017-11-01,-0.840000000,-1.440000000,0.220000000,0.020000000\n2017-12-01,-0.850000000,-1.540000000,0.250000000,-0.090000000\n2018-01-01,-0.860000000,-1.250000000,0.030000000,0.290000000\n2018-02-01,-0.730000000,-0.430000000,-0.270000000,-0.270000000\n2018-03-01,-0.730000000,-1.300000000,-0.130000000,-0.180000000\n2018-04-01,-0.360000000,-0.810000000,-0.330000000,-0.030000000\n2018-05-01,-0.120000000,-0.460000000,-0.600000000,-0.070000000\n2018-06-01,0.120000000,-0.470000000,-0.720000000,0.150000000\n2018-07-01,0.270000000,-0.080000000,-0.480000000,0.120000000\n2018-08-01,0.050000000,-0.540000000,-0.320000000,0.210000000\n2018-09-01,0.300000000,0.120000000,-0.040000000,0.210000000\n2018-10-01,0.840000000,0.300000000,-0.130000000,0.310000000\n2018-11-01,1.000000000,0.690000000,-0.320000000,0.210000000\n2018-12-01,0.970000000,0.790000000,-0.330000000,0.400000000\n2019-01-01,0.510000000,0.670000000,-0.170000000,0.410000000\n2019-02-01,0.620000000,0.660000000,-0.040000000,0.410000000\n2019-03-01,0.810000000,-0.180000000,-0.150000000,0.670000000\n2019-04-01,0.670000000,0.360000000,-0.170000000,0.340000000\n2019-05-01,0.610000000,0.240000000,-0.080000000,0.310000000\n2019-06-01,0.660000000,-0.110000000,0.060000000,0.270000000\n2019-07-01,0.410000000,-0.360000000,0.090000000,0.240000000\n2019-08-01,0.190000000,-0.290000000,0.090000000,0.220000000\n2019-09-01,0.110000000,-0.680000000,0.200000000,0.330000000\n2019-10-01,0.550000000,-0.710000000,0.010000000,0.250000000\n2019-11-01,0.740000000,0.020000000,-0.050000000,0.870000000\n2019-12-01,0.510000000,-0.190000000,-0.040000000,0.800000000\n2020-01-01,0.640000000,-0.120000000,0.010000000,0.950000000\n2020-02-01,0.480000000,-0.030000000,0.320000000,0.570000000\n2020-03-01,0.360000000,-0.440000000,0.580000000,0.310000000\n2020-04-01,0.490000000,0.030000000,0.460000000,0.410000000\n2020-05-01,-0.120000000,-0.240000000,0.320000000,0.530000000\n2020-06-01,-0.210000000,-0.430000000,0.470000000,0.290000000\n2020-07-01,-0.040000000,-1.090000000,0.380000000,0.290000000\n2020-08-01,-0.420000000,-0.430000000,0.250000000,0.300000000\n2020-09-01,-0.660000000,-0.850000000,0.150000000,0.090000000\n', 'research/lagged_sources/official_hashes.json': '[\n  {\n    "nome": "treino_tp.nc",\n    "sha256": "012bebbe0e38aefdf4f9e2432c3ca22f71d94ea1a1b5421da9d9e5048632cab0",\n    "bytes": 186685299\n  },\n  {\n    "nome": "treino_tp_alvo.nc",\n    "sha256": "915f67f2627e37399830d1dd92b209ce1160082b0645662675eab17df957abe4",\n    "bytes": 186509303\n  },\n  {\n    "nome": "teste_features.nc",\n    "sha256": "70bcb5c9bb9c011c8901660834b42dc57ff7e54a4f34161d489ec5e9f05c1f23",\n    "bytes": 39057097\n  },\n  {\n    "nome": "sample_submission.csv",\n    "sha256": "34fe449ddb309b0579a87786f76049863833d82c26f183d18e943dc76f1e4c8c",\n    "bytes": 48145117\n  },\n  {\n    "nome": "treino_cloud_cover.nc",\n    "sha256": "607ee1210615a0ca8a5c954737617e668a5f016eb881554cb1468fda74b4fde1",\n    "bytes": 190610590\n  },\n  {\n    "nome": "treino_geopotential_850.nc",\n    "sha256": "ca3baca0177e2e90b42901803f348a4b4affe88bee202b76af3f293fa7035c17",\n    "bytes": 159707312\n  },\n  {\n    "nome": "treino_rel_hum_850.nc",\n    "sha256": "90f62b44b1b461259e7539249914e26cf09993f3bcd572ba4871931e45264b19",\n    "bytes": 180398147\n  },\n  {\n    "nome": "treino_shum_850.nc",\n    "sha256": "35cf8071b68d1e924ad87e0f8bd801d0a66d1d78ae75fc3cb623bde85c3f47bb",\n    "bytes": 175651693\n  },\n  {\n    "nome": "treino_surface_pressure.nc",\n    "sha256": "f00a5e44a03ea923ac0f693b975ef71cd347f8ad3d07692c602e3ac9a25c10fb",\n    "bytes": 160228025\n  },\n  {\n    "nome": "treino_t2.nc",\n    "sha256": "7d8c53b438289cddf87a11414ad10c8167f52ff33c6eff459a3288975079ef23",\n    "bytes": 175952066\n  },\n  {\n    "nome": "treino_temperature_850.nc",\n    "sha256": "1db7b4fdeba87b6a25f7fb6f1e40abec50964234dd9cc918cd34f902b8e41bac",\n    "bytes": 172474226\n  },\n  {\n    "nome": "treino_u_850.nc",\n    "sha256": "73c898f27bea25cf2ab0914bbe5c29acc9794a93f8b6b3affa39854bebccf761",\n    "bytes": 188451602\n  },\n  {\n    "nome": "treino_v_850.nc",\n    "sha256": "7bab583972e5718bb0e0fe0fe6a056bd56c968af1f1497205182f7621e26c241",\n    "bytes": 197354831\n  }\n]', 'research/spatial_unet/__init__.py': '"""Monthly spatial rainfall experiment."""\n', 'research/spatial_unet/evidence/seven_blocks/A/baseline_check.json': '{\n  "reproduced": true,\n  "rmse_archived": 1.7415222874502263,\n  "rmse_current": 1.7415222874502265,\n  "tolerance": 1e-06\n}\n', 'research/spatial_unet/evidence/seven_blocks/A/complete.json': '{\n  "files": {\n    "baseline_check.json": "e14a23353cd8d4a73ed10f51337c8ca9aac882a12e654dc4979efc3108d062b0",\n    "hybrid/arvores_multissistema.txt": "af6545e3858a0079c267974b3505e47476c4230364bad3ea009d45d9ef97106a",\n    "hybrid/arvores_seas5.txt": "dabc6a10efd7d2db4f49d487fd012000b4d760181dee3c1f85e7ddb196325b6f",\n    "hybrid/clima_atmosfera_indices.npz": "41f4ab7f405b7f6e87440f9e8069cd8db3ca88357980c744159d3cd604f696a0",\n    "hybrid/clima_cfsv2.nc": "ceb237b3d8ce7c14c6b279014d53fe37b6da919d2bb96c9e5e8bd2223ffcfcbb",\n    "hybrid/clima_chuva.nc": "911f9c242d15d6f42d146e1758b8867d3b1bf4c9576a087581593db43895c854",\n    "hybrid/clima_seas5.nc": "8e7748210852ae4ff02872e13c57f914d765a9b231b300f832b577fd8929973a",\n    "hybrid/fit.json": "8f55c5ca472389590e4ae08f621524f5fbe616278e1eeb3b3ec53f4e9ad10d62",\n    "hybrid/ridge.npz": "715581aceda0eb2dccad1cd9f83c7783f36753911906302a345e9fa8eeed442b",\n    "inner/context.json": "676a517a25f215bc59fc5db7eb00256ab035e25258512c59d9e0a274ec9c2722",\n    "inner/network.pt": "cba2eb23bb56539d30ff34ccb60264473fa39b0a82add44bedcf97027c37428d",\n    "inner/rainfall_climatology.nc": "553b75b2b7de4e2622a61274533209307e4feeb1bb1e92365ddb88c0a1424a27",\n    "inner/references.npz": "09c5bdc5920c2ca269114b1cf7db5080bc81bf986fa4a2ae15aef7d8344db61b",\n    "inner/scaling.npz": "c0518ef5e38d117c9af500dd4571ba35691949115548207f872bca2a41c2cad0",\n    "inner/training.json": "72363f855838e9d4185559d95a00ccc80aa4fe055ebc71763b8c3da5cca09ab2",\n    "inner/training_history.json": "3bf2693365a121adaba2828a1e941cd30d298886bf95ec27f96db262e720c950",\n    "monthly_metrics.csv": "e94feeaa24c019b6843a190806ca108dc5ff32556c9a82e5abc996e0e7a279b2",\n    "predictions.nc": "9e3a1643a5b5ee4d6b5306caacf141ea0038817384c78bc5d1fca4ab1bc51924",\n    "source_calendar.csv": "d128b8d7afa6626f8bef46d11c0db715d22a6d0b791554fe858b052c250ec991",\n    "split.json": "7b13475e8dd96144a7c4f91724bb81cc0c68d9ad10be33f475089fe284b4cb97",\n    "unet/context.json": "1ef3da22657165feee22bd4f91cb4cb053d77f7041db183bd897570bc051167c",\n    "unet/network.pt": "699c5d5d93b68b2dc4950c8f3903b2d1907fc0fc34c172c3d62639e323c21472",\n    "unet/rainfall_climatology.nc": "6e144638a6f81d3b1701855239602e8f4de95f943b4e2471ac4d3d7f251b6c4c",\n    "unet/references.npz": "49d96fa44c94332a2db7b9af3aff8115161269e8e135896b89033bdfec119d8f",\n    "unet/scaling.npz": "9e26b76cb4bec1262a6b1fe3edf6fd90ee2f6bb31e73ab031c334eaa379518ca",\n    "unet/training.json": "90d834c5f3c9894531f5cd9829c2059e0e288b63f89928f0495ff19b397c1bbc",\n    "unet/training_history.json": "7ebb09bab80a6e31d2e63c1073cc9c6438575321d050f86ad9f39a297e57fec0"\n  },\n  "signature": "02d1ffc0a7205f2bf372673d337cf5daa95f125d43a95836b953c10ae101244c"\n}\n', 'research/spatial_unet/evidence/seven_blocks/A/inner/training.json': '{\n  "device": "cuda",\n  "inner_selection": true,\n  "parameter_count": 122945,\n  "selected_epochs": 7,\n  "train_end": "2012-06-01",\n  "train_months": 333,\n  "train_start": "1984-10-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/A/inner/training_history.json': '[\n  {\n    "epoch": 1,\n    "inner_rmse": 1.7712802125536742,\n    "seconds": 19.75755717600032,\n    "training_mse": 2.9899380672443376\n  },\n  {\n    "epoch": 2,\n    "inner_rmse": 1.7630924563095887,\n    "seconds": 20.007791307999923,\n    "training_mse": 2.915609723454839\n  },\n  {\n    "epoch": 3,\n    "inner_rmse": 1.7754288953341917,\n    "seconds": 19.413215710999793,\n    "training_mse": 2.8952149313849374\n  },\n  {\n    "epoch": 4,\n    "inner_rmse": 1.7571765697573876,\n    "seconds": 19.28588767200017,\n    "training_mse": 2.8812605969540708\n  },\n  {\n    "epoch": 5,\n    "inner_rmse": 1.7762455499499508,\n    "seconds": 19.487949802999992,\n    "training_mse": 2.8690792440294146\n  },\n  {\n    "epoch": 6,\n    "inner_rmse": 1.7528391751699306,\n    "seconds": 19.568974601000264,\n    "training_mse": 2.8556866710250444\n  },\n  {\n    "epoch": 7,\n    "inner_rmse": 1.7496555925870572,\n    "seconds": 19.52801611599989,\n    "training_mse": 2.835033578557653\n  },\n  {\n    "epoch": 8,\n    "inner_rmse": 1.753744376738498,\n    "seconds": 19.426557943999796,\n    "training_mse": 2.8239873357721277\n  },\n  {\n    "epoch": 9,\n    "inner_rmse": 1.759452976039331,\n    "seconds": 19.453914257999713,\n    "training_mse": 2.794945277251281\n  },\n  {\n    "epoch": 10,\n    "inner_rmse": 1.7548599046526339,\n    "seconds": 19.462336864000463,\n    "training_mse": 2.7887543722673938\n  },\n  {\n    "epoch": 11,\n    "inner_rmse": 1.7632706687661237,\n    "seconds": 19.492270013000052,\n    "training_mse": 2.7843282566414223\n  },\n  {\n    "epoch": 12,\n    "inner_rmse": 1.7558846795555714,\n    "seconds": 19.461337081999773,\n    "training_mse": 2.778459931279088\n  },\n  {\n    "epoch": 13,\n    "inner_rmse": 1.7630468431198976,\n    "seconds": 19.490186054000333,\n    "training_mse": 2.7664224020353667\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/A/monthly_metrics.csv': 'modelo,bloco,mes_alvo,ano,mes,regiao,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area\nunet,A,2015-01-01,2015,1,dominio_inteiro,78561,311546.4526023889,18436.235942290165,98088.75103783142,291485.9741968437,67475.78503925544\nunet,A,2015-01-01,2015,1,lat_menor_que_menos35,26100,26917.201875741328,2446.7586306780577,18345.609639763832,18206.72146025173,17451.68248953376\nunet,A,2015-01-01,2015,1,lat_menos35_a_menos15,20880,127689.4168564303,22726.032870612107,33458.57665587682,117705.38217239577,18808.59412325443\nunet,A,2015-01-01,2015,1,lat_maior_igual_menos15,31581,156939.8338702173,-6736.555559,46284.56474219076,155573.87056419614,31215.50842646724\nunet,A,2015-02-01,2015,2,dominio_inteiro,78561,203544.6223357332,-14983.39374711318,78860.07697574282,185295.26048726134,67475.78503925544\nunet,A,2015-02-01,2015,2,lat_menor_que_menos35,26100,39520.93509132315,3475.987204760313,23445.286394685507,29138.145324760284,17451.68248953376\nunet,A,2015-02-01,2015,2,lat_menos35_a_menos15,20880,64204.34245012082,-6248.943753156345,21628.591405874584,56770.4559741484,18808.59412325443\nunet,A,2015-02-01,2015,2,lat_maior_igual_menos15,31581,99819.34479428922,-12210.437198717147,33786.19917518273,99386.65918835266,31215.50842646724\nunet,A,2015-03-01,2015,3,dominio_inteiro,78561,259400.3806475407,-24324.825755835045,88877.14326277422,243845.99746701433,67475.78503925544\nunet,A,2015-03-01,2015,3,lat_menor_que_menos35,26100,34708.88729648968,5088.084681242704,21238.400357693434,24042.80488060962,17451.68248953376\nunet,A,2015-03-01,2015,3,lat_menos35_a_menos15,20880,38276.73761183958,393.17547213612124,19278.799804706592,34048.70455670922,18808.59412325443\nunet,A,2015-03-01,2015,3,lat_maior_igual_menos15,31581,186414.75573921142,-29806.08590921387,48359.94310037419,185754.48802969547,31215.50842646724\nunet,A,2015-04-01,2015,4,dominio_inteiro,78561,191727.94404797882,4262.284766949248,79805.29556294112,177875.03888522892,67475.78503925544\nunet,A,2015-04-01,2015,4,lat_menor_que_menos35,26100,23580.79492809798,4257.360865294933,18531.526672542095,16963.67065172708,17451.68248953376\nunet,A,2015-04-01,2015,4,lat_menos35_a_menos15,20880,63323.75152024657,2986.071832270827,23875.70102639636,56721.782386086474,18808.59412325443\nunet,A,2015-04-01,2015,4,lat_maior_igual_menos15,31581,104823.39759963426,-2981.147930616513,37398.06786400266,104189.58584741538,31215.50842646724\nunet,A,2015-05-01,2015,5,dominio_inteiro,78561,264709.1231924013,-10346.46994244866,85843.57149085216,247910.03541795578,67475.78503925544\nunet,A,2015-05-01,2015,5,lat_menor_que_menos35,26100,43876.43448794457,-5893.699926137924,20578.56926381588,30934.606336311204,17451.68248953376\nunet,A,2015-05-01,2015,5,lat_menos35_a_menos15,20880,26917.030834495145,-764.4662962546572,16122.075051167049,24159.523966520064,18808.59412325443\nunet,A,2015-05-01,2015,5,lat_maior_igual_menos15,31581,193915.65786996158,-3688.3037200560793,49142.927175869234,192815.9051151245,31215.50842646724\nunet,A,2015-06-01,2015,6,dominio_inteiro,78561,210996.43890778034,302.92936778999865,72363.0469926279,195420.92469087505,67475.78503925544\nunet,A,2015-06-01,2015,6,lat_menor_que_menos35,26100,38659.063846908575,-680.9289585053921,21252.976875334978,27236.362503714518,17451.68248953376\nunet,A,2015-06-01,2015,6,lat_menos35_a_menos15,20880,24291.344484291414,8473.626484077424,14573.940460961312,21016.477164354437,18808.59412325443\nunet,A,2015-06-01,2015,6,lat_maior_igual_menos15,31581,148046.03057658035,-7489.768157782033,36536.129656331614,147168.08502280607,31215.50842646724\nunet,A,2015-07-01,2015,7,dominio_inteiro,78561,277139.8500588777,-32760.7680657804,78541.72523906827,255867.69850298794,67475.78503925544\nunet,A,2015-07-01,2015,7,lat_menor_que_menos35,26100,45231.784402698024,-11862.146902352571,23042.26330050826,32838.59962871157,17451.68248953376\nunet,A,2015-07-01,2015,7,lat_menos35_a_menos15,20880,61258.91376877022,-6619.74538975209,19337.288309164345,53539.34137384516,18808.59412325443\nunet,A,2015-07-01,2015,7,lat_maior_igual_menos15,31581,170649.1518874095,-14278.87577367574,36162.173629395664,169489.7575004312,31215.50842646724\nunet,A,2015-08-01,2015,8,dominio_inteiro,78561,208062.78713894374,-17874.13830997236,77001.08364512213,189071.5607506358,67475.78503925544\nunet,A,2015-08-01,2015,8,lat_menor_que_menos35,26100,44159.306466778035,-19359.158669143915,22950.180367141962,32359.374328911254,17451.68248953376\nunet,A,2015-08-01,2015,8,lat_menos35_a_menos15,20880,45930.32350308484,2467.6400042977184,20589.49575402774,39726.66256865345,18808.59412325443\nunet,A,2015-08-01,2015,8,lat_maior_igual_menos15,31581,117973.15716908085,-982.619645126164,33461.407523952425,116985.52385307115,31215.50842646724\nunet,A,2015-09-01,2015,9,dominio_inteiro,78561,201843.77465325617,13719.7994524315,82408.61099518836,188554.9011069441,67475.78503925544\nunet,A,2015-09-01,2015,9,lat_menor_que_menos35,26100,26168.448956101933,2055.446819782257,18722.621868133545,18851.254795760702,17451.68248953376\nunet,A,2015-09-01,2015,9,lat_menos35_a_menos15,20880,39561.80604164421,-4103.214113853872,20015.601162947714,34887.04567691007,18808.59412325443\nunet,A,2015-09-01,2015,9,lat_maior_igual_menos15,31581,136113.51965551003,15767.566746503115,43670.387964107096,134816.60063427332,31215.50842646724\nunet,A,2015-10-01,2015,10,dominio_inteiro,78561,180889.01022359065,16448.09778118506,76687.29520744458,167551.85474291316,67475.78503925544\nunet,A,2015-10-01,2015,10,lat_menor_que_menos35,26100,25249.78908786086,637.2061715722084,18891.838076770306,17827.915812593765,17451.68248953376\nunet,A,2015-10-01,2015,10,lat_menos35_a_menos15,20880,42206.116233285444,6864.956786952913,20406.171262152493,37297.86075941096,18808.59412325443\nunet,A,2015-10-01,2015,10,lat_maior_igual_menos15,31581,113433.10490244435,8945.93482265994,37389.28586852178,112426.07817090841,31215.50842646724\nunet,A,2015-11-01,2015,11,dominio_inteiro,78561,174949.22851007807,7500.699267160147,70525.09234129265,165043.65641646556,67475.78503925544\nunet,A,2015-11-01,2015,11,lat_menor_que_menos35,26100,10960.882268117268,5229.076561808586,12787.39594578743,7826.566307658894,17451.68248953376\nunet,A,2015-11-01,2015,11,lat_menos35_a_menos15,20880,65721.68643546166,-4973.893010593951,23295.212699756026,59785.84691670457,18808.59412325443\nunet,A,2015-11-01,2015,11,lat_maior_igual_menos15,31581,98266.65980649914,7245.515715945512,34442.48369574919,97431.2431921021,31215.50842646724\nunet,A,2015-12-01,2015,12,dominio_inteiro,78561,338819.3974446844,33081.186417832505,96810.07403515419,320496.5652708919,67475.78503925544\nunet,A,2015-12-01,2015,12,lat_menor_que_menos35,26100,24205.307183018864,-386.5185890644789,16417.92509098351,17379.934995985714,17451.68248953376\nunet,A,2015-12-01,2015,12,lat_menos35_a_menos15,20880,110118.0039173972,12607.63467336027,32186.96699885046,100224.95646325691,18808.59412325443\nunet,A,2015-12-01,2015,12,lat_maior_igual_menos15,31581,204496.08634426832,20860.070333536714,48205.18194532022,202891.6738116492,31215.50842646724\nunet,A,2016-01-01,2016,1,dominio_inteiro,78561,503089.9891328763,5848.562802436994,122881.17838010727,472869.4339776356,67475.78503925544\nunet,A,2016-01-01,2016,1,lat_menor_que_menos35,26100,41950.921120983105,4411.307314641774,21365.98494976014,28668.699516164037,17451.68248953376\nunet,A,2016-01-01,2016,1,lat_menos35_a_menos15,20880,152572.29937609797,-11860.074247046141,36334.166585933184,138968.95365205154,18808.59412325443\nunet,A,2016-01-01,2016,1,lat_maior_igual_menos15,31581,308566.7686357952,13297.329734841362,65181.02684441395,305231.78080942,31215.50842646724\nunet,A,2016-02-01,2016,2,dominio_inteiro,78561,193151.3029296618,9557.743664006703,79250.82226833235,180044.20145121112,67475.78503925544\nunet,A,2016-02-01,2016,2,lat_menor_que_menos35,26100,24600.489830941813,-7095.722238257527,17834.231153652072,16940.418245762852,17451.68248953376\nunet,A,2016-02-01,2016,2,lat_menos35_a_menos15,20880,48938.70211704222,5536.054408763535,21163.18747073505,44459.376297439696,18808.59412325443\nunet,A,2016-02-01,2016,2,lat_maior_igual_menos15,31581,119612.11098167775,11117.411493500695,40253.40364394523,118644.40690800859,31215.50842646724\nunet,A,2016-03-01,2016,3,dominio_inteiro,78561,357941.52511887794,-3038.4935534936376,100942.0592742064,337801.31679326494,67475.78503925544\nunet,A,2016-03-01,2016,3,lat_menor_que_menos35,26100,37231.08441773731,6895.612743228674,23161.13417300582,27268.970894508773,17451.68248953376\nunet,A,2016-03-01,2016,3,lat_menos35_a_menos15,20880,77280.03915248944,4957.1542818821035,27049.735220537055,68174.1684250566,18808.59412325443\nunet,A,2016-03-01,2016,3,lat_maior_igual_menos15,31581,243430.40154865116,-14891.260578604415,50731.18988066353,242358.17747369953,31215.50842646724\nunet,A,2016-04-01,2016,4,dominio_inteiro,78561,301701.81001624325,-4165.197077458724,95743.70821539871,276758.91649914713,67475.78503925544\nunet,A,2016-04-01,2016,4,lat_menor_que_menos35,26100,45022.96601248482,-4703.071849256754,21941.85375317931,32569.548321046714,17451.68248953376\nunet,A,2016-04-01,2016,4,lat_menos35_a_menos15,20880,91332.52161203569,957.3791208369657,27759.06080831308,79646.72315851202,18808.59412325443\nunet,A,2016-04-01,2016,4,lat_maior_igual_menos15,31581,165346.32239172276,-419.5043490389362,46042.79365390632,164542.64501958838,31215.50842646724\nunet,A,2016-05-01,2016,5,dominio_inteiro,78561,290408.8144299522,32162.120465276763,96993.48947594129,251744.8608240601,67475.78503925544\nunet,A,2016-05-01,2016,5,lat_menor_que_menos35,26100,103534.25169520867,27255.645448863506,36079.90729743242,69409.10451249903,17451.68248953376\nunet,A,2016-05-01,2016,5,lat_menos35_a_menos15,20880,31383.671953487796,-1723.7319216169417,17722.114989254624,27685.029791973036,18808.59412325443\nunet,A,2016-05-01,2016,5,lat_maior_igual_menos15,31581,155490.89078125576,6630.206938030198,43191.46718925424,154650.72651958803,31215.50842646724\nunet,A,2016-06-01,2016,6,dominio_inteiro,78561,226837.1205608314,35262.97221466899,84578.83423833176,199189.11292391043,67475.78503925544\nunet,A,2016-06-01,2016,6,lat_menor_que_menos35,26100,74728.5895549205,22724.62756639719,30362.86092299223,54289.71754391682,17451.68248953376\nunet,A,2016-06-01,2016,6,lat_menos35_a_menos15,20880,56049.61046437671,1650.4155161604285,22842.046625796705,49457.96207476307,18808.59412325443\nunet,A,2016-06-01,2016,6,lat_maior_igual_menos15,31581,96058.9205415342,10887.92913211137,31373.92668954283,95441.43330523054,31215.50842646724\nunet,A,2016-07-01,2016,7,dominio_inteiro,78561,166714.92892097883,3771.4037655685097,70720.60744672082,152847.58938417377,67475.78503925544\nunet,A,2016-07-01,2016,7,lat_menor_que_menos35,26100,40066.30664207697,-467.5464161336422,23362.332249194384,29378.490455639534,17451.68248953376\nunet,A,2016-07-01,2016,7,lat_menos35_a_menos15,20880,16819.843139618977,1153.2365053389221,12611.32601406239,14532.991287400957,18808.59412325443\nunet,A,2016-07-01,2016,7,lat_maior_igual_menos15,31581,109828.77913928287,3085.7136763632298,34746.94918346405,108936.10764113328,31215.50842646724\nunet,A,2016-08-01,2016,8,dominio_inteiro,78561,162486.6608585893,10125.393725652248,67139.0593944462,150294.09309469245,67475.78503925544\nunet,A,2016-08-01,2016,8,lat_menor_que_menos35,26100,30712.775460958168,1794.312391102314,18921.714326918125,22483.754212795062,17451.68248953376\nunet,A,2016-08-01,2016,8,lat_menos35_a_menos15,20880,25436.359121971574,-5740.15016373992,15219.707887836732,22519.99050516823,18808.59412325443\nunet,A,2016-08-01,2016,8,lat_maior_igual_menos15,31581,106337.52627565959,14071.231498289853,32997.637179691344,105290.34837672915,31215.50842646724\nunet,A,2016-09-01,2016,9,dominio_inteiro,78561,169508.50986663986,13463.852064465638,74962.04150672024,154145.68311027478,67475.78503925544\nunet,A,2016-09-01,2016,9,lat_menor_que_menos35,26100,32178.850977132264,8439.457458794117,21943.89932566881,21785.438840583138,17451.68248953376\nunet,A,2016-09-01,2016,9,lat_menos35_a_menos15,20880,34738.33632846155,9464.086823337246,17229.992684372235,30820.67675727336,18808.59412325443\nunet,A,2016-09-01,2016,9,lat_maior_igual_menos15,31581,102591.32256104604,-4439.6922176657245,35788.149496679194,101539.56751241829,31215.50842646724\nunet,A,2016-10-01,2016,10,dominio_inteiro,78561,171793.25337368436,1891.880940596573,66977.28891174216,160911.1043405485,67475.78503925544\nunet,A,2016-10-01,2016,10,lat_menor_que_menos35,26100,23220.927974619888,190.08014169335365,16995.378389924765,16997.843762016244,17451.68248953376\nunet,A,2016-10-01,2016,10,lat_menos35_a_menos15,20880,29316.60872463792,1088.2701414255425,15545.193735633977,26169.423466666536,18808.59412325443\nunet,A,2016-10-01,2016,10,lat_maior_igual_menos15,31581,119255.71667442659,613.5306574776769,34436.71678618342,117743.83711186575,31215.50842646724\nunet,A,2016-11-01,2016,11,dominio_inteiro,78561,263591.0409575006,4769.723295981064,85353.36107619666,251535.080658512,67475.78503925544\nunet,A,2016-11-01,2016,11,lat_menor_que_menos35,26100,20828.040443583683,-7852.7263720184565,17935.956150993705,14331.392781983555,17451.68248953376\nunet,A,2016-11-01,2016,11,lat_menos35_a_menos15,20880,29102.109644514978,5979.434379735962,16367.54818486236,26411.599348563785,18808.59412325443\nunet,A,2016-11-01,2016,11,lat_maior_igual_menos15,31581,213660.89086940192,6643.015288263559,51049.85674034059,210792.08852796466,31215.50842646724\nunet,A,2016-12-01,2016,12,dominio_inteiro,78561,208377.4320635052,-17289.714249939658,82193.87006712519,196596.15033057742,67475.78503925544\nunet,A,2016-12-01,2016,12,lat_menor_que_menos35,26100,21984.690413363962,-3861.928366035223,15996.25605365634,15485.401574665255,17451.68248953376\nunet,A,2016-12-01,2016,12,lat_menos35_a_menos15,20880,42695.40102130122,8717.34884083923,20467.76890338771,38448.079983946285,18808.59412325443\nunet,A,2016-12-01,2016,12,lat_maior_igual_menos15,31581,143697.34062884003,-22145.134724743664,45729.845110081136,142662.66877196592,31215.50842646724\nhybrid,A,2015-01-01,2015,1,dominio_inteiro,78561,309159.82628768886,16868.150301014437,98616.89339834316,289242.78390116134,67475.78503925544\nhybrid,A,2015-01-01,2015,1,lat_menor_que_menos35,26100,28010.491572363204,42.25867288392243,18571.946973340426,18981.456566900284,17451.68248953376\nhybrid,A,2015-01-01,2015,1,lat_menos35_a_menos15,20880,120393.62519171633,21031.820372569287,32415.358778215086,110878.02555825045,18808.59412325443\nhybrid,A,2015-01-01,2015,1,lat_maior_igual_menos15,31581,160755.7095236093,-4205.928744438774,47629.587646787644,159383.30177601063,31215.50842646724\nhybrid,A,2015-02-01,2015,2,dominio_inteiro,78561,193284.67740125337,-13180.08281381888,76316.42551937522,174850.90038453756,67475.78503925544\nhybrid,A,2015-02-01,2015,2,lat_menor_que_menos35,26100,39477.34591532923,1851.6719711032806,23110.226692277654,29108.80749022255,17451.68248953376\nhybrid,A,2015-02-01,2015,2,lat_menos35_a_menos15,20880,66262.74843682555,-7940.432857599054,21755.267891041374,58600.03866266295,18808.59412325443\nhybrid,A,2015-02-01,2015,2,lat_maior_igual_menos15,31581,87544.5830490986,-7091.32192732311,31450.930936056197,87142.05423165203,31215.50842646724\nhybrid,A,2015-03-01,2015,3,dominio_inteiro,78561,237269.49039420977,-19126.70849678214,85843.42520812823,221832.35445834344,67475.78503925544\nhybrid,A,2015-03-01,2015,3,lat_menor_que_menos35,26100,34202.128047781014,4487.321127287162,20871.712418945215,23804.03764232526,17451.68248953376\nhybrid,A,2015-03-01,2015,3,lat_menos35_a_menos15,20880,39848.36844563254,-325.37706626452433,19692.463705910155,35408.53291524434,18808.59412325443\nhybrid,A,2015-03-01,2015,3,lat_maior_igual_menos15,31581,163218.99390079628,-23288.65255780478,45279.24908327285,162619.78390077382,31215.50842646724\nhybrid,A,2015-04-01,2015,4,dominio_inteiro,78561,195727.88643015551,5623.975769205887,81132.07504379866,181955.66152329778,67475.78503925544\nhybrid,A,2015-04-01,2015,4,lat_menor_que_menos35,26100,22978.04729260719,3153.981349287423,18318.455596539872,16654.25597642293,17451.68248953376\nhybrid,A,2015-04-01,2015,4,lat_menos35_a_menos15,20880,64376.27311275031,2849.347661787924,24032.95374994118,57613.00245919031,18808.59412325443\nhybrid,A,2015-04-01,2015,4,lat_maior_igual_menos15,31581,108373.56602479806,-379.35324186945945,38780.665697317614,107688.40308768452,31215.50842646724\nhybrid,A,2015-05-01,2015,5,dominio_inteiro,78561,266629.63863571564,-5891.360632881565,87646.44677907134,250738.48154764622,67475.78503925544\nhybrid,A,2015-05-01,2015,5,lat_menor_que_menos35,26100,41014.83978461791,-5260.661431431029,20368.77093729645,28952.38334446377,17451.68248953376\nhybrid,A,2015-05-01,2015,5,lat_menos35_a_menos15,20880,26087.55109867783,-405.50385037410615,16105.994395207588,23422.31123667417,18808.59412325443\nhybrid,A,2015-05-01,2015,5,lat_maior_igual_menos15,31581,199527.2477524199,-225.19535107642878,51171.681446567294,198363.78696650828,31215.50842646724\nhybrid,A,2015-06-01,2015,6,dominio_inteiro,78561,219442.62561789475,928.6799085736936,74305.07156407602,202142.5250346067,67475.78503925544\nhybrid,A,2015-06-01,2015,6,lat_menor_que_menos35,26100,43249.0904885946,-280.76738914506404,22748.110709714412,30474.91743131869,17451.68248953376\nhybrid,A,2015-06-01,2015,6,lat_menos35_a_menos15,20880,26497.957862959865,8803.966688827906,15126.83671100319,22845.683238781363,18808.59412325443\nhybrid,A,2015-06-01,2015,6,lat_maior_igual_menos15,31581,149695.57726634026,-7594.5193911091455,36430.124143358415,148821.92436450665,31215.50842646724\nhybrid,A,2015-07-01,2015,7,dominio_inteiro,78561,269570.3888915997,-32382.754456330134,78092.28690262348,248933.6986620463,67475.78503925544\nhybrid,A,2015-07-01,2015,7,lat_menor_que_menos35,26100,42795.70513795142,-10875.793198629162,22433.514049462705,31078.50463131333,17451.68248953376\nhybrid,A,2015-07-01,2015,7,lat_menos35_a_menos15,20880,62312.433840027865,-7062.644483536833,19628.488091009775,54494.88107256178,18808.59412325443\nhybrid,A,2015-07-01,2015,7,lat_maior_igual_menos15,31581,164462.2499136204,-14444.316774164143,36030.284762151015,163360.31295817118,31215.50842646724\nhybrid,A,2015-08-01,2015,8,dominio_inteiro,78561,199796.47197919124,-18979.978432213185,74778.07446245839,180616.44839370702,67475.78503925544\nhybrid,A,2015-08-01,2015,8,lat_menor_que_menos35,26100,45144.96030618925,-19448.316317913188,23318.33199377694,33038.60444981203,17451.68248953376\nhybrid,A,2015-08-01,2015,8,lat_menos35_a_menos15,20880,44854.117096048736,1123.208270589577,19923.715280111886,38695.58206173099,18808.59412325443\nhybrid,A,2015-08-01,2015,8,lat_maior_igual_menos15,31581,109797.39457695326,-654.8703848895725,31536.02718856955,108882.26188216404,31215.50842646724\nhybrid,A,2015-09-01,2015,9,dominio_inteiro,78561,188274.92971736434,9414.050415057016,78348.59441395363,174950.13013610768,67475.78503925544\nhybrid,A,2015-09-01,2015,9,lat_menor_que_menos35,26100,26160.963755741086,1647.7357287227064,18865.5831174482,18865.89215730488,17451.68248953376\nhybrid,A,2015-09-01,2015,9,lat_menos35_a_menos15,20880,40933.08528151391,-5870.781395540536,19800.802127994262,36089.95369661777,18808.59412325443\nhybrid,A,2015-09-01,2015,9,lat_maior_igual_menos15,31581,121180.88068010935,13637.096081874844,39682.20916851118,119994.28428218504,31215.50842646724\nhybrid,A,2015-10-01,2015,10,dominio_inteiro,78561,162361.16545108147,10253.978616586486,71735.31437441736,150245.48363571885,67475.78503925544\nhybrid,A,2015-10-01,2015,10,lat_menor_que_menos35,26100,22480.79498886319,-481.0323729769948,17761.597619583183,15996.1413228143,17451.68248953376\nhybrid,A,2015-10-01,2015,10,lat_menos35_a_menos15,20880,39139.15681587736,4973.106364975886,19326.08805737728,34444.238567331646,18808.59412325443\nhybrid,A,2015-10-01,2015,10,lat_maior_igual_menos15,31581,100741.21364634091,5761.904624587593,34647.62869745691,99805.10374557287,31215.50842646724\nhybrid,A,2015-11-01,2015,11,dominio_inteiro,78561,170616.71301593984,5475.880608013025,70554.40936315442,160249.25253170053,67475.78503925544\nhybrid,A,2015-11-01,2015,11,lat_menor_que_menos35,26100,12394.217708395336,4344.626411681377,13467.301952890228,8944.534517912973,17451.68248953376\nhybrid,A,2015-11-01,2015,11,lat_menos35_a_menos15,20880,65502.266531767666,-4070.50472219829,23558.631216038673,59381.53885078632,18808.59412325443\nhybrid,A,2015-11-01,2015,11,lat_maior_igual_menos15,31581,92720.22877577685,5201.758918529937,33528.47619422553,91923.17916300126,31215.50842646724\nhybrid,A,2015-12-01,2015,12,dominio_inteiro,78561,307410.1447660888,26103.39216668271,90636.3639425029,290441.67515202996,67475.78503925544\nhybrid,A,2015-12-01,2015,12,lat_menor_que_menos35,26100,21762.089821775655,-2336.303851667325,15520.120632706377,15833.350599975834,17451.68248953376\nhybrid,A,2015-12-01,2015,12,lat_menos35_a_menos15,20880,104617.45442370136,12821.216262489528,31113.570888635502,94973.26361022111,18808.59412325443\nhybrid,A,2015-12-01,2015,12,lat_maior_igual_menos15,31581,181030.6005206118,15618.479755860502,44002.672421161005,179635.06094183298,31215.50842646724\nhybrid,A,2016-01-01,2016,1,dominio_inteiro,78561,506354.38642885623,-6922.651462731766,118891.51665335768,475032.7725174133,67475.78503925544\nhybrid,A,2016-01-01,2016,1,lat_menor_que_menos35,26100,36626.40591955088,1673.4618704312425,19765.906738769983,25221.643967841104,17451.68248953376\nhybrid,A,2016-01-01,2016,1,lat_menos35_a_menos15,20880,172588.63404968393,-9153.205956817446,38709.99802011789,156251.758774259,18808.59412325443\nhybrid,A,2016-01-01,2016,1,lat_maior_igual_menos15,31581,297139.3464596215,557.0926236544371,60415.61189446982,293559.36977531324,31215.50842646724\nhybrid,A,2016-02-01,2016,2,dominio_inteiro,78561,192191.39350493977,-180.278573948475,78506.79870906712,177632.73181094558,67475.78503925544\nhybrid,A,2016-02-01,2016,2,lat_menor_que_menos35,26100,28537.006137652705,-8153.471171829679,19222.656081050154,19475.915491868993,17451.68248953376\nhybrid,A,2016-02-01,2016,2,lat_menos35_a_menos15,20880,48006.84584613615,5033.4818528548185,20768.453998711717,43340.15781738171,18808.59412325443\nhybrid,A,2016-02-01,2016,2,lat_maior_igual_menos15,31581,115647.54152115091,2939.7107450263843,38515.68862930526,114816.65850169488,31215.50842646724\nhybrid,A,2016-03-01,2016,3,dominio_inteiro,78561,370302.672762742,-8497.68924093571,103829.78773749681,347071.480603305,67475.78503925544\nhybrid,A,2016-03-01,2016,3,lat_menor_que_menos35,26100,46065.46332877333,4204.848398937497,26650.19318277503,32909.21889051563,17451.68248953376\nhybrid,A,2016-03-01,2016,3,lat_menos35_a_menos15,20880,74435.287828396,5801.861437349427,26516.90788772231,65369.9140329977,18808.59412325443\nhybrid,A,2016-03-01,2016,3,lat_maior_igual_menos15,31581,249801.92160557263,-18504.399077222635,50662.68666699947,248792.3476797916,31215.50842646724\nhybrid,A,2016-04-01,2016,4,dominio_inteiro,78561,297934.5271496615,-12320.852647578817,93677.72039932857,275951.472133559,67475.78503925544\nhybrid,A,2016-04-01,2016,4,lat_menor_que_menos35,26100,38285.203650223964,-7670.7996932468195,20899.519591391556,27656.588391737103,17451.68248953376\nhybrid,A,2016-04-01,2016,4,lat_menos35_a_menos15,20880,84299.2930219866,2173.9883207389307,26797.80537782493,73667.82208496472,18808.59412325443\nhybrid,A,2016-04-01,2016,4,lat_maior_igual_menos15,31581,175350.03047745096,-6824.041275070927,45980.39543011208,174627.0616568572,31215.50842646724\nhybrid,A,2016-05-01,2016,5,dominio_inteiro,78561,277557.7095503318,21025.50870254243,91711.18917435949,241761.53493671608,67475.78503925544\nhybrid,A,2016-05-01,2016,5,lat_menor_que_menos35,26100,95741.55255974468,23981.888200315378,32572.91959177353,64574.25116221368,17451.68248953376\nhybrid,A,2016-05-01,2016,5,lat_menos35_a_menos15,20880,31827.296378199295,-890.0993078355232,17229.57983852608,28031.0941968212,18808.59412325443\nhybrid,A,2016-05-01,2016,5,lat_maior_igual_menos15,31581,149988.86061238777,-2066.2801899374203,41908.68974405988,149156.1895776812,31215.50842646724\nhybrid,A,2016-06-01,2016,6,dominio_inteiro,78561,221916.95281921295,27812.097269908372,81642.18880382512,194731.04039052676,67475.78503925544\nhybrid,A,2016-06-01,2016,6,lat_menor_que_menos35,26100,75110.60750408037,21112.596230558785,29003.87386662387,54947.40442935779,17451.68248953376\nhybrid,A,2016-06-01,2016,6,lat_menos35_a_menos15,20880,55174.56724682175,388.9949019000365,22433.513108104617,48723.39837890887,18808.59412325443\nhybrid,A,2016-06-01,2016,6,lat_maior_igual_menos15,31581,91631.77806831081,6310.506137449551,30204.801829096636,91060.23758226007,31215.50842646724\nhybrid,A,2016-07-01,2016,7,dominio_inteiro,78561,160239.30895054547,2930.3950553266454,67634.87571210676,146939.9056893179,67475.78503925544\nhybrid,A,2016-07-01,2016,7,lat_menor_que_menos35,26100,37898.93517908853,1558.730926130167,22788.3614984113,27577.468067773727,17451.68248953376\nhybrid,A,2016-07-01,2016,7,lat_menos35_a_menos15,20880,15288.288648864469,476.69548370044924,11619.02432265328,13169.82184144026,18808.59412325443\nhybrid,A,2016-07-01,2016,7,lat_maior_igual_menos15,31581,107052.08512259249,894.968645496029,33227.489891042176,106192.6157801039,31215.50842646724\nhybrid,A,2016-08-01,2016,8,dominio_inteiro,78561,162931.9887021141,11012.442980788503,66295.52502514434,150693.12099956334,67475.78503925544\nhybrid,A,2016-08-01,2016,8,lat_menor_que_menos35,26100,30992.898102710915,2373.9755226386965,18945.75335748362,22657.608459969557,17451.68248953376\nhybrid,A,2016-08-01,2016,8,lat_menos35_a_menos15,20880,24691.418345681916,-5538.7561174274215,14847.157787107059,21869.10312642344,18808.59412325443\nhybrid,A,2016-08-01,2016,8,lat_maior_igual_menos15,31581,107247.67225372123,14177.223575577227,32502.61388055366,106166.40941317033,31215.50842646724\nhybrid,A,2016-09-01,2016,9,dominio_inteiro,78561,172946.76532320946,12922.431056285734,75253.1699761308,156690.9662955851,67475.78503925544\nhybrid,A,2016-09-01,2016,9,lat_menor_que_menos35,26100,34968.770261812984,9047.748419041602,22672.185651593656,23777.888057500913,17451.68248953376\nhybrid,A,2016-09-01,2016,9,lat_menos35_a_menos15,20880,34897.603211455644,9194.141323989641,17354.089235155123,30904.929472207787,18808.59412325443\nhybrid,A,2016-09-01,2016,9,lat_maior_igual_menos15,31581,103080.39184994085,-5319.45868674551,35226.895089382015,102008.14876587642,31215.50842646724\nhybrid,A,2016-10-01,2016,10,dominio_inteiro,78561,167845.76186907847,439.6262262192647,65520.59756821633,156677.95544669952,67475.78503925544\nhybrid,A,2016-10-01,2016,10,lat_menor_que_menos35,26100,24580.561751385332,1709.727716691284,17504.8910954511,17869.582305702803,17451.68248953376\nhybrid,A,2016-10-01,2016,10,lat_menos35_a_menos15,20880,28055.155769717538,838.2374417035261,15193.013689565163,25049.2249228639,18808.59412325443\nhybrid,A,2016-10-01,2016,10,lat_maior_igual_menos15,31581,115210.0443479756,-2108.3389321755444,32822.69278320007,113759.14821813279,31215.50842646724\nhybrid,A,2016-11-01,2016,11,dominio_inteiro,78561,261035.47770885075,2434.510984859029,82649.70187656253,249652.9548943964,67475.78503925544\nhybrid,A,2016-11-01,2016,11,lat_menor_que_menos35,26100,19472.1119186688,-5644.673502237397,17098.29268363177,13546.467143256068,17451.68248953376\nhybrid,A,2016-11-01,2016,11,lat_menos35_a_menos15,20880,28319.02510072738,4499.598383654288,16111.822508851445,25704.60512498586,18808.59412325443\nhybrid,A,2016-11-01,2016,11,lat_maior_igual_menos15,31581,213244.34068945455,3579.5861034421396,49439.58668407932,210401.88262615446,31215.50842646724\nhybrid,A,2016-12-01,2016,12,dominio_inteiro,78561,207622.6316233841,-12555.452142850005,82809.20991186511,195998.31936435122,67475.78503925544\nhybrid,A,2016-12-01,2016,12,lat_menor_que_menos35,26100,21038.9202097412,-3257.7943017304083,15691.32374809693,14849.1355071277,17451.68248953376\nhybrid,A,2016-12-01,2016,12,lat_menos35_a_menos15,20880,44985.704230318515,9937.85921553329,21122.810348329567,40585.27981631437,18808.59412325443\nhybrid,A,2016-12-01,2016,12,lat_maior_igual_menos15,31581,141598.0071833244,-19235.517056652883,45995.0758154386,140563.90404090914,31215.50842646724\nclimatology,A,2015-01-01,2015,1,dominio_inteiro,78561,312149.4281093196,28208.903654305264,99707.32546293968,292204.3065607041,67475.78503925544\nclimatology,A,2015-01-01,2015,1,lat_menor_que_menos35,26100,30530.17082324761,1067.3163519725204,18755.42299681157,20678.699242484166,17451.68248953376\nclimatology,A,2015-01-01,2015,1,lat_menos35_a_menos15,20880,108546.55968136714,19439.344901744276,31281.37656782614,99768.65170883675,18808.59412325443\nclimatology,A,2015-01-01,2015,1,lat_maior_igual_menos15,31581,173072.69760470485,7702.242400588468,49670.525898301974,171756.95560938315,31215.50842646724\nclimatology,A,2015-02-01,2015,2,dominio_inteiro,78561,194744.10696707646,-138.33887726801913,78821.23629791255,175534.5321355579,67475.78503925544\nclimatology,A,2015-02-01,2015,2,lat_menor_que_menos35,26100,42061.80659655563,2676.0101874172688,23938.49206480384,30850.40059255021,17451.68248953376\nclimatology,A,2015-02-01,2015,2,lat_menos35_a_menos15,20880,65696.67302764393,-8182.424511341262,21824.153298234218,58070.33259849638,18808.59412325443\nclimatology,A,2015-02-01,2015,2,lat_maior_igual_menos15,31581,86985.6273428769,5368.075446655974,33058.59093487449,86613.79894451132,31215.50842646724\nclimatology,A,2015-03-01,2015,3,dominio_inteiro,78561,213267.69211054698,-8999.520260943333,81857.57925353595,197969.34219763806,67475.78503925544\nclimatology,A,2015-03-01,2015,3,lat_menor_que_menos35,26100,34760.51789106137,3646.9766337275505,20910.523748278618,24111.978174622625,17451.68248953376\nclimatology,A,2015-03-01,2015,3,lat_menos35_a_menos15,20880,37911.76905929552,-221.03775847540237,19254.205295181135,33791.98226242978,18808.59412325443\nclimatology,A,2015-03-01,2015,3,lat_maior_igual_menos15,31581,140595.40516019013,-12425.45913619548,41692.8502100762,140065.38176058567,31215.50842646724\nclimatology,A,2015-04-01,2015,4,dominio_inteiro,78561,216251.20772494155,12830.605457594618,86314.20750094578,202992.77858347556,67475.78503925544\nclimatology,A,2015-04-01,2015,4,lat_menor_que_menos35,26100,22275.312826726928,2588.0573678314686,17929.87242653966,16187.583884362288,17451.68248953376\nclimatology,A,2015-04-01,2015,4,lat_menos35_a_menos15,20880,62028.632950069696,644.298425488174,23441.70678307861,55699.072641990555,18808.59412325443\nclimatology,A,2015-04-01,2015,4,lat_maior_igual_menos15,31581,131947.26194814488,9598.249664274976,44942.628291327506,131106.12205712273,31215.50842646724\nclimatology,A,2015-05-01,2015,5,dominio_inteiro,78561,334749.0189158285,-6203.289924692363,98484.1557868747,318962.7483269717,67475.78503925544\nclimatology,A,2015-05-01,2015,5,lat_menor_que_menos35,26100,38784.74790449707,-5078.5259937644005,19877.353371679783,27477.698625858113,17451.68248953376\nclimatology,A,2015-05-01,2015,5,lat_menos35_a_menos15,20880,29454.020494319702,-4859.345497308299,16882.449850973673,26544.523225506007,18808.59412325443\nclimatology,A,2015-05-01,2015,5,lat_maior_igual_menos15,31581,266510.25051701174,3734.581566380337,61724.35256422125,264940.52647560765,31215.50842646724\nclimatology,A,2015-06-01,2015,6,dominio_inteiro,78561,238464.28620658783,9248.98515542224,76011.6435988918,225779.9040330444,67475.78503925544\nclimatology,A,2015-06-01,2015,6,lat_menor_que_menos35,26100,30401.595471536486,-1062.627030313015,19802.57276713848,21288.039431392142,17451.68248953376\nclimatology,A,2015-06-01,2015,6,lat_menos35_a_menos15,20880,17314.782434260193,4421.009831496514,12462.229528048076,14897.350670020714,18808.59412325443\nclimatology,A,2015-06-01,2015,6,lat_maior_igual_menos15,31581,190747.90830079117,5890.602354238741,43746.84130370524,189594.51393163158,31215.50842646724\nclimatology,A,2015-07-01,2015,7,dominio_inteiro,78561,308947.0633916516,-24145.166838202626,83361.35061912797,286505.3122640068,67475.78503925544\nclimatology,A,2015-07-01,2015,7,lat_menor_que_menos35,26100,45471.93899937726,-12322.36436522007,23022.13496118784,33068.99034427144,17451.68248953376\nclimatology,A,2015-07-01,2015,7,lat_menos35_a_menos15,20880,71244.71803015134,-10628.315095733851,20060.377490632236,62547.65887897642,18808.59412325443\nclimatology,A,2015-07-01,2015,7,lat_maior_igual_menos15,31581,192230.40636212294,-1194.4873772487044,40278.8381673079,190888.66304075895,31215.50842646724\nclimatology,A,2015-08-01,2015,8,dominio_inteiro,78561,219914.65884839825,-10203.421165345935,78954.28989452939,199374.03616793343,67475.78503925544\nclimatology,A,2015-08-01,2015,8,lat_menor_que_menos35,26100,49557.34155443202,-20364.900391459465,23761.58234834671,36746.551239784436,17451.68248953376\nclimatology,A,2015-08-01,2015,8,lat_menos35_a_menos15,20880,45619.07691984135,-1846.298598308349,19686.55038906657,39136.370251073706,18808.59412325443\nclimatology,A,2015-08-01,2015,8,lat_maior_igual_menos15,31581,124738.24037412487,12007.777824421879,35506.157157116104,123491.11467707533,31215.50842646724\nclimatology,A,2015-09-01,2015,9,dominio_inteiro,78561,239864.71040445278,23697.18699628464,89827.4978222379,225494.0638529153,67475.78503925544\nclimatology,A,2015-09-01,2015,9,lat_menor_que_menos35,26100,27653.8233076941,1829.3095400631428,19003.6602576077,19907.9154178874,17451.68248953376\nclimatology,A,2015-09-01,2015,9,lat_menos35_a_menos15,20880,42928.99476407518,-9142.709236247232,19905.899813655997,37815.02843901849,18808.59412325443\nclimatology,A,2015-09-01,2015,9,lat_maior_igual_menos15,31581,169281.8923326835,31010.58669246873,50917.937750974204,167771.11999600945,31215.50842646724\nclimatology,A,2015-10-01,2015,10,dominio_inteiro,78561,242175.8444665499,27711.321957787266,89180.96017596754,226509.514129906,67475.78503925544\nclimatology,A,2015-10-01,2015,10,lat_menor_que_menos35,26100,30260.179262893704,221.60397627949715,19759.188917189837,21288.733577183728,17451.68248953376\nclimatology,A,2015-10-01,2015,10,lat_menos35_a_menos15,20880,42287.7064213879,2675.507231589174,20179.763529778225,37190.052769028196,18808.59412325443\nclimatology,A,2015-10-01,2015,10,lat_maior_igual_menos15,31581,169627.95878226828,24814.210749918595,49242.00772899948,168030.7277836941,31215.50842646724\nclimatology,A,2015-11-01,2015,11,dominio_inteiro,78561,247801.66021212,21709.17250457959,84427.62147858588,236220.67616077766,67475.78503925544\nclimatology,A,2015-11-01,2015,11,lat_menor_que_menos35,26100,11381.20606382334,3616.950760245323,12442.052436292171,8003.996111096029,17451.68248953376\nclimatology,A,2015-11-01,2015,11,lat_menos35_a_menos15,20880,78504.44262111465,-5360.422122815682,25124.625736205257,71573.21651293294,18808.59412325443\nclimatology,A,2015-11-01,2015,11,lat_maior_igual_menos15,31581,157916.01152718204,23452.64386714995,46860.94330608845,156643.4635367487,31215.50842646724\nclimatology,A,2015-12-01,2015,12,dominio_inteiro,78561,445777.99325118895,48679.66334673,110733.66653722088,425367.2693864044,67475.78503925544\nclimatology,A,2015-12-01,2015,12,lat_menor_que_menos35,26100,23406.851075458853,-143.4295174330473,16029.18509413302,16904.382420929956,17451.68248953376\nclimatology,A,2015-12-01,2015,12,lat_menos35_a_menos15,20880,128813.17714701414,8639.836129691277,33779.41328750487,117095.38781728447,18808.59412325443\nclimatology,A,2015-12-01,2015,12,lat_maior_igual_menos15,31581,293557.9650287159,40183.25673447177,60925.068155582994,291367.49914819014,31215.50842646724\nclimatology,A,2016-01-01,2016,1,dominio_inteiro,78561,609105.1441551499,18528.184278694913,134992.09299099818,576560.9621173741,67475.78503925544\nclimatology,A,2016-01-01,2016,1,lat_menor_que_menos35,26100,49373.24692462531,3325.378714643419,21633.887139596045,33637.85711389993,17451.68248953376\nclimatology,A,2016-01-01,2016,1,lat_menos35_a_menos15,20880,156618.5260066338,-14566.340904530138,36508.863419497386,143232.35365454396,18808.59412325443\nclimatology,A,2016-01-01,2016,1,lat_maior_igual_menos15,31581,403113.3712238908,29769.14646858163,76849.34243190475,399690.7513489301,31215.50842646724\nclimatology,A,2016-02-01,2016,2,dominio_inteiro,78561,250932.66952528516,23286.379979543504,88800.27923372958,237773.09157472846,67475.78503925544\nclimatology,A,2016-02-01,2016,2,lat_menor_que_menos35,26100,22422.80141562545,-6813.650075644255,17453.118269711733,15570.211800223253,17451.68248953376\nclimatology,A,2016-02-01,2016,2,lat_menos35_a_menos15,20880,50593.437118754126,1815.6080585439922,20950.773567967233,45739.46234375007,18808.59412325443\nclimatology,A,2016-02-01,2016,2,lat_maior_igual_menos15,31581,177916.4309909056,28284.421996643767,50396.38739605062,176463.41743075513,31215.50842646724\nclimatology,A,2016-03-01,2016,3,dominio_inteiro,78561,292165.31593030854,6503.647845135769,94739.59368187073,272901.6864685957,67475.78503925544\nclimatology,A,2016-03-01,2016,3,lat_menor_que_menos35,26100,35558.518484194705,7472.402735412121,22530.902639985085,26027.936331437806,17451.68248953376\nclimatology,A,2016-03-01,2016,3,lat_menos35_a_menos15,20880,74340.71573642967,1433.2629312218633,25523.678363709478,65562.74974120625,18808.59412325443\nclimatology,A,2016-03-01,2016,3,lat_maior_igual_menos15,31581,182266.08170968413,-2402.017821498215,46685.012678176165,181311.00039595165,31215.50842646724\nclimatology,A,2016-04-01,2016,4,dominio_inteiro,78561,328052.507256704,1530.7435496356338,100158.45633947663,300087.59105933784,67475.78503925544\nclimatology,A,2016-04-01,2016,4,lat_menor_que_menos35,26100,53188.133514543224,-4156.247289150953,23092.354052454233,38204.56233801167,17451.68248953376\nclimatology,A,2016-04-01,2016,4,lat_menos35_a_menos15,20880,88716.76915163264,-3252.031454272568,26029.489264300093,76685.2960292963,18808.59412325443\nclimatology,A,2016-04-01,2016,4,lat_maior_igual_menos15,31581,186147.60459052812,8939.022293059155,51036.613022722304,185197.7326920299,31215.50842646724\nclimatology,A,2016-05-01,2016,5,dominio_inteiro,78561,349656.33163273276,36439.57239525393,103085.41849388927,306476.3868540545,67475.78503925544\nclimatology,A,2016-05-01,2016,5,lat_menor_que_menos35,26100,119328.14515362665,28445.600059092045,37354.09995537996,80553.78023549232,17451.68248953376\nclimatology,A,2016-05-01,2016,5,lat_menos35_a_menos15,20880,30933.694715573714,-5033.576072869822,17288.972727978602,27275.81870306463,18808.59412325443\nclimatology,A,2016-05-01,2016,5,lat_maior_igual_menos15,31581,199394.49176353237,13027.548409031704,48442.34581053071,198646.78791549758,31215.50842646724\nclimatology,A,2016-06-01,2016,6,dominio_inteiro,78561,254032.5506173816,37756.14534218982,90707.79810203519,221673.1653621745,67475.78503925544\nclimatology,A,2016-06-01,2016,6,lat_menor_que_menos35,26100,97290.81994210654,24471.133040487766,32523.693340718746,71516.78136485371,17451.68248953376\nclimatology,A,2016-06-01,2016,6,lat_menos35_a_menos15,20880,50826.78827259774,114.28337580803782,21795.3106117025,44850.85041812736,18808.59412325443\nclimatology,A,2016-06-01,2016,6,lat_maior_igual_menos15,31581,105914.94240267733,13170.728925894015,36388.79414961394,105305.53357919345,31215.50842646724\nclimatology,A,2016-07-01,2016,7,dominio_inteiro,78561,169672.265647549,7424.695527520031,70250.08332901075,156747.60125545211,67475.78503925544\nclimatology,A,2016-07-01,2016,7,lat_menor_que_menos35,26100,36283.073699759276,1924.2462762594223,22513.454357862473,26380.410464973836,17451.68248953376\nclimatology,A,2016-07-01,2016,7,lat_menos35_a_menos15,20880,15623.635012903642,1691.690321136266,12091.039264895022,13550.713071178756,18808.59412325443\nclimatology,A,2016-07-01,2016,7,lat_maior_igual_menos15,31581,117765.55693488606,3808.7589301243424,35645.58970625326,116816.47771929952,31215.50842646724\nclimatology,A,2016-08-01,2016,8,dominio_inteiro,78561,153907.67658746112,11701.487968565198,64658.76098685269,142488.13672462665,67475.78503925544\nclimatology,A,2016-08-01,2016,8,lat_menor_que_menos35,26100,30114.366687801303,1883.3715506792068,18664.08187508583,21994.98450407015,17451.68248953376\nclimatology,A,2016-08-01,2016,8,lat_menos35_a_menos15,20880,20235.838200406968,-3104.8263540456537,13698.177338977112,17866.82826317405,18808.59412325443\nclimatology,A,2016-08-01,2016,8,lat_maior_igual_menos15,31581,103557.47169925284,12922.942771931645,32296.50177278975,102626.32395738245,31215.50842646724\nclimatology,A,2016-09-01,2016,9,dominio_inteiro,78561,168193.07616373646,10919.065048042452,75471.6186001075,152018.00272031047,67475.78503925544\nclimatology,A,2016-09-01,2016,9,lat_menor_que_menos35,26100,35370.30323910802,10384.779053002596,23464.666693776846,24026.94313929342,17451.68248953376\nclimatology,A,2016-09-01,2016,9,lat_menos35_a_menos15,20880,33577.73031667686,9270.882148640463,17277.203990100184,29789.914818009347,18808.59412325443\nclimatology,A,2016-09-01,2016,9,lat_maior_igual_menos15,31581,99245.04260795155,-8736.596153600607,34729.747916230466,98201.14476300769,31215.50842646724\nclimatology,A,2016-10-01,2016,10,dominio_inteiro,78561,168632.82490783214,-3219.0502803719137,65476.40010439954,157900.99805171473,67475.78503925544\nclimatology,A,2016-10-01,2016,10,lat_menor_que_menos35,26100,22043.952227639296,1365.7156095802784,17155.281127244234,16090.268150310341,17451.68248953376\nclimatology,A,2016-10-01,2016,10,lat_menos35_a_menos15,20880,30884.608179870855,-101.14415085711516,15785.837941625854,27550.549767097014,18808.59412325443\nclimatology,A,2016-10-01,2016,10,lat_maior_igual_menos15,31581,115704.26450032197,-4483.621739095077,32535.28103552945,114260.18013430737,31215.50842646724\nclimatology,A,2016-11-01,2016,11,dominio_inteiro,78561,273594.3833968317,-3505.24399932666,84760.04446559417,261985.27087248198,67475.78503925544\nclimatology,A,2016-11-01,2016,11,lat_menor_que_menos35,26100,18836.652421003902,-4995.691527962685,16553.751399934292,13304.140421912944,17451.68248953376\nclimatology,A,2016-11-01,2016,11,lat_menos35_a_menos15,20880,30750.07933561065,4684.909832140373,16891.469115205284,27863.995361480513,18808.59412325443\nclimatology,A,2016-11-01,2016,11,lat_maior_igual_menos15,31581,224007.6516402171,-3194.462303504348,51314.82395045459,220817.13508908852,31215.50842646724\nclimatology,A,2016-12-01,2016,12,dominio_inteiro,78561,251740.61596121258,-21822.137190379377,91473.57720230584,239410.12827801862,67475.78503925544\nclimatology,A,2016-12-01,2016,12,lat_menor_que_menos35,26100,22157.85465077391,-3716.236829444766,15972.235428020358,15554.352691394048,17451.68248953376\nclimatology,A,2016-12-01,2016,12,lat_menos35_a_menos15,20880,44956.770782124295,11229.689416434441,21279.198397171393,40614.8492050237,18808.59412325443\nclimatology,A,2016-12-01,2016,12,lat_maior_igual_menos15,31581,184625.99052831437,-29335.589777369052,54222.14337711409,183240.92638160085,31215.50842646724\n', 'research/spatial_unet/evidence/seven_blocks/A/source_calendar.csv': 'time_alvo,limite_emissao,atmosfera,indices,sst,inicializacao_sazonal,ultimo_alvo_treino,disponibilidade_historica_comprovada\n2015-01-01,2014-12-31 23:59:59.999999999,2014-09-01,2014-10-01,2014-11-01,2014-12-01,2014-09-01,False\n2015-02-01,2015-01-31 23:59:59.999999999,2014-10-01,2014-11-01,2014-12-01,2015-01-01,2014-09-01,False\n2015-03-01,2015-02-28 23:59:59.999999999,2014-11-01,2014-12-01,2015-01-01,2015-02-01,2014-09-01,False\n2015-04-01,2015-03-31 23:59:59.999999999,2014-12-01,2015-01-01,2015-02-01,2015-03-01,2014-09-01,False\n2015-05-01,2015-04-30 23:59:59.999999999,2015-01-01,2015-02-01,2015-03-01,2015-04-01,2014-09-01,False\n2015-06-01,2015-05-31 23:59:59.999999999,2015-02-01,2015-03-01,2015-04-01,2015-05-01,2014-09-01,False\n2015-07-01,2015-06-30 23:59:59.999999999,2015-03-01,2015-04-01,2015-05-01,2015-06-01,2014-09-01,False\n2015-08-01,2015-07-31 23:59:59.999999999,2015-04-01,2015-05-01,2015-06-01,2015-07-01,2014-09-01,False\n2015-09-01,2015-08-31 23:59:59.999999999,2015-05-01,2015-06-01,2015-07-01,2015-08-01,2014-09-01,False\n2015-10-01,2015-09-30 23:59:59.999999999,2015-06-01,2015-07-01,2015-08-01,2015-09-01,2014-09-01,False\n2015-11-01,2015-10-31 23:59:59.999999999,2015-07-01,2015-08-01,2015-09-01,2015-10-01,2014-09-01,False\n2015-12-01,2015-11-30 23:59:59.999999999,2015-08-01,2015-09-01,2015-10-01,2015-11-01,2014-09-01,False\n2016-01-01,2015-12-31 23:59:59.999999999,2015-09-01,2015-10-01,2015-11-01,2015-12-01,2014-09-01,False\n2016-02-01,2016-01-31 23:59:59.999999999,2015-10-01,2015-11-01,2015-12-01,2016-01-01,2014-09-01,False\n2016-03-01,2016-02-29 23:59:59.999999999,2015-11-01,2015-12-01,2016-01-01,2016-02-01,2014-09-01,False\n2016-04-01,2016-03-31 23:59:59.999999999,2015-12-01,2016-01-01,2016-02-01,2016-03-01,2014-09-01,False\n2016-05-01,2016-04-30 23:59:59.999999999,2016-01-01,2016-02-01,2016-03-01,2016-04-01,2014-09-01,False\n2016-06-01,2016-05-31 23:59:59.999999999,2016-02-01,2016-03-01,2016-04-01,2016-05-01,2014-09-01,False\n2016-07-01,2016-06-30 23:59:59.999999999,2016-03-01,2016-04-01,2016-05-01,2016-06-01,2014-09-01,False\n2016-08-01,2016-07-31 23:59:59.999999999,2016-04-01,2016-05-01,2016-06-01,2016-07-01,2014-09-01,False\n2016-09-01,2016-08-31 23:59:59.999999999,2016-05-01,2016-06-01,2016-07-01,2016-08-01,2014-09-01,False\n2016-10-01,2016-09-30 23:59:59.999999999,2016-06-01,2016-07-01,2016-08-01,2016-09-01,2014-09-01,False\n2016-11-01,2016-10-31 23:59:59.999999999,2016-07-01,2016-08-01,2016-09-01,2016-10-01,2014-09-01,False\n2016-12-01,2016-11-30 23:59:59.999999999,2016-08-01,2016-09-01,2016-10-01,2016-11-01,2014-09-01,False\n', 'research/spatial_unet/evidence/seven_blocks/A/split.json': '{\n  "historical_publication_vintages_verified": false,\n  "independent_holdout": false,\n  "inner_reference": [\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06"\n  ],\n  "inner_train": [\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06"\n  ],\n  "inner_validation": [\n    "2012-10",\n    "2012-11",\n    "2012-12",\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06",\n    "2014-07",\n    "2014-08",\n    "2014-09"\n  ],\n  "outer": [\n    "2015-01",\n    "2015-02",\n    "2015-03",\n    "2015-04",\n    "2015-05",\n    "2015-06",\n    "2015-07",\n    "2015-08",\n    "2015-09",\n    "2015-10",\n    "2015-11",\n    "2015-12",\n    "2016-01",\n    "2016-02",\n    "2016-03",\n    "2016-04",\n    "2016-05",\n    "2016-06",\n    "2016-07",\n    "2016-08",\n    "2016-09",\n    "2016-10",\n    "2016-11",\n    "2016-12"\n  ],\n  "train": [\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09",\n    "2012-10",\n    "2012-11",\n    "2012-12",\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06",\n    "2014-07",\n    "2014-08",\n    "2014-09"\n  ]\n}\n', 'research/spatial_unet/evidence/seven_blocks/A/unet/training.json': '{\n  "device": "cuda",\n  "inner_selection": false,\n  "parameter_count": 122945,\n  "selected_epochs": 7,\n  "train_end": "2014-09-01",\n  "train_months": 360,\n  "train_start": "1984-10-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/A/unet/training_history.json': '[\n  {\n    "epoch": 1,\n    "seconds": 21.1854687999994,\n    "training_mse": 2.9884720967875587\n  },\n  {\n    "epoch": 2,\n    "seconds": 21.62386910800069,\n    "training_mse": 2.9127535694175295\n  },\n  {\n    "epoch": 3,\n    "seconds": 21.035209890999795,\n    "training_mse": 2.8991818752553726\n  },\n  {\n    "epoch": 4,\n    "seconds": 20.90057044399964,\n    "training_mse": 2.880743959214952\n  },\n  {\n    "epoch": 5,\n    "seconds": 21.01977664399965,\n    "training_mse": 2.862917951742808\n  },\n  {\n    "epoch": 6,\n    "seconds": 21.10321722899971,\n    "training_mse": 2.849092134502199\n  },\n  {\n    "epoch": 7,\n    "seconds": 21.038484412999424,\n    "training_mse": 2.828175741434097\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/B/baseline_check.json': '{\n  "reproduced": true,\n  "rmse_archived": 1.802321626064037,\n  "rmse_current": 1.802321626064037,\n  "tolerance": 1e-06\n}\n', 'research/spatial_unet/evidence/seven_blocks/B/complete.json': '{\n  "files": {\n    "baseline_check.json": "f1d3970e916b9b4a2ccdec3cc2f7f1476d1d4b6fec65a8ae6c7bae72194f2a37",\n    "hybrid/arvores_multissistema.txt": "41e33c4c0b51d59bfddb2b4015bcb5b23c068c008be208593c39ccbdcc0d896a",\n    "hybrid/arvores_seas5.txt": "ffb6a3f56a4609291bf43fb5443af6b47cc685981b9d55dbc1bec2338b4da2ba",\n    "hybrid/clima_atmosfera_indices.npz": "c7cb894cf5ecda90773fe9f81fe6bb71b145b939cfd529da59ca588847a5042c",\n    "hybrid/clima_cfsv2.nc": "c5a785abfb71656fc0d2e670f5e22d03848a742099f90fda944974bdd92a0975",\n    "hybrid/clima_chuva.nc": "bfba6e4c5ae8e3f5af9d09caa265a29268a528995756a7fc060dfa63a2c2761d",\n    "hybrid/clima_seas5.nc": "7e4f0aa4815df36d80d412da49c8e24ff052e41fb378ccae45ff0f7c3961de98",\n    "hybrid/fit.json": "8e7c0f7909120dd1b5987354c9d5712c079d46d3cb4498159218a1e6cbd483ce",\n    "hybrid/ridge.npz": "7e9c60782b6ca771744f6d4e9ec2c17b592ac0a3fe169a7e804998188bf11ded",\n    "inner/context.json": "9a5ad03e2a42ff3baca1a42809ca2a07bdbc2248b7b1a375fadbcef23e45aa97",\n    "inner/network.pt": "44b74305ca15fe0c7bdc8ce38ec008f437609e4e4ed18eb701b8e33c95ad3450",\n    "inner/rainfall_climatology.nc": "b558ee269a9cc9862fd2fe9761c612907708ee08a2802f6fb6327da575d3868b",\n    "inner/references.npz": "ce98a7fe2167f6438ae2bfef6b0b9d5bb0e1679c22008f2f371c3eb62ddc3f30",\n    "inner/scaling.npz": "e677d7cfeb999fbc26685b041906834ebc8cb45ac97436c9ec1f66db2c8430ed",\n    "inner/training.json": "86fb5a91dfbcd0ee802b032691767bbe20429316f8ffd55fcc0111626f197ee8",\n    "inner/training_history.json": "a57824d77f62ed7657c61a93057c187b997ec0fb6340c5240ee1be6135e5f2c1",\n    "monthly_metrics.csv": "8553057907dcccc2740682c99771b9d010737cd32145493c88f68847a9ba156f",\n    "predictions.nc": "96031fca5d47d5bcf54442ea69dbdcc09e7f439e2d6359f59c141740112fbe26",\n    "source_calendar.csv": "fca88bee094820b02b12ad18bc65e9b1090125af011a407174e89400fb2f6e70",\n    "split.json": "9f91c64097ede8c47da2404ca89cee79f38e14e508b5bce309925187967a0953",\n    "unet/context.json": "53bfc4ba55e6d4fedad64f3453fe2839c4ed841054cfb48f5a58b44566687611",\n    "unet/network.pt": "a565e2ad79b299546a38c9ec5a7aff1ac53c818f36563b6262d13d1900d0ce81",\n    "unet/rainfall_climatology.nc": "911adb1768903bcbd69bdf9db8438df265193fbba290a4719bc4c23c9858df84",\n    "unet/references.npz": "f07856369b0d88799682212792ccdd4265faf039df5d6e7c81d8e19699d8dc4a",\n    "unet/scaling.npz": "4d5330cb3066f35489395dc2009a72f8544300e455e9c365f5593a17ef2ca233",\n    "unet/training.json": "df8c77187b46c19cb97ce24af3e2f11dc87c69c0993d812b1c9abad0c8f5ced7",\n    "unet/training_history.json": "73cfbf2afb218971a24bbfe5ebe960a1e9a0087bbc538d26bc64a2b87d70b01e"\n  },\n  "signature": "02d1ffc0a7205f2bf372673d337cf5daa95f125d43a95836b953c10ae101244c"\n}\n', 'research/spatial_unet/evidence/seven_blocks/B/inner/training.json': '{\n  "device": "cuda",\n  "inner_selection": true,\n  "parameter_count": 122945,\n  "selected_epochs": 18,\n  "train_end": "2014-06-01",\n  "train_months": 333,\n  "train_start": "1986-10-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/B/inner/training_history.json': '[\n  {\n    "epoch": 1,\n    "inner_rmse": 1.778806926589282,\n    "seconds": 19.639476860000286,\n    "training_mse": 3.0099655243011565\n  },\n  {\n    "epoch": 2,\n    "inner_rmse": 1.7712689773862185,\n    "seconds": 20.07072769699971,\n    "training_mse": 2.933716454663434\n  },\n  {\n    "epoch": 3,\n    "inner_rmse": 1.7861319416038337,\n    "seconds": 19.453902355000537,\n    "training_mse": 2.910236569138261\n  },\n  {\n    "epoch": 4,\n    "inner_rmse": 1.7691727840187466,\n    "seconds": 19.293456784000227,\n    "training_mse": 2.894323601021065\n  },\n  {\n    "epoch": 5,\n    "inner_rmse": 1.7720932236733118,\n    "seconds": 19.44305742600045,\n    "training_mse": 2.879559138873676\n  },\n  {\n    "epoch": 6,\n    "inner_rmse": 1.7608935356715512,\n    "seconds": 19.598693853999976,\n    "training_mse": 2.8552058745432904\n  },\n  {\n    "epoch": 7,\n    "inner_rmse": 1.7673833564011492,\n    "seconds": 19.545314753999264,\n    "training_mse": 2.844598355593982\n  },\n  {\n    "epoch": 8,\n    "inner_rmse": 1.763510166772885,\n    "seconds": 19.446623996999733,\n    "training_mse": 2.831284891735684\n  },\n  {\n    "epoch": 9,\n    "inner_rmse": 1.7611581970698424,\n    "seconds": 19.440514134000296,\n    "training_mse": 2.8164948629545377\n  },\n  {\n    "epoch": 10,\n    "inner_rmse": 1.7784977908957151,\n    "seconds": 19.44277926399991,\n    "training_mse": 2.8075655179696755\n  },\n  {\n    "epoch": 11,\n    "inner_rmse": 1.7615440698642955,\n    "seconds": 19.516847406000124,\n    "training_mse": 2.803824467701955\n  },\n  {\n    "epoch": 12,\n    "inner_rmse": 1.756234196918273,\n    "seconds": 19.473689344000377,\n    "training_mse": 2.789495304182127\n  },\n  {\n    "epoch": 13,\n    "inner_rmse": 1.765797494887212,\n    "seconds": 19.521901633999732,\n    "training_mse": 2.78766796252391\n  },\n  {\n    "epoch": 14,\n    "inner_rmse": 1.7716742913280492,\n    "seconds": 19.430071863000194,\n    "training_mse": 2.7866153545207806\n  },\n  {\n    "epoch": 15,\n    "inner_rmse": 1.757828702255769,\n    "seconds": 19.485858747999373,\n    "training_mse": 2.777735281634975\n  },\n  {\n    "epoch": 16,\n    "inner_rmse": 1.7660143861754205,\n    "seconds": 19.45488290600042,\n    "training_mse": 2.7729832095068856\n  },\n  {\n    "epoch": 17,\n    "inner_rmse": 1.7588906568613127,\n    "seconds": 19.45346010600042,\n    "training_mse": 2.771338681200961\n  },\n  {\n    "epoch": 18,\n    "inner_rmse": 1.7522502428057598,\n    "seconds": 19.421554929000195,\n    "training_mse": 2.761320597416646\n  },\n  {\n    "epoch": 19,\n    "inner_rmse": 1.7551752951904391,\n    "seconds": 19.46675108799991,\n    "training_mse": 2.7587322873754188\n  },\n  {\n    "epoch": 20,\n    "inner_rmse": 1.7657007614158025,\n    "seconds": 19.464806402999784,\n    "training_mse": 2.7539617936532417\n  },\n  {\n    "epoch": 21,\n    "inner_rmse": 1.755588430767563,\n    "seconds": 19.485170437000306,\n    "training_mse": 2.7576677749106833\n  },\n  {\n    "epoch": 22,\n    "inner_rmse": 1.7566419179222779,\n    "seconds": 19.483981640999446,\n    "training_mse": 2.74997379543545\n  },\n  {\n    "epoch": 23,\n    "inner_rmse": 1.768334372107406,\n    "seconds": 19.50227697399987,\n    "training_mse": 2.7501153817047945\n  },\n  {\n    "epoch": 24,\n    "inner_rmse": 1.7636430600335362,\n    "seconds": 19.451298571000734,\n    "training_mse": 2.7393956413498155\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/B/monthly_metrics.csv': 'modelo,bloco,mes_alvo,ano,mes,regiao,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area\nunet,B,2017-01-01,2017,1,dominio_inteiro,78561,242644.75073974943,7032.0513317249715,89050.34101918712,228746.92754117685,67475.78503925544\nunet,B,2017-01-01,2017,1,lat_menor_que_menos35,26100,22260.172169140544,-8871.983211509883,16576.76945308596,15454.354624828931,17451.68248953376\nunet,B,2017-01-01,2017,1,lat_menos35_a_menos15,20880,70220.62513612212,14590.487100686878,26409.104170221835,64154.19180988103,18808.59412325443\nunet,B,2017-01-01,2017,1,lat_maior_igual_menos15,31581,150163.95343448676,1313.547442547977,46064.46739587933,149138.3811064669,31215.50842646724\nunet,B,2017-02-01,2017,2,dominio_inteiro,78561,286488.2174097556,-10807.36615701206,92529.4147395473,265508.2376978103,67475.78503925544\nunet,B,2017-02-01,2017,2,lat_menor_que_menos35,26100,53324.56317413356,-9621.321484871209,25181.440910719335,38830.93282340227,17451.68248953376\nunet,B,2017-02-01,2017,2,lat_menos35_a_menos15,20880,50948.44009972896,7520.497333308682,21126.21054256521,45468.03677694528,18808.59412325443\nunet,B,2017-02-01,2017,2,lat_maior_igual_menos15,31581,182215.21413589315,-8706.542005449533,46221.76328626275,181209.2680974627,31215.50842646724\nunet,B,2017-03-01,2017,3,dominio_inteiro,78561,233607.80196739256,-7659.981453272514,83513.97941623162,217330.83416430725,67475.78503925544\nunet,B,2017-03-01,2017,3,lat_menor_que_menos35,26100,34660.401373256806,220.70669075846672,22148.84399560094,23178.943010522136,17451.68248953376\nunet,B,2017-03-01,2017,3,lat_menos35_a_menos15,20880,38740.826836137,-197.741462867707,19163.739636372775,34722.40944413488,18808.59412325443\nunet,B,2017-03-01,2017,3,lat_maior_igual_menos15,31581,160206.57375799876,-7682.946681163274,42201.39578425791,159429.48170965022,31215.50842646724\nunet,B,2017-04-01,2017,4,dominio_inteiro,78561,198938.9202863575,8960.828345444053,83851.34873490408,180887.63949192097,67475.78503925544\nunet,B,2017-04-01,2017,4,lat_menor_que_menos35,26100,44454.12759151091,-3890.7695977389812,26498.892794579268,30512.335587412497,17451.68248953376\nunet,B,2017-04-01,2017,4,lat_menos35_a_menos15,20880,32518.635045201263,-3963.9343250766397,17038.865879163146,29047.78542572201,18808.59412325443\nunet,B,2017-04-01,2017,4,lat_maior_igual_menos15,31581,121966.15764964529,16815.532268259674,40313.59006116167,121327.51847878649,31215.50842646724\nunet,B,2017-05-01,2017,5,dominio_inteiro,78561,290779.4341097316,-16052.648471184075,94618.95654378086,272748.6865811709,67475.78503925544\nunet,B,2017-05-01,2017,5,lat_menor_que_menos35,26100,33640.41196908233,1612.9799498915672,21539.726315438747,24365.315716597855,17451.68248953376\nunet,B,2017-05-01,2017,5,lat_menos35_a_menos15,20880,65508.01966849383,-12549.139214795083,22198.769835758954,57876.66815391649,18808.59412325443\nunet,B,2017-05-01,2017,5,lat_maior_igual_menos15,31581,191631.00247215547,-5116.489206280559,50880.46039258316,190506.70271065654,31215.50842646724\nunet,B,2017-06-01,2017,6,dominio_inteiro,78561,251132.54289347527,-19329.317824417725,80849.74928830005,238768.8461705322,67475.78503925544\nunet,B,2017-06-01,2017,6,lat_menor_que_menos35,26100,30322.67530733658,-4231.037890970707,20779.867235958576,22250.54670353223,17451.68248953376\nunet,B,2017-06-01,2017,6,lat_menos35_a_menos15,20880,22368.668309017437,4542.238399984315,14011.835537431762,19201.877393335806,18808.59412325443\nunet,B,2017-06-01,2017,6,lat_maior_igual_menos15,31581,198441.19927712125,-19640.518333431333,46058.046514909714,197316.42207366417,31215.50842646724\nunet,B,2017-07-01,2017,7,dominio_inteiro,78561,283170.8483852249,2258.701902870089,90443.32446298748,262269.7538607845,67475.78503925544\nunet,B,2017-07-01,2017,7,lat_menor_que_menos35,26100,51384.38679444323,-212.00645849108696,25573.226892977953,36672.86526320568,17451.68248953376\nunet,B,2017-07-01,2017,7,lat_menos35_a_menos15,20880,36570.50166109032,14355.151140972972,18328.362179607153,31714.864119451267,18808.59412325443\nunet,B,2017-07-01,2017,7,lat_maior_igual_menos15,31581,195215.9599296914,-11884.442779611796,46541.73539040238,193882.02447812763,31215.50842646724\nunet,B,2017-08-01,2017,8,dominio_inteiro,78561,151009.28369056797,-1242.5295619494282,67177.13741656998,135319.98862109933,67475.78503925544\nunet,B,2017-08-01,2017,8,lat_menor_que_menos35,26100,44629.12682892973,-12337.286505192518,23903.19718542695,31940.775034429345,17451.68248953376\nunet,B,2017-08-01,2017,8,lat_menos35_a_menos15,20880,17080.060422102953,-3057.289867915213,11346.191157475114,14823.131123060472,18808.59412325443\nunet,B,2017-08-01,2017,8,lat_maior_igual_menos15,31581,89300.09643953532,14152.046811158303,31927.749073667917,88556.0824636095,31215.50842646724\nunet,B,2017-09-01,2017,9,dominio_inteiro,78561,236625.37082677794,-473.35939553845674,84075.2255397914,217432.9909960613,67475.78503925544\nunet,B,2017-09-01,2017,9,lat_menor_que_menos35,26100,36839.53194140668,-6856.7028895020485,21146.817159950733,25018.411352418927,17451.68248953376\nunet,B,2017-09-01,2017,9,lat_menos35_a_menos15,20880,50962.478015407425,14037.036763007753,22395.050381827168,45037.59199949745,18808.59412325443\nunet,B,2017-09-01,2017,9,lat_maior_igual_menos15,31581,148823.3608699639,-7653.693269044161,40533.3579980135,147376.98764414492,31215.50842646724\nunet,B,2017-10-01,2017,10,dominio_inteiro,78561,256421.34797782364,-10388.120638483204,90098.68851932045,235806.60889627636,67475.78503925544\nunet,B,2017-10-01,2017,10,lat_menor_que_menos35,26100,48620.2177060927,-16380.062720835209,23350.61711281538,34989.965129122116,17451.68248953376\nunet,B,2017-10-01,2017,10,lat_menos35_a_menos15,20880,45167.29671727408,3974.814281917177,21251.08133395668,40258.03607882772,18808.59412325443\nunet,B,2017-10-01,2017,10,lat_maior_igual_menos15,31581,162633.83355445685,2017.1278004348278,45496.99007254839,160558.60768832656,31215.50842646724\nunet,B,2017-11-01,2017,11,dominio_inteiro,78561,195726.60227854774,16331.31278682733,75726.47942983685,183563.9457386582,67475.78503925544\nunet,B,2017-11-01,2017,11,lat_menor_que_menos35,26100,22284.847539365233,-6278.322268456221,15982.621541708708,16332.697777678244,17451.68248953376\nunet,B,2017-11-01,2017,11,lat_menos35_a_menos15,20880,41712.34223647022,2665.518776024226,20068.94832637487,36697.43760898387,18808.59412325443\nunet,B,2017-11-01,2017,11,lat_maior_igual_menos15,31581,131729.41250271228,19944.116279259324,39674.90956175327,130533.81035199609,31215.50842646724\nunet,B,2017-12-01,2017,12,dominio_inteiro,78561,245439.9249373419,2599.6834507193416,80423.92796088196,230924.43223326767,67475.78503925544\nunet,B,2017-12-01,2017,12,lat_menor_que_menos35,26100,15060.537450093201,1210.1673496589065,13674.806396998465,10490.786055972074,17451.68248953376\nunet,B,2017-12-01,2017,12,lat_menos35_a_menos15,20880,110033.76020652863,-2111.670448726043,27942.90703473054,100981.66664270147,18808.59412325443\nunet,B,2017-12-01,2017,12,lat_maior_igual_menos15,31581,120345.62728072007,3501.1865497864783,38806.21452915296,119451.9795345941,31215.50842646724\nunet,B,2018-01-01,2018,1,dominio_inteiro,78561,209269.94706710795,12713.83268953301,79531.34016261157,196761.16957464686,67475.78503925544\nunet,B,2018-01-01,2018,1,lat_menor_que_menos35,26100,23186.264596153655,-3260.446881197393,17402.625885732472,15860.552003097424,17451.68248953376\nunet,B,2018-01-01,2018,1,lat_menos35_a_menos15,20880,41706.875873744706,2183.3620863277465,19103.505007280968,37552.11016216481,18808.59412325443\nunet,B,2018-01-01,2018,1,lat_maior_igual_menos15,31581,144376.80659720965,13790.917484402657,43025.20926959813,143348.50740938462,31215.50842646724\nunet,B,2018-02-01,2018,2,dominio_inteiro,78561,440947.19378154917,2007.4438436722849,114352.89734317572,422875.83344740194,67475.78503925544\nunet,B,2018-02-01,2018,2,lat_menor_que_menos35,26100,26508.25947627801,5268.88451564312,20533.176599770784,18864.25971542671,17451.68248953376\nunet,B,2018-02-01,2018,2,lat_menos35_a_menos15,20880,99631.87291402023,102.12240766943432,30574.74675059528,90453.1331117041,18808.59412325443\nunet,B,2018-02-01,2018,2,lat_maior_igual_menos15,31581,314807.061391251,-3363.5630796402693,63244.97399280965,313558.44062027114,31215.50842646724\nunet,B,2018-03-01,2018,3,dominio_inteiro,78561,260814.99753825713,26712.022390490398,92008.80548655055,241468.69070975453,67475.78503925544\nunet,B,2018-03-01,2018,3,lat_menor_que_menos35,26100,39903.91265953508,-11523.826949238777,22774.275585234165,27708.47237004631,17451.68248953376\nunet,B,2018-03-01,2018,3,lat_menos35_a_menos15,20880,62269.682119692276,5927.65626292862,23610.437029877678,55636.934648322116,18808.59412325443\nunet,B,2018-03-01,2018,3,lat_maior_igual_menos15,31581,158641.40275902976,32308.193076800555,45624.09287143871,158123.28369138605,31215.50842646724\nunet,B,2018-04-01,2018,4,dominio_inteiro,78561,249894.14169055625,4059.520018667914,87863.47371476796,234559.24848832024,67475.78503925544\nunet,B,2018-04-01,2018,4,lat_menor_que_menos35,26100,34782.782183381416,-6806.3753806352615,21150.324463129044,24464.819442108907,17451.68248953376\nunet,B,2018-04-01,2018,4,lat_menos35_a_menos15,20880,42118.94239155313,15347.762474150397,20498.986330031417,37632.4872368786,18808.59412325443\nunet,B,2018-04-01,2018,4,lat_maior_igual_menos15,31581,172992.41711562173,-4481.867074847221,46214.162921607494,172461.94180933273,31215.50842646724\nunet,B,2018-05-01,2018,5,dominio_inteiro,78561,483260.9037659125,-18669.036039704457,97499.67450709641,470734.18428425945,67475.78503925544\nunet,B,2018-05-01,2018,5,lat_menor_que_menos35,26100,26763.585101766093,-4490.787813127041,18441.079581797123,19640.139869658837,17451.68248953376\nunet,B,2018-05-01,2018,5,lat_menos35_a_menos15,20880,35953.66489632723,3926.6605421481654,18408.854888639413,31418.157759433685,18808.59412325443\nunet,B,2018-05-01,2018,5,lat_maior_igual_menos15,31581,420543.65376781917,-18104.90876872558,60649.740036659874,419675.886655167,31215.50842646724\nunet,B,2018-06-01,2018,6,dominio_inteiro,78561,205568.53138567653,18215.71850599069,73410.55051854346,197770.9270727979,67475.78503925544\nunet,B,2018-06-01,2018,6,lat_menor_que_menos35,26100,18743.89686766545,1371.650680243969,16313.209503233433,13353.95177180364,17451.68248953376\nunet,B,2018-06-01,2018,6,lat_menos35_a_menos15,20880,12810.510515435151,8340.29142142646,11923.45303173922,11228.033535479844,18808.59412325443\nunet,B,2018-06-01,2018,6,lat_maior_igual_menos15,31581,174014.12400257593,8503.776404320262,45173.88798357081,173188.9417655144,31215.50842646724\nunet,B,2018-07-01,2018,7,dominio_inteiro,78561,151797.2041143665,16266.59627404809,68161.01412378252,140079.73638007845,67475.78503925544\nunet,B,2018-07-01,2018,7,lat_menor_que_menos35,26100,28690.71030505231,7775.935383141041,19358.201551496983,20758.7804343219,17451.68248953376\nunet,B,2018-07-01,2018,7,lat_menos35_a_menos15,20880,24666.298592003946,-731.5658805891871,14593.609326176345,21532.48094980458,18808.59412325443\nunet,B,2018-07-01,2018,7,lat_maior_igual_menos15,31581,98440.19521731023,9222.226771496236,34209.20324610919,97788.47499595197,31215.50842646724\nunet,B,2018-08-01,2018,8,dominio_inteiro,78561,165752.00570082583,6605.912129406817,66834.45034938771,153207.7012483179,67475.78503925544\nunet,B,2018-08-01,2018,8,lat_menor_que_menos35,26100,28115.195021754964,1215.3067404031754,20453.866007208824,19225.63026962832,17451.68248953376\nunet,B,2018-08-01,2018,8,lat_menos35_a_menos15,20880,26613.901659034193,-3787.5065567046404,15258.700327431783,23828.425050794365,18808.59412325443\nunet,B,2018-08-01,2018,8,lat_maior_igual_menos15,31581,111022.90902003668,9178.111945708282,31121.884014747106,110153.6459278952,31215.50842646724\nunet,B,2018-09-01,2018,9,dominio_inteiro,78561,172430.64528916325,-11817.397909873165,71553.42780044582,157199.4066075854,67475.78503925544\nunet,B,2018-09-01,2018,9,lat_menor_que_menos35,26100,37215.763458228146,-9801.586607068777,22227.666309565306,26441.60212901342,17451.68248953376\nunet,B,2018-09-01,2018,9,lat_menos35_a_menos15,20880,32549.671302961626,-4298.041479093023,15291.413195396774,28946.116016730793,18808.59412325443\nunet,B,2018-09-01,2018,9,lat_maior_igual_menos15,31581,102665.21052797347,2282.2301762886345,34034.34829548374,101811.68846184119,31215.50842646724\nunet,B,2018-10-01,2018,10,dominio_inteiro,78561,222855.89158956578,-8663.319135260303,73567.37703170348,211904.81521675506,67475.78503925544\nunet,B,2018-10-01,2018,10,lat_menor_que_menos35,26100,20517.727043988438,-646.5204650461674,16091.529117375612,14556.009719324225,17451.68248953376\nunet,B,2018-10-01,2018,10,lat_menos35_a_menos15,20880,34020.423167846144,-8556.304964907933,17576.12320625875,30491.241552618674,18808.59412325443\nunet,B,2018-10-01,2018,10,lat_maior_igual_menos15,31581,168317.7413777312,539.5062946937978,39899.724708069116,166857.56394481217,31215.50842646724\nunet,B,2018-11-01,2018,11,dominio_inteiro,78561,304998.12986100704,-29864.537151699653,91706.57973638759,292507.7803498019,67475.78503925544\nunet,B,2018-11-01,2018,11,lat_menor_que_menos35,26100,25179.596864257597,-9653.262329399586,18267.94929844141,18102.014876913967,17451.68248953376\nunet,B,2018-11-01,2018,11,lat_menos35_a_menos15,20880,44946.687425428565,139.04712076322176,19365.726146051893,41355.19628011497,18808.59412325443\nunet,B,2018-11-01,2018,11,lat_maior_igual_menos15,31581,234871.84557132088,-20350.32194306329,54072.90429189429,233050.56919277302,31215.50842646724\nunet,B,2018-12-01,2018,12,dominio_inteiro,78561,425547.1630781202,21146.462848380208,110391.13412777334,412141.31060236617,67475.78503925544\nunet,B,2018-12-01,2018,12,lat_menor_que_menos35,26100,17186.82786893328,3425.0645519793034,16146.873903930187,11946.754224884038,17451.68248953376\nunet,B,2018-12-01,2018,12,lat_menos35_a_menos15,20880,72331.00604530593,13496.078883741051,26709.35846586898,65678.91924026658,18808.59412325443\nunet,B,2018-12-01,2018,12,lat_maior_igual_menos15,31581,336029.329163881,4225.319412659854,67534.90175797418,334515.6371372156,31215.50842646724\nhybrid,B,2017-01-01,2017,1,dominio_inteiro,78561,241947.17176402023,-1578.1049487042678,89024.15611360785,226971.60068450906,67475.78503925544\nhybrid,B,2017-01-01,2017,1,lat_menor_que_menos35,26100,26757.998007200775,-9640.994332255283,17362.734700732122,18558.404601948114,17451.68248953376\nhybrid,B,2017-01-01,2017,1,lat_menos35_a_menos15,20880,65099.0057578484,13485.899972946987,25526.43425635509,59272.3597598607,18808.59412325443\nhybrid,B,2017-01-01,2017,1,lat_maior_igual_menos15,31581,150090.16799897104,-5423.010589395973,46134.98715652063,149140.83632270026,31215.50842646724\nhybrid,B,2017-02-01,2017,2,dominio_inteiro,78561,285810.9716761311,-16548.780484121122,91336.44573545971,265608.10589805833,67475.78503925544\nhybrid,B,2017-02-01,2017,2,lat_menor_que_menos35,26100,51905.00256117272,-9761.759518329758,24376.91756653008,37787.47098700631,17451.68248953376\nhybrid,B,2017-02-01,2017,2,lat_menos35_a_menos15,20880,46910.55032961114,7192.496794129737,20341.723883428116,41894.70047454598,18808.59412325443\nhybrid,B,2017-02-01,2017,2,lat_maior_igual_menos15,31581,186995.4187853473,-13979.517759921098,46617.804285501516,185925.93443650607,31215.50842646724\nhybrid,B,2017-03-01,2017,3,dominio_inteiro,78561,230190.68764606124,-18735.406529544878,82162.04554652564,214252.0450767228,67475.78503925544\nhybrid,B,2017-03-01,2017,3,lat_menor_que_menos35,26100,34690.15137512831,64.49537116012061,22209.622154243232,23227.703751320092,17451.68248953376\nhybrid,B,2017-03-01,2017,3,lat_menos35_a_menos15,20880,36400.07965151395,-811.117178965691,18229.85662204926,32648.180607876053,18808.59412325443\nhybrid,B,2017-03-01,2017,3,lat_maior_igual_menos15,31581,159100.45661941898,-17988.78472173931,41722.56677023313,158376.16071752668,31215.50842646724\nhybrid,B,2017-04-01,2017,4,dominio_inteiro,78561,178148.3645344378,-93.33363117656268,79450.81459938326,160205.75810832233,67475.78503925544\nhybrid,B,2017-04-01,2017,4,lat_menor_que_menos35,26100,44424.701176708375,-5536.281751955382,26464.43138452549,30443.92271526367,17451.68248953376\nhybrid,B,2017-04-01,2017,4,lat_menos35_a_menos15,20880,31879.359858690455,-4012.1944292302305,16486.065803134803,28491.699735647115,18808.59412325443\nhybrid,B,2017-04-01,2017,4,lat_maior_igual_menos15,31581,101844.30349903899,9455.14255000905,36500.31741172296,101270.13565741156,31215.50842646724\nhybrid,B,2017-05-01,2017,5,dominio_inteiro,78561,265359.3166485098,-17257.596837274927,89651.68461872148,247226.03423341404,67475.78503925544\nhybrid,B,2017-05-01,2017,5,lat_menor_que_menos35,26100,33989.56756278055,1231.1083215393564,21271.133774676295,24560.802596136597,17451.68248953376\nhybrid,B,2017-05-01,2017,5,lat_menos35_a_menos15,20880,65808.84792682085,-12157.814921103036,22263.125487061257,58119.450167297604,18808.59412325443\nhybrid,B,2017-05-01,2017,5,lat_maior_igual_menos15,31581,165560.90115890844,-6330.890237711249,46117.42535698394,164545.78146997982,31215.50842646724\nhybrid,B,2017-06-01,2017,6,dominio_inteiro,78561,218316.4294739323,-16397.80313447181,75799.55661576526,207514.42181546715,67475.78503925544\nhybrid,B,2017-06-01,2017,6,lat_menor_que_menos35,26100,26042.80283055164,-5884.054772145763,19002.4509465871,19191.03829981351,17451.68248953376\nhybrid,B,2017-06-01,2017,6,lat_menos35_a_menos15,20880,20788.29105444336,4374.242847641321,13968.606331459538,17892.45313976603,18808.59412325443\nhybrid,B,2017-06-01,2017,6,lat_maior_igual_menos15,31581,171485.3355889373,-14887.991209967367,42828.49933771862,170430.9303758876,31215.50842646724\nhybrid,B,2017-07-01,2017,7,dominio_inteiro,78561,294951.5492607523,-1434.8904519676753,92579.47571328643,272642.07255513617,67475.78503925544\nhybrid,B,2017-07-01,2017,7,lat_menor_que_menos35,26100,53491.97118469264,-3488.926019372685,26008.59050247366,37786.395102033945,17451.68248953376\nhybrid,B,2017-07-01,2017,7,lat_menos35_a_menos15,20880,39744.9100438133,15854.697909245673,19290.463639952788,34534.951370181865,18808.59412325443\nhybrid,B,2017-07-01,2017,7,lat_maior_igual_menos15,31581,201714.66803224632,-13800.662341840663,47280.421570859995,200320.7260829204,31215.50842646724\nhybrid,B,2017-08-01,2017,8,dominio_inteiro,78561,147856.40672087765,117.65616693703214,66233.0553617031,131826.67357024123,67475.78503925544\nhybrid,B,2017-08-01,2017,8,lat_menor_que_menos35,26100,45906.781294237895,-12288.174950626688,24020.208546542268,32810.27917359314,17451.68248953376\nhybrid,B,2017-08-01,2017,8,lat_menos35_a_menos15,20880,16503.824230212012,-1566.1774758136748,11177.61243086518,14299.67829517814,18808.59412325443\nhybrid,B,2017-08-01,2017,8,lat_maior_igual_menos15,31581,85445.80119642774,13972.008593377397,31035.23438429564,84716.71610146994,31215.50842646724\nhybrid,B,2017-09-01,2017,9,dominio_inteiro,78561,228042.38070698915,1169.0773629221285,82128.18711687284,209919.52344115474,67475.78503925544\nhybrid,B,2017-09-01,2017,9,lat_menor_que_menos35,26100,34401.11388177776,-6410.265272450457,20567.07934566046,23407.640378083637,17451.68248953376\nhybrid,B,2017-09-01,2017,9,lat_menos35_a_menos15,20880,49381.273347499264,14270.818991877899,22026.82929478176,43644.898340121414,18808.59412325443\nhybrid,B,2017-09-01,2017,9,lat_maior_igual_menos15,31581,144259.99347771212,-6691.476356505314,39534.27847643063,142866.98472294968,31215.50842646724\nhybrid,B,2017-10-01,2017,10,dominio_inteiro,78561,253681.43636331128,-12150.505717245878,87846.6451202133,233974.7824581592,67475.78503925544\nhybrid,B,2017-10-01,2017,10,lat_menor_que_menos35,26100,45740.171026886586,-15019.78851615654,22697.24167131628,32953.787856473005,17451.68248953376\nhybrid,B,2017-10-01,2017,10,lat_menos35_a_menos15,20880,43900.91826951671,3350.3492666018515,20577.46880098623,39032.73624118726,18808.59412325443\nhybrid,B,2017-10-01,2017,10,lat_maior_igual_menos15,31581,164040.34706690803,-481.0664676911879,44571.934647910806,161988.25836049893,31215.50842646724\nhybrid,B,2017-11-01,2017,11,dominio_inteiro,78561,190242.1725013196,14703.864118656058,73756.59934814028,178183.22732663844,67475.78503925544\nhybrid,B,2017-11-01,2017,11,lat_menor_que_menos35,26100,23659.89953438442,-5323.673110947878,16407.525788743733,17234.593507391375,17451.68248953376\nhybrid,B,2017-11-01,2017,11,lat_menos35_a_menos15,20880,38873.402796221286,3037.5936465638715,19412.258129518534,34331.49172123169,18808.59412325443\nhybrid,B,2017-11-01,2017,11,lat_maior_igual_menos15,31581,127708.8701707139,16989.94358304006,37936.81542987803,126617.14209801538,31215.50842646724\nhybrid,B,2017-12-01,2017,12,dominio_inteiro,78561,246292.7032087809,-8049.969810504087,78956.43825528557,232142.04983890048,67475.78503925544\nhybrid,B,2017-12-01,2017,12,lat_menor_que_menos35,26100,14843.592276422653,1549.6894905384288,13518.713260158165,10295.777624526167,17451.68248953376\nhybrid,B,2017-12-01,2017,12,lat_menos35_a_menos15,20880,106184.93779794019,-3102.062667756046,27221.90631287174,97525.49714287106,18808.59412325443\nhybrid,B,2017-12-01,2017,12,lat_maior_igual_menos15,31581,125264.17313441807,-6497.596633286471,38215.81868225565,124320.77507150325,31215.50842646724\nhybrid,B,2018-01-01,2018,1,dominio_inteiro,78561,191979.01590516462,6710.674503678698,74650.62957649658,179947.352007918,67475.78503925544\nhybrid,B,2018-01-01,2018,1,lat_menor_que_menos35,26100,21386.403205718158,-1711.6299337594003,16953.353886684003,14633.934933834931,17451.68248953376\nhybrid,B,2018-01-01,2018,1,lat_menos35_a_menos15,20880,42372.42978898625,1260.0024257651817,18805.97321264264,38049.41810951564,18808.59412325443\nhybrid,B,2018-01-01,2018,1,lat_maior_igual_menos15,31581,128220.18291046025,7162.302011672917,38891.302477169935,127263.99896456745,31215.50842646724\nhybrid,B,2018-02-01,2018,2,dominio_inteiro,78561,448153.5811371636,-5488.698344605353,115769.99919009095,429737.28762652766,67475.78503925544\nhybrid,B,2018-02-01,2018,2,lat_menor_que_menos35,26100,26984.88000318349,5497.065906681161,20789.307226951525,19181.628296263407,17451.68248953376\nhybrid,B,2018-02-01,2018,2,lat_menos35_a_menos15,20880,101387.6027703475,-1221.8149247827505,30490.403942677796,92129.5298904955,18808.59412325443\nhybrid,B,2018-02-01,2018,2,lat_maior_igual_menos15,31581,319781.09836363257,-9763.949326503764,64490.288020461616,318426.1294397688,31215.50842646724\nhybrid,B,2018-03-01,2018,3,dominio_inteiro,78561,239096.50694181345,17895.364078479437,87779.92538309429,219106.90960278467,67475.78503925544\nhybrid,B,2018-03-01,2018,3,lat_menor_que_menos35,26100,42287.05845934145,-11977.545211596605,22867.947746797756,29441.24331808038,17451.68248953376\nhybrid,B,2018-03-01,2018,3,lat_menos35_a_menos15,20880,62073.260429470516,5299.585666111785,23525.19797885746,55416.93346097821,18808.59412325443\nhybrid,B,2018-03-01,2018,3,lat_maior_igual_menos15,31581,134736.18805300145,24573.323623964254,41386.77965743907,134248.73282372614,31215.50842646724\nhybrid,B,2018-04-01,2018,4,dominio_inteiro,78561,260502.1855080932,-10499.695416413986,85861.97330829367,245074.69468663554,67475.78503925544\nhybrid,B,2018-04-01,2018,4,lat_menor_que_menos35,26100,36617.271345935726,-8837.268380258954,20792.2353765561,25703.664775089772,17451.68248953376\nhybrid,B,2018-04-01,2018,4,lat_menos35_a_menos15,20880,37851.54935883051,13527.630505031162,19478.215494794873,33858.55726321449,18808.59412325443\nhybrid,B,2018-04-01,2018,4,lat_maior_igual_menos15,31581,186033.36480332696,-15190.057541186192,45591.5224369427,185512.47264833128,31215.50842646724\nhybrid,B,2018-05-01,2018,5,dominio_inteiro,78561,533779.687478226,-25568.82487781967,101805.9072494227,520618.9366549028,67475.78503925544\nhybrid,B,2018-05-01,2018,5,lat_menor_que_menos35,26100,27939.66185194936,-5247.582500137475,18876.744431126415,20532.23651496093,17451.68248953376\nhybrid,B,2018-05-01,2018,5,lat_menos35_a_menos15,20880,38662.492238857296,4338.723215043184,19023.220450594177,33814.02322210852,18808.59412325443\nhybrid,B,2018-05-01,2018,5,lat_maior_igual_menos15,31581,467177.5333874194,-24659.965592725384,63905.94236770212,466272.67691783333,31215.50842646724\nhybrid,B,2018-06-01,2018,6,dominio_inteiro,78561,221456.09798710622,8178.963961399824,73684.23203754857,213463.8888093064,67475.78503925544\nhybrid,B,2018-06-01,2018,6,lat_menor_que_menos35,26100,19791.34717894511,-805.2930704701163,16328.157999631836,14176.768006018612,17451.68248953376\nhybrid,B,2018-06-01,2018,6,lat_menos35_a_menos15,20880,11492.048286080671,7265.599368385536,11346.67736806486,10071.86447392906,18808.59412325443\nhybrid,B,2018-06-01,2018,6,lat_maior_igual_menos15,31581,190172.70252208045,1718.657663484406,46009.39666985188,189215.25632935876,31215.50842646724\nhybrid,B,2018-07-01,2018,7,dominio_inteiro,78561,151828.26060430214,8082.464526920389,65593.7231071679,140140.81542836974,67475.78503925544\nhybrid,B,2018-07-01,2018,7,lat_menor_que_menos35,26100,28150.3425430039,6873.043816460292,18949.656276611375,20282.79965104413,17451.68248953376\nhybrid,B,2018-07-01,2018,7,lat_menos35_a_menos15,20880,24733.29722386016,-2666.3090250606238,14235.180829002573,21517.899919014308,18808.59412325443\nhybrid,B,2018-07-01,2018,7,lat_maior_igual_menos15,31581,98944.62083743808,3875.7297355207224,32408.88600155394,98340.11585831131,31215.50842646724\nhybrid,B,2018-08-01,2018,8,dominio_inteiro,78561,163564.89277026252,5148.378549886133,65880.31253298878,151673.14052159563,67475.78503925544\nhybrid,B,2018-08-01,2018,8,lat_menor_que_menos35,26100,26115.94227901654,2707.5161217466234,19875.903161359984,17928.552786648197,17451.68248953376\nhybrid,B,2018-08-01,2018,8,lat_menos35_a_menos15,20880,27299.883069430492,-4610.42964229522,15340.078036313987,24482.48908267414,18808.59412325443\nhybrid,B,2018-08-01,2018,8,lat_maior_igual_menos15,31581,110149.06742181549,7051.292070434729,30664.331335314822,109262.09865227331,31215.50842646724\nhybrid,B,2018-09-01,2018,9,dominio_inteiro,78561,165443.8474768502,-6180.057739674893,70280.49161072209,150790.2938303422,67475.78503925544\nhybrid,B,2018-09-01,2018,9,lat_menor_que_menos35,26100,37064.64458367063,-7428.3399360967405,21990.83669097082,26588.59844454874,17451.68248953376\nhybrid,B,2018-09-01,2018,9,lat_menos35_a_menos15,20880,30783.337381707744,-3286.6589585998427,15214.932233493404,27398.209330571695,18808.59412325443\nhybrid,B,2018-09-01,2018,9,lat_maior_igual_menos15,31581,97595.86551147181,4534.94115502169,33074.72268625787,96803.48605522177,31215.50842646724\nhybrid,B,2018-10-01,2018,10,dominio_inteiro,78561,229816.77798289567,-3232.8889223136252,74453.71437656366,218507.4352328673,67475.78503925544\nhybrid,B,2018-10-01,2018,10,lat_menor_que_menos35,26100,21631.593181114575,227.6852076626712,16567.142501434104,15259.533185955803,17451.68248953376\nhybrid,B,2018-10-01,2018,10,lat_menos35_a_menos15,20880,32958.07631829598,-8577.521213662436,17160.624049743106,29518.60522301857,18808.59412325443\nhybrid,B,2018-10-01,2018,10,lat_maior_igual_menos15,31581,175227.10848348512,5116.947083686137,40725.94782538645,173729.29682389292,31215.50842646724\nhybrid,B,2018-11-01,2018,11,dominio_inteiro,78561,305594.9429194481,-30520.608885636822,91365.19484911443,293236.0045411632,67475.78503925544\nhybrid,B,2018-11-01,2018,11,lat_menor_que_menos35,26100,24672.87161735849,-9050.909340318507,18164.966877055558,17770.52677785344,17451.68248953376\nhybrid,B,2018-11-01,2018,11,lat_menos35_a_menos15,20880,46226.383919537606,129.7786608123797,19481.29848671591,42552.28671505209,18808.59412325443\nhybrid,B,2018-11-01,2018,11,lat_maior_igual_menos15,31581,234695.68738255196,-21599.478206130694,53718.92948534296,232913.19104825766,31215.50842646724\nhybrid,B,2018-12-01,2018,12,dominio_inteiro,78561,432616.5678504091,21890.18713888577,113250.50490874697,418983.8223378304,67475.78503925544\nhybrid,B,2018-12-01,2018,12,lat_menor_que_menos35,26100,18845.837208540477,5850.365560615149,16980.55589341364,13276.534339922333,17451.68248953376\nhybrid,B,2018-12-01,2018,12,lat_menos35_a_menos15,20880,69592.16295086316,14000.281723321394,26473.180200455998,63080.46615524484,18808.59412325443\nhybrid,B,2018-12-01,2018,12,lat_maior_igual_menos15,31581,344178.5676910054,2039.5398549492256,69796.76881487733,342626.8218426632,31215.50842646724\nclimatology,B,2017-01-01,2017,1,dominio_inteiro,78561,265153.86003862426,-11815.83213771705,92527.29926201876,248961.199792326,67475.78503925544\nclimatology,B,2017-01-01,2017,1,lat_menor_que_menos35,26100,28874.531920237725,-9488.122841708362,17428.369055338204,19932.711210133664,17451.68248953376\nclimatology,B,2017-01-01,2017,1,lat_menos35_a_menos15,20880,69766.07171788395,14042.209896327811,26341.994990358013,63492.15115488792,18808.59412325443\nclimatology,B,2017-01-01,2017,1,lat_maior_igual_menos15,31581,166513.25640050258,-16369.9191923365,48756.93521632254,165536.33742730442,31215.50842646724\nclimatology,B,2017-02-01,2017,2,dominio_inteiro,78561,352381.35063382523,-27246.10316254024,99125.5923201096,332142.90027211234,67475.78503925544\nclimatology,B,2017-02-01,2017,2,lat_menor_que_menos35,26100,52455.413288914686,-9765.776393309236,24744.541235163808,38094.63752391028,17451.68248953376\nclimatology,B,2017-02-01,2017,2,lat_menos35_a_menos15,20880,44739.93538900235,7808.395015144604,20195.92323655903,39983.018574456,18808.59412325443\nclimatology,B,2017-02-01,2017,2,lat_maior_igual_menos15,31581,255186.00195590823,-25288.721784375608,54185.127848386765,254065.244173746,31215.50842646724\nclimatology,B,2017-03-01,2017,3,dominio_inteiro,78561,295488.1692790766,-25739.46726258844,90566.17850663513,278755.86150793557,67475.78503925544\nclimatology,B,2017-03-01,2017,3,lat_menor_que_menos35,26100,36456.44340417866,533.5766760408878,22617.617451161146,24340.980425729318,17451.68248953376\nclimatology,B,2017-03-01,2017,3,lat_menos35_a_menos15,20880,37504.61197976896,-221.27673146501184,18330.727984216064,33893.083634744755,18808.59412325443\nclimatology,B,2017-03-01,2017,3,lat_maior_igual_menos15,31581,221527.11389512903,-26051.767207164317,49617.83307125792,220521.79744746146,31215.50842646724\nclimatology,B,2017-04-01,2017,4,dominio_inteiro,78561,171289.97997201423,-2017.9720057258382,77723.51284177229,153108.2376115548,67475.78503925544\nclimatology,B,2017-04-01,2017,4,lat_menor_que_menos35,26100,45145.637586652425,-4480.020628601313,26570.534740000963,30914.67220634873,17451.68248953376\nclimatology,B,2017-04-01,2017,4,lat_menos35_a_menos15,20880,32259.77823081058,-4016.509031219408,16619.217527656816,28878.075301882203,18808.59412325443\nclimatology,B,2017-04-01,2017,4,lat_maior_igual_menos15,31581,93884.56415455126,6478.557654094882,34533.76057411451,93315.49010332387,31215.50842646724\nclimatology,B,2017-05-01,2017,5,dominio_inteiro,78561,246902.3459844724,-14694.863255612552,86897.58603209257,229318.80285406092,67475.78503925544\nclimatology,B,2017-05-01,2017,5,lat_menor_que_menos35,26100,34522.678794003616,486.4438464641571,21290.25344133377,24936.29388534369,17451.68248953376\nclimatology,B,2017-05-01,2017,5,lat_menos35_a_menos15,20880,60799.69863078571,-10892.941920403391,21937.208039961755,53693.742822486485,18808.59412325443\nclimatology,B,2017-05-01,2017,5,lat_maior_igual_menos15,31581,151579.9685596831,-4288.365181673318,43670.124550797045,150688.76614623074,31215.50842646724\nclimatology,B,2017-06-01,2017,6,dominio_inteiro,78561,218008.96772811172,-19143.41622293042,75416.0427167411,206987.07183898354,67475.78503925544\nclimatology,B,2017-06-01,2017,6,lat_menor_que_menos35,26100,26420.532014547,-8288.25672006607,19206.098320961,19369.651169443892,17451.68248953376\nclimatology,B,2017-06-01,2017,6,lat_menos35_a_menos15,20880,20382.823999967928,2826.3931277561933,13950.03456860967,17550.590838003794,18808.59412325443\nclimatology,B,2017-06-01,2017,6,lat_maior_igual_menos15,31581,171205.61171359677,-13681.552630620543,42259.90982717043,170066.82983153584,31215.50842646724\nclimatology,B,2017-07-01,2017,7,dominio_inteiro,78561,341200.96849592746,-4610.852165881544,99673.39693460241,315846.117061214,67475.78503925544\nclimatology,B,2017-07-01,2017,7,lat_menor_que_menos35,26100,59149.60264439894,-4594.132374346256,27454.390453398228,41412.41568946236,17451.68248953376\nclimatology,B,2017-07-01,2017,7,lat_menos35_a_menos15,20880,46755.86875348284,17489.196147273295,21063.865538763814,40716.86167693624,18808.59412325443\nclimatology,B,2017-07-01,2017,7,lat_maior_igual_menos15,31581,235295.49709804563,-17505.915938808583,51155.14094244037,233716.83969481543,31215.50842646724\nclimatology,B,2017-08-01,2017,8,dominio_inteiro,78561,152969.04679236567,-4626.983887849608,67185.81749581802,134065.66227697962,67475.78503925544\nclimatology,B,2017-08-01,2017,8,lat_menor_que_menos35,26100,54757.91137818964,-15034.492233514786,24971.14463043213,38963.73747987684,17451.68248953376\nclimatology,B,2017-08-01,2017,8,lat_menos35_a_menos15,20880,17978.872961728513,-1882.032593972981,11640.909896172583,15566.578397500602,18808.59412325443\nclimatology,B,2017-08-01,2017,8,lat_maior_igual_menos15,31581,80232.26245244751,12289.540939638158,30573.762969213305,79535.34639960219,31215.50842646724\nclimatology,B,2017-09-01,2017,9,dominio_inteiro,78561,252545.21277728747,-5324.051452637184,85957.01602147426,232550.38762781568,67475.78503925544\nclimatology,B,2017-09-01,2017,9,lat_menor_que_menos35,26100,37744.43133929741,-8551.46613496542,21157.194396436214,25604.006490760225,17451.68248953376\nclimatology,B,2017-09-01,2017,9,lat_menos35_a_menos15,20880,54327.51965205774,15584.723145144992,23265.843319722917,48039.565994726465,18808.59412325443\nclimatology,B,2017-09-01,2017,9,lat_maior_igual_menos15,31581,160473.26178593232,-12357.308462816756,41533.97830531513,158906.815142329,31215.50842646724\nclimatology,B,2017-10-01,2017,10,dominio_inteiro,78561,265866.13231473503,-15579.200049057108,90514.82907317209,245018.29786389304,67475.78503925544\nclimatology,B,2017-10-01,2017,10,lat_menor_que_menos35,26100,48882.77920475682,-16646.813583433628,23536.555596232414,35104.10000092781,17451.68248953376\nclimatology,B,2017-10-01,2017,10,lat_menos35_a_menos15,20880,44464.99337654595,4032.5508708735288,20932.475830528914,39567.3509125968,18808.59412325443\nclimatology,B,2017-10-01,2017,10,lat_maior_igual_menos15,31581,172518.35973343224,-2964.937336497009,46045.79764641076,170346.84695036843,31215.50842646724\nclimatology,B,2017-11-01,2017,11,dominio_inteiro,78561,200380.43315702977,8243.649848244473,75552.93644370415,187045.79649113133,67475.78503925544\nclimatology,B,2017-11-01,2017,11,lat_menor_que_menos35,26100,27053.84492284164,-5104.824485480785,17397.603453457355,19690.13407908449,17451.68248953376\nclimatology,B,2017-11-01,2017,11,lat_menos35_a_menos15,20880,40040.2518488457,403.45471298819757,19445.76802837939,35151.00538625153,18808.59412325443\nclimatology,B,2017-11-01,2017,11,lat_maior_igual_menos15,31581,133286.33638534247,12945.01962073706,38709.56496186741,132204.65702579528,31215.50842646724\nclimatology,B,2017-12-01,2017,12,dominio_inteiro,78561,252532.70872569922,-9728.722707695793,79698.78685329948,238495.11094597346,67475.78503925544\nclimatology,B,2017-12-01,2017,12,lat_menor_que_menos35,26100,13972.46637708923,796.7333087474108,13423.49569453299,9670.089338687663,17451.68248953376\nclimatology,B,2017-12-01,2017,12,lat_menos35_a_menos15,20880,107896.99549628732,-3349.3873431230895,27508.992858455516,99127.28426647227,18808.59412325443\nclimatology,B,2017-12-01,2017,12,lat_maior_igual_menos15,31581,130663.24685232269,-7176.068673320115,38766.29830031097,129697.7373408135,31215.50842646724\nclimatology,B,2018-01-01,2018,1,dominio_inteiro,78561,207901.64231984114,-2381.915092123789,78143.9618286899,195195.31750364092,67475.78503925544\nclimatology,B,2018-01-01,2018,1,lat_menor_que_menos35,26100,22998.811355023856,-2990.9199684783816,17396.052200950682,15774.310213974046,17451.68248953376\nclimatology,B,2018-01-01,2018,1,lat_menos35_a_menos15,20880,44596.61280258888,556.0353128927527,19706.959826517734,40234.785833308764,18808.59412325443\nclimatology,B,2018-01-01,2018,1,lat_maior_igual_menos15,31581,140306.2181622284,52.96956346184015,41040.94980122149,139186.2214563581,31215.50842646724\nclimatology,B,2018-02-01,2018,2,dominio_inteiro,78561,409152.5196886454,-10002.737874759478,110011.22340769891,391865.75151912693,67475.78503925544\nclimatology,B,2018-02-01,2018,2,lat_menor_que_menos35,26100,26286.875455185826,4522.032978639007,20467.22005482018,18626.863006719046,17451.68248953376\nclimatology,B,2018-02-01,2018,2,lat_menos35_a_menos15,20880,91106.92172040703,52.10076465632301,29461.260044975323,82610.56931220816,18808.59412325443\nclimatology,B,2018-02-01,2018,2,lat_maior_igual_menos15,31581,291758.72251305246,-14576.871618054807,60082.74330790341,290628.31920019974,31215.50842646724\nclimatology,B,2018-03-01,2018,3,dominio_inteiro,78561,224225.54644120566,16564.384818710387,86631.42034820467,202871.49184462632,67475.78503925544\nclimatology,B,2018-03-01,2018,3,lat_menor_que_menos35,26100,45528.576224869685,-13007.246306747198,23778.297769099474,31490.989215694965,17451.68248953376\nclimatology,B,2018-03-01,2018,3,lat_menos35_a_menos15,20880,63162.72896151083,5466.683893229812,23835.664335723966,56314.483563867085,18808.59412325443\nclimatology,B,2018-03-01,2018,3,lat_maior_igual_menos15,31581,115534.24125482512,24104.947232227772,39017.45824338123,115066.01906506429,31215.50842646724\nclimatology,B,2018-04-01,2018,4,dominio_inteiro,78561,292187.6166449332,-5372.469915271737,91821.85548223276,275149.68697543617,67475.78503925544\nclimatology,B,2018-04-01,2018,4,lat_menor_que_menos35,26100,37623.94087284049,-8080.473051697016,21447.839502424,26323.96419570089,17451.68248953376\nclimatology,B,2018-04-01,2018,4,lat_menos35_a_menos15,20880,47984.34300619953,15541.070009307936,21303.889044232666,42766.92070990237,18808.59412325443\nclimatology,B,2018-04-01,2018,4,lat_maior_igual_menos15,31581,206579.33276589317,-12833.066872882657,49070.12693557609,206058.8020698329,31215.50842646724\nclimatology,B,2018-05-01,2018,5,dominio_inteiro,78561,549095.7939787448,-20850.323613278568,105341.94690451212,535803.9513680092,67475.78503925544\nclimatology,B,2018-05-01,2018,5,lat_menor_que_menos35,26100,27956.088112991816,-5540.432603120804,18931.63858151436,20509.018305681933,17451.68248953376\nclimatology,B,2018-05-01,2018,5,lat_menos35_a_menos15,20880,38551.59516535451,4577.620060797781,19029.703769709915,33692.73526072831,18808.59412325443\nclimatology,B,2018-05-01,2018,5,lat_maior_igual_menos15,31581,482588.1107003984,-19887.511070955545,67380.60455328785,481602.19780159905,31215.50842646724\nclimatology,B,2018-06-01,2018,6,dominio_inteiro,78561,211435.19103281788,13623.21204245044,73244.69010648923,202937.45748809178,67475.78503925544\nclimatology,B,2018-06-01,2018,6,lat_menor_que_menos35,26100,21339.63373299547,-254.67594003677368,16857.436897039413,15315.247487994584,17451.68248953376\nclimatology,B,2018-06-01,2018,6,lat_menos35_a_menos15,20880,11685.685247578964,7532.646179514006,11344.216291246936,10240.47585443877,18808.59412325443\nclimatology,B,2018-06-01,2018,6,lat_maior_igual_menos15,31581,178409.87205224342,6345.241802973207,45043.03691820288,177381.7341456584,31215.50842646724\nclimatology,B,2018-07-01,2018,7,dominio_inteiro,78561,173863.7755279609,14978.575782116503,71470.13307909109,160556.78408496766,67475.78503925544\nclimatology,B,2018-07-01,2018,7,lat_menor_que_menos35,26100,32998.022048839215,6724.070598065853,19975.29981559515,23965.06198201216,17451.68248953376\nclimatology,B,2018-07-01,2018,7,lat_menos35_a_menos15,20880,25815.052701854496,-1592.7301985519007,14674.612816835754,22516.243021368893,18808.59412325443\nclimatology,B,2018-07-01,2018,7,lat_maior_igual_menos15,31581,115050.70077726721,9847.23538260255,36820.22044666018,114075.47908158659,31215.50842646724\nclimatology,B,2018-08-01,2018,8,dominio_inteiro,78561,191073.83063440997,7720.256560148438,72034.56551479199,177954.14440592934,67475.78503925544\nclimatology,B,2018-08-01,2018,8,lat_menor_que_menos35,26100,27993.095065988313,543.3867185115814,20163.129323005676,19074.403912930225,17451.68248953376\nclimatology,B,2018-08-01,2018,8,lat_menos35_a_menos15,20880,28424.74023186667,-5315.3154461458325,15624.85904490389,25488.22220843537,18808.59412325443\nclimatology,B,2018-08-01,2018,8,lat_maior_igual_menos15,31581,134655.99533655497,12492.18528778269,36246.577146882424,133391.51828456373,31215.50842646724\nclimatology,B,2018-09-01,2018,9,dominio_inteiro,78561,172694.52392458197,-9109.146667480934,71048.90913102916,157115.23209824745,67475.78503925544\nclimatology,B,2018-09-01,2018,9,lat_menor_que_menos35,26100,39717.6882461183,-9691.652444779873,22839.68607646227,28333.459092542158,17451.68248953376\nclimatology,B,2018-09-01,2018,9,lat_menos35_a_menos15,20880,29848.30904793858,-3273.589461666532,14993.980787877925,26564.180362081264,18808.59412325443\nclimatology,B,2018-09-01,2018,9,lat_maior_igual_menos15,31581,103128.52663052511,3856.0952389654703,33215.24226668896,102217.59264362403,31215.50842646724\nclimatology,B,2018-10-01,2018,10,dominio_inteiro,78561,213796.95990595227,-7756.746550216776,72761.17109129947,203071.7324665811,67475.78503925544\nclimatology,B,2018-10-01,2018,10,lat_menor_que_menos35,26100,20457.522777849066,-1424.07722479105,16071.670528948307,14550.047832236885,17451.68248953376\nclimatology,B,2018-10-01,2018,10,lat_menos35_a_menos15,20880,34463.36792023055,-8666.476190588874,17536.215965915442,30979.296598524463,18808.59412325443\nclimatology,B,2018-10-01,2018,10,lat_maior_igual_menos15,31581,158876.06920787267,2333.8068651631474,39153.284596435726,157542.38803581978,31215.50842646724\nclimatology,B,2018-11-01,2018,11,dominio_inteiro,78561,346397.4397987085,-31739.475838401035,97858.07483000154,333059.88233969884,67475.78503925544\nclimatology,B,2018-11-01,2018,11,lat_menor_que_menos35,26100,26110.61577953532,-10252.127113044262,18551.67908102274,18811.456732303755,17451.68248953376\nclimatology,B,2018-11-01,2018,11,lat_menos35_a_menos15,20880,50305.68859209127,512.3414317381976,20510.037677653454,46308.732853040274,18808.59412325443\nclimatology,B,2018-11-01,2018,11,lat_maior_igual_menos15,31581,269981.13542708184,-21999.69015709497,58796.35807132535,267939.6927543548,31215.50842646724\nclimatology,B,2018-12-01,2018,12,dominio_inteiro,78561,479099.7412714694,16489.449365668464,118593.53882366722,464919.82290487667,67475.78503925544\nclimatology,B,2018-12-01,2018,12,lat_menor_que_menos35,26100,18816.770598785406,4397.475648835301,16866.606430068612,13213.34536623293,17451.68248953376\nclimatology,B,2018-12-01,2018,12,lat_menos35_a_menos15,20880,72383.67653040896,13507.056062028278,26890.70870902692,65560.18674735,18808.59412325443\nclimatology,B,2018-12-01,2018,12,lat_maior_igual_menos15,31581,387899.29414227494,-1415.0823451951146,74836.22368457168,386146.2907912937,31215.50842646724\n', 'research/spatial_unet/evidence/seven_blocks/B/source_calendar.csv': 'time_alvo,limite_emissao,atmosfera,indices,sst,inicializacao_sazonal,ultimo_alvo_treino,disponibilidade_historica_comprovada\n2017-01-01,2016-12-31 23:59:59.999999999,2016-09-01,2016-10-01,2016-11-01,2016-12-01,2016-09-01,False\n2017-02-01,2017-01-31 23:59:59.999999999,2016-10-01,2016-11-01,2016-12-01,2017-01-01,2016-09-01,False\n2017-03-01,2017-02-28 23:59:59.999999999,2016-11-01,2016-12-01,2017-01-01,2017-02-01,2016-09-01,False\n2017-04-01,2017-03-31 23:59:59.999999999,2016-12-01,2017-01-01,2017-02-01,2017-03-01,2016-09-01,False\n2017-05-01,2017-04-30 23:59:59.999999999,2017-01-01,2017-02-01,2017-03-01,2017-04-01,2016-09-01,False\n2017-06-01,2017-05-31 23:59:59.999999999,2017-02-01,2017-03-01,2017-04-01,2017-05-01,2016-09-01,False\n2017-07-01,2017-06-30 23:59:59.999999999,2017-03-01,2017-04-01,2017-05-01,2017-06-01,2016-09-01,False\n2017-08-01,2017-07-31 23:59:59.999999999,2017-04-01,2017-05-01,2017-06-01,2017-07-01,2016-09-01,False\n2017-09-01,2017-08-31 23:59:59.999999999,2017-05-01,2017-06-01,2017-07-01,2017-08-01,2016-09-01,False\n2017-10-01,2017-09-30 23:59:59.999999999,2017-06-01,2017-07-01,2017-08-01,2017-09-01,2016-09-01,False\n2017-11-01,2017-10-31 23:59:59.999999999,2017-07-01,2017-08-01,2017-09-01,2017-10-01,2016-09-01,False\n2017-12-01,2017-11-30 23:59:59.999999999,2017-08-01,2017-09-01,2017-10-01,2017-11-01,2016-09-01,False\n2018-01-01,2017-12-31 23:59:59.999999999,2017-09-01,2017-10-01,2017-11-01,2017-12-01,2016-09-01,False\n2018-02-01,2018-01-31 23:59:59.999999999,2017-10-01,2017-11-01,2017-12-01,2018-01-01,2016-09-01,False\n2018-03-01,2018-02-28 23:59:59.999999999,2017-11-01,2017-12-01,2018-01-01,2018-02-01,2016-09-01,False\n2018-04-01,2018-03-31 23:59:59.999999999,2017-12-01,2018-01-01,2018-02-01,2018-03-01,2016-09-01,False\n2018-05-01,2018-04-30 23:59:59.999999999,2018-01-01,2018-02-01,2018-03-01,2018-04-01,2016-09-01,False\n2018-06-01,2018-05-31 23:59:59.999999999,2018-02-01,2018-03-01,2018-04-01,2018-05-01,2016-09-01,False\n2018-07-01,2018-06-30 23:59:59.999999999,2018-03-01,2018-04-01,2018-05-01,2018-06-01,2016-09-01,False\n2018-08-01,2018-07-31 23:59:59.999999999,2018-04-01,2018-05-01,2018-06-01,2018-07-01,2016-09-01,False\n2018-09-01,2018-08-31 23:59:59.999999999,2018-05-01,2018-06-01,2018-07-01,2018-08-01,2016-09-01,False\n2018-10-01,2018-09-30 23:59:59.999999999,2018-06-01,2018-07-01,2018-08-01,2018-09-01,2016-09-01,False\n2018-11-01,2018-10-31 23:59:59.999999999,2018-07-01,2018-08-01,2018-09-01,2018-10-01,2016-09-01,False\n2018-12-01,2018-11-30 23:59:59.999999999,2018-08-01,2018-09-01,2018-10-01,2018-11-01,2016-09-01,False\n', 'research/spatial_unet/evidence/seven_blocks/B/split.json': '{\n  "historical_publication_vintages_verified": false,\n  "independent_holdout": false,\n  "inner_reference": [\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09",\n    "2012-10",\n    "2012-11",\n    "2012-12",\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06"\n  ],\n  "inner_train": [\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09",\n    "2012-10",\n    "2012-11",\n    "2012-12",\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06"\n  ],\n  "inner_validation": [\n    "2014-10",\n    "2014-11",\n    "2014-12",\n    "2015-01",\n    "2015-02",\n    "2015-03",\n    "2015-04",\n    "2015-05",\n    "2015-06",\n    "2015-07",\n    "2015-08",\n    "2015-09",\n    "2015-10",\n    "2015-11",\n    "2015-12",\n    "2016-01",\n    "2016-02",\n    "2016-03",\n    "2016-04",\n    "2016-05",\n    "2016-06",\n    "2016-07",\n    "2016-08",\n    "2016-09"\n  ],\n  "outer": [\n    "2017-01",\n    "2017-02",\n    "2017-03",\n    "2017-04",\n    "2017-05",\n    "2017-06",\n    "2017-07",\n    "2017-08",\n    "2017-09",\n    "2017-10",\n    "2017-11",\n    "2017-12",\n    "2018-01",\n    "2018-02",\n    "2018-03",\n    "2018-04",\n    "2018-05",\n    "2018-06",\n    "2018-07",\n    "2018-08",\n    "2018-09",\n    "2018-10",\n    "2018-11",\n    "2018-12"\n  ],\n  "train": [\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09",\n    "2012-10",\n    "2012-11",\n    "2012-12",\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06",\n    "2014-07",\n    "2014-08",\n    "2014-09",\n    "2014-10",\n    "2014-11",\n    "2014-12",\n    "2015-01",\n    "2015-02",\n    "2015-03",\n    "2015-04",\n    "2015-05",\n    "2015-06",\n    "2015-07",\n    "2015-08",\n    "2015-09",\n    "2015-10",\n    "2015-11",\n    "2015-12",\n    "2016-01",\n    "2016-02",\n    "2016-03",\n    "2016-04",\n    "2016-05",\n    "2016-06",\n    "2016-07",\n    "2016-08",\n    "2016-09"\n  ]\n}\n', 'research/spatial_unet/evidence/seven_blocks/B/unet/training.json': '{\n  "device": "cuda",\n  "inner_selection": false,\n  "parameter_count": 122945,\n  "selected_epochs": 18,\n  "train_end": "2016-09-01",\n  "train_months": 360,\n  "train_start": "1986-10-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/B/unet/training_history.json': '[\n  {\n    "epoch": 1,\n    "seconds": 21.156687685999714,\n    "training_mse": 2.9878124011887444\n  },\n  {\n    "epoch": 2,\n    "seconds": 21.559255651000058,\n    "training_mse": 2.9246882445282405\n  },\n  {\n    "epoch": 3,\n    "seconds": 20.979700005999803,\n    "training_mse": 2.88786483142111\n  },\n  {\n    "epoch": 4,\n    "seconds": 20.84568666300038,\n    "training_mse": 2.8613802750905353\n  },\n  {\n    "epoch": 5,\n    "seconds": 20.99462336399938,\n    "training_mse": 2.8557185583644444\n  },\n  {\n    "epoch": 6,\n    "seconds": 21.12322513500021,\n    "training_mse": 2.8425669994619156\n  },\n  {\n    "epoch": 7,\n    "seconds": 21.038377572000172,\n    "training_mse": 2.82469196650717\n  },\n  {\n    "epoch": 8,\n    "seconds": 20.975847251999767,\n    "training_mse": 2.817902671628528\n  },\n  {\n    "epoch": 9,\n    "seconds": 21.02240421100032,\n    "training_mse": 2.811295298073027\n  },\n  {\n    "epoch": 10,\n    "seconds": 21.07702205899932,\n    "training_mse": 2.804443675941891\n  },\n  {\n    "epoch": 11,\n    "seconds": 21.093725016999997,\n    "training_mse": 2.8028161075380114\n  },\n  {\n    "epoch": 12,\n    "seconds": 21.031772584000464,\n    "training_mse": 2.7992443011866674\n  },\n  {\n    "epoch": 13,\n    "seconds": 21.014354213999468,\n    "training_mse": 2.796034289068646\n  },\n  {\n    "epoch": 14,\n    "seconds": 21.00559704800071,\n    "training_mse": 2.786867368221283\n  },\n  {\n    "epoch": 15,\n    "seconds": 21.00739377700029,\n    "training_mse": 2.786985679467519\n  },\n  {\n    "epoch": 16,\n    "seconds": 21.01935904900074,\n    "training_mse": 2.778667981094784\n  },\n  {\n    "epoch": 17,\n    "seconds": 21.062115311000525,\n    "training_mse": 2.7789894580841064\n  },\n  {\n    "epoch": 18,\n    "seconds": 21.065742275999582,\n    "training_mse": 2.768290713098314\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/C/baseline_check.json': '{\n  "reproduced": true,\n  "rmse_archived": 1.7408015775162955,\n  "rmse_current": 1.7408015775162955,\n  "tolerance": 1e-06\n}\n', 'research/spatial_unet/evidence/seven_blocks/C/complete.json': '{\n  "files": {\n    "baseline_check.json": "7f122d971f5a90098d901fe995d1a7c33700c97be5ebea93d3668d9e584207ad",\n    "hybrid/arvores_multissistema.txt": "f8ccd61ddcf38d26b29fcdb537ca5fb124673d654e23fb21798009eee1845fcf",\n    "hybrid/arvores_seas5.txt": "8bb0696db3e9fce2a677bfac639286fcf0785fd0dd3f773897f3b4b1ef067245",\n    "hybrid/clima_atmosfera_indices.npz": "642867f0f3cf4c122568556a1208c78ee17544dd9393dfd4f3535e89b9e18f98",\n    "hybrid/clima_cfsv2.nc": "e9bd6b85712271ad9c7a397ef415986736a27f1126ea7a1f4623cc8e9827b851",\n    "hybrid/clima_chuva.nc": "029dfeada18076ec0ceda34d2216a963f24baba0d41a5dad94020eebf3bb0583",\n    "hybrid/clima_seas5.nc": "273a8cd0bd6fc1de48e5b33cf440ec98aea8fbbe266ff4886397e748e1c41406",\n    "hybrid/fit.json": "577ea29297b3fb26e045bf08192d2bf19bbe035e1b7faad0190be3f2f0a1a084",\n    "hybrid/ridge.npz": "fbc8b00a04e6c1882f5e05ba1af9f5b5755c74d835f42b62a48c6cb7d539a32d",\n    "inner/context.json": "8c3edfeeee8b38f5eaa4c1519cbab6587386694f678f9020320a84b385f5f51a",\n    "inner/network.pt": "ddd4255b52fecd8247ffb2a3d218e880e9cb1c5705e99bb1b5202079f5d89e53",\n    "inner/rainfall_climatology.nc": "28ed7423e129d0133a1b2490a1cf86da59fa4b90db27d9675b20c3fc71caabee",\n    "inner/references.npz": "0cbe4d64fc45be2c713115bcf406868951a93ea48374a95fb7764a2993ccbb9a",\n    "inner/scaling.npz": "77902be3dde8d9d5648d12c0915ff85df886bed29b10a6c8f4b07a2ac3589c9a",\n    "inner/training.json": "9d8fb4da86c2dcc78793842be18bd8e5a4f7ff5f3e30d8fc3511c54bae7e1a31",\n    "inner/training_history.json": "588792d87882e50dba74e53f279ff5006c71e4b3a29ebb47a1ee9ec0272b42f9",\n    "monthly_metrics.csv": "7e1e251ba0121ca2f2f0e610aa1ed25b964cd0b82cbdb1c01947218063ee4eff",\n    "predictions.nc": "f232d723a3f53c2472df5bd53dcff3b26fa6501b7e6da1be8aa6df7b93bb89d0",\n    "source_calendar.csv": "546f288ede5e7bb85086c835b6f94bde96f77bf14f4fe2259a1933d1cec609a4",\n    "split.json": "913001edf8a679143de0b4a95bf96f57d359eb22a2183a0a7e4f4d2d25e02aba",\n    "unet/context.json": "86d1acac07bc68d5ecbd83de97e50940fd1c799fac695914c821d3fc9efe8255",\n    "unet/network.pt": "868ad8ef27684f6c222c05a41044f5a7907736db9f5b0b3359c970a36e4163be",\n    "unet/rainfall_climatology.nc": "7674a9178da2e636b77821b019956382b415ed51d4be118a27f47756ae74b6ba",\n    "unet/references.npz": "d168e83201ab65f06b3def3c1de0c27a8aeedca0f4428d2a583eeb6fc0bb9b15",\n    "unet/scaling.npz": "1d7abe7ac491bdb1ab7bc861e1059508a5c6cd6691e7be54961c32f86bd4ba39",\n    "unet/training.json": "9f08d5f86d56687d2fc626545a93d151f602216d68bcf9a9919bf12ed7c47e52",\n    "unet/training_history.json": "b5f1f29b8884f21e05eec676a5ac873c8916c46ab1f71412b32613217aa128e1"\n  },\n  "signature": "02d1ffc0a7205f2bf372673d337cf5daa95f125d43a95836b953c10ae101244c"\n}\n', 'research/spatial_unet/evidence/seven_blocks/C/inner/training.json': '{\n  "device": "cuda",\n  "inner_selection": true,\n  "parameter_count": 122945,\n  "selected_epochs": 9,\n  "train_end": "2016-06-01",\n  "train_months": 333,\n  "train_start": "1988-10-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/C/inner/training_history.json': '[\n  {\n    "epoch": 1,\n    "inner_rmse": 1.7591281429763135,\n    "seconds": 19.728291931000058,\n    "training_mse": 3.0378665387093484\n  },\n  {\n    "epoch": 2,\n    "inner_rmse": 1.7614023095414766,\n    "seconds": 19.96156579000035,\n    "training_mse": 2.97100872392053\n  },\n  {\n    "epoch": 3,\n    "inner_rmse": 1.7495467349799434,\n    "seconds": 19.37741855599961,\n    "training_mse": 2.9487219909289935\n  },\n  {\n    "epoch": 4,\n    "inner_rmse": 1.7534674301755717,\n    "seconds": 19.315887695999663,\n    "training_mse": 2.9263696706330813\n  },\n  {\n    "epoch": 5,\n    "inner_rmse": 1.7521995461855266,\n    "seconds": 19.40916002100039,\n    "training_mse": 2.9047985814355157\n  },\n  {\n    "epoch": 6,\n    "inner_rmse": 1.7533214684478964,\n    "seconds": 19.552338843999678,\n    "training_mse": 2.886346938731792\n  },\n  {\n    "epoch": 7,\n    "inner_rmse": 1.7487585996445478,\n    "seconds": 19.443191819000276,\n    "training_mse": 2.8695634134538897\n  },\n  {\n    "epoch": 8,\n    "inner_rmse": 1.753961255174765,\n    "seconds": 19.367101862999334,\n    "training_mse": 2.856911823198244\n  },\n  {\n    "epoch": 9,\n    "inner_rmse": 1.7484659404446357,\n    "seconds": 19.369313098999555,\n    "training_mse": 2.8483661284317843\n  },\n  {\n    "epoch": 10,\n    "inner_rmse": 1.7569722289524852,\n    "seconds": 19.467225366000093,\n    "training_mse": 2.836138508341334\n  },\n  {\n    "epoch": 11,\n    "inner_rmse": 1.7516601454551886,\n    "seconds": 19.501485456999944,\n    "training_mse": 2.824634332556624\n  },\n  {\n    "epoch": 12,\n    "inner_rmse": 1.7592010469820893,\n    "seconds": 19.47231639599977,\n    "training_mse": 2.8231138552989328\n  },\n  {\n    "epoch": 13,\n    "inner_rmse": 1.7614668558096813,\n    "seconds": 19.521704615999624,\n    "training_mse": 2.8219792270087622\n  },\n  {\n    "epoch": 14,\n    "inner_rmse": 1.754532725015126,\n    "seconds": 19.4558247519999,\n    "training_mse": 2.812719688043222\n  },\n  {\n    "epoch": 15,\n    "inner_rmse": 1.7518152860474476,\n    "seconds": 19.43291339299958,\n    "training_mse": 2.8014593589771257\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/C/monthly_metrics.csv': 'modelo,bloco,mes_alvo,ano,mes,regiao,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area\nunet,C,2019-01-01,2019,1,dominio_inteiro,78561,301973.2279745722,20905.250723651145,91033.32802750776,282874.7817978409,67475.78503925544\nunet,C,2019-01-01,2019,1,lat_menor_que_menos35,26100,12689.29570162352,-933.5228843986988,12920.74570466578,9105.543556163788,17451.68248953376\nunet,C,2019-01-01,2019,1,lat_menos35_a_menos15,20880,152443.47296267317,15723.373372922186,36670.60157187143,137993.48701347836,18808.59412325443\nunet,C,2019-01-01,2019,1,lat_maior_igual_menos15,31581,136840.45931027553,6115.400235127658,41441.98075097054,135775.7512281987,31215.50842646724\nunet,C,2019-02-01,2019,2,dominio_inteiro,78561,182858.0519069073,7155.299069399014,74506.56770906784,170104.60793326102,67475.78503925544\nunet,C,2019-02-01,2019,2,lat_menor_que_menos35,26100,23839.984954542302,3771.3135499432683,18689.563171379268,16726.145805595515,17451.68248953376\nunet,C,2019-02-01,2019,2,lat_menos35_a_menos15,20880,48539.58651197055,4420.666228195652,21273.675528382882,43485.13153074495,18808.59412325443\nunet,C,2019-02-01,2019,2,lat_maior_igual_menos15,31581,110478.48044039447,-1036.6807087399065,34543.329009305686,109893.33059692054,31215.50842646724\nunet,C,2019-03-01,2019,3,dominio_inteiro,78561,202779.32232830382,12332.46943497099,78089.93122257479,192935.59184517077,67475.78503925544\nunet,C,2019-03-01,2019,3,lat_menor_que_menos35,26100,16951.855876174694,5559.076595887542,16080.735479518771,12368.397272130487,17451.68248953376\nunet,C,2019-03-01,2019,3,lat_menos35_a_menos15,20880,55198.13144537837,2048.825755123049,22534.935528744012,50505.553809642544,18808.59412325443\nunet,C,2019-03-01,2019,3,lat_maior_igual_menos15,31581,130629.33500675077,4724.567083960399,39474.260214312,130061.64076339772,31215.50842646724\nunet,C,2019-04-01,2019,4,dominio_inteiro,78561,227909.56051365228,-3465.7634289134294,78153.17357497104,214691.710665038,67475.78503925544\nunet,C,2019-04-01,2019,4,lat_menor_que_menos35,26100,25147.391001054653,4622.904065340757,19367.97638168931,18249.35811959269,17451.68248953376\nunet,C,2019-04-01,2019,4,lat_menos35_a_menos15,20880,49450.88192391075,-2701.4781957268715,20375.069316484034,43575.804448200004,18808.59412325443\nunet,C,2019-04-01,2019,4,lat_maior_igual_menos15,31581,153311.28758868683,-5387.189298527315,38410.12787679769,152866.5480972453,31215.50842646724\nunet,C,2019-05-01,2019,5,dominio_inteiro,78561,302285.70406586025,-4720.007640993223,88747.89789971523,291424.25944446534,67475.78503925544\nunet,C,2019-05-01,2019,5,lat_menor_que_menos35,26100,16189.46026144694,-1660.1822972595692,15116.748966544867,11137.030677604827,17451.68248953376\nunet,C,2019-05-01,2019,5,lat_menos35_a_menos15,20880,39098.24282092716,-5980.29139418155,17587.285957194865,35044.19400599772,18808.59412325443\nunet,C,2019-05-01,2019,5,lat_maior_igual_menos15,31581,246998.00098348613,2920.466050447896,56043.8629759755,245243.03476086282,31215.50842646724\nunet,C,2019-06-01,2019,6,dominio_inteiro,78561,246429.8641827259,4373.4192004054785,75755.72658364475,234681.66390384853,67475.78503925544\nunet,C,2019-06-01,2019,6,lat_menor_que_menos35,26100,21709.82674892698,-7299.568296611309,16085.360145926476,16065.43495576819,17451.68248953376\nunet,C,2019-06-01,2019,6,lat_menos35_a_menos15,20880,38414.32721478386,9204.677455432713,17455.828534163535,33229.926924047155,18808.59412325443\nunet,C,2019-06-01,2019,6,lat_maior_igual_menos15,31581,186305.71021901505,2468.3100415840745,42214.53790355474,185386.30202403315,31215.50842646724\nunet,C,2019-07-01,2019,7,dominio_inteiro,78561,195068.43405755615,5878.580750964582,67642.92451225221,185237.9390846424,67475.78503925544\nunet,C,2019-07-01,2019,7,lat_menor_que_menos35,26100,20224.258724382795,-568.4461459219456,16070.169848054647,14711.014543729209,17451.68248953376\nunet,C,2019-07-01,2019,7,lat_menos35_a_menos15,20880,25368.23198598932,6013.76428125985,14639.395540511236,22146.055785596087,18808.59412325443\nunet,C,2019-07-01,2019,7,lat_maior_igual_menos15,31581,149475.943347184,433.26261562667787,36933.35912368633,148380.86875531712,31215.50842646724\nunet,C,2019-08-01,2019,8,dominio_inteiro,78561,160894.6612522762,24805.55314274784,64252.306289090775,152213.5614973358,67475.78503925544\nunet,C,2019-08-01,2019,8,lat_menor_que_menos35,26100,20378.307753816873,4442.387547761202,16667.793918579817,14876.960175073986,17451.68248953376\nunet,C,2019-08-01,2019,8,lat_menos35_a_menos15,20880,17750.54806353226,9102.43151262682,13445.249079289846,15542.796953751582,18808.59412325443\nunet,C,2019-08-01,2019,8,lat_maior_igual_menos15,31581,122765.80543492705,11260.73408235982,34139.26329122111,121793.80436851025,31215.50842646724\nunet,C,2019-09-01,2019,9,dominio_inteiro,78561,157485.35750858707,6231.4036952396855,67328.0487259524,147594.30418949126,67475.78503925544\nunet,C,2019-09-01,2019,9,lat_menor_que_menos35,26100,21131.645186493864,-1301.6706279218197,16870.42052039504,15447.111456182942,17451.68248953376\nunet,C,2019-09-01,2019,9,lat_menos35_a_menos15,20880,23530.668681688192,4751.491234777495,14522.945685679093,20634.75625831444,18808.59412325443\nunet,C,2019-09-01,2019,9,lat_maior_igual_menos15,31581,112823.04364040503,2781.58308838401,35934.68251987826,111512.43647499385,31215.50842646724\nunet,C,2019-10-01,2019,10,dominio_inteiro,78561,194228.63060286065,25860.198613252956,73948.90859512053,183408.0105108217,67475.78503925544\nunet,C,2019-10-01,2019,10,lat_menor_que_menos35,26100,15140.39291795564,5649.438559025526,12732.124586075544,10686.120540076885,17451.68248953376\nunet,C,2019-10-01,2019,10,lat_menos35_a_menos15,20880,46829.12719553636,11867.85225981893,20617.51747067459,41701.12155400929,18808.59412325443\nunet,C,2019-10-01,2019,10,lat_maior_igual_menos15,31581,132259.11048936867,8342.9077944085,40599.2665383704,131020.76841673555,31215.50842646724\nunet,C,2019-11-01,2019,11,dominio_inteiro,78561,239731.3799348943,18998.33549076598,82294.70024021436,227612.1581997355,67475.78503925544\nunet,C,2019-11-01,2019,11,lat_menor_que_menos35,26100,23985.509037581924,3076.824264675379,18461.74425584078,16562.952998536184,17451.68248953376\nunet,C,2019-11-01,2019,11,lat_menos35_a_menos15,20880,25524.72688704094,6936.17429409083,15615.552101411857,23162.061886385563,18808.59412325443\nunet,C,2019-11-01,2019,11,lat_maior_igual_menos15,31581,190221.14401027144,8985.336931999773,48217.40388296172,187887.14331481376,31215.50842646724\nunet,C,2019-12-01,2019,12,dominio_inteiro,78561,252515.83352614162,11053.941660713404,86741.51136180758,238259.0208462099,67475.78503925544\nunet,C,2019-12-01,2019,12,lat_menor_que_menos35,26100,31843.16863235898,-4686.092725560069,17548.28347109258,23932.716230765338,17451.68248953376\nunet,C,2019-12-01,2019,12,lat_menos35_a_menos15,20880,50706.87017229054,11991.516199445352,21341.411882197484,45983.38915389369,18808.59412325443\nunet,C,2019-12-01,2019,12,lat_maior_igual_menos15,31581,169965.79472149213,3748.5181868281215,47851.81600851752,168342.91546155082,31215.50842646724\nunet,C,2020-01-01,2020,1,dominio_inteiro,78561,260935.7474483561,10965.855986076058,84257.16101891978,246361.16271652584,67475.78503925544\nunet,C,2020-01-01,2020,1,lat_menor_que_menos35,26100,12723.202139482604,-2906.3710008449852,13769.274933818728,8717.34852540595,17451.68248953376\nunet,C,2020-01-01,2020,1,lat_menos35_a_menos15,20880,107801.03089068047,-4448.955951110809,28594.753172677592,98078.193537641,18808.59412325443\nunet,C,2020-01-01,2020,1,lat_maior_igual_menos15,31581,140411.51441819302,18321.182938031852,41893.13291242346,139565.62065347887,31215.50842646724\nunet,C,2020-02-01,2020,2,dominio_inteiro,78561,201983.4129370108,25935.70246918127,78299.4656252414,189663.75074244765,67475.78503925544\nunet,C,2020-02-01,2020,2,lat_menor_que_menos35,26100,17800.457334369403,2082.066530480981,15176.6805177778,12675.527440977257,17451.68248953376\nunet,C,2020-02-01,2020,2,lat_menos35_a_menos15,20880,65753.3181245501,2137.9149307347834,23669.43329044804,59307.5700890871,18808.59412325443\nunet,C,2020-02-01,2020,2,lat_maior_igual_menos15,31581,118429.6374780913,21715.721007965505,39453.35181701556,117680.6532123833,31215.50842646724\nunet,C,2020-03-01,2020,3,dominio_inteiro,78561,246801.6276229199,37507.40986313112,90369.05605696328,231633.9000790366,67475.78503925544\nunet,C,2020-03-01,2020,3,lat_menor_que_menos35,26100,28070.35371427458,12531.88315217197,20208.546378538013,20128.352153357722,17451.68248953376\nunet,C,2020-03-01,2020,3,lat_menos35_a_menos15,20880,67403.38561362775,11019.13175484538,26348.718409210443,61116.11057634097,18808.59412325443\nunet,C,2020-03-01,2020,3,lat_maior_igual_menos15,31581,151327.8882950176,13956.39495611377,43811.791269214824,150389.43734933788,31215.50842646724\nunet,C,2020-04-01,2020,4,dominio_inteiro,78561,351439.1522901953,17812.8039734466,102005.73489603493,334167.6329329621,67475.78503925544\nunet,C,2020-04-01,2020,4,lat_menor_que_menos35,26100,42271.58718920463,2048.0740590393543,23909.423951655626,30184.717589776414,17451.68248953376\nunet,C,2020-04-01,2020,4,lat_menos35_a_menos15,20880,40031.69705806422,8419.76169539988,19446.02291046083,35963.82071807067,18808.59412325443\nunet,C,2020-04-01,2020,4,lat_maior_igual_menos15,31581,269135.8680429264,7344.968219007365,58650.28803391848,268019.09462511505,31215.50842646724\nunet,C,2020-05-01,2020,5,dominio_inteiro,78561,345306.25480116985,19150.017561445013,99749.47563964315,329290.7421898217,67475.78503925544\nunet,C,2020-05-01,2020,5,lat_menor_que_menos35,26100,44695.513293779484,-4995.5330775380135,23598.62782484293,32564.691914287916,17451.68248953376\nunet,C,2020-05-01,2020,5,lat_menos35_a_menos15,20880,21461.2529283763,3798.7030003517866,15336.929036810994,18949.004456654686,18808.59412325443\nunet,C,2020-05-01,2020,5,lat_maior_igual_menos15,31581,279149.4885790141,20346.84763863124,60813.91877798922,277777.04581887915,31215.50842646724\nunet,C,2020-06-01,2020,6,dominio_inteiro,78561,208287.78679630923,-1124.1319643482566,76828.39076467603,190933.64258366678,67475.78503925544\nunet,C,2020-06-01,2020,6,lat_menor_que_menos35,26100,50316.69163069077,-5621.967046990991,25612.251721039414,36284.10799688219,17451.68248953376\nunet,C,2020-06-01,2020,6,lat_menos35_a_menos15,20880,17720.513426354497,-1950.9742219224572,11924.888088800013,15276.410761049661,18808.59412325443\nunet,C,2020-06-01,2020,6,lat_maior_igual_menos15,31581,140250.58173926393,6448.809304565191,39291.25095483661,139373.12382573492,31215.50842646724\nunet,C,2020-07-01,2020,7,dominio_inteiro,78561,233967.4257218103,14694.607725575566,83992.87533052266,210676.2924242658,67475.78503925544\nunet,C,2020-07-01,2020,7,lat_menor_que_menos35,26100,67853.74057176622,-5460.914539813995,30577.341741383076,47544.88727139041,17451.68248953376\nunet,C,2020-07-01,2020,7,lat_menos35_a_menos15,20880,14336.39058470177,1753.374348178506,10562.596738651395,12378.549130932057,18808.59412325443\nunet,C,2020-07-01,2020,7,lat_maior_igual_menos15,31581,151777.2945653423,18402.147917211056,42852.936850488186,150752.85602194327,31215.50842646724\nunet,C,2020-08-01,2020,8,dominio_inteiro,78561,186328.01738961093,5521.07774328813,67112.00241834298,176228.75064912785,67475.78503925544\nunet,C,2020-08-01,2020,8,lat_menor_que_menos35,26100,20368.223801897984,-5680.6845523417,17674.52337655425,13902.561822352516,17451.68248953376\nunet,C,2020-08-01,2020,8,lat_menos35_a_menos15,20880,25859.080866387652,-1260.6626853495836,13869.868450984359,23251.12715610208,18808.59412325443\nunet,C,2020-08-01,2020,8,lat_maior_igual_menos15,31581,140100.7127213253,12462.424980979413,35567.61059080437,139075.06167067325,31215.50842646724\nunet,C,2020-09-01,2020,9,dominio_inteiro,78561,129505.71977935328,31722.228866172954,63951.19924190454,122791.80951356245,67475.78503925544\nunet,C,2020-09-01,2020,9,lat_menor_que_menos35,26100,10980.926487777062,2572.1610728502274,13027.313111305237,7725.192508984139,17451.68248953376\nunet,C,2020-09-01,2020,9,lat_menos35_a_menos15,20880,24596.11148788226,10849.274826439098,14632.838593663648,22208.36709445513,18808.59412325443\nunet,C,2020-09-01,2020,9,lat_maior_igual_menos15,31581,93928.68180369395,18300.79296688363,36291.04753693566,92858.24991012315,31215.50842646724\nunet,C,2020-10-01,2020,10,dominio_inteiro,78561,213976.14285126893,35547.96956767887,82964.58676495403,202281.59335811494,67475.78503925544\nunet,C,2020-10-01,2020,10,lat_menor_que_menos35,26100,15772.886301676142,5376.5599500238895,14633.116991788149,11460.90620971394,17451.68248953376\nunet,C,2020-10-01,2020,10,lat_menos35_a_menos15,20880,63810.86096407965,3594.8305506296456,23336.716698866338,57800.89433124164,18808.59412325443\nunet,C,2020-10-01,2020,10,lat_maior_igual_menos15,31581,134392.39558551315,26576.579067025334,44994.753074299544,133019.79281715935,31215.50842646724\nunet,C,2020-11-01,2020,11,dominio_inteiro,78561,412762.65620438295,11507.53330987133,108555.11779973842,398101.9024981921,67475.78503925544\nunet,C,2020-11-01,2020,11,lat_menor_que_menos35,26100,17414.484114652943,11073.822181284428,15293.06030189991,12794.29583720615,17451.68248953376\nunet,C,2020-11-01,2020,11,lat_menos35_a_menos15,20880,73541.41787217426,11601.80590592511,25803.85832396336,67743.00913375251,18808.59412325443\nunet,C,2020-11-01,2020,11,lat_maior_igual_menos15,31581,321806.7542175557,-11168.094777338207,67458.19917387515,317564.5975272334,31215.50842646724\nunet,C,2020-12-01,2020,12,dominio_inteiro,78561,292139.90731297684,13343.57329193689,91720.12979806028,279377.4023197039,67475.78503925544\nunet,C,2020-12-01,2020,12,lat_menor_que_menos35,26100,11087.48222393098,283.2961043715477,12818.507601559162,7838.021322162543,17451.68248953376\nunet,C,2020-12-01,2020,12,lat_menos35_a_menos15,20880,80214.1135941443,13294.422755332664,27985.39676634781,72648.77716373504,18808.59412325443\nunet,C,2020-12-01,2020,12,lat_maior_igual_menos15,31581,200838.3114949015,-234.14556776732206,50916.22543015331,198890.60383380635,31215.50842646724\nhybrid,C,2019-01-01,2019,1,dominio_inteiro,78561,305527.424472293,20637.29276329605,91849.6664797912,285628.03164088656,67475.78503925544\nhybrid,C,2019-01-01,2019,1,lat_menor_que_menos35,26100,14491.45141902028,-2785.197181173273,13957.2942695575,10299.615880669564,17451.68248953376\nhybrid,C,2019-01-01,2019,1,lat_menos35_a_menos15,20880,152815.46696964375,15305.79550192257,36169.12292823992,138237.82316296856,18808.59412325443\nhybrid,C,2019-01-01,2019,1,lat_maior_igual_menos15,31581,138220.506083629,8116.694442546748,41723.249281993776,137090.59259724844,31215.50842646724\nhybrid,C,2019-02-01,2019,2,dominio_inteiro,78561,196049.29102673833,11586.95865008859,76431.57896763817,183892.42787763604,67475.78503925544\nhybrid,C,2019-02-01,2019,2,lat_menor_que_menos35,26100,22285.5252910379,3265.3422831089815,18072.7825650587,15702.2178258068,17451.68248953376\nhybrid,C,2019-02-01,2019,2,lat_menos35_a_menos15,20880,48193.55931423087,3350.1822609831943,20889.73996186828,43195.02464599945,18808.59412325443\nhybrid,C,2019-02-01,2019,2,lat_maior_igual_menos15,31581,125570.20642146953,4971.434105996413,37469.05644071119,124995.18540582982,31215.50842646724\nhybrid,C,2019-03-01,2019,3,dominio_inteiro,78561,242935.0277748423,13105.44338357593,83958.70259482725,232852.48182619704,67475.78503925544\nhybrid,C,2019-03-01,2019,3,lat_menor_que_menos35,26100,16582.686771522756,5000.425071933133,15752.427137942464,12080.88380986413,17451.68248953376\nhybrid,C,2019-03-01,2019,3,lat_menos35_a_menos15,20880,58974.713405715636,-333.7442281741162,22885.31809973428,54021.11100912736,18808.59412325443\nhybrid,C,2019-03-01,2019,3,lat_maior_igual_menos15,31581,167377.62759760392,8438.762539816915,45320.95735715051,166750.48700720555,31215.50842646724\nhybrid,C,2019-04-01,2019,4,dominio_inteiro,78561,243351.7548442327,-4250.70707369427,79529.95221066364,230632.3582836526,67475.78503925544\nhybrid,C,2019-04-01,2019,4,lat_menor_que_menos35,26100,24332.405104451347,4789.496192541841,19086.992254159995,17675.023785256722,17451.68248953376\nhybrid,C,2019-04-01,2019,4,lat_menos35_a_menos15,20880,47540.30381682524,-3180.2413607387793,19934.131167930875,41931.69337436039,18808.59412325443\nhybrid,C,2019-04-01,2019,4,lat_maior_igual_menos15,31581,171479.04592295605,-5859.961905497332,40508.82878857276,171025.64112403552,31215.50842646724\nhybrid,C,2019-05-01,2019,5,dominio_inteiro,78561,271564.7990988635,-994.546184271012,84755.33728044937,261139.3615648005,67475.78503925544\nhybrid,C,2019-05-01,2019,5,lat_menor_que_menos35,26100,15691.68583628834,117.5609733308236,14786.895819346695,10895.895662545341,17451.68248953376\nhybrid,C,2019-05-01,2019,5,lat_menos35_a_menos15,20880,39661.040639879495,-5190.472694273971,17455.831506074705,35553.37335063245,18808.59412325443\nhybrid,C,2019-05-01,2019,5,lat_maior_igual_menos15,31581,216212.0726226957,4078.365536672133,52512.609955027976,214690.09255162266,31215.50842646724\nhybrid,C,2019-06-01,2019,6,dominio_inteiro,78561,248401.60953884848,4800.824122758822,75333.33070105372,236722.90881930484,67475.78503925544\nhybrid,C,2019-06-01,2019,6,lat_menor_que_menos35,26100,21148.72107561997,-6404.4431835615405,16037.913419846724,15585.702650027226,17451.68248953376\nhybrid,C,2019-06-01,2019,6,lat_menos35_a_menos15,20880,38730.35934001242,10377.50388843124,17819.78542447277,33541.33447187407,18808.59412325443\nhybrid,C,2019-06-01,2019,6,lat_maior_igual_menos15,31581,188522.5291232161,827.7634178891226,41475.63185673421,187595.87169740355,31215.50842646724\nhybrid,C,2019-07-01,2019,7,dominio_inteiro,78561,178932.79951629016,9677.827313657974,65785.85962739558,169603.54948668752,67475.78503925544\nhybrid,C,2019-07-01,2019,7,lat_menor_que_menos35,26100,19828.022457593797,-569.0100420884175,16073.224668038823,14320.772528445785,17451.68248953376\nhybrid,C,2019-07-01,2019,7,lat_menos35_a_menos15,20880,22426.07339787508,5056.829761822079,13897.444531416575,19597.75532172893,18808.59412325443\nhybrid,C,2019-07-01,2019,7,lat_maior_igual_menos15,31581,136678.70366082125,5190.007593924312,35815.19042794018,135685.02163651277,31215.50842646724\nhybrid,C,2019-08-01,2019,8,dominio_inteiro,78561,171308.01070478276,28167.196604578,66233.37559141824,162002.5399634442,67475.78503925544\nhybrid,C,2019-08-01,2019,8,lat_menor_que_menos35,26100,21629.575460494176,5132.813006180798,17084.491977598118,15754.335617217803,17451.68248953376\nhybrid,C,2019-08-01,2019,8,lat_menos35_a_menos15,20880,19209.559476120245,10023.196969892642,13990.881035022634,16852.57320236224,18808.59412325443\nhybrid,C,2019-08-01,2019,8,lat_maior_igual_menos15,31581,130468.87576816833,13011.186628504558,35158.00257879749,129395.63114386413,31215.50842646724\nhybrid,C,2019-09-01,2019,9,dominio_inteiro,78561,158945.1515172635,4481.348692194959,66792.65241091602,148755.5584961497,67475.78503925544\nhybrid,C,2019-09-01,2019,9,lat_menor_que_menos35,26100,22211.737528601596,-1034.0094288270543,17057.326084570963,16374.757851348553,17451.68248953376\nhybrid,C,2019-09-01,2019,9,lat_menos35_a_menos15,20880,24999.546184687257,4289.496764063412,14919.720309254912,21950.535003855133,18808.59412325443\nhybrid,C,2019-09-01,2019,9,lat_maior_igual_menos15,31581,111733.86780397466,1225.8613569586005,34815.60601709016,110430.265640946,31215.50842646724\nhybrid,C,2019-10-01,2019,10,dominio_inteiro,78561,180010.30436482935,21655.09996088866,71912.95485931632,169849.43855233787,67475.78503925544\nhybrid,C,2019-10-01,2019,10,lat_menor_que_menos35,26100,13467.046189688854,5529.8138372551875,12661.53022818226,9435.501967064436,17451.68248953376\nhybrid,C,2019-10-01,2019,10,lat_menos35_a_menos15,20880,46889.706678054004,12126.532557193916,20751.180152738845,41837.82430458284,18808.59412325443\nhybrid,C,2019-10-01,2019,10,lat_maior_igual_menos15,31581,119653.5514970865,3998.7535664395564,38500.24447839522,118576.11228069055,31215.50842646724\nhybrid,C,2019-11-01,2019,11,dominio_inteiro,78561,238059.51778050806,10028.097856087246,81734.2063460761,226298.486673225,67475.78503925544\nhybrid,C,2019-11-01,2019,11,lat_menor_que_menos35,26100,23773.42434401215,1334.0539993656782,17435.273875084327,16446.634541555402,17451.68248953376\nhybrid,C,2019-11-01,2019,11,lat_menos35_a_menos15,20880,25348.32397290054,7129.596996730279,15330.663226605375,23122.746322465522,18808.59412325443\nhybrid,C,2019-11-01,2019,11,lat_maior_igual_menos15,31581,188937.76946359535,1564.4468599912898,48968.2692443864,186729.10580920405,31215.50842646724\nhybrid,C,2019-12-01,2019,12,dominio_inteiro,78561,261457.21862657112,8318.965465619087,87777.2808192023,247258.84547342168,67475.78503925544\nhybrid,C,2019-12-01,2019,12,lat_menor_que_menos35,26100,30895.273105246815,-5248.717753167193,17232.721748089687,23194.610880720353,17451.68248953376\nhybrid,C,2019-12-01,2019,12,lat_menos35_a_menos15,20880,52457.345286091804,13257.660725958105,21728.681513160554,47579.29239296046,18808.59412325443\nhybrid,C,2019-12-01,2019,12,lat_maior_igual_menos15,31581,178104.6002352325,310.02249282817684,48815.87755795205,176484.94219974085,31215.50842646724\nhybrid,C,2020-01-01,2020,1,dominio_inteiro,78561,258454.62113865514,11791.286481516345,82340.75037982195,244245.5035609148,67475.78503925544\nhybrid,C,2020-01-01,2020,1,lat_menor_que_menos35,26100,11215.122923019675,-840.2303273249265,12750.678608893108,7765.891494681743,17451.68248953376\nhybrid,C,2020-01-01,2020,1,lat_menos35_a_menos15,20880,109264.5668392448,-4749.841720067204,28139.814579564496,99306.24280204161,18808.59412325443\nhybrid,C,2020-01-01,2020,1,lat_maior_igual_menos15,31581,137974.93137639068,17381.35852890847,41450.25719136434,137173.36926419142,31215.50842646724\nhybrid,C,2020-02-01,2020,2,dominio_inteiro,78561,202495.2261128883,17075.461311018727,77531.9010795569,189522.60116036807,67475.78503925544\nhybrid,C,2020-02-01,2020,2,lat_menor_que_menos35,26100,17805.13932364529,1475.8198889229668,15133.042977980987,12731.742624722148,17451.68248953376\nhybrid,C,2020-02-01,2020,2,lat_menos35_a_menos15,20880,75240.47104981416,-1503.5076874814094,24417.738496215658,67941.12848872374,18808.59412325443\nhybrid,C,2020-02-01,2020,2,lat_maior_igual_menos15,31581,109449.61573942883,17103.149109577167,37981.119605360254,108849.73004692221,31215.50842646724\nhybrid,C,2020-03-01,2020,3,dominio_inteiro,78561,245890.5569677961,26891.486073008302,89449.74665598052,231392.03552291828,67475.78503925544\nhybrid,C,2020-03-01,2020,3,lat_menor_que_menos35,26100,24385.962780980066,10910.13848023798,19210.452463772304,17716.093987509634,17451.68248953376\nhybrid,C,2020-03-01,2020,3,lat_menos35_a_menos15,20880,72744.39826114608,9198.322167221775,27335.964824039438,65907.40408422904,18808.59412325443\nhybrid,C,2020-03-01,2020,3,lat_maior_igual_menos15,31581,148760.19592566992,6783.025425548549,42903.32936816877,147768.5374511796,31215.50842646724\nhybrid,C,2020-04-01,2020,4,dominio_inteiro,78561,341374.13390057685,11061.791338608129,99322.10579985348,325662.8815393138,67475.78503925544\nhybrid,C,2020-04-01,2020,4,lat_menor_que_menos35,26100,35908.40312369319,2737.6691158478543,22123.43991358847,25650.453735422398,17451.68248953376\nhybrid,C,2020-04-01,2020,4,lat_menos35_a_menos15,20880,42813.36575163655,10484.120780022591,20159.082775479605,38491.57250741284,18808.59412325443\nhybrid,C,2020-04-01,2020,4,lat_maior_igual_menos15,31581,262652.36502524716,-2159.9985572623177,57039.583110785396,261520.85529647858,31215.50842646724\nhybrid,C,2020-05-01,2020,5,dominio_inteiro,78561,325679.6732899968,16072.956665370479,96172.48564656387,308900.54229411006,67475.78503925544\nhybrid,C,2020-05-01,2020,5,lat_menor_que_menos35,26100,47227.45494868108,-3988.9330835447427,23636.94984717056,34647.1457577887,17451.68248953376\nhybrid,C,2020-05-01,2020,5,lat_menos35_a_menos15,20880,23728.44620717924,7998.005158255606,15911.87949181813,20833.302844993035,18808.59412325443\nhybrid,C,2020-05-01,2020,5,lat_maior_igual_menos15,31581,254723.77213413647,12063.884590659614,56623.65630757519,253420.0936913283,31215.50842646724\nhybrid,C,2020-06-01,2020,6,dominio_inteiro,78561,188309.3138567385,-4760.941016771399,73078.82250146965,173844.7174468825,67475.78503925544\nhybrid,C,2020-06-01,2020,6,lat_menor_que_menos35,26100,39782.789577620555,-6208.666141321222,22627.7552312951,28554.009317189928,17451.68248953376\nhybrid,C,2020-06-01,2020,6,lat_menos35_a_menos15,20880,18059.29527357504,-558.9460373880328,12480.200020251406,15641.010992288193,18808.59412325443\nhybrid,C,2020-06-01,2020,6,lat_maior_igual_menos15,31581,130467.22900554293,2006.671161937854,37970.867249923154,129649.69713740438,31215.50842646724\nhybrid,C,2020-07-01,2020,7,dominio_inteiro,78561,210209.9515408445,18937.170170314865,80396.83710792547,188451.2335984007,67475.78503925544\nhybrid,C,2020-07-01,2020,7,lat_menor_que_menos35,26100,61554.059322284564,-3549.825370885228,29556.94058403664,42533.801003694585,17451.68248953376\nhybrid,C,2020-07-01,2020,7,lat_menos35_a_menos15,20880,13789.47593826787,4886.508150823552,11008.094842473734,11991.739600698029,18808.59412325443\nhybrid,C,2020-07-01,2020,7,lat_maior_igual_menos15,31581,134866.41628029203,17600.48739037654,39831.80168141509,133925.69299400804,31215.50842646724\nhybrid,C,2020-08-01,2020,8,dominio_inteiro,78561,196465.63927157436,1941.5862633591546,68616.53687531817,185497.44101799,67475.78503925544\nhybrid,C,2020-08-01,2020,8,lat_menor_que_menos35,26100,23771.996814124064,-5860.455479045906,19155.683116320655,16276.156849596207,17451.68248953376\nhybrid,C,2020-08-01,2020,8,lat_menos35_a_menos15,20880,24758.550394613892,-1398.0306961635717,13741.40680198778,22286.32851645509,18808.59412325443\nhybrid,C,2020-08-01,2020,8,lat_maior_igual_menos15,31581,147935.0920628364,9200.072438568632,35719.446957009735,146934.9556519387,31215.50842646724\nhybrid,C,2020-09-01,2020,9,dominio_inteiro,78561,124905.03819336784,26499.94762604047,61546.96067622476,117898.39115865505,67475.78503925544\nhybrid,C,2020-09-01,2020,9,lat_menor_que_menos35,26100,11379.055541056276,1035.222031901274,13271.133255935829,7893.91076767575,17451.68248953376\nhybrid,C,2020-09-01,2020,9,lat_menos35_a_menos15,20880,26032.749502016777,10775.284825848521,14543.965391269656,23535.277178003245,18808.59412325443\nhybrid,C,2020-09-01,2020,9,lat_maior_igual_menos15,31581,87493.23315029478,14689.440768290679,33731.86202901928,86469.20321297605,31215.50842646724\nhybrid,C,2020-10-01,2020,10,dominio_inteiro,78561,211997.8317546921,29717.625171850592,82097.33039851388,199592.29096743476,67475.78503925544\nhybrid,C,2020-10-01,2020,10,lat_menor_que_menos35,26100,16779.25737650499,3412.1340124246794,15370.671513962472,11996.084416241805,17451.68248953376\nhybrid,C,2020-10-01,2020,10,lat_menos35_a_menos15,20880,66976.14948256759,4014.400890116253,23788.304603556826,60692.28891963575,18808.59412325443\nhybrid,C,2020-10-01,2020,10,lat_maior_igual_menos15,31581,128242.42489561955,22291.090269309654,42938.35428099458,126903.91763155721,31215.50842646724\nhybrid,C,2020-11-01,2020,11,dominio_inteiro,78561,421106.4439484315,7018.978642641537,109351.27474528035,406363.134496248,67475.78503925544\nhybrid,C,2020-11-01,2020,11,lat_menor_que_menos35,26100,17387.786792762206,10880.647813343052,15203.545787391186,12868.094353744109,17451.68248953376\nhybrid,C,2020-11-01,2020,11,lat_menos35_a_menos15,20880,74677.60703343846,11255.645805083921,26066.845651729724,68698.87438092593,18808.59412325443\nhybrid,C,2020-11-01,2020,11,lat_maior_igual_menos15,31581,329041.05012223084,-15117.31497578544,68080.88330615943,324796.16576157795,31215.50842646724\nhybrid,C,2020-12-01,2020,12,dominio_inteiro,78561,290260.16113362875,16645.517332486845,91420.87016944721,277555.2293616957,67475.78503925544\nhybrid,C,2020-12-01,2020,12,lat_menor_que_menos35,26100,10743.214704371303,1368.3244398363934,12704.094139478184,7608.036489560605,17451.68248953376\nhybrid,C,2020-12-01,2020,12,lat_menos35_a_menos15,20880,80881.93040939466,14222.334097865478,28022.84158641944,73259.8709699965,18808.59412325443\nhybrid,C,2020-12-01,2020,12,lat_maior_igual_menos15,31581,198635.01601986284,1054.8587947849737,50693.93444354959,196687.32190213865,31215.50842646724\nclimatology,C,2019-01-01,2019,1,dominio_inteiro,78561,337709.8513413746,24150.561004502582,100741.24785853166,317364.0926850967,67475.78503925544\nclimatology,C,2019-01-01,2019,1,lat_menor_que_menos35,26100,14901.587675241131,-3164.741459533572,14069.420881465077,10587.618390352452,17451.68248953376\nclimatology,C,2019-01-01,2019,1,lat_menos35_a_menos15,20880,152510.9239677052,15580.839622201282,36233.31827929977,137755.0411027252,18808.59412325443\nclimatology,C,2019-01-01,2019,1,lat_maior_igual_menos15,31581,170297.33969842835,11734.462841834873,50438.50869776681,169021.43319201906,31215.50842646724\nclimatology,C,2019-02-01,2019,2,dominio_inteiro,78561,268263.65189458453,15773.232464539702,89297.28790858097,255357.40632837705,67475.78503925544\nclimatology,C,2019-02-01,2019,2,lat_menor_que_menos35,26100,23070.16057107017,4426.460570082068,18251.671732887626,16300.682729362594,17451.68248953376\nclimatology,C,2019-02-01,2019,2,lat_menos35_a_menos15,20880,51612.27589281821,2673.726712834905,21571.366775299306,46228.402801567354,18808.59412325443\nclimatology,C,2019-02-01,2019,2,lat_maior_igual_menos15,31581,193581.2154306962,8673.045181622729,49474.24940039404,192828.32079744714,31215.50842646724\nclimatology,C,2019-03-01,2019,3,dominio_inteiro,78561,336491.4614365721,18854.13737474382,97531.69584493339,325175.3449291726,67475.78503925544\nclimatology,C,2019-03-01,2019,3,lat_menor_que_menos35,26100,19637.663508478905,8142.345657259226,17354.5698992908,14231.169034831224,17451.68248953376\nclimatology,C,2019-03-01,2019,3,lat_menos35_a_menos15,20880,61220.08852746229,-1420.8769983444363,23057.815660921857,56092.381172956186,18808.59412325443\nclimatology,C,2019-03-01,2019,3,lat_maior_igual_menos15,31581,255633.70940063082,12132.66871582903,57119.310284720734,254851.79472138514,31215.50842646724\nclimatology,C,2019-04-01,2019,4,dominio_inteiro,78561,338452.360547524,862.117072425317,91233.40344332578,325793.08204583084,67475.78503925544\nclimatology,C,2019-04-01,2019,4,lat_menor_que_menos35,26100,25332.5876912783,4477.246882528067,19477.929170638323,18281.37728308466,17451.68248953376\nclimatology,C,2019-04-01,2019,4,lat_menos35_a_menos15,20880,42897.54008996585,-4671.421862205025,18774.315870690625,37928.09801694557,18808.59412325443\nclimatology,C,2019-04-01,2019,4,lat_maior_igual_menos15,31581,270222.23276627983,1056.2920521022752,52981.15840199683,269583.60674580064,31215.50842646724\nclimatology,C,2019-05-01,2019,5,dominio_inteiro,78561,255647.74729078298,-4794.413395746145,82483.44799468713,245145.5755392852,67475.78503925544\nclimatology,C,2019-05-01,2019,5,lat_menor_que_menos35,26100,15813.064674526844,1350.6214265227318,14945.961643755436,10995.273180787222,17451.68248953376\nclimatology,C,2019-05-01,2019,5,lat_menos35_a_menos15,20880,42385.101119152314,-5342.434566982556,17987.574258121196,37978.59880589736,18808.59412325443\nclimatology,C,2019-05-01,2019,5,lat_maior_igual_menos15,31581,197449.5814971038,-802.600255286321,49549.9120928105,196171.70355260064,31215.50842646724\nclimatology,C,2019-06-01,2019,6,dominio_inteiro,78561,272032.7730948317,8343.696128884796,79932.52684946498,260748.74550231063,67475.78503925544\nclimatology,C,2019-06-01,2019,6,lat_menor_que_menos35,26100,19003.1138972903,-5054.995585262775,15496.500529944897,14064.607634193522,17451.68248953376\nclimatology,C,2019-06-01,2019,6,lat_menos35_a_menos15,20880,38203.63117634062,10414.138548244257,17914.049282047432,33090.479343735526,18808.59412325443\nclimatology,C,2019-06-01,2019,6,lat_maior_igual_menos15,31581,214826.02802120076,2984.553165903315,46521.97703747265,213593.6585243816,31215.50842646724\nclimatology,C,2019-07-01,2019,7,dominio_inteiro,78561,176278.0759120135,8334.36610634718,64416.81542631332,167292.12659855827,67475.78503925544\nclimatology,C,2019-07-01,2019,7,lat_menor_que_menos35,26100,19232.184696007156,-321.30846697092056,15882.631517231464,13876.555835408944,17451.68248953376\nclimatology,C,2019-07-01,2019,7,lat_menos35_a_menos15,20880,19729.93159876157,2287.0196558376774,12954.243266626261,17164.000547425207,18808.59412325443\nclimatology,C,2019-07-01,2019,7,lat_maior_igual_menos15,31581,137315.95961724478,6368.654917480424,35579.94064245559,136251.57021572412,31215.50842646724\nclimatology,C,2019-08-01,2019,8,dominio_inteiro,78561,190831.25024629824,29549.00163162453,68931.37163298251,180816.40942017123,67475.78503925544\nclimatology,C,2019-08-01,2019,8,lat_menor_que_menos35,26100,23450.635121152627,4782.920839428902,17403.90231370926,17208.199021789922,17451.68248953376\nclimatology,C,2019-08-01,2019,8,lat_menos35_a_menos15,20880,20033.391855029546,10476.297752616927,14341.541163174435,17545.598821567986,18808.59412325443\nclimatology,C,2019-08-01,2019,8,lat_maior_igual_menos15,31581,147347.22327011608,14289.783039578702,37185.92815609882,146062.61157681333,31215.50842646724\nclimatology,C,2019-09-01,2019,9,dominio_inteiro,78561,161573.81308304492,10806.101565275574,68993.49741681083,150928.34048056422,67475.78503925544\nclimatology,C,2019-09-01,2019,9,lat_menor_que_menos35,26100,23609.200675960354,-883.232611477375,17822.050087034702,17466.659673263035,17451.68248953376\nclimatology,C,2019-09-01,2019,9,lat_menos35_a_menos15,20880,27539.163281199868,6712.161077083787,15997.484781308332,24287.995205608706,18808.59412325443\nclimatology,C,2019-09-01,2019,9,lat_maior_igual_menos15,31581,110425.4491258847,4977.173099669162,35173.962548467796,109173.6856016925,31215.50842646724\nclimatology,C,2019-10-01,2019,10,dominio_inteiro,78561,188493.5399405468,24860.013877149642,73846.75083679377,178669.31908833203,67475.78503925544\nclimatology,C,2019-10-01,2019,10,lat_menor_que_menos35,26100,10645.1168967628,2936.864397317171,11756.578639775515,7442.622743706526,17451.68248953376\nclimatology,C,2019-10-01,2019,10,lat_menos35_a_menos15,20880,50893.902498479525,13147.20324862373,22159.6113193263,45474.13861306773,18808.59412325443\nclimatology,C,2019-10-01,2019,10,lat_maior_igual_menos15,31581,126954.52054530449,8775.946231208742,39930.560877691954,125752.55773155778,31215.50842646724\nclimatology,C,2019-11-01,2019,11,dominio_inteiro,78561,257155.13108365797,10718.451543180676,86424.15345219737,243847.30825772567,67475.78503925544\nclimatology,C,2019-11-01,2019,11,lat_menor_que_menos35,26100,27930.506783729033,100.21627923846245,18130.36123022437,19297.171073367936,17451.68248953376\nclimatology,C,2019-11-01,2019,11,lat_menos35_a_menos15,20880,29203.844316486313,8731.647106329383,16421.79641496231,26766.522114475018,18808.59412325443\nclimatology,C,2019-11-01,2019,11,lat_maior_igual_menos15,31581,200020.77998344263,1886.5881576128304,51871.99580701068,197783.61506988274,31215.50842646724\nclimatology,C,2019-12-01,2019,12,dominio_inteiro,78561,306215.86395865784,2099.6523155007744,96097.67427721585,291295.07726520975,67475.78503925544\nclimatology,C,2019-12-01,2019,12,lat_menor_que_menos35,26100,31110.407278492832,-6641.252813801169,17132.987995371222,23188.970855205236,17451.68248953376\nclimatology,C,2019-12-01,2019,12,lat_menos35_a_menos15,20880,55120.674296003745,14686.121495011845,22579.6376420554,49957.189752336584,18808.59412325443\nclimatology,C,2019-12-01,2019,12,lat_maior_igual_menos15,31581,219984.7823841612,-5945.216365709901,56385.048639789224,218148.91665766796,31215.50842646724\nclimatology,C,2020-01-01,2020,1,dominio_inteiro,78561,295393.2211672633,8190.439849717426,91054.53365090617,280300.1939992824,67475.78503925544\nclimatology,C,2020-01-01,2020,1,lat_menor_que_menos35,26100,12267.814440897255,-2921.581211730838,13409.596490740776,8428.229050911255,17451.68248953376\nclimatology,C,2020-01-01,2020,1,lat_menos35_a_menos15,20880,113289.99974682048,-4264.6880267245,28843.073922795593,102941.37992603402,18808.59412325443\nclimatology,C,2020-01-01,2020,1,lat_maior_igual_menos15,31581,169835.40697954557,15376.709088172764,48801.863237369806,168930.58502233712,31215.50842646724\nclimatology,C,2020-02-01,2020,2,dominio_inteiro,78561,260889.3506005627,14895.703320252593,88668.81747512042,246163.10459721973,67475.78503925544\nclimatology,C,2020-02-01,2020,2,lat_menor_que_menos35,26100,18735.404324655854,1822.8190600723028,15631.00231795013,13444.534914396332,17451.68248953376\nclimatology,C,2020-02-01,2020,2,lat_menos35_a_menos15,20880,90288.94780045752,-2647.1217521309154,26610.002124071936,81539.14881727034,18808.59412325443\nclimatology,C,2020-02-01,2020,2,lat_maior_igual_menos15,31581,151864.9984754493,15720.006012311205,46427.813033098355,151179.42086555308,31215.50842646724\nclimatology,C,2020-03-01,2020,3,dominio_inteiro,78561,300275.5124865427,23692.585899218917,100205.21962724626,283743.4656318173,67475.78503925544\nclimatology,C,2020-03-01,2020,3,lat_menor_que_menos35,26100,25675.026761648893,10079.735031038523,19724.414277523756,18778.132775719587,17451.68248953376\nclimatology,C,2020-03-01,2020,3,lat_menos35_a_menos15,20880,88311.91613319752,10110.096436103806,30403.159015217796,80028.45617884216,18808.59412325443\nclimatology,C,2020-03-01,2020,3,lat_maior_igual_menos15,31581,186288.56959169626,3502.7544320765883,50077.64633450471,184936.87667725555,31215.50842646724\nclimatology,C,2020-04-01,2020,4,dominio_inteiro,78561,372903.3472250092,5037.278434119653,103231.99148262618,358299.01225226256,67475.78503925544\nclimatology,C,2020-04-01,2020,4,lat_menor_que_menos35,26100,30333.546949679807,2325.296046346426,20733.285859376192,21817.06016156659,17451.68248953376\nclimatology,C,2020-04-01,2020,4,lat_menos35_a_menos15,20880,48666.86939506802,12111.77744809771,21714.009306533728,43797.65326236842,18808.59412325443\nclimatology,C,2020-04-01,2020,4,lat_maior_igual_menos15,31581,293902.9308802613,-9399.795060324483,60784.69631671626,292684.2988283275,31215.50842646724\nclimatology,C,2020-05-01,2020,5,dominio_inteiro,78561,339932.13970702677,7122.176341192331,96021.52470161347,322540.74636075477,67475.78503925544\nclimatology,C,2020-05-01,2020,5,lat_menor_que_menos35,26100,47851.21037567448,-4325.072085440159,23444.743474543095,35029.62187657862,17451.68248953376\nclimatology,C,2020-05-01,2020,5,lat_menos35_a_menos15,20880,27170.455729840724,10523.507434360217,17170.58207326336,23831.372092661593,18808.59412325443\nclimatology,C,2020-05-01,2020,5,lat_maior_igual_menos15,31581,264910.47360151156,923.7409922722727,55406.199153807014,263679.75239151454,31215.50842646724\nclimatology,C,2020-06-01,2020,6,dominio_inteiro,78561,199136.20495098538,-8932.320106466766,75609.1597142038,184820.14870967704,67475.78503925544\nclimatology,C,2020-06-01,2020,6,lat_menor_que_menos35,26100,38320.036709245425,-5608.048487484455,22349.83239787817,27330.00263094078,17451.68248953376\nclimatology,C,2020-06-01,2020,6,lat_menos35_a_menos15,20880,18870.8951605702,77.01849209191278,13101.997116508428,16392.404005996275,18808.59412325443\nclimatology,C,2020-06-01,2020,6,lat_maior_igual_menos15,31581,141945.27308116975,-3401.290111074224,40157.330199817196,141097.74207273996,31215.50842646724\nclimatology,C,2020-07-01,2020,7,dominio_inteiro,78561,190591.14577485138,9495.346330956556,75614.19163199794,171028.04544418625,67475.78503925544\nclimatology,C,2020-07-01,2020,7,lat_menor_que_menos35,26100,54682.545404038254,-3292.3053236603737,28307.174234092236,37855.45469296188,17451.68248953376\nclimatology,C,2020-07-01,2020,7,lat_menos35_a_menos15,20880,15182.96236125679,7398.963854446076,12262.74776529055,13325.899206829738,18808.59412325443\nclimatology,C,2020-07-01,2020,7,lat_maior_igual_menos15,31581,120725.63800955632,5388.687800170854,35044.26963261515,119846.69154439463,31215.50842646724\nclimatology,C,2020-08-01,2020,8,dominio_inteiro,78561,218950.33349075334,-2137.0087290932424,73427.71358867874,207155.62093103895,67475.78503925544\nclimatology,C,2020-08-01,2020,8,lat_menor_que_menos35,26100,26633.351905398453,-3989.9368265867233,20099.59360897541,18428.34537246932,17451.68248953376\nclimatology,C,2020-08-01,2020,8,lat_menos35_a_menos15,20880,23524.459278321134,293.1770250778645,13964.422336356714,21116.78831066274,18808.59412325443\nclimatology,C,2020-08-01,2020,8,lat_maior_igual_menos15,31581,168792.52230703374,1559.7510724156164,39363.69764334662,167610.48724790686,31215.50842646724\nclimatology,C,2020-09-01,2020,9,dominio_inteiro,78561,140965.61213095387,25939.965014372254,66210.65451488341,132388.20204513345,67475.78503925544\nclimatology,C,2020-09-01,2020,9,lat_menor_que_menos35,26100,14484.73172542109,595.4260646700859,14772.543067276478,10001.415236165016,17451.68248953376\nclimatology,C,2020-09-01,2020,9,lat_menos35_a_menos15,20880,31776.992393921926,12815.749181331834,16491.398959406884,28808.499430579246,18808.59412325443\nclimatology,C,2020-09-01,2020,9,lat_maior_igual_menos15,31581,94703.88801161086,12528.789768370334,34946.71248820005,93578.28737838921,31215.50842646724\nclimatology,C,2020-10-01,2020,10,dominio_inteiro,78561,227996.19573441063,26969.688330884994,86267.94030935827,213342.92437406298,67475.78503925544\nclimatology,C,2020-10-01,2020,10,lat_menor_que_menos35,26100,19991.33580856434,3575.0022146999836,17036.32691398263,14261.224992927588,17451.68248953376\nclimatology,C,2020-10-01,2020,10,lat_menos35_a_menos15,20880,79664.08292727853,3603.261750820995,25940.23888208854,72128.84777235585,18808.59412325443\nclimatology,C,2020-10-01,2020,10,lat_maior_igual_menos15,31581,128340.7769985678,19791.424365364015,43291.3745132871,126952.85160877954,31215.50842646724\nclimatology,C,2020-11-01,2020,11,dominio_inteiro,78561,472738.730262526,380.6944325849763,117942.14237121659,454888.65231703286,67475.78503925544\nclimatology,C,2020-11-01,2020,11,lat_menor_que_menos35,26100,22530.841552143913,13675.32194635272,17655.92827704549,16774.156538606716,17451.68248953376\nclimatology,C,2020-11-01,2020,11,lat_menos35_a_menos15,20880,86075.80282504161,11315.892788092078,28313.328420718924,78886.34410261786,18808.59412325443\nclimatology,C,2020-11-01,2020,11,lat_maior_igual_menos15,31581,364132.0858853405,-24610.520301859826,71972.88567345217,359228.1516758083,31215.50842646724\nclimatology,C,2020-12-01,2020,12,dominio_inteiro,78561,327167.8181757311,2423.989321116009,94207.64214620006,313877.61177219683,67475.78503925544\nclimatology,C,2020-12-01,2020,12,lat_menor_que_menos35,26100,10387.823450568623,1933.0822844654322,12426.616008922458,7399.505298897277,17451.68248953376\nclimatology,C,2020-12-01,2020,12,lat_menos35_a_menos15,20880,82886.45353696955,12459.457905534306,28418.61266480398,74805.22447726024,18808.59412325443\nclimatology,C,2020-12-01,2020,12,lat_maior_igual_menos15,31581,233893.54118819293,-11968.550868883729,53362.41347247362,231672.8819960393,31215.50842646724\n', 'research/spatial_unet/evidence/seven_blocks/C/source_calendar.csv': 'time_alvo,limite_emissao,atmosfera,indices,sst,inicializacao_sazonal,ultimo_alvo_treino,disponibilidade_historica_comprovada\n2019-01-01,2018-12-31 23:59:59.999999999,2018-09-01,2018-10-01,2018-11-01,2018-12-01,2018-09-01,False\n2019-02-01,2019-01-31 23:59:59.999999999,2018-10-01,2018-11-01,2018-12-01,2019-01-01,2018-09-01,False\n2019-03-01,2019-02-28 23:59:59.999999999,2018-11-01,2018-12-01,2019-01-01,2019-02-01,2018-09-01,False\n2019-04-01,2019-03-31 23:59:59.999999999,2018-12-01,2019-01-01,2019-02-01,2019-03-01,2018-09-01,False\n2019-05-01,2019-04-30 23:59:59.999999999,2019-01-01,2019-02-01,2019-03-01,2019-04-01,2018-09-01,False\n2019-06-01,2019-05-31 23:59:59.999999999,2019-02-01,2019-03-01,2019-04-01,2019-05-01,2018-09-01,False\n2019-07-01,2019-06-30 23:59:59.999999999,2019-03-01,2019-04-01,2019-05-01,2019-06-01,2018-09-01,False\n2019-08-01,2019-07-31 23:59:59.999999999,2019-04-01,2019-05-01,2019-06-01,2019-07-01,2018-09-01,False\n2019-09-01,2019-08-31 23:59:59.999999999,2019-05-01,2019-06-01,2019-07-01,2019-08-01,2018-09-01,False\n2019-10-01,2019-09-30 23:59:59.999999999,2019-06-01,2019-07-01,2019-08-01,2019-09-01,2018-09-01,False\n2019-11-01,2019-10-31 23:59:59.999999999,2019-07-01,2019-08-01,2019-09-01,2019-10-01,2018-09-01,False\n2019-12-01,2019-11-30 23:59:59.999999999,2019-08-01,2019-09-01,2019-10-01,2019-11-01,2018-09-01,False\n2020-01-01,2019-12-31 23:59:59.999999999,2019-09-01,2019-10-01,2019-11-01,2019-12-01,2018-09-01,False\n2020-02-01,2020-01-31 23:59:59.999999999,2019-10-01,2019-11-01,2019-12-01,2020-01-01,2018-09-01,False\n2020-03-01,2020-02-29 23:59:59.999999999,2019-11-01,2019-12-01,2020-01-01,2020-02-01,2018-09-01,False\n2020-04-01,2020-03-31 23:59:59.999999999,2019-12-01,2020-01-01,2020-02-01,2020-03-01,2018-09-01,False\n2020-05-01,2020-04-30 23:59:59.999999999,2020-01-01,2020-02-01,2020-03-01,2020-04-01,2018-09-01,False\n2020-06-01,2020-05-31 23:59:59.999999999,2020-02-01,2020-03-01,2020-04-01,2020-05-01,2018-09-01,False\n2020-07-01,2020-06-30 23:59:59.999999999,2020-03-01,2020-04-01,2020-05-01,2020-06-01,2018-09-01,False\n2020-08-01,2020-07-31 23:59:59.999999999,2020-04-01,2020-05-01,2020-06-01,2020-07-01,2018-09-01,False\n2020-09-01,2020-08-31 23:59:59.999999999,2020-05-01,2020-06-01,2020-07-01,2020-08-01,2018-09-01,False\n2020-10-01,2020-09-30 23:59:59.999999999,2020-06-01,2020-07-01,2020-08-01,2020-09-01,2018-09-01,False\n2020-11-01,2020-10-31 23:59:59.999999999,2020-07-01,2020-08-01,2020-09-01,2020-10-01,2018-09-01,False\n2020-12-01,2020-11-30 23:59:59.999999999,2020-08-01,2020-09-01,2020-10-01,2020-11-01,2018-09-01,False\n', 'research/spatial_unet/evidence/seven_blocks/C/split.json': '{\n  "historical_publication_vintages_verified": false,\n  "independent_holdout": false,\n  "inner_reference": [\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09",\n    "2012-10",\n    "2012-11",\n    "2012-12",\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06",\n    "2014-07",\n    "2014-08",\n    "2014-09",\n    "2014-10",\n    "2014-11",\n    "2014-12",\n    "2015-01",\n    "2015-02",\n    "2015-03",\n    "2015-04",\n    "2015-05",\n    "2015-06",\n    "2015-07",\n    "2015-08",\n    "2015-09",\n    "2015-10",\n    "2015-11",\n    "2015-12",\n    "2016-01",\n    "2016-02",\n    "2016-03",\n    "2016-04",\n    "2016-05",\n    "2016-06"\n  ],\n  "inner_train": [\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09",\n    "2012-10",\n    "2012-11",\n    "2012-12",\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06",\n    "2014-07",\n    "2014-08",\n    "2014-09",\n    "2014-10",\n    "2014-11",\n    "2014-12",\n    "2015-01",\n    "2015-02",\n    "2015-03",\n    "2015-04",\n    "2015-05",\n    "2015-06",\n    "2015-07",\n    "2015-08",\n    "2015-09",\n    "2015-10",\n    "2015-11",\n    "2015-12",\n    "2016-01",\n    "2016-02",\n    "2016-03",\n    "2016-04",\n    "2016-05",\n    "2016-06"\n  ],\n  "inner_validation": [\n    "2016-10",\n    "2016-11",\n    "2016-12",\n    "2017-01",\n    "2017-02",\n    "2017-03",\n    "2017-04",\n    "2017-05",\n    "2017-06",\n    "2017-07",\n    "2017-08",\n    "2017-09",\n    "2017-10",\n    "2017-11",\n    "2017-12",\n    "2018-01",\n    "2018-02",\n    "2018-03",\n    "2018-04",\n    "2018-05",\n    "2018-06",\n    "2018-07",\n    "2018-08",\n    "2018-09"\n  ],\n  "outer": [\n    "2019-01",\n    "2019-02",\n    "2019-03",\n    "2019-04",\n    "2019-05",\n    "2019-06",\n    "2019-07",\n    "2019-08",\n    "2019-09",\n    "2019-10",\n    "2019-11",\n    "2019-12",\n    "2020-01",\n    "2020-02",\n    "2020-03",\n    "2020-04",\n    "2020-05",\n    "2020-06",\n    "2020-07",\n    "2020-08",\n    "2020-09",\n    "2020-10",\n    "2020-11",\n    "2020-12"\n  ],\n  "train": [\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09",\n    "2012-10",\n    "2012-11",\n    "2012-12",\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06",\n    "2014-07",\n    "2014-08",\n    "2014-09",\n    "2014-10",\n    "2014-11",\n    "2014-12",\n    "2015-01",\n    "2015-02",\n    "2015-03",\n    "2015-04",\n    "2015-05",\n    "2015-06",\n    "2015-07",\n    "2015-08",\n    "2015-09",\n    "2015-10",\n    "2015-11",\n    "2015-12",\n    "2016-01",\n    "2016-02",\n    "2016-03",\n    "2016-04",\n    "2016-05",\n    "2016-06",\n    "2016-07",\n    "2016-08",\n    "2016-09",\n    "2016-10",\n    "2016-11",\n    "2016-12",\n    "2017-01",\n    "2017-02",\n    "2017-03",\n    "2017-04",\n    "2017-05",\n    "2017-06",\n    "2017-07",\n    "2017-08",\n    "2017-09",\n    "2017-10",\n    "2017-11",\n    "2017-12",\n    "2018-01",\n    "2018-02",\n    "2018-03",\n    "2018-04",\n    "2018-05",\n    "2018-06",\n    "2018-07",\n    "2018-08",\n    "2018-09"\n  ]\n}\n', 'research/spatial_unet/evidence/seven_blocks/C/unet/training.json': '{\n  "device": "cuda",\n  "inner_selection": false,\n  "parameter_count": 122945,\n  "selected_epochs": 9,\n  "train_end": "2018-09-01",\n  "train_months": 360,\n  "train_start": "1988-10-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/C/unet/training_history.json': '[\n  {\n    "epoch": 1,\n    "seconds": 21.13679869599946,\n    "training_mse": 3.0321092877123093\n  },\n  {\n    "epoch": 2,\n    "seconds": 21.591719371999716,\n    "training_mse": 2.967309398783578\n  },\n  {\n    "epoch": 3,\n    "seconds": 21.02662261400019,\n    "training_mse": 2.946812054846022\n  },\n  {\n    "epoch": 4,\n    "seconds": 20.906304211000133,\n    "training_mse": 2.918182904190487\n  },\n  {\n    "epoch": 5,\n    "seconds": 21.036090586999308,\n    "training_mse": 2.8951851275232103\n  },\n  {\n    "epoch": 6,\n    "seconds": 21.120757787999537,\n    "training_mse": 2.8838846683502197\n  },\n  {\n    "epoch": 7,\n    "seconds": 21.061988432999897,\n    "training_mse": 2.8686109284559884\n  },\n  {\n    "epoch": 8,\n    "seconds": 20.994901000999562,\n    "training_mse": 2.8524727993541292\n  },\n  {\n    "epoch": 9,\n    "seconds": 21.007944725000016,\n    "training_mse": 2.8524118065834045\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/H1/baseline_check.json': '{\n  "reproduced": true,\n  "rmse_archived": 1.7413644424984764,\n  "rmse_current": 1.7413644424984764,\n  "tolerance": 1e-06\n}\n', 'research/spatial_unet/evidence/seven_blocks/H1/complete.json': '{\n  "files": {\n    "baseline_check.json": "eaf22b2e51fc3a65d5f9777c76871e1d678a67e77c35f37d4e28fb928b9a228f",\n    "hybrid/arvores_multissistema.txt": "747d94cacf280ae1c409c7b1999abee64247b810c20f18b125c90eb2bb3fcedc",\n    "hybrid/arvores_seas5.txt": "aec8c7bba9a1519be539e9e1dbf5528ade4de6fe67c2808ee28c8cac76d7dc46",\n    "hybrid/clima_atmosfera_indices.npz": "120c7a0e9656067a3e40283558a89b4d703c4ebdfef5fb2f62ccaf792293a2e0",\n    "hybrid/clima_cfsv2.nc": "1e1e3f34b44300297f32da44201573fce44dcbf2f72bcf77b2d101f8a0e45e8e",\n    "hybrid/clima_chuva.nc": "53f3c9b97cfd222fa279514b45fc9a12343ecf8e3deab898d48353051ee0f533",\n    "hybrid/clima_seas5.nc": "231ffc4706c67f6c1c93e0df68c619ddb56cd30144ce581a19716b2e30e270b2",\n    "hybrid/fit.json": "5f21f82ff87762ecc1c2d9c38b7231aa61fc5a6e7a9dd2265cd9edc3c17c047a",\n    "hybrid/ridge.npz": "643c5c113afb02ed31b4f367d9069ed05d804cebffd08c35ce9fa74a7c8ddedb",\n    "inner/context.json": "20b0cf183a2c0651ec09126371cfceba4d5b93e3e937536b29ef40224a037c43",\n    "inner/network.pt": "2b94f273913430d93fbab3bdb6de8afd1a11507a80954bfb2687f6ea952c6971",\n    "inner/rainfall_climatology.nc": "c9316f57ea2fed9f341db15ca896e72e3f24635df08fc7af2e9a78527616e4b4",\n    "inner/references.npz": "584c68f11d8551a2011d3f2b5f82d66c3c3f0bef0f5a6e47f4606931e6a32870",\n    "inner/scaling.npz": "8700c4eecaedfc317cf7090222a0188dfc1898c772ac748979423239d288d5ca",\n    "inner/training.json": "6c2f05c250aa4c694bd9b96bbdc76d497e9bd18b7ec2c814f5141a7f3712082c",\n    "inner/training_history.json": "03d6e27f74a03baf79da6b6cf37428bea999c084f454ed4874b89a8b8dddc77c",\n    "monthly_metrics.csv": "73eaea553f587b1b664ced3760bb1e47c33b25d271ce595c6fc7abf986efb862",\n    "predictions.nc": "06491506d0d591fedbf6a2a98cf8b06ea9bc4cf8e8fd2d2c25e815da68c0036a",\n    "source_calendar.csv": "321b876fcfb1e4c8ef4a4e0e51c0c45b4319c47f09160ad6c08c157c396c402d",\n    "split.json": "2339fc58697d911f5631261d50aa0f2242ac54eabdce38a0d2d9a47a31d2fea7",\n    "unet/context.json": "98fcb35a3397b6f3059f7bcfb0a105e6dd7e1d1de4bb98a5e5504b9f03332cab",\n    "unet/network.pt": "8078e75b700b6b1bbf1e9b46fef31f8c9c547f7ca5ac4d8eb0835b46c419cfba",\n    "unet/rainfall_climatology.nc": "c01e47105ea99d965139b9dae144b81cf7cebaddd5657eedbc2346f7847463a5",\n    "unet/references.npz": "e46d47eccfeb4f4dbd76a3d911fdd2fd4cdaa777f177ea6b6a08d9fade405ffa",\n    "unet/scaling.npz": "7803bd8d976680d0766dcdd3de107f5c38708e0d5d20ed811e4b7b3bf00bebe9",\n    "unet/training.json": "6c856cbb90035b6e4bbf6a1db33f7acea11edeec23c9787a8ad54f7d39af81f2",\n    "unet/training_history.json": "3013155d59efc48bb61ae1eb8e5924c18ca7f4c366a6a6eee7ffd8245e53e6b4"\n  },\n  "signature": "02d1ffc0a7205f2bf372673d337cf5daa95f125d43a95836b953c10ae101244c"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H1/inner/training.json': '{\n  "device": "cuda",\n  "inner_selection": true,\n  "parameter_count": 122945,\n  "selected_epochs": 5,\n  "train_end": "2004-06-01",\n  "train_months": 268,\n  "train_start": "1982-03-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H1/inner/training_history.json': '[\n  {\n    "epoch": 1,\n    "inner_rmse": 1.7724098190523738,\n    "seconds": 17.969769533999965,\n    "training_mse": 2.9276454377530228\n  },\n  {\n    "epoch": 2,\n    "inner_rmse": 1.7683325262052747,\n    "seconds": 14.774183966999999,\n    "training_mse": 2.843834393949651\n  },\n  {\n    "epoch": 3,\n    "inner_rmse": 1.7518715534247575,\n    "seconds": 14.976493392999942,\n    "training_mse": 2.8268477551972686\n  },\n  {\n    "epoch": 4,\n    "inner_rmse": 1.7510614007390781,\n    "seconds": 15.133748663000006,\n    "training_mse": 2.8158734485284604\n  },\n  {\n    "epoch": 5,\n    "inner_rmse": 1.7427899055832419,\n    "seconds": 15.394628152999985,\n    "training_mse": 2.79629879834047\n  },\n  {\n    "epoch": 6,\n    "inner_rmse": 1.7456995577940988,\n    "seconds": 15.794932774000017,\n    "training_mse": 2.780257347804397\n  },\n  {\n    "epoch": 7,\n    "inner_rmse": 1.7472134094402356,\n    "seconds": 15.99700683399999,\n    "training_mse": 2.7526057371452675\n  },\n  {\n    "epoch": 8,\n    "inner_rmse": 1.7485489842063011,\n    "seconds": 15.799470548000045,\n    "training_mse": 2.7459465968075083\n  },\n  {\n    "epoch": 9,\n    "inner_rmse": 1.7511633493537815,\n    "seconds": 15.645117784999911,\n    "training_mse": 2.730145499777438\n  },\n  {\n    "epoch": 10,\n    "inner_rmse": 1.757567533140066,\n    "seconds": 15.61672369200005,\n    "training_mse": 2.719780470897902\n  },\n  {\n    "epoch": 11,\n    "inner_rmse": 1.7527373577908627,\n    "seconds": 15.646614275000047,\n    "training_mse": 2.700898989812652\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/H1/monthly_metrics.csv': 'modelo,bloco,mes_alvo,ano,mes,regiao,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area\nunet,H1,2007-01-01,2007,1,dominio_inteiro,78561,252813.4586426329,18448.496235519648,88414.97943687811,237316.30797678037,67475.78503925544\nunet,H1,2007-01-01,2007,1,lat_menor_que_menos35,26100,26462.825601996312,2095.497803494334,18153.74264059961,19149.22142323976,17451.68248953376\nunet,H1,2007-01-01,2007,1,lat_menos35_a_menos15,20880,85497.08698674057,-7006.864868156612,26552.83303138241,78296.7864987408,18808.59412325443\nunet,H1,2007-01-01,2007,1,lat_maior_igual_menos15,31581,140853.54605389605,23359.863300181925,43708.403764896095,139870.30005479982,31215.50842646724\nunet,H1,2007-02-01,2007,2,dominio_inteiro,78561,360761.7660705571,7698.652322448092,106820.17769112601,344039.3896353772,67475.78503925544\nunet,H1,2007-02-01,2007,2,lat_menor_que_menos35,26100,30563.031467764242,1674.086950391531,21405.077570170164,22082.89284501517,17451.68248953376\nunet,H1,2007-02-01,2007,2,lat_menos35_a_menos15,20880,72792.07795065016,-3476.3757311000954,24625.276310508838,66510.06339753348,18808.59412325443\nunet,H1,2007-02-01,2007,2,lat_maior_igual_menos15,31581,257406.65665214276,9500.941103156656,60789.82381044701,255446.43339282853,31215.50842646724\nunet,H1,2007-03-01,2007,3,dominio_inteiro,78561,348828.40855413565,-5968.005813099444,96364.90710223466,328407.1508377838,67475.78503925544\nunet,H1,2007-03-01,2007,3,lat_menor_que_menos35,26100,27285.61358248799,-4160.3887540996075,17901.088426560163,20005.303436440852,17451.68248953376\nunet,H1,2007-03-01,2007,3,lat_menos35_a_menos15,20880,106735.83601645773,5834.602982737124,29760.819073960185,94716.68293197516,18808.59412325443\nunet,H1,2007-03-01,2007,3,lat_maior_igual_menos15,31581,214806.95895518997,-7642.22004173696,48702.99960171431,213685.16446936785,31215.50842646724\nunet,H1,2007-04-01,2007,4,dominio_inteiro,78561,165096.3600663377,-5292.969720738009,69047.37445710786,153600.81182662296,67475.78503925544\nunet,H1,2007-04-01,2007,4,lat_menor_que_menos35,26100,28383.69673955764,-5622.285224378109,18717.755958855152,19197.856729476673,17451.68248953376\nunet,H1,2007-04-01,2007,4,lat_menos35_a_menos15,20880,16678.434241157112,1989.2018703650683,12843.281386451796,14817.827276949072,18808.59412325443\nunet,H1,2007-04-01,2007,4,lat_maior_igual_menos15,31581,120034.2290856229,-1659.886366724968,37486.33711180091,119585.1278201972,31215.50842646724\nunet,H1,2007-05-01,2007,5,dominio_inteiro,78561,287413.25092370604,153.2139742206782,86596.29968598671,269107.79796610895,67475.78503925544\nunet,H1,2007-05-01,2007,5,lat_menor_que_menos35,26100,54125.03281328517,7623.974949121475,24042.053723335266,40143.951692890456,17451.68248953376\nunet,H1,2007-05-01,2007,5,lat_menos35_a_menos15,20880,24520.768217753433,-2296.9469949621707,14333.061806039885,21265.280398450584,18808.59412325443\nunet,H1,2007-05-01,2007,5,lat_maior_igual_menos15,31581,208767.44989266744,-5173.813979938626,48221.18415661156,207698.56587476796,31215.50842646724\nunet,H1,2007-06-01,2007,6,dominio_inteiro,78561,205519.33649806224,19126.228018213063,75186.26703513786,194213.76651498227,67475.78503925544\nunet,H1,2007-06-01,2007,6,lat_menor_que_menos35,26100,25767.015706782367,-578.3614877462387,18033.71031987667,18197.419035090683,17451.68248953376\nunet,H1,2007-06-01,2007,6,lat_menos35_a_menos15,20880,23636.163637463786,7048.884733568877,14738.460158031434,21033.502252999915,18808.59412325443\nunet,H1,2007-06-01,2007,6,lat_maior_igual_menos15,31581,156116.15715381608,12655.704772390425,42414.09655722976,154982.84522689163,31215.50842646724\nunet,H1,2007-07-01,2007,7,dominio_inteiro,78561,149238.89861256097,12234.074669528753,63922.61194750294,138560.78499562005,67475.78503925544\nunet,H1,2007-07-01,2007,7,lat_menor_que_menos35,26100,25986.251940510138,2143.531771183014,17973.991917729378,18822.977525025133,17451.68248953376\nunet,H1,2007-07-01,2007,7,lat_menos35_a_menos15,20880,21042.369011006824,1059.4323908053339,14419.020162764937,18338.62081262341,18808.59412325443\nunet,H1,2007-07-01,2007,7,lat_maior_igual_menos15,31581,102210.277661044,9031.110507540405,31529.599867008626,101399.18665797151,31215.50842646724\nunet,H1,2007-08-01,2007,8,dominio_inteiro,78561,221302.3842899484,-12683.838275609538,74177.69929883815,209743.04377446085,67475.78503925544\nunet,H1,2007-08-01,2007,8,lat_menor_que_menos35,26100,22928.142497509947,-5735.418234139681,17855.790032237768,15681.514381189292,17451.68248953376\nunet,H1,2007-08-01,2007,8,lat_menos35_a_menos15,20880,17691.752138472635,8003.9231703057885,13665.011781081557,15560.429269452561,18808.59412325443\nunet,H1,2007-08-01,2007,8,lat_maior_igual_menos15,31581,180682.48965396578,-14952.343211775646,42656.89748551883,178501.10012381902,31215.50842646724\nunet,H1,2007-09-01,2007,9,dominio_inteiro,78561,181428.9830705232,18108.263396089897,74070.29799768515,170393.80981937083,67475.78503925544\nunet,H1,2007-09-01,2007,9,lat_menor_que_menos35,26100,20114.87092122244,-1023.1936174631119,16597.725458681583,14393.433644336,17451.68248953376\nunet,H1,2007-09-01,2007,9,lat_menos35_a_menos15,20880,41603.97745322888,15684.894368072972,20744.1748815123,37619.41016297104,18808.59412325443\nunet,H1,2007-09-01,2007,9,lat_maior_igual_menos15,31581,119710.13469607188,3446.5626454800367,36728.39765749127,118380.96601206379,31215.50842646724\nunet,H1,2007-10-01,2007,10,dominio_inteiro,78561,253938.02036231197,-10477.31696032174,83719.24412300996,238912.4584402804,67475.78503925544\nunet,H1,2007-10-01,2007,10,lat_menor_que_menos35,26100,29628.942004762124,-10842.99503466487,19255.32359597087,20951.28454573254,17451.68248953376\nunet,H1,2007-10-01,2007,10,lat_menos35_a_menos15,20880,42961.31524497089,11233.42501117289,20466.539695683867,38296.238981925475,18808.59412325443\nunet,H1,2007-10-01,2007,10,lat_maior_igual_menos15,31581,181347.76311257895,-10867.74693682976,43997.38083135523,179664.9349126224,31215.50842646724\nunet,H1,2007-11-01,2007,11,dominio_inteiro,78561,222083.01462066523,-8713.816287845373,82677.9575066287,205510.2982016753,67475.78503925544\nunet,H1,2007-11-01,2007,11,lat_menor_que_menos35,26100,39189.50034745125,-10820.485256537795,21896.186531618237,28462.498112937894,17451.68248953376\nunet,H1,2007-11-01,2007,11,lat_menos35_a_menos15,20880,52570.944958543216,-954.8631404228508,21304.9020461943,47904.73260557337,18808.59412325443\nunet,H1,2007-11-01,2007,11,lat_maior_igual_menos15,31581,130322.56931467075,3061.5321091152728,39476.86892881617,129143.067483164,31215.50842646724\nunet,H1,2007-12-01,2007,12,dominio_inteiro,78561,243984.5134591145,-10370.180082648993,77661.0670581758,235208.73615909528,67475.78503925544\nunet,H1,2007-12-01,2007,12,lat_menor_que_menos35,26100,11648.337744779583,3480.2519223093987,12533.555408030748,8452.447252831022,17451.68248953376\nunet,H1,2007-12-01,2007,12,lat_menos35_a_menos15,20880,47071.58297257039,4831.098565489054,20424.96999296546,42710.22327649822,18808.59412325443\nunet,H1,2007-12-01,2007,12,lat_maior_igual_menos15,31581,185264.5927417645,-18681.530570447445,44702.541657179594,184046.065629766,31215.50842646724\nunet,H1,2008-01-01,2008,1,dominio_inteiro,78561,260768.89982382304,-5998.767056576908,83053.14672931284,246966.05682832075,67475.78503925544\nunet,H1,2008-01-01,2008,1,lat_menor_que_menos35,26100,22937.826092380375,5098.337396338582,16999.83294056356,15788.337603333126,17451.68248953376\nunet,H1,2008-01-01,2008,1,lat_menos35_a_menos15,20880,70125.01275669417,-7936.318449765444,24032.944899350405,64274.397235100405,18808.59412325443\nunet,H1,2008-01-01,2008,1,lat_maior_igual_menos15,31581,167706.06097474849,-3160.786003150046,42020.36888939887,166903.3219898872,31215.50842646724\nunet,H1,2008-02-01,2008,2,dominio_inteiro,78561,264229.0320022445,-30242.968831948936,89690.52494549751,248880.8135186854,67475.78503925544\nunet,H1,2008-02-01,2008,2,lat_menor_que_menos35,26100,20404.805272223923,-4378.107685953379,17093.22754687071,14083.093630720216,17451.68248953376\nunet,H1,2008-02-01,2008,2,lat_menos35_a_menos15,20880,86233.45976803076,-12667.171028725803,26398.819857500494,77951.24739567954,18808.59412325443\nunet,H1,2008-02-01,2008,2,lat_maior_igual_menos15,31581,157590.76696198984,-13197.690117269754,46198.47754112631,156846.47249228565,31215.50842646724\nunet,H1,2008-03-01,2008,3,dominio_inteiro,78561,181977.57412923552,-9047.934868486598,74996.05804523267,170566.96762731034,67475.78503925544\nunet,H1,2008-03-01,2008,3,lat_menor_que_menos35,26100,24358.66304042079,2532.4881079494953,17350.04538449645,16898.209009800554,17451.68248953376\nunet,H1,2008-03-01,2008,3,lat_menos35_a_menos15,20880,33839.77319081383,-2412.3516279459,17480.37010142207,30430.033417943712,18808.59412325443\nunet,H1,2008-03-01,2008,3,lat_maior_igual_menos15,31581,123779.1378980009,-9168.071348490193,40165.64255931415,123238.72519956605,31215.50842646724\nunet,H1,2008-04-01,2008,4,dominio_inteiro,78561,252259.2190736635,-18003.46873152256,90748.93311448395,235232.79480357337,67475.78503925544\nunet,H1,2008-04-01,2008,4,lat_menor_que_menos35,26100,37493.66980990829,-10395.034633606672,23941.07970765233,25948.250467347265,17451.68248953376\nunet,H1,2008-04-01,2008,4,lat_menos35_a_menos15,20880,46440.89005179891,-5046.841512978077,21079.592295363545,41522.04863444158,18808.59412325443\nunet,H1,2008-04-01,2008,4,lat_maior_igual_menos15,31581,168324.6592119563,-2561.592584937811,45728.26111146808,167762.4957017845,31215.50842646724\nunet,H1,2008-05-01,2008,5,dominio_inteiro,78561,215031.5626674791,-19149.519126832485,84136.63640220463,197920.4895762667,67475.78503925544\nunet,H1,2008-05-01,2008,5,lat_menor_que_menos35,26100,45177.744736863475,-12082.053809702396,24719.830011427402,33279.669168578934,17451.68248953376\nunet,H1,2008-05-01,2008,5,lat_menos35_a_menos15,20880,34131.19405372784,-1004.9922387003899,18123.57882720232,29584.69186289107,18808.59412325443\nunet,H1,2008-05-01,2008,5,lat_maior_igual_menos15,31581,135722.62387688778,-6062.473078429699,41293.22756357491,135056.1285447967,31215.50842646724\nunet,H1,2008-06-01,2008,6,dominio_inteiro,78561,234849.60196399863,-7948.989264555275,70800.99016778916,226603.32907340195,67475.78503925544\nunet,H1,2008-06-01,2008,6,lat_menor_que_menos35,26100,25699.171518119463,8548.69237202406,16702.891327917576,18964.13918160272,17451.68248953376\nunet,H1,2008-06-01,2008,6,lat_menos35_a_menos15,20880,8192.70606574705,789.7091324329376,8740.034673601389,7250.24341784209,18808.59412325443\nunet,H1,2008-06-01,2008,6,lat_maior_igual_menos15,31581,200957.7243801321,-17287.390769012272,45358.0641662702,200388.94647395716,31215.50842646724\nunet,H1,2008-07-01,2008,7,dominio_inteiro,78561,173410.53219465053,-12764.508380331099,69812.79586554319,161686.5099548414,67475.78503925544\nunet,H1,2008-07-01,2008,7,lat_menor_que_menos35,26100,32023.586437656784,-4333.430282115936,19291.957330584526,23778.171811986715,17451.68248953376\nunet,H1,2008-07-01,2008,7,lat_menos35_a_menos15,20880,19713.12515706609,5039.66334592551,13719.295711435378,17287.12917339758,18808.59412325443\nunet,H1,2008-07-01,2008,7,lat_maior_igual_menos15,31581,121673.82059992764,-13470.741444140673,36801.54282352328,120621.20896945712,31215.50842646724\nunet,H1,2008-08-01,2008,8,dominio_inteiro,78561,165995.21971491646,-23316.478407558054,69089.51243906096,149557.96373041108,67475.78503925544\nunet,H1,2008-08-01,2008,8,lat_menor_que_menos35,26100,43828.427337302965,-15735.867733180523,23124.73421895504,31570.42584970325,17451.68248953376\nunet,H1,2008-08-01,2008,8,lat_menos35_a_menos15,20880,24123.360400318656,-6127.64440186508,13792.35401671566,20957.62876563068,18808.59412325443\nunet,H1,2008-08-01,2008,8,lat_maior_igual_menos15,31581,98043.43197729482,-1452.9662725124508,32172.424203390256,97029.90911507716,31215.50842646724\nunet,H1,2008-09-01,2008,9,dominio_inteiro,78561,159598.80869326173,23687.700018286705,71104.89432542026,144502.3412318436,67475.78503925544\nunet,H1,2008-09-01,2008,9,lat_menor_que_menos35,26100,36052.22759052036,15947.633003592491,23285.939923465252,24651.055725191793,17451.68248953376\nunet,H1,2008-09-01,2008,9,lat_menos35_a_menos15,20880,27574.26399990929,3326.102458421141,16012.803222868592,24864.928100762478,18808.59412325443\nunet,H1,2008-09-01,2008,9,lat_maior_igual_menos15,31581,95972.31710283208,4413.964556273073,31806.151179086417,94986.35740588934,31215.50842646724\nunet,H1,2008-10-01,2008,10,dominio_inteiro,78561,246158.8577339774,-20616.75280497782,80927.96937676333,234077.31494825036,67475.78503925544\nunet,H1,2008-10-01,2008,10,lat_menor_que_menos35,26100,17480.976240147513,-918.5834477543831,16049.3257086277,11981.843883965706,17451.68248953376\nunet,H1,2008-10-01,2008,10,lat_menos35_a_menos15,20880,38323.524942714575,4416.784178590402,18245.52721139975,34629.75712475741,18808.59412325443\nunet,H1,2008-10-01,2008,10,lat_maior_igual_menos15,31581,190354.3565511153,-24114.95353581384,46633.11645673588,187465.71393952728,31215.50842646724\nunet,H1,2008-11-01,2008,11,dominio_inteiro,78561,402723.2148283168,-14333.584229953587,99536.02734342963,385998.3699036818,67475.78503925544\nunet,H1,2008-11-01,2008,11,lat_menor_que_menos35,26100,25000.801484920168,2227.4017636030912,17441.29197151959,18481.814463995586,17451.68248953376\nunet,H1,2008-11-01,2008,11,lat_menos35_a_menos15,20880,77018.7576589833,1552.6302951201797,25448.116070337594,70050.48602253616,18808.59412325443\nunet,H1,2008-11-01,2008,11,lat_maior_igual_menos15,31581,300703.6556844134,-18113.616288676858,56646.61930157244,297466.06941715005,31215.50842646724\nunet,H1,2008-12-01,2008,12,dominio_inteiro,78561,282680.4896460468,-15056.961801089346,91926.29347421974,268645.85929289827,67475.78503925544\nunet,H1,2008-12-01,2008,12,lat_menor_que_menos35,26100,17433.089368650973,1962.0406858026981,14942.940013945103,12395.853865326577,17451.68248953376\nunet,H1,2008-12-01,2008,12,lat_menos35_a_menos15,20880,71583.72778936684,-1958.8050823435187,25736.574589900672,64187.52239400077,18808.59412325443\nunet,H1,2008-12-01,2008,12,lat_maior_igual_menos15,31581,193663.67248802897,-15060.197404548526,51246.778870373964,192062.48303357096,31215.50842646724\nhybrid,H1,2007-01-01,2007,1,dominio_inteiro,78561,247371.8533826878,9808.628173226409,86103.47420881575,232199.84585912994,67475.78503925544\nhybrid,H1,2007-01-01,2007,1,lat_menor_que_menos35,26100,26162.612395096046,-1354.7238455315025,17364.527072542318,19063.40839389228,17451.68248953376\nhybrid,H1,2007-01-01,2007,1,lat_menos35_a_menos15,20880,85536.59677162045,-8398.189636808558,26326.624820496385,78377.20422264625,18808.59412325443\nhybrid,H1,2007-01-01,2007,1,lat_maior_igual_menos15,31581,135672.64421597135,19561.54165556647,42412.32231577704,134759.2332425914,31215.50842646724\nhybrid,H1,2007-02-01,2007,2,dominio_inteiro,78561,359498.84446164785,5965.6187525906425,106999.38267974643,342837.6068081609,67475.78503925544\nhybrid,H1,2007-02-01,2007,2,lat_menor_que_menos35,26100,30350.85156678474,-870.5537334463689,21314.00389373394,21822.686624117767,17451.68248953376\nhybrid,H1,2007-02-01,2007,2,lat_menos35_a_menos15,20880,71913.26949772578,-5414.587784988617,24111.547635030296,65734.1249053567,18808.59412325443\nhybrid,H1,2007-02-01,2007,2,lat_maior_igual_menos15,31581,257234.72339713736,12250.760271025625,61573.83115098219,255280.7952786865,31215.50842646724\nhybrid,H1,2007-03-01,2007,3,dominio_inteiro,78561,342829.37710381486,-5750.586884216427,94918.29366406091,322418.6516122843,67475.78503925544\nhybrid,H1,2007-03-01,2007,3,lat_menor_que_menos35,26100,27097.56716302532,-4641.829187572824,17999.06693257982,19832.369310394515,17451.68248953376\nhybrid,H1,2007-03-01,2007,3,lat_menos35_a_menos15,20880,106574.95280885862,6967.091799083235,29334.156532237204,94518.49385745701,18808.59412325443\nhybrid,H1,2007-03-01,2007,3,lat_maior_igual_menos15,31581,209156.85713193088,-8075.849495726837,47585.0701992439,208067.78844443275,31215.50842646724\nhybrid,H1,2007-04-01,2007,4,dominio_inteiro,78561,172049.86406907855,-8194.39899422024,70208.98633188875,161266.67541777578,67475.78503925544\nhybrid,H1,2007-04-01,2007,4,lat_menor_que_menos35,26100,25744.449901811826,-5418.96797974371,18167.60714415604,17326.08768250006,17451.68248953376\nhybrid,H1,2007-04-01,2007,4,lat_menos35_a_menos15,20880,17586.305022216024,2607.212680826405,13110.516812493883,15680.64966048741,18808.59412325443\nhybrid,H1,2007-04-01,2007,4,lat_maior_igual_menos15,31581,128719.1091450507,-5382.643695302934,38930.86237523883,128259.93807478831,31215.50842646724\nhybrid,H1,2007-05-01,2007,5,dominio_inteiro,78561,294955.3499662207,-3578.235869997109,86705.0691039824,277062.310022586,67475.78503925544\nhybrid,H1,2007-05-01,2007,5,lat_menor_que_menos35,26100,52506.773589039076,8512.160876651935,24205.40950206102,38929.00610575738,17451.68248953376\nhybrid,H1,2007-05-01,2007,5,lat_menos35_a_menos15,20880,24331.536128578937,372.90171756122413,14552.009045287356,21110.59877063224,18808.59412325443\nhybrid,H1,2007-05-01,2007,5,lat_maior_igual_menos15,31581,218117.04024860263,-12463.298464210267,47947.65055663402,217022.7051461964,31215.50842646724\nhybrid,H1,2007-06-01,2007,6,dominio_inteiro,78561,198802.65269031958,18002.130323190016,74778.81813615443,187512.15126944816,67475.78503925544\nhybrid,H1,2007-06-01,2007,6,lat_menor_que_menos35,26100,26403.65535933653,441.1568678925549,18329.643224756594,18845.46877574396,17451.68248953376\nhybrid,H1,2007-06-01,2007,6,lat_menos35_a_menos15,20880,24467.031977751158,7834.247522961542,15329.492031212203,21796.305755637935,18808.59412325443\nhybrid,H1,2007-06-01,2007,6,lat_maior_igual_menos15,31581,147931.96535323188,9726.72593233592,41119.68288018563,146870.3767380663,31215.50842646724\nhybrid,H1,2007-07-01,2007,7,dominio_inteiro,78561,147013.587768294,11275.759907066216,62845.78288240236,136087.69950723817,67475.78503925544\nhybrid,H1,2007-07-01,2007,7,lat_menor_que_menos35,26100,26615.041049772706,4313.882635055082,18418.590012352663,19293.516272091754,17451.68248953376\nhybrid,H1,2007-07-01,2007,7,lat_menos35_a_menos15,20880,21510.00689467107,840.5112400796013,14435.334645733274,18739.974367622977,18808.59412325443\nhybrid,H1,2007-07-01,2007,7,lat_maior_igual_menos15,31581,98888.53982385024,6121.366031931535,29991.85822431643,98054.20886752346,31215.50842646724\nhybrid,H1,2007-08-01,2007,8,dominio_inteiro,78561,237370.33473302564,-16973.72084128895,74891.47512447764,225906.38587287956,67475.78503925544\nhybrid,H1,2007-08-01,2007,8,lat_menor_que_menos35,26100,22555.024432402548,-4090.0662890990816,17801.305853145,15571.670846789173,17451.68248953376\nhybrid,H1,2007-08-01,2007,8,lat_menos35_a_menos15,20880,16716.091943576957,7192.490399253211,13287.601252990018,14647.520332602475,18808.59412325443\nhybrid,H1,2007-08-01,2007,8,lat_maior_igual_menos15,31581,198099.2183570461,-20076.14495144308,43802.568018342616,195687.1946934879,31215.50842646724\nhybrid,H1,2007-09-01,2007,9,dominio_inteiro,78561,182662.72371413477,15478.071335028813,73384.451743157,171716.25465894915,67475.78503925544\nhybrid,H1,2007-09-01,2007,9,lat_menor_que_menos35,26100,20663.650103437983,683.4913154607443,17027.026792186247,14795.501307057024,17451.68248953376\nhybrid,H1,2007-09-01,2007,9,lat_menos35_a_menos15,20880,38279.25935670685,14710.434198625047,19660.151790080818,34591.2174556057,18808.59412325443\nhybrid,H1,2007-09-01,2007,9,lat_maior_igual_menos15,31581,123719.81425398994,84.1458209430175,36697.27316088995,122329.53589628646,31215.50842646724\nhybrid,H1,2007-10-01,2007,10,dominio_inteiro,78561,242313.47289844125,-8678.714828053562,81506.95012180824,228453.0125549106,67475.78503925544\nhybrid,H1,2007-10-01,2007,10,lat_menor_que_menos35,26100,26389.429932278068,-7878.542784730325,18263.810419279627,18850.23321420887,17451.68248953376\nhybrid,H1,2007-10-01,2007,10,lat_menos35_a_menos15,20880,43917.081770279474,11432.920639614627,20791.424629309546,39169.627864232076,18808.59412325443\nhybrid,H1,2007-10-01,2007,10,lat_maior_igual_menos15,31581,172006.96119588375,-12233.092682937866,42451.71507321908,170433.15147646965,31215.50842646724\nhybrid,H1,2007-11-01,2007,11,dominio_inteiro,78561,215774.7949227936,-10716.084280856567,79954.95402890039,200838.7903523884,67475.78503925544\nhybrid,H1,2007-11-01,2007,11,lat_menor_que_menos35,26100,34527.98074241266,-8745.290358241573,20516.31276268868,25119.740832408646,17451.68248953376\nhybrid,H1,2007-11-01,2007,11,lat_menos35_a_menos15,20880,51007.88882319153,-1227.9324597080617,20698.97816730068,46672.676343793246,18808.59412325443\nhybrid,H1,2007-11-01,2007,11,lat_maior_igual_menos15,31581,130238.92535718942,-742.8614629069316,38739.66309891103,129046.37317618649,31215.50842646724\nhybrid,H1,2007-12-01,2007,12,dominio_inteiro,78561,240417.08532083847,903.8561499402165,79546.66933317317,231216.58510506293,67475.78503925544\nhybrid,H1,2007-12-01,2007,12,lat_menor_que_menos35,26100,13119.034387936194,7841.4482082226605,13642.054470090361,9622.725837175783,17451.68248953376\nhybrid,H1,2007-12-01,2007,12,lat_menos35_a_menos15,20880,47848.24575341133,5026.404672003516,20516.6324096599,43393.31516162481,18808.59412325443\nhybrid,H1,2007-12-01,2007,12,lat_maior_igual_menos15,31581,179449.80517949094,-11963.99673028596,45387.9824534229,178200.54410626236,31215.50842646724\nhybrid,H1,2008-01-01,2008,1,dominio_inteiro,78561,278662.15320065664,-7385.055837150856,86139.88902761467,264674.34548966907,67475.78503925544\nhybrid,H1,2008-01-01,2008,1,lat_menor_que_menos35,26100,21903.315196872743,7997.018149272404,17729.5541197328,15113.350457186767,17451.68248953376\nhybrid,H1,2008-01-01,2008,1,lat_menos35_a_menos15,20880,77574.32613023795,-11037.23652763594,24407.075737986863,71299.34944476151,18808.59412325443\nhybrid,H1,2008-01-01,2008,1,lat_maior_igual_menos15,31581,179184.5118735459,-4344.837458787321,44003.25916989501,178261.64558772076,31215.50842646724\nhybrid,H1,2008-02-01,2008,2,dominio_inteiro,78561,267388.9932692175,-20374.920738653866,89812.9017487055,252320.5624968345,67475.78503925544\nhybrid,H1,2008-02-01,2008,2,lat_menor_que_menos35,26100,19095.607044523356,254.7429678779593,16607.33912711237,13532.136956788905,17451.68248953376\nhybrid,H1,2008-02-01,2008,2,lat_menos35_a_menos15,20880,91513.4073706234,-12998.514426962543,27013.66670830955,82770.28369585953,18808.59412325443\nhybrid,H1,2008-02-01,2008,2,lat_maior_igual_menos15,31581,156779.97885407077,-7631.149279569277,46191.895913283566,156018.14184418606,31215.50842646724\nhybrid,H1,2008-03-01,2008,3,dominio_inteiro,78561,179298.0719556198,3056.620826691734,75699.43883168192,167654.56662481948,67475.78503925544\nhybrid,H1,2008-03-01,2008,3,lat_menor_que_menos35,26100,25925.753811107996,7994.784889567592,18365.13972270027,18169.57810422738,17451.68248953376\nhybrid,H1,2008-03-01,2008,3,lat_menos35_a_menos15,20880,33169.64879420774,464.73751153924354,17263.265982672496,29819.330873672916,18808.59412325443\nhybrid,H1,2008-03-01,2008,3,lat_maior_igual_menos15,31581,120202.66935030406,-5402.9015744151,40071.033126309165,119665.65764691916,31215.50842646724\nhybrid,H1,2008-04-01,2008,4,dominio_inteiro,78561,249750.03847668183,959.6199748740121,91671.76126548843,233824.95836444938,67475.78503925544\nhybrid,H1,2008-04-01,2008,4,lat_menor_que_menos35,26100,34079.9646516092,-5518.1255373032145,22674.423516101524,23974.2537584584,17451.68248953376\nhybrid,H1,2008-04-01,2008,4,lat_menos35_a_menos15,20880,47752.821804032326,-2505.8286110651115,21213.52582216156,42602.49326229134,18808.59412325443\nhybrid,H1,2008-04-01,2008,4,lat_maior_igual_menos15,31581,167917.2520210403,8983.574123242339,47783.81192722534,167248.2113436996,31215.50842646724\nhybrid,H1,2008-05-01,2008,5,dominio_inteiro,78561,213943.64669586334,-1591.3393848997428,83880.21354109826,197567.7719482514,67475.78503925544\nhybrid,H1,2008-05-01,2008,5,lat_menor_que_menos35,26100,43371.276184040435,-8180.177682693014,23304.449957588124,32110.265968122625,17451.68248953376\nhybrid,H1,2008-05-01,2008,5,lat_menos35_a_menos15,20880,34820.924014038654,3792.3338890329283,18804.53094201391,30389.99669537355,18808.59412325443\nhybrid,H1,2008-05-01,2008,5,lat_maior_igual_menos15,31581,135751.44649778426,2796.5044087603437,41771.23264149623,135067.50928475524,31215.50842646724\nhybrid,H1,2008-06-01,2008,6,dominio_inteiro,78561,224137.0068071983,6133.487604565688,72298.79879116117,215012.33111135673,67475.78503925544\nhybrid,H1,2008-06-01,2008,6,lat_menor_que_menos35,26100,28220.820134574853,12240.940325896381,18276.588113970196,20734.752373685573,17451.68248953376\nhybrid,H1,2008-06-01,2008,6,lat_menos35_a_menos15,20880,9095.03276653358,4099.554493543168,9340.829230581281,8037.004436036803,18808.59412325443\nhybrid,H1,2008-06-01,2008,6,lat_maior_igual_menos15,31581,186821.15390608986,-10207.007214873865,44681.381446609696,186240.57430163433,31215.50842646724\nhybrid,H1,2008-07-01,2008,7,dominio_inteiro,78561,166626.28689454147,-8416.533905566204,69551.25434296097,155197.147385264,67475.78503925544\nhybrid,H1,2008-07-01,2008,7,lat_menor_que_menos35,26100,31168.976319377816,-3925.488211466394,18749.049680245233,23106.941799362132,17451.68248953376\nhybrid,H1,2008-07-01,2008,7,lat_menos35_a_menos15,20880,19285.05187374275,5966.890383585181,13559.775171317364,16959.92058832684,18808.59412325443\nhybrid,H1,2008-07-01,2008,7,lat_maior_igual_menos15,31581,116172.25870142091,-10457.93607768499,37242.42949139837,115130.28499757501,31215.50842646724\nhybrid,H1,2008-08-01,2008,8,dominio_inteiro,78561,169878.6933011625,-25427.351834315952,69358.27713788148,152914.87669246588,67475.78503925544\nhybrid,H1,2008-08-01,2008,8,lat_menor_que_menos35,26100,46519.388080601886,-16239.348327008665,23387.72905325809,33769.761318112665,17451.68248953376\nhybrid,H1,2008-08-01,2008,8,lat_menos35_a_menos15,20880,24137.147744762726,-6621.975937865486,13766.478350739822,20967.09696146107,18808.59412325443\nhybrid,H1,2008-08-01,2008,8,lat_maior_igual_menos15,31581,99222.15747579793,-2566.0275694418006,32204.06973388356,98178.01841289215,31215.50842646724\nhybrid,H1,2008-09-01,2008,9,dominio_inteiro,78561,152758.99858349675,21787.39545987801,69422.32391489856,138811.9538206217,67475.78503925544\nhybrid,H1,2008-09-01,2008,9,lat_menor_que_menos35,26100,32448.53533195437,15210.079552160005,22638.553955921932,22085.160398116906,17451.68248953376\nhybrid,H1,2008-09-01,2008,9,lat_menos35_a_menos15,20880,26740.803606842215,3288.0087543391737,15482.642794483698,24104.58541941241,18808.59412325443\nhybrid,H1,2008-09-01,2008,9,lat_maior_igual_menos15,31581,93569.65964470016,3289.3071533788343,31301.12716449293,92622.20800309238,31215.50842646724\nhybrid,H1,2008-10-01,2008,10,dominio_inteiro,78561,253368.2225685853,-25081.23208359232,82136.1302645384,241207.56299183465,67475.78503925544\nhybrid,H1,2008-10-01,2008,10,lat_menor_que_menos35,26100,17112.500314409754,-1645.86801985617,15881.815089967029,11692.178783621215,17451.68248953376\nhybrid,H1,2008-10-01,2008,10,lat_menos35_a_menos15,20880,38634.282042287516,4009.8609793779297,18267.914715900344,34934.77865445724,18808.59412325443\nhybrid,H1,2008-10-01,2008,10,lat_maior_igual_menos15,31581,197621.44021188805,-27445.225043114082,47986.40045867102,194580.6055537562,31215.50842646724\nhybrid,H1,2008-11-01,2008,11,dominio_inteiro,78561,393039.690534977,-10801.445609082573,98720.84818631083,376829.0238282456,67475.78503925544\nhybrid,H1,2008-11-01,2008,11,lat_menor_que_menos35,26100,23962.616134421303,4704.447905630284,17137.289730743978,17774.30820263725,17451.68248953376\nhybrid,H1,2008-11-01,2008,11,lat_menos35_a_menos15,20880,75452.6690668634,2545.988733032037,24813.691615008906,68584.22559607537,18808.59412325443\nhybrid,H1,2008-11-01,2008,11,lat_maior_igual_menos15,31581,293624.4053336922,-18051.8822477449,56769.86684055796,290470.49002953304,31215.50842646724\nhybrid,H1,2008-12-01,2008,12,dominio_inteiro,78561,287475.24634941004,-3638.8816534392618,94101.35695478848,273116.90895495383,67475.78503925544\nhybrid,H1,2008-12-01,2008,12,lat_menor_que_menos35,26100,18410.678907689537,6111.29748731799,15791.029435490469,13164.570867821447,17451.68248953376\nhybrid,H1,2008-12-01,2008,12,lat_menos35_a_menos15,20880,72013.12663506568,584.027916446649,25216.333433426673,64522.17700958129,18808.59412325443\nhybrid,H1,2008-12-01,2008,12,lat_maior_igual_menos15,31581,197051.44080665475,-10334.2070572039,53093.99408587134,195430.16107755108,31215.50842646724\nclimatology,H1,2007-01-01,2007,1,dominio_inteiro,78561,243727.01710497116,15843.211771037662,85997.42456723785,229197.5254961048,67475.78503925544\nclimatology,H1,2007-01-01,2007,1,lat_menor_que_menos35,26100,27648.846527206602,-337.2764429003,17793.275801360607,20049.812313113292,17451.68248953376\nclimatology,H1,2007-01-01,2007,1,lat_menos35_a_menos15,20880,71603.04432652556,-7327.6487991584,24054.193490367732,65410.670115764326,18808.59412325443\nclimatology,H1,2007-01-01,2007,1,lat_maior_igual_menos15,31581,144475.12625123898,23508.137013096362,44149.95527550951,143737.0430672272,31215.50842646724\nclimatology,H1,2007-02-01,2007,2,dominio_inteiro,78561,380213.93210152956,4923.407014134107,108777.75360213337,363791.8104677605,67475.78503925544\nclimatology,H1,2007-02-01,2007,2,lat_menor_que_menos35,26100,29069.556796687568,-1050.6165217757225,20857.72514501214,20912.25030703441,17451.68248953376\nclimatology,H1,2007-02-01,2007,2,lat_menos35_a_menos15,20880,69583.02872058829,-4453.401524867164,23653.187537211226,63520.03614057136,18808.59412325443\nclimatology,H1,2007-02-01,2007,2,lat_maior_igual_menos15,31581,281561.34658425365,10427.425060776994,64266.84091991,279359.5240201548,31215.50842646724\nclimatology,H1,2007-03-01,2007,3,dominio_inteiro,78561,428567.50484308985,-7735.020480127074,104197.39545666892,407636.9493910697,67475.78503925544\nclimatology,H1,2007-03-01,2007,3,lat_menor_que_menos35,26100,28773.304318282615,-2714.8787795603275,18538.88986930251,21103.52155633196,17451.68248953376\nclimatology,H1,2007-03-01,2007,3,lat_menos35_a_menos15,20880,108711.26379481913,9185.178616856225,30424.261229357682,96637.08970761567,18808.59412325443\nclimatology,H1,2007-03-01,2007,3,lat_maior_igual_menos15,31581,291082.93672998814,-14205.320317422971,55234.24435800873,289896.338127122,31215.50842646724\nclimatology,H1,2007-04-01,2007,4,dominio_inteiro,78561,198839.32986509567,-7737.425438072532,74125.25543967634,188384.4457872083,67475.78503925544\nclimatology,H1,2007-04-01,2007,4,lat_menor_que_menos35,26100,24131.024999544356,-5417.824785768986,17784.00160139799,16260.827711906531,17451.68248953376\nclimatology,H1,2007-04-01,2007,4,lat_menos35_a_menos15,20880,19424.4337569788,4075.882695797831,14247.773349933326,17425.13644090478,18808.59412325443\nclimatology,H1,2007-04-01,2007,4,lat_maior_igual_menos15,31581,155283.87110857252,-6395.4833481013775,42093.48048834503,154698.48163439697,31215.50842646724\nclimatology,H1,2007-05-01,2007,5,dominio_inteiro,78561,319806.0905065427,-2130.2817009962164,90310.57968776533,301843.3054957601,67475.78503925544\nclimatology,H1,2007-05-01,2007,5,lat_menor_que_menos35,26100,52122.0872481024,8601.958325326443,24139.53222411871,38651.1807650426,17451.68248953376\nclimatology,H1,2007-05-01,2007,5,lat_menos35_a_menos15,20880,24732.94216662096,1878.313825224992,15111.586761561688,21503.926543390957,18808.59412325443\nclimatology,H1,2007-05-01,2007,5,lat_maior_igual_menos15,31581,242951.0610918193,-12610.553851547651,51059.460702084936,241688.19818732655,31215.50842646724\nclimatology,H1,2007-06-01,2007,6,dominio_inteiro,78561,232107.83685730665,17366.973557863617,79701.7865167989,219559.76442127812,67475.78503925544\nclimatology,H1,2007-06-01,2007,6,lat_menor_que_menos35,26100,29957.42504705068,1179.1344250440598,19352.62331020832,21478.71793050471,17451.68248953376\nclimatology,H1,2007-06-01,2007,6,lat_menos35_a_menos15,20880,28074.9605597259,9513.728531720117,16473.561044869944,25066.34110018168,18808.59412325443\nclimatology,H1,2007-06-01,2007,6,lat_maior_igual_menos15,31581,174075.4512505301,6674.110601099441,43875.60216172063,173014.7053905917,31215.50842646724\nclimatology,H1,2007-07-01,2007,7,dominio_inteiro,78561,150896.34385984277,10427.913365219254,63252.95252099866,140408.50196331224,67475.78503925544\nclimatology,H1,2007-07-01,2007,7,lat_menor_que_menos35,26100,25581.143458812556,4373.583749234676,18316.70811790228,18915.69129488197,17451.68248953376\nclimatology,H1,2007-07-01,2007,7,lat_menos35_a_menos15,20880,22965.825549668178,1425.0530860153958,15000.481356787495,19979.16195155094,18808.59412325443\nclimatology,H1,2007-07-01,2007,7,lat_maior_igual_menos15,31581,102349.37485136204,4629.276529969182,29935.763046308886,101513.64871687934,31215.50842646724\nclimatology,H1,2007-08-01,2007,8,dominio_inteiro,78561,242527.53258621367,-14137.86157321278,78267.27791145071,230328.2078698625,67475.78503925544\nclimatology,H1,2007-08-01,2007,8,lat_menor_que_menos35,26100,24309.90242647448,-3295.783916950226,18652.674415409565,16972.93805861251,17451.68248953376\nclimatology,H1,2007-08-01,2007,8,lat_menos35_a_menos15,20880,20692.33956464365,8701.776312023401,14959.555256038904,18194.977632352537,18808.59412325443\nclimatology,H1,2007-08-01,2007,8,lat_maior_igual_menos15,31581,197525.29059509555,-19543.853968285955,44655.048240002245,195160.29217889748,31215.50842646724\nclimatology,H1,2007-09-01,2007,9,dominio_inteiro,78561,200496.53228639477,16615.985685997177,77282.64531149622,189002.35853484026,67475.78503925544\nclimatology,H1,2007-09-01,2007,9,lat_menor_que_menos35,26100,19640.004547156186,278.1726956963539,16526.48151499033,14143.778523161754,17451.68248953376\nclimatology,H1,2007-09-01,2007,9,lat_menos35_a_menos15,20880,47411.430672498514,16886.485593121033,22085.919619994238,42908.92481454127,18808.59412325443\nclimatology,H1,2007-09-01,2007,9,lat_maior_igual_menos15,31581,133445.0970667401,-548.6726028202102,38670.24417651165,131949.6551971372,31215.50842646724\nclimatology,H1,2007-10-01,2007,10,dominio_inteiro,78561,285492.0385889042,-12099.202158599888,88800.93521408804,270304.4456658929,67475.78503925544\nclimatology,H1,2007-10-01,2007,10,lat_menor_que_menos35,26100,28882.91592540309,-8407.735891610384,19095.38244268298,20415.407304831315,17451.68248953376\nclimatology,H1,2007-10-01,2007,10,lat_menos35_a_menos15,20880,42702.924635459596,11853.630294781382,20466.25896072827,38074.192673145335,18808.59412325443\nclimatology,H1,2007-10-01,2007,10,lat_maior_igual_menos15,31581,213906.19802804152,-15545.096561770886,49239.29381067678,211814.8456879162,31215.50842646724\nclimatology,H1,2007-11-01,2007,11,dominio_inteiro,78561,242678.89324291714,-10119.527681245374,83669.53161110342,226760.68865970644,67475.78503925544\nclimatology,H1,2007-11-01,2007,11,lat_menor_que_menos35,26100,36206.7032221293,-8017.184385359287,20876.723991155624,26369.70296053145,17451.68248953376\nclimatology,H1,2007-11-01,2007,11,lat_menos35_a_menos15,20880,50878.48363095695,-429.7985962116145,21292.373873246695,46179.12154176003,18808.59412325443\nclimatology,H1,2007-11-01,2007,11,lat_maior_igual_menos15,31581,155593.7063898309,-1672.5446996744722,41500.43374670111,154211.86415741497,31215.50842646724\nclimatology,H1,2007-12-01,2007,12,dominio_inteiro,78561,284223.39650069433,-11999.046629584627,83563.64087892114,275266.66441132367,67475.78503925544\nclimatology,H1,2007-12-01,2007,12,lat_menor_que_menos35,26100,12331.36142680303,6736.180936038494,13135.07036498189,9008.765985041668,17451.68248953376\nclimatology,H1,2007-12-01,2007,12,lat_menos35_a_menos15,20880,45376.118692841104,7057.077880699886,20191.869700984797,41139.59816835951,18808.59412325443\nclimatology,H1,2007-12-01,2007,12,lat_maior_igual_menos15,31581,226515.91638105022,-25792.305446323007,50236.700812954456,225118.3002579225,31215.50842646724\nclimatology,H1,2008-01-01,2008,1,dominio_inteiro,78561,242953.0153082427,-8417.19281575433,78745.60602247319,229610.59970512363,67475.78503925544\nclimatology,H1,2008-01-01,2008,1,lat_menor_que_menos35,26100,21887.345487353192,8805.85099850595,17736.367943152785,15097.211633453388,17451.68248953376\nclimatology,H1,2008-01-01,2008,1,lat_menos35_a_menos15,20880,73154.46975292853,-8296.578089929884,24293.84951831377,67360.56824024435,18808.59412325443\nclimatology,H1,2008-01-01,2008,1,lat_maior_igual_menos15,31581,147911.20006796098,-8926.465724330395,36715.388561006635,147152.81983142588,31215.50842646724\nclimatology,H1,2008-02-01,2008,2,dominio_inteiro,78561,266391.7184622529,-32129.33880495769,88193.51568793715,252541.57830696192,67475.78503925544\nclimatology,H1,2008-02-01,2008,2,lat_menor_que_menos35,26100,18364.34781894657,-271.5799998641014,16473.153055727482,12719.535509253428,17451.68248953376\nclimatology,H1,2008-02-01,2008,2,lat_menos35_a_menos15,20880,75531.4422110039,-11001.848180140601,23959.559364259476,68073.14328353057,18808.59412325443\nclimatology,H1,2008-02-01,2008,2,lat_maior_igual_menos15,31581,172495.9284323025,-20855.910624952987,47760.80326795019,171748.89951417793,31215.50842646724\nclimatology,H1,2008-03-01,2008,3,dominio_inteiro,78561,212902.666425059,-11139.059863061644,79830.64821486454,200737.8392784362,67475.78503925544\nclimatology,H1,2008-03-01,2008,3,lat_menor_que_menos35,26100,25347.42188750056,8235.902088969946,18307.125267237425,17897.531201115664,17451.68248953376\nclimatology,H1,2008-03-01,2008,3,lat_menos35_a_menos15,20880,39414.66273797304,2322.2578557478264,19331.009762276895,35319.50255915911,18808.59412325443\nclimatology,H1,2008-03-01,2008,3,lat_maior_igual_menos15,31581,148140.58179958537,-21697.219807779416,42192.513185350224,147520.80551816148,31215.50842646724\nclimatology,H1,2008-04-01,2008,4,dominio_inteiro,78561,271383.59018433234,-19543.699852135032,90237.78462948464,254853.58992405553,67475.78503925544\nclimatology,H1,2008-04-01,2008,4,lat_menor_que_menos35,26100,35017.719354915345,-7016.405146181583,22665.577894866467,24493.91266910714,17451.68248953376\nclimatology,H1,2008-04-01,2008,4,lat_menos35_a_menos15,20880,49131.46881795756,262.7113090790808,21422.568476172164,43656.67697599446,18808.59412325443\nclimatology,H1,2008-04-01,2008,4,lat_maior_igual_menos15,31581,187234.40201145946,-12790.00601503253,46149.63825844601,186703.00027895396,31215.50842646724\nclimatology,H1,2008-05-01,2008,5,dominio_inteiro,78561,233811.0362449589,-20748.008980659302,86766.8065010109,218723.00553748177,67475.78503925544\nclimatology,H1,2008-05-01,2008,5,lat_menor_que_menos35,26100,38535.63697826373,-8164.969393789768,21978.67532080412,28480.41782891854,17451.68248953376\nclimatology,H1,2008-05-01,2008,5,lat_menos35_a_menos15,20880,33873.12924065627,5280.230329131242,18741.78897436103,29656.879672688792,18808.59412325443\nclimatology,H1,2008-05-01,2008,5,lat_maior_igual_menos15,31581,161402.2700260389,-17863.269916000776,46046.34220584575,160585.70803587444,31215.50842646724\nclimatology,H1,2008-06-01,2008,6,dominio_inteiro,78561,343702.3253436185,-9133.707365598297,83157.09653278603,335796.7231600436,67475.78503925544\nclimatology,H1,2008-06-01,2008,6,lat_menor_que_menos35,26100,22122.829989863098,10493.152674555779,17070.662221491337,16131.322660775802,17451.68248953376\nclimatology,H1,2008-06-01,2008,6,lat_menos35_a_menos15,20880,9964.536107307313,5169.131668927148,9985.661499906331,8795.87235258479,18808.59412325443\nclimatology,H1,2008-06-01,2008,6,lat_maior_igual_menos15,31581,311614.959246448,-24795.991709081223,56100.772811388364,310869.52814668306,31215.50842646724\nclimatology,H1,2008-07-01,2008,7,dominio_inteiro,78561,202389.43532092072,-14233.103099014144,73353.39336034516,190581.57061324775,67475.78503925544\nclimatology,H1,2008-07-01,2008,7,lat_menor_que_menos35,26100,32840.98449074884,-2544.8847261071205,19180.24965852499,24379.773268991805,17451.68248953376\nclimatology,H1,2008-07-01,2008,7,lat_menos35_a_menos15,20880,18778.383891046895,7106.161560746841,13805.262562404387,16496.439820003587,18808.59412325443\nclimatology,H1,2008-07-01,2008,7,lat_maior_igual_menos15,31581,150770.06693912498,-18794.379933653865,40367.88113941578,149705.35752425238,31215.50842646724\nclimatology,H1,2008-08-01,2008,8,dominio_inteiro,78561,177224.75576310736,-24708.01004123036,70670.25877622701,162274.2963262592,67475.78503925544\nclimatology,H1,2008-08-01,2008,8,lat_menor_que_menos35,26100,38401.22111897534,-13103.597561359406,21590.254137933254,27633.731845042435,17451.68248953376\nclimatology,H1,2008-08-01,2008,8,lat_menos35_a_menos15,20880,22801.38009639949,-4688.051416248083,13868.86794539541,19851.316658209278,18808.59412325443\nclimatology,H1,2008-08-01,2008,8,lat_maior_igual_menos15,31581,116022.15454773253,-6916.36106362287,35211.13669289835,114789.2478230075,31215.50842646724\nclimatology,H1,2008-09-01,2008,9,dominio_inteiro,78561,171040.6249864829,22363.788420372177,73198.65619157394,154530.9881283636,67475.78503925544\nclimatology,H1,2008-09-01,2008,9,lat_menor_que_menos35,26100,39047.636356052855,18813.80769020319,24947.269702076912,26504.8554519648,17451.68248953376\nclimatology,H1,2008-09-01,2008,9,lat_menos35_a_menos15,20880,29103.563551683263,4823.266522686463,16376.204480972607,26237.556910185554,18808.59412325443\nclimatology,H1,2008-09-01,2008,9,lat_maior_igual_menos15,31581,102889.4250787468,-1273.2857925174758,31875.182008524425,101788.57576621324,31215.50842646724\nclimatology,H1,2008-10-01,2008,10,dominio_inteiro,78561,265237.37701537355,-22013.823511749302,82533.74683959762,253117.57441044127,67475.78503925544\nclimatology,H1,2008-10-01,2008,10,lat_menor_que_menos35,26100,17667.230904527678,1502.155236929655,16581.832956045866,12247.702468853538,17451.68248953376\nclimatology,H1,2008-10-01,2008,10,lat_menos35_a_menos15,20880,35589.451797538284,3591.4366149718117,17662.917542148178,32065.92647552006,18808.59412325443\nclimatology,H1,2008-10-01,2008,10,lat_maior_igual_menos15,31581,211980.69431330764,-27107.41536365077,48288.996341403574,208803.9454660677,31215.50842646724\nclimatology,H1,2008-11-01,2008,11,dominio_inteiro,78561,464514.7775405204,-16214.614732636976,106180.55195427078,445455.70594311436,67475.78503925544\nclimatology,H1,2008-11-01,2008,11,lat_menor_que_menos35,26100,26513.79531179548,5932.419649064541,18043.570642530918,19694.754996845135,17451.68248953376\nclimatology,H1,2008-11-01,2008,11,lat_menos35_a_menos15,20880,90256.81185128196,2587.735659769831,27363.319153449942,81957.53911334972,18808.59412325443\nclimatology,H1,2008-11-01,2008,11,lat_maior_igual_menos15,31581,347744.17037744296,-24734.770041471347,60773.662158289924,343803.41183291946,31215.50842646724\nclimatology,H1,2008-12-01,2008,12,dominio_inteiro,78561,340364.0115669921,-17408.50192896207,99638.83595277718,324138.7239854619,67475.78503925544\nclimatology,H1,2008-12-01,2008,12,lat_menor_que_menos35,26100,20085.418237149985,6360.594426333904,16491.94972488284,14367.285776439385,17451.68248953376\nclimatology,H1,2008-12-01,2008,12,lat_menos35_a_menos15,20880,82099.06095032733,2529.9906690909993,27000.50321219093,73492.2940635363,18808.59412325443\nclimatology,H1,2008-12-01,2008,12,lat_maior_igual_menos15,31581,238179.5323795147,-26299.087024386972,56146.38301570341,236279.14414548624,31215.50842646724\n', 'research/spatial_unet/evidence/seven_blocks/H1/source_calendar.csv': 'time_alvo,limite_emissao,atmosfera,indices,sst,inicializacao_sazonal,ultimo_alvo_treino,disponibilidade_historica_comprovada\n2007-01-01,2006-12-31 23:59:59.999999999,2006-09-01,2006-10-01,2006-11-01,2006-12-01,2006-09-01,False\n2007-02-01,2007-01-31 23:59:59.999999999,2006-10-01,2006-11-01,2006-12-01,2007-01-01,2006-09-01,False\n2007-03-01,2007-02-28 23:59:59.999999999,2006-11-01,2006-12-01,2007-01-01,2007-02-01,2006-09-01,False\n2007-04-01,2007-03-31 23:59:59.999999999,2006-12-01,2007-01-01,2007-02-01,2007-03-01,2006-09-01,False\n2007-05-01,2007-04-30 23:59:59.999999999,2007-01-01,2007-02-01,2007-03-01,2007-04-01,2006-09-01,False\n2007-06-01,2007-05-31 23:59:59.999999999,2007-02-01,2007-03-01,2007-04-01,2007-05-01,2006-09-01,False\n2007-07-01,2007-06-30 23:59:59.999999999,2007-03-01,2007-04-01,2007-05-01,2007-06-01,2006-09-01,False\n2007-08-01,2007-07-31 23:59:59.999999999,2007-04-01,2007-05-01,2007-06-01,2007-07-01,2006-09-01,False\n2007-09-01,2007-08-31 23:59:59.999999999,2007-05-01,2007-06-01,2007-07-01,2007-08-01,2006-09-01,False\n2007-10-01,2007-09-30 23:59:59.999999999,2007-06-01,2007-07-01,2007-08-01,2007-09-01,2006-09-01,False\n2007-11-01,2007-10-31 23:59:59.999999999,2007-07-01,2007-08-01,2007-09-01,2007-10-01,2006-09-01,False\n2007-12-01,2007-11-30 23:59:59.999999999,2007-08-01,2007-09-01,2007-10-01,2007-11-01,2006-09-01,False\n2008-01-01,2007-12-31 23:59:59.999999999,2007-09-01,2007-10-01,2007-11-01,2007-12-01,2006-09-01,False\n2008-02-01,2008-01-31 23:59:59.999999999,2007-10-01,2007-11-01,2007-12-01,2008-01-01,2006-09-01,False\n2008-03-01,2008-02-29 23:59:59.999999999,2007-11-01,2007-12-01,2008-01-01,2008-02-01,2006-09-01,False\n2008-04-01,2008-03-31 23:59:59.999999999,2007-12-01,2008-01-01,2008-02-01,2008-03-01,2006-09-01,False\n2008-05-01,2008-04-30 23:59:59.999999999,2008-01-01,2008-02-01,2008-03-01,2008-04-01,2006-09-01,False\n2008-06-01,2008-05-31 23:59:59.999999999,2008-02-01,2008-03-01,2008-04-01,2008-05-01,2006-09-01,False\n2008-07-01,2008-06-30 23:59:59.999999999,2008-03-01,2008-04-01,2008-05-01,2008-06-01,2006-09-01,False\n2008-08-01,2008-07-31 23:59:59.999999999,2008-04-01,2008-05-01,2008-06-01,2008-07-01,2006-09-01,False\n2008-09-01,2008-08-31 23:59:59.999999999,2008-05-01,2008-06-01,2008-07-01,2008-08-01,2006-09-01,False\n2008-10-01,2008-09-30 23:59:59.999999999,2008-06-01,2008-07-01,2008-08-01,2008-09-01,2006-09-01,False\n2008-11-01,2008-10-31 23:59:59.999999999,2008-07-01,2008-08-01,2008-09-01,2008-10-01,2006-09-01,False\n2008-12-01,2008-11-30 23:59:59.999999999,2008-08-01,2008-09-01,2008-10-01,2008-11-01,2006-09-01,False\n', 'research/spatial_unet/evidence/seven_blocks/H1/split.json': '{\n  "historical_publication_vintages_verified": false,\n  "independent_holdout": false,\n  "inner_reference": [\n    "1976-10",\n    "1976-11",\n    "1976-12",\n    "1977-01",\n    "1977-02",\n    "1977-03",\n    "1977-04",\n    "1977-05",\n    "1977-06",\n    "1977-07",\n    "1977-08",\n    "1977-09",\n    "1977-10",\n    "1977-11",\n    "1977-12",\n    "1978-01",\n    "1978-02",\n    "1978-03",\n    "1978-04",\n    "1978-05",\n    "1978-06",\n    "1978-07",\n    "1978-08",\n    "1978-09",\n    "1978-10",\n    "1978-11",\n    "1978-12",\n    "1979-01",\n    "1979-02",\n    "1979-03",\n    "1979-04",\n    "1979-05",\n    "1979-06",\n    "1979-07",\n    "1979-08",\n    "1979-09",\n    "1979-10",\n    "1979-11",\n    "1979-12",\n    "1980-01",\n    "1980-02",\n    "1980-03",\n    "1980-04",\n    "1980-05",\n    "1980-06",\n    "1980-07",\n    "1980-08",\n    "1980-09",\n    "1980-10",\n    "1980-11",\n    "1980-12",\n    "1981-01",\n    "1981-02",\n    "1981-03",\n    "1981-04",\n    "1981-05",\n    "1981-06",\n    "1981-07",\n    "1981-08",\n    "1981-09",\n    "1981-10",\n    "1981-11",\n    "1981-12",\n    "1982-01",\n    "1982-02",\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06"\n  ],\n  "inner_train": [\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06"\n  ],\n  "inner_validation": [\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09"\n  ],\n  "outer": [\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12"\n  ],\n  "train": [\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09"\n  ]\n}\n', 'research/spatial_unet/evidence/seven_blocks/H1/unet/training.json': '{\n  "device": "cuda",\n  "inner_selection": false,\n  "parameter_count": 122945,\n  "selected_epochs": 5,\n  "train_end": "2006-09-01",\n  "train_months": 295,\n  "train_start": "1982-03-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H1/unet/training_history.json': '[\n  {\n    "epoch": 1,\n    "seconds": 17.419955856,\n    "training_mse": 2.9858263670387917\n  },\n  {\n    "epoch": 2,\n    "seconds": 17.79643877700005,\n    "training_mse": 2.916958699792118\n  },\n  {\n    "epoch": 3,\n    "seconds": 17.406965393000064,\n    "training_mse": 2.8713089514586887\n  },\n  {\n    "epoch": 4,\n    "seconds": 17.191613972000027,\n    "training_mse": 2.8330801244509423\n  },\n  {\n    "epoch": 5,\n    "seconds": 17.210546843999964,\n    "training_mse": 2.809321471392098\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/H2/baseline_check.json': '{\n  "reproduced": true,\n  "rmse_archived": 1.7991379822396203,\n  "rmse_current": 1.7991379822396205,\n  "tolerance": 1e-06\n}\n', 'research/spatial_unet/evidence/seven_blocks/H2/complete.json': '{\n  "files": {\n    "baseline_check.json": "1a086678d5bd3a8347fac1a42cb69dcd3e5adae04290c9dad83715f94a05367e",\n    "hybrid/arvores_multissistema.txt": "0f1b6b02b9199716d891b9848de77214a3a31ba2dd40bf419a1b87daf05cfc22",\n    "hybrid/arvores_seas5.txt": "26e093b88baee1ef5f36978e8219fb7f3e7141c114f60c3ad10553833a0d64fb",\n    "hybrid/clima_atmosfera_indices.npz": "b568d12f5c568add8e1f1c022f06741da20f9ebe1afa55fadbc8f20c691690f7",\n    "hybrid/clima_cfsv2.nc": "b45adea35e37737f4305e616de4f36ae093af8e647850b351ee07dd3a374fb65",\n    "hybrid/clima_chuva.nc": "4306382a052ca87877577eddf0604a939d7f7e0c336201cd7a4d8ec728ff6c1c",\n    "hybrid/clima_seas5.nc": "c242281a24006c5e436e6421c03db00eb3a6bb61aae1af2ebf6993bfa2c6734e",\n    "hybrid/fit.json": "4d0f6b4c45b641fbc852c731a9ef333d6521ce4d765839f3478b17b66b2c8d1e",\n    "hybrid/ridge.npz": "60f9a5f1291c3bf65ffde4735692a249a1a11ef2d84c80ff830bbc117fd9a683",\n    "inner/context.json": "cfc79749c8c7537767d59633b9ba42416278e84c05bf131d3931704b28744e8a",\n    "inner/network.pt": "4cc338194a413092fd10b54ad241122e25cba5bd8cde28f2d49a4f938172bfe1",\n    "inner/rainfall_climatology.nc": "605791af4c79f09bbb0676d46af03e617c51fab46a23f1a9a41acdc81b11f701",\n    "inner/references.npz": "97bde12b608d39b649d82f5c43f9c27e117d679a4dd6fe4bcffc3bfdb4eb88bd",\n    "inner/scaling.npz": "646a080fcba41ccb7554975154c28d5b866b2aa1cc22328f568ef1e7ba73ab31",\n    "inner/training.json": "5ee6d6965db0b3886a2f2bf0391fc9dacd1d3a58c19f9f26af13720dac29a248",\n    "inner/training_history.json": "dfe2400a2c50455d20669d19208df4aea35130d25ff16c12476291ba8cbb9489",\n    "monthly_metrics.csv": "0fcd88f5c2c791aa09776186eae38f9fd06c54e3c7c13d9486a3c672ce755f9a",\n    "predictions.nc": "3178166fc4e65d3ef861defa0383b1f8b994d5b4700c6c642536b56135ad57db",\n    "source_calendar.csv": "40a890485099c431c3418b091adafd8d103cc20192bce2d24d18e280a7068507",\n    "split.json": "585fff2facadac324229427d0c7b3ed990b20051cf7dc3f1632f1b99b6daffee",\n    "unet/context.json": "e12e2c1791d54c5769aaba7dc9e51e78f9761185e08a4c55cb7b454af70083c2",\n    "unet/network.pt": "fea390656ca021b077cfdf604231a113f1bd7004aaa109a5d47a183bfe1a4d7e",\n    "unet/rainfall_climatology.nc": "e47169990e35d319730d02265d0ad1b15289df6a3054ee66304a0f46081cfd80",\n    "unet/references.npz": "505ae54649aa14b4400221a2b07cde4988f039208809e21890df6396c3447532",\n    "unet/scaling.npz": "2fa10bd3d4f9bf23e29d1de2f0597ee6f47bf6350debc2f3a6185fcd7fa3813f",\n    "unet/training.json": "72c64d5d55baa6dcc719dfc38c5d65df70e8f3f052f525cfde085703284d3535",\n    "unet/training_history.json": "79910009d5bd562eb266ae0c58fe33930d990af9b96a9fea35c5338c8b071d58"\n  },\n  "signature": "02d1ffc0a7205f2bf372673d337cf5daa95f125d43a95836b953c10ae101244c"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H2/inner/training.json': '{\n  "device": "cuda",\n  "inner_selection": true,\n  "parameter_count": 122945,\n  "selected_epochs": 31,\n  "train_end": "2006-06-01",\n  "train_months": 292,\n  "train_start": "1982-03-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H2/inner/training_history.json': '[\n  {\n    "epoch": 1,\n    "inner_rmse": 1.7160838103578375,\n    "seconds": 17.35282724299998,\n    "training_mse": 2.9492187736785573\n  },\n  {\n    "epoch": 2,\n    "inner_rmse": 1.7013101776230153,\n    "seconds": 17.61587770699998,\n    "training_mse": 2.8479768201096416\n  },\n  {\n    "epoch": 3,\n    "inner_rmse": 1.710167564147156,\n    "seconds": 17.133877340000026,\n    "training_mse": 2.8169877014748037\n  },\n  {\n    "epoch": 4,\n    "inner_rmse": 1.7115863284717479,\n    "seconds": 16.956513607000034,\n    "training_mse": 2.7947859894739437\n  },\n  {\n    "epoch": 5,\n    "inner_rmse": 1.7088020965571127,\n    "seconds": 17.001184399999943,\n    "training_mse": 2.781704223319276\n  },\n  {\n    "epoch": 6,\n    "inner_rmse": 1.7035055923172657,\n    "seconds": 17.17776091799999,\n    "training_mse": 2.7533620310156315\n  },\n  {\n    "epoch": 7,\n    "inner_rmse": 1.6961841707777743,\n    "seconds": 17.181981079000025,\n    "training_mse": 2.7382009641764915\n  },\n  {\n    "epoch": 8,\n    "inner_rmse": 1.693469556628337,\n    "seconds": 17.142665437999995,\n    "training_mse": 2.733973882786215\n  },\n  {\n    "epoch": 9,\n    "inner_rmse": 1.7088107188583082,\n    "seconds": 17.04955747400004,\n    "training_mse": 2.7250993880507064\n  },\n  {\n    "epoch": 10,\n    "inner_rmse": 1.6989122873583749,\n    "seconds": 17.088684078000142,\n    "training_mse": 2.7132440650299805\n  },\n  {\n    "epoch": 11,\n    "inner_rmse": 1.6872485271479447,\n    "seconds": 17.104408853999985,\n    "training_mse": 2.7017225655790877\n  },\n  {\n    "epoch": 12,\n    "inner_rmse": 1.6897413874319267,\n    "seconds": 17.109063442999968,\n    "training_mse": 2.6972855092727976\n  },\n  {\n    "epoch": 13,\n    "inner_rmse": 1.6925758727181621,\n    "seconds": 17.11208512999997,\n    "training_mse": 2.6917051109549117\n  },\n  {\n    "epoch": 14,\n    "inner_rmse": 1.699922542574144,\n    "seconds": 17.118353866000007,\n    "training_mse": 2.689362801917612\n  },\n  {\n    "epoch": 15,\n    "inner_rmse": 1.7010396087412285,\n    "seconds": 17.09363638900004,\n    "training_mse": 2.6743104539505422\n  },\n  {\n    "epoch": 16,\n    "inner_rmse": 1.695022408871176,\n    "seconds": 17.10746793500016,\n    "training_mse": 2.675780120777757\n  },\n  {\n    "epoch": 17,\n    "inner_rmse": 1.6861351065396544,\n    "seconds": 17.093800346999842,\n    "training_mse": 2.666918686807972\n  },\n  {\n    "epoch": 18,\n    "inner_rmse": 1.6920882278242306,\n    "seconds": 17.085230065000133,\n    "training_mse": 2.6678464624979723\n  },\n  {\n    "epoch": 19,\n    "inner_rmse": 1.6926106187423333,\n    "seconds": 17.098964798999987,\n    "training_mse": 2.661620078021533\n  },\n  {\n    "epoch": 20,\n    "inner_rmse": 1.684984263066817,\n    "seconds": 17.082188578000114,\n    "training_mse": 2.6520411274204516\n  },\n  {\n    "epoch": 21,\n    "inner_rmse": 1.6887272631193557,\n    "seconds": 17.071532552999997,\n    "training_mse": 2.650541953844567\n  },\n  {\n    "epoch": 22,\n    "inner_rmse": 1.711067270828508,\n    "seconds": 17.07430837999982,\n    "training_mse": 2.6384226598151743\n  },\n  {\n    "epoch": 23,\n    "inner_rmse": 1.6863638485333814,\n    "seconds": 17.091333364000093,\n    "training_mse": 2.643302670080368\n  },\n  {\n    "epoch": 24,\n    "inner_rmse": 1.6817058611264915,\n    "seconds": 17.068144705000122,\n    "training_mse": 2.634634097145028\n  },\n  {\n    "epoch": 25,\n    "inner_rmse": 1.6876467897171312,\n    "seconds": 17.105544388999988,\n    "training_mse": 2.6253789418364226\n  },\n  {\n    "epoch": 26,\n    "inner_rmse": 1.6802359360382977,\n    "seconds": 17.0581933819999,\n    "training_mse": 2.623683120289894\n  },\n  {\n    "epoch": 27,\n    "inner_rmse": 1.6823927971269081,\n    "seconds": 17.05577739199998,\n    "training_mse": 2.6216745327596795\n  },\n  {\n    "epoch": 28,\n    "inner_rmse": 1.686683852804504,\n    "seconds": 17.049376748999975,\n    "training_mse": 2.619477779081423\n  },\n  {\n    "epoch": 29,\n    "inner_rmse": 1.7061369590969762,\n    "seconds": 17.06555176100005,\n    "training_mse": 2.6086261868476868\n  },\n  {\n    "epoch": 30,\n    "inner_rmse": 1.684302638470296,\n    "seconds": 17.060937099000057,\n    "training_mse": 2.6042220665983957\n  },\n  {\n    "epoch": 31,\n    "inner_rmse": 1.6750720110018267,\n    "seconds": 17.039852586000052,\n    "training_mse": 2.6026552686952567\n  },\n  {\n    "epoch": 32,\n    "inner_rmse": 1.7194153857412235,\n    "seconds": 17.04825169700007,\n    "training_mse": 2.5918308823076015\n  },\n  {\n    "epoch": 33,\n    "inner_rmse": 1.7078345073404868,\n    "seconds": 17.039033585000197,\n    "training_mse": 2.595689936043465\n  },\n  {\n    "epoch": 34,\n    "inner_rmse": 1.6795206554712074,\n    "seconds": 17.042666213000075,\n    "training_mse": 2.583162702109716\n  },\n  {\n    "epoch": 35,\n    "inner_rmse": 1.6967376383768584,\n    "seconds": 17.047268004999978,\n    "training_mse": 2.5780129832764196\n  },\n  {\n    "epoch": 36,\n    "inner_rmse": 1.7087597022049386,\n    "seconds": 17.043333790999895,\n    "training_mse": 2.5785623894978875\n  },\n  {\n    "epoch": 37,\n    "inner_rmse": 1.6813116156509087,\n    "seconds": 17.07287820900001,\n    "training_mse": 2.5819692472889\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/H2/monthly_metrics.csv': 'modelo,bloco,mes_alvo,ano,mes,regiao,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area\nunet,H2,2009-01-01,2009,1,dominio_inteiro,78561,252775.18806960073,-7492.205401293933,87362.77266860381,238498.17941015362,67475.78503925544\nunet,H2,2009-01-01,2009,1,lat_menor_que_menos35,26100,22290.225649015218,3900.2423256859183,17365.437888644636,15564.080229951578,17451.68248953376\nunet,H2,2009-01-01,2009,1,lat_menos35_a_menos15,20880,63001.67278930362,-1574.2689732406288,22980.11569623463,56174.58202428862,18808.59412325443\nunet,H2,2009-01-01,2009,1,lat_maior_igual_menos15,31581,167483.2896312819,-9818.178753739223,47017.21908372454,166759.51715591337,31215.50842646724\nunet,H2,2009-02-01,2009,2,dominio_inteiro,78561,308663.7248854171,-27675.303101310506,91099.3575753998,289652.4090661859,67475.78503925544\nunet,H2,2009-02-01,2009,2,lat_menor_que_menos35,26100,45272.772563858285,-11006.337470911443,21137.252879418433,30815.518930203812,17451.68248953376\nunet,H2,2009-02-01,2009,2,lat_menos35_a_menos15,20880,35038.77685192427,-522.2408129535615,17969.70650364086,31743.408215008014,18808.59412325443\nunet,H2,2009-02-01,2009,2,lat_maior_igual_menos15,31581,228352.1754696346,-16146.724817445502,51992.39819234051,227093.48192097404,31215.50842646724\nunet,H2,2009-03-01,2009,3,dominio_inteiro,78561,243517.82153647678,3485.3097239732742,87781.70903331414,227063.5446490277,67475.78503925544\nunet,H2,2009-03-01,2009,3,lat_menor_que_menos35,26100,30599.297346938634,4356.90840035677,20823.277727782726,21517.796651747056,17451.68248953376\nunet,H2,2009-03-01,2009,3,lat_menos35_a_menos15,20880,65833.21734696315,2454.941270377487,23555.544664792717,59095.14232936069,18808.59412325443\nunet,H2,2009-03-01,2009,3,lat_maior_igual_menos15,31581,147085.30684257502,-3326.5399467609823,43402.886640738696,146450.60566791994,31215.50842646724\nunet,H2,2009-04-01,2009,4,dominio_inteiro,78561,321442.63969077327,5453.027328731492,105451.25901330821,297659.25565095415,67475.78503925544\nunet,H2,2009-04-01,2009,4,lat_menor_que_menos35,26100,36156.39772776807,-1120.4592598676682,23599.58771121502,25746.0487435356,17451.68248953376\nunet,H2,2009-04-01,2009,4,lat_menos35_a_menos15,20880,119542.18666117653,7106.9811562029645,35084.05777457822,107198.78786194958,18808.59412325443\nunet,H2,2009-04-01,2009,4,lat_maior_igual_menos15,31581,165744.05530182863,-533.4945676038042,46767.61352751497,164714.419045469,31215.50842646724\nunet,H2,2009-05-01,2009,5,dominio_inteiro,78561,313477.6579485381,4791.977613204159,94903.47136777919,299500.4429143658,67475.78503925544\nunet,H2,2009-05-01,2009,5,lat_menor_que_menos35,26100,33582.13394278009,-9619.361582994461,19803.767753481865,23151.25393154172,17451.68248953376\nunet,H2,2009-05-01,2009,5,lat_menos35_a_menos15,20880,18906.31044644715,10561.05554807838,14811.404545762576,16754.90282230188,18808.59412325443\nunet,H2,2009-05-01,2009,5,lat_maior_igual_menos15,31581,260989.21355931085,3850.2836481202394,60288.29906853475,259594.28616052217,31215.50842646724\nunet,H2,2009-06-01,2009,6,dominio_inteiro,78561,180603.22649096476,-5799.068077055737,75976.9869262483,167057.49342544124,67475.78503925544\nunet,H2,2009-06-01,2009,6,lat_menor_que_menos35,26100,27619.321419287935,1043.959107041359,19789.211562991142,18661.417275652653,17451.68248953376\nunet,H2,2009-06-01,2009,6,lat_menos35_a_menos15,20880,32657.15095576597,-1849.5845496617258,17848.2353653796,28811.90412973418,18808.59412325443\nunet,H2,2009-06-01,2009,6,lat_maior_igual_menos15,31581,120326.75411591087,-4993.442634435371,38339.53999787755,119584.1720200544,31215.50842646724\nunet,H2,2009-07-01,2009,7,dominio_inteiro,78561,243520.5822422428,-2584.1661373898387,80999.38411192968,227437.35451325116,67475.78503925544\nunet,H2,2009-07-01,2009,7,lat_menor_que_menos35,26100,30712.569448055478,2946.4365941286087,19536.82999920845,22647.82332575654,17451.68248953376\nunet,H2,2009-07-01,2009,7,lat_menos35_a_menos15,20880,53071.78910506266,-5200.1809330917895,19068.691548392177,46432.273274770414,18808.59412325443\nunet,H2,2009-07-01,2009,7,lat_maior_igual_menos15,31581,159736.22368912466,-330.4217984266579,42393.86256432906,158357.25791272428,31215.50842646724\nunet,H2,2009-08-01,2009,8,dominio_inteiro,78561,300578.2727562928,-15681.596033963375,82170.16801118385,282567.2974906463,67475.78503925544\nunet,H2,2009-08-01,2009,8,lat_menor_que_menos35,26100,49077.954077497845,-4766.146117448807,22983.931181311607,36068.57540198745,17451.68248953376\nunet,H2,2009-08-01,2009,8,lat_menos35_a_menos15,20880,18708.886912887276,2639.0984672065824,13871.450474703684,16388.555966143656,18808.59412325443\nunet,H2,2009-08-01,2009,8,lat_maior_igual_menos15,31581,232791.43176590762,-13554.54838372115,45314.78635516856,230110.16612251516,31215.50842646724\nunet,H2,2009-09-01,2009,9,dominio_inteiro,78561,213558.70593290194,17248.44657676015,78056.81599185336,198147.0656366532,67475.78503925544\nunet,H2,2009-09-01,2009,9,lat_menor_que_menos35,26100,23438.968741158136,2512.560649037361,18053.301958560944,16097.028542488752,17451.68248953376\nunet,H2,2009-09-01,2009,9,lat_menos35_a_menos15,20880,53049.147468749776,-5280.492792001925,20156.130390890874,46059.543299062585,18808.59412325443\nunet,H2,2009-09-01,2009,9,lat_maior_igual_menos15,31581,137070.58972299405,20016.378719724715,39847.383642401546,135990.49379510185,31215.50842646724\nunet,H2,2009-10-01,2009,10,dominio_inteiro,78561,262602.4326809088,-6345.9652700359,84913.73419024143,247718.32540821415,67475.78503925544\nunet,H2,2009-10-01,2009,10,lat_menor_que_menos35,26100,20454.24497468247,-5417.502565443516,16456.007870554924,14486.523596470885,17451.68248953376\nunet,H2,2009-10-01,2009,10,lat_menos35_a_menos15,20880,76476.8468209062,1524.2582192579284,26026.24129428249,69354.90203874954,18808.59412325443\nunet,H2,2009-10-01,2009,10,lat_maior_igual_menos15,31581,165671.34088532015,-2452.720923850313,42431.48502540402,163876.89977299373,31215.50842646724\nunet,H2,2009-11-01,2009,11,dominio_inteiro,78561,250992.81744403334,8793.866504421458,88109.27393817343,228542.04283316788,67475.78503925544\nunet,H2,2009-11-01,2009,11,lat_menor_que_menos35,26100,32205.70215186721,-9782.13667473197,18931.598241955042,23471.793158092096,17451.68248953376\nunet,H2,2009-11-01,2009,11,lat_menos35_a_menos15,20880,112591.45603577007,3576.157637571916,29735.229500556365,99945.15931291587,18808.59412325443\nunet,H2,2009-11-01,2009,11,lat_maior_igual_menos15,31581,106195.65925639609,14999.845541581511,39442.44619566202,105125.09036215988,31215.50842646724\nunet,H2,2009-12-01,2009,12,dominio_inteiro,78561,224108.00703721977,-6306.368300301023,85911.86869810335,211043.52646642024,67475.78503925544\nunet,H2,2009-12-01,2009,12,lat_menor_que_menos35,26100,22036.485044750025,3157.2808807194233,18278.057145923376,15510.887633034821,17451.68248953376\nunet,H2,2009-12-01,2009,12,lat_menos35_a_menos15,20880,61274.749300657204,-6794.31643362809,23013.846792852506,55753.60264339016,18808.59412325443\nunet,H2,2009-12-01,2009,12,lat_maior_igual_menos15,31581,140796.77269181257,-2669.3327473923564,44619.96475932747,139779.03618999527,31215.50842646724\nunet,H2,2010-01-01,2010,1,dominio_inteiro,78561,310482.71718484507,10916.894601400942,96361.86459398642,278216.2336552829,67475.78503925544\nunet,H2,2010-01-01,2010,1,lat_menor_que_menos35,26100,46808.105150564836,-8125.709176555276,21413.184300526977,31354.815453833347,17451.68248953376\nunet,H2,2010-01-01,2010,1,lat_menos35_a_menos15,20880,159362.6420440874,13704.170227313414,37880.90222997032,143629.69069746742,18808.59412325443\nunet,H2,2010-01-01,2010,1,lat_maior_igual_menos15,31581,104311.96999019281,5338.433550642803,37067.778063489124,103231.72750398214,31215.50842646724\nunet,H2,2010-02-01,2010,2,dominio_inteiro,78561,411888.9693455194,-14878.256913150195,118658.84778606752,378215.4700743398,67475.78503925544\nunet,H2,2010-02-01,2010,2,lat_menor_que_menos35,26100,65555.92576422979,-22060.954790830612,30608.34701511264,46623.50365002842,17451.68248953376\nunet,H2,2010-02-01,2010,2,lat_menos35_a_menos15,20880,127251.69278443101,8333.110782240983,34640.95030528167,113780.55092159913,18808.59412325443\nunet,H2,2010-02-01,2010,2,lat_maior_igual_menos15,31581,219081.35079685858,-1150.412904560566,53409.55046567321,217811.41550271225,31215.50842646724\nunet,H2,2010-03-01,2010,3,dominio_inteiro,78561,270165.29108471633,12272.99496731488,87384.11032321723,256266.21604912027,67475.78503925544\nunet,H2,2010-03-01,2010,3,lat_menor_que_menos35,26100,23119.281519897177,4752.453780591488,17950.13478344679,16661.6150590767,17451.68248953376\nunet,H2,2010-03-01,2010,3,lat_menos35_a_menos15,20880,61950.08683648727,978.0963851395063,22459.572378006298,55342.330364562695,18808.59412325443\nunet,H2,2010-03-01,2010,3,lat_maior_igual_menos15,31581,185095.92272833188,6542.444801583886,46974.403161764145,184262.27062548086,31215.50842646724\nunet,H2,2010-04-01,2010,4,dominio_inteiro,78561,282211.63473744306,-10259.33283507824,91664.00197537988,270041.66818224086,67475.78503925544\nunet,H2,2010-04-01,2010,4,lat_menor_que_menos35,26100,21642.41445642103,8683.114920586348,18097.85258999467,15721.041907837007,17451.68248953376\nunet,H2,2010-04-01,2010,4,lat_menos35_a_menos15,20880,49361.37545578342,1475.7950978726149,21200.009690396488,44289.59296254215,18808.59412325443\nunet,H2,2010-04-01,2010,4,lat_maior_igual_menos15,31581,211207.84482523863,-20418.242853537202,52366.13969498873,210031.03331186177,31215.50842646724\nunet,H2,2010-05-01,2010,5,dominio_inteiro,78561,336710.03023876715,-7254.105060597882,101989.43120237999,312070.2187973988,67475.78503925544\nunet,H2,2010-05-01,2010,5,lat_menor_que_menos35,26100,55737.535595730806,2142.207574903965,29076.51276165247,37512.41083796219,17451.68248953376\nunet,H2,2010-05-01,2010,5,lat_menos35_a_menos15,20880,44208.6158917236,-7205.279453735799,20483.482352335006,39352.67811422702,18808.59412325443\nunet,H2,2010-05-01,2010,5,lat_maior_igual_menos15,31581,236763.8787513128,-2191.033181766048,52429.43608839251,235205.12984520962,31215.50842646724\nunet,H2,2010-06-01,2010,6,dominio_inteiro,78561,189890.27978514013,-7711.088956305757,72682.19079930149,178090.1664009557,67475.78503925544\nunet,H2,2010-06-01,2010,6,lat_menor_que_menos35,26100,28106.446462428892,-7358.528256639838,18883.433763459325,19000.33527752437,17451.68248953376\nunet,H2,2010-06-01,2010,6,lat_menos35_a_menos15,20880,12905.08771492087,-1508.7884822189808,9996.9714743644,11578.908187761623,18808.59412325443\nunet,H2,2010-06-01,2010,6,lat_maior_igual_menos15,31581,148878.74560779036,1156.2277825530618,43801.785561477765,147510.9229356697,31215.50842646724\nunet,H2,2010-07-01,2010,7,dominio_inteiro,78561,203509.6828355332,-23725.17064587027,75636.83005420119,190651.16501901278,67475.78503925544\nunet,H2,2010-07-01,2010,7,lat_menor_que_menos35,26100,32788.62593551107,-8085.702098578215,20452.92788091302,24130.03137807725,17451.68248953376\nunet,H2,2010-07-01,2010,7,lat_menos35_a_menos15,20880,24315.41209606056,-4001.6971445046365,14403.02068638429,21314.089618580605,18808.59412325443\nunet,H2,2010-07-01,2010,7,lat_maior_igual_menos15,31581,146405.64480396156,-11637.771402787417,40780.881486903876,145207.04402235494,31215.50842646724\nunet,H2,2010-08-01,2010,8,dominio_inteiro,78561,151884.50422159318,-735.2084204740822,60803.19099851325,144360.85917108384,67475.78503925544\nunet,H2,2010-08-01,2010,8,lat_menor_que_menos35,26100,14713.7236826521,683.1315050721169,14874.925982415676,10112.294029593319,17451.68248953376\nunet,H2,2010-08-01,2010,8,lat_menos35_a_menos15,20880,11297.088463161412,2209.0845644064248,9543.833298187703,9799.619313599622,18808.59412325443\nunet,H2,2010-08-01,2010,8,lat_maior_igual_menos15,31581,125873.69207577966,-3627.424489952624,36384.43171790987,124448.94582789093,31215.50842646724\nunet,H2,2010-09-01,2010,9,dominio_inteiro,78561,172594.39970487115,-2511.071307050064,68268.53683690354,164107.0921501505,67475.78503925544\nunet,H2,2010-09-01,2010,9,lat_menor_que_menos35,26100,15221.388315031922,-2138.4564770162106,14282.190099149942,11331.786009228643,17451.68248953376\nunet,H2,2010-09-01,2010,9,lat_menos35_a_menos15,20880,28940.892157601866,784.2905997540802,16005.746022190899,25616.059075186324,18808.59412325443\nunet,H2,2010-09-01,2010,9,lat_maior_igual_menos15,31581,128432.11923223735,-1156.9054297879338,37980.6007155627,127159.24706573556,31215.50842646724\nunet,H2,2010-10-01,2010,10,dominio_inteiro,78561,218031.50037622295,16930.098161182366,83140.51503183227,202901.46656055516,67475.78503925544\nunet,H2,2010-10-01,2010,10,lat_menor_que_menos35,26100,30391.73629005658,3417.641065120697,21232.840186834335,20665.79419243334,17451.68248953376\nunet,H2,2010-10-01,2010,10,lat_menos35_a_menos15,20880,28160.86932849235,-308.744157881476,16062.987876701169,24816.778599074147,18808.59412325443\nunet,H2,2010-10-01,2010,10,lat_maior_igual_menos15,31581,159478.89475767402,13821.201253943145,45844.686968296766,157418.89376904772,31215.50842646724\nunet,H2,2010-11-01,2010,11,dominio_inteiro,78561,238016.35107847207,-4407.297063706443,81231.2699619662,227352.68495591776,67475.78503925544\nunet,H2,2010-11-01,2010,11,lat_menor_que_menos35,26100,14043.001905674684,1018.6371804177761,14512.655189663172,9768.755240421951,17451.68248953376\nunet,H2,2010-11-01,2010,11,lat_menos35_a_menos15,20880,43691.934555828775,1107.088742947206,20343.709991509095,39418.357035501554,18808.59412325443\nunet,H2,2010-11-01,2010,11,lat_maior_igual_menos15,31581,180281.4146169686,-6533.022987071425,46374.904780793935,178165.57267999428,31215.50842646724\nunet,H2,2010-12-01,2010,12,dominio_inteiro,78561,251949.05321977675,-11890.75306247361,83736.90349962376,235581.9637891309,67475.78503925544\nunet,H2,2010-12-01,2010,12,lat_menor_que_menos35,26100,20059.01410821,-3312.72611887753,17319.11436136067,13557.766112419175,17451.68248953376\nunet,H2,2010-12-01,2010,12,lat_menos35_a_menos15,20880,77777.2022020167,-4256.665392065421,24520.393145797774,69264.45646032748,18808.59412325443\nunet,H2,2010-12-01,2010,12,lat_maior_igual_menos15,31581,154112.83690955004,-4321.361551530659,41897.39599246532,152759.74121638422,31215.50842646724\nhybrid,H2,2009-01-01,2009,1,dominio_inteiro,78561,244494.02690625738,-10931.310783796838,85289.94137703218,230798.30908050443,67475.78503925544\nhybrid,H2,2009-01-01,2009,1,lat_menor_que_menos35,26100,20753.924304751417,4348.935675655181,16520.165896865947,14651.357276316267,17451.68248953376\nhybrid,H2,2009-01-01,2009,1,lat_menos35_a_menos15,20880,64364.89077576593,198.14407106463295,23574.313720898543,57481.57214392594,18808.59412325443\nhybrid,H2,2009-01-01,2009,1,lat_maior_igual_menos15,31581,159375.21182574003,-15478.390530516652,45195.46175926769,158665.37966026223,31215.50842646724\nhybrid,H2,2009-02-01,2009,2,dominio_inteiro,78561,293429.39688239194,-26709.430864794333,89470.9584839497,274654.9507469542,67475.78503925544\nhybrid,H2,2009-02-01,2009,2,lat_menor_que_menos35,26100,44808.09501447987,-11310.538137672918,21267.58001096014,30512.6406203976,17451.68248953376\nhybrid,H2,2009-02-01,2009,2,lat_menos35_a_menos15,20880,34246.408012694126,2527.0295149196277,17905.40755694292,30919.804874063062,18808.59412325443\nhybrid,H2,2009-02-01,2009,2,lat_maior_igual_menos15,31581,214374.89385521793,-17925.92224204104,50297.97091604662,213222.50525249352,31215.50842646724\nhybrid,H2,2009-03-01,2009,3,dominio_inteiro,78561,232936.40525577095,-5336.524801242313,84753.22503953219,218678.06917668215,67475.78503925544\nhybrid,H2,2009-03-01,2009,3,lat_menor_que_menos35,26100,23834.935084902838,1230.4679904291856,18951.709844328798,16227.240380698253,17451.68248953376\nhybrid,H2,2009-03-01,2009,3,lat_menos35_a_menos15,20880,61445.03600226213,1437.7248065248273,22991.91628752211,55432.54742568433,18808.59412325443\nhybrid,H2,2009-03-01,2009,3,lat_maior_igual_menos15,31581,147656.43416860595,-8004.717598196328,42809.5989076813,147018.28137029958,31215.50842646724\nhybrid,H2,2009-04-01,2009,4,dominio_inteiro,78561,372977.51831406483,-3757.4410302897695,111805.89762421958,349284.6502551462,67475.78503925544\nhybrid,H2,2009-04-01,2009,4,lat_menor_que_menos35,26100,34604.5410229298,-3673.207116077894,22898.090094217685,24559.059777492374,17451.68248953376\nhybrid,H2,2009-04-01,2009,4,lat_menos35_a_menos15,20880,118098.36989266687,7564.7798834274345,34982.919560772534,105806.6894371496,18808.59412325443\nhybrid,H2,2009-04-01,2009,4,lat_maior_igual_menos15,31581,220274.60739846822,-7649.013797639309,53924.88796922934,218918.90104050422,31215.50842646724\nhybrid,H2,2009-05-01,2009,5,dominio_inteiro,78561,375688.49128918425,5176.976483186643,102217.42378635085,361051.66744710715,67475.78503925544\nhybrid,H2,2009-05-01,2009,5,lat_menor_que_menos35,26100,33990.64149986057,-8442.965950998028,19611.01357996045,23529.465932081093,17451.68248953376\nhybrid,H2,2009-05-01,2009,5,lat_menos35_a_menos15,20880,22633.52511470721,12450.922105816113,16304.608544319879,20016.65908570033,18808.59412325443\nhybrid,H2,2009-05-01,2009,5,lat_maior_igual_menos15,31581,319064.3246746164,1169.0203283685632,66301.8016620705,317505.5424293258,31215.50842646724\nhybrid,H2,2009-06-01,2009,6,dominio_inteiro,78561,183349.12046636496,-1072.977751542725,77224.06209841013,169291.66413481522,67475.78503925544\nhybrid,H2,2009-06-01,2009,6,lat_menor_que_menos35,26100,29545.327257216744,3362.9587666051048,20367.436124713753,20142.360225710134,17451.68248953376\nhybrid,H2,2009-06-01,2009,6,lat_menos35_a_menos15,20880,33451.11221536132,-568.5863810093567,18257.875153643083,29456.175170192437,18808.59412325443\nhybrid,H2,2009-06-01,2009,6,lat_maior_igual_menos15,31581,120352.68099378687,-3867.3501371384737,38598.750820053305,119693.12873891267,31215.50842646724\nhybrid,H2,2009-07-01,2009,7,dominio_inteiro,78561,220852.81085370743,-4392.930208228265,76676.32460416973,204640.16506400885,67475.78503925544\nhybrid,H2,2009-07-01,2009,7,lat_menor_que_menos35,26100,32186.911648798152,2442.506741140715,19696.602358271324,23798.636556543875,17451.68248953376\nhybrid,H2,2009-07-01,2009,7,lat_menos35_a_menos15,20880,53306.45246867735,-5530.258667492319,18988.193037991303,46670.587348243644,18808.59412325443\nhybrid,H2,2009-07-01,2009,7,lat_maior_igual_menos15,31581,135359.44673623194,-1305.1782818766605,37991.5292079071,134170.94115922134,31215.50842646724\nhybrid,H2,2009-08-01,2009,8,dominio_inteiro,78561,302197.9372406525,-26010.81512389564,77883.56771631716,284048.8976338566,67475.78503925544\nhybrid,H2,2009-08-01,2009,8,lat_menor_que_menos35,26100,50282.606482934934,-7367.387094147688,22662.802639485213,36930.976562267315,17451.68248953376\nhybrid,H2,2009-08-01,2009,8,lat_menos35_a_menos15,20880,16803.094562389142,-225.8165316608605,12290.243479331959,14696.295037186628,18808.59412325443\nhybrid,H2,2009-08-01,2009,8,lat_maior_igual_menos15,31581,235112.23619532847,-18417.61149808709,42930.521597499974,232421.62603440264,31215.50842646724\nhybrid,H2,2009-09-01,2009,9,dominio_inteiro,78561,201925.16493415902,18926.80352368347,77335.92362834171,186261.31087372525,67475.78503925544\nhybrid,H2,2009-09-01,2009,9,lat_menor_que_menos35,26100,23841.15196325225,1367.3686219042506,17716.720873594695,16329.808516686513,17451.68248953376\nhybrid,H2,2009-09-01,2009,9,lat_menos35_a_menos15,20880,54186.12685764566,-6939.640316003013,20238.048241314897,47023.87575736983,18808.59412325443\nhybrid,H2,2009-09-01,2009,9,lat_maior_igual_menos15,31581,123897.8861132611,24499.07521778223,39381.15451343212,122907.62659966893,31215.50842646724\nhybrid,H2,2009-10-01,2009,10,dominio_inteiro,78561,246392.67405736927,-3946.7502378281642,82946.65156191532,231603.7605442048,67475.78503925544\nhybrid,H2,2009-10-01,2009,10,lat_menor_que_menos35,26100,19742.215771706382,-5289.523746016151,16020.165091335599,13926.398606322013,17451.68248953376\nhybrid,H2,2009-10-01,2009,10,lat_menos35_a_menos15,20880,79342.1818645179,48.885942884742235,26340.850302647723,71945.4554227182,18808.59412325443\nhybrid,H2,2009-10-01,2009,10,lat_maior_igual_menos15,31581,147308.27642114498,1293.8875653032446,40585.636167931996,145731.90651516456,31215.50842646724\nhybrid,H2,2009-11-01,2009,11,dominio_inteiro,78561,263579.5004868824,6102.938008426183,89958.67456501159,240081.24171569844,67475.78503925544\nhybrid,H2,2009-11-01,2009,11,lat_menor_que_menos35,26100,34207.64988796648,-11376.989936737786,19707.19919324393,25104.809491149877,17451.68248953376\nhybrid,H2,2009-11-01,2009,11,lat_menos35_a_menos15,20880,118731.3924987365,2917.4315084482296,30341.809244021984,105432.42478823528,18808.59412325443\nhybrid,H2,2009-11-01,2009,11,lat_maior_igual_menos15,31581,110640.4581001794,14562.49643671574,39909.66612774568,109544.00743631329,31215.50842646724\nhybrid,H2,2009-12-01,2009,12,dominio_inteiro,78561,213732.66728744746,-5326.297976184293,83195.35112790941,201078.7069484338,67475.78503925544\nhybrid,H2,2009-12-01,2009,12,lat_menor_que_menos35,26100,19957.78583157752,2990.3365948523533,16771.177824571987,14132.6849950395,17451.68248953376\nhybrid,H2,2009-12-01,2009,12,lat_menos35_a_menos15,20880,63581.38267664273,-7968.294322226781,23287.919793615234,57715.07897017457,18808.59412325443\nhybrid,H2,2009-12-01,2009,12,lat_maior_igual_menos15,31581,130193.49877922723,-348.3402488098627,43136.25350972219,129230.94298321973,31215.50842646724\nhybrid,H2,2010-01-01,2010,1,dominio_inteiro,78561,291892.2451084307,5141.231579086716,93338.01891003837,259914.80823057264,67475.78503925544\nhybrid,H2,2010-01-01,2010,1,lat_menor_que_menos35,26100,49588.91291069535,-11844.126978361339,21233.04153706701,32901.32274743076,17451.68248953376\nhybrid,H2,2010-01-01,2010,1,lat_menos35_a_menos15,20880,143158.3563030285,10357.649629972184,35128.41603199282,128867.39031144237,18808.59412325443\nhybrid,H2,2010-01-01,2010,1,lat_maior_igual_menos15,31581,99144.97589470682,6627.708927475873,36976.561340978544,98146.0951716995,31215.50842646724\nhybrid,H2,2010-02-01,2010,2,dominio_inteiro,78561,375794.50545459054,-8316.206989039958,113295.46911097852,343932.64513289864,67475.78503925544\nhybrid,H2,2010-02-01,2010,2,lat_menor_que_menos35,26100,69058.3631297384,-23066.1907173652,31646.568577974584,49919.91111637377,17451.68248953376\nhybrid,H2,2010-02-01,2010,2,lat_menos35_a_menos15,20880,105585.09257193371,5626.25217965402,30843.10900318733,93957.68716192912,18808.59412325443\nhybrid,H2,2010-02-01,2010,2,lat_maior_igual_menos15,31581,201151.0497529184,9123.73154867122,50805.79152981663,200055.04685459568,31215.50842646724\nhybrid,H2,2010-03-01,2010,3,dominio_inteiro,78561,264491.80339588213,13413.705005831323,85696.38346613004,251528.49159408474,67475.78503925544\nhybrid,H2,2010-03-01,2010,3,lat_menor_que_menos35,26100,20214.062865475782,2640.688486412772,16377.125149874093,14689.777943990144,17451.68248953376\nhybrid,H2,2010-03-01,2010,3,lat_menos35_a_menos15,20880,64015.473049255976,-967.9819687071986,22924.40337319235,57324.404834580775,18808.59412325443\nhybrid,H2,2010-03-01,2010,3,lat_maior_igual_menos15,31581,180262.26748115034,11740.998488125751,46394.8549430636,179514.3088155138,31215.50842646724\nhybrid,H2,2010-04-01,2010,4,dominio_inteiro,78561,300040.8376626026,-15465.434104064316,94038.55756296775,287872.6167665436,67475.78503925544\nhybrid,H2,2010-04-01,2010,4,lat_menor_que_menos35,26100,22071.617423666263,5878.566368042117,18154.776862080533,16145.1122870057,17451.68248953376\nhybrid,H2,2010-04-01,2010,4,lat_menos35_a_menos15,20880,48973.277875377564,1850.0621582923964,21589.29687996671,44052.289230125505,18808.59412325443\nhybrid,H2,2010-04-01,2010,4,lat_maior_igual_menos15,31581,228995.94236355874,-23194.06263039883,54294.48382092049,227675.21524941243,31215.50842646724\nhybrid,H2,2010-05-01,2010,5,dominio_inteiro,78561,319259.7224293634,-11376.097743972285,97999.52407219581,295349.6738408664,67475.78503925544\nhybrid,H2,2010-05-01,2010,5,lat_menor_que_menos35,26100,54641.685523063694,431.1589422601651,27584.2194545572,36776.77845838159,17451.68248953376\nhybrid,H2,2010-05-01,2010,5,lat_menos35_a_menos15,20880,43157.15969412999,-5666.339318715694,19912.98248603875,38448.74474325697,18808.59412325443\nhybrid,H2,2010-05-01,2010,5,lat_maior_igual_menos15,31581,221460.87721216975,-6140.917367516753,50502.32213159987,220124.15063922785,31215.50842646724\nhybrid,H2,2010-06-01,2010,6,dominio_inteiro,78561,160902.27205473208,-5932.578973992791,67687.39006566312,148988.05754239333,67475.78503925544\nhybrid,H2,2010-06-01,2010,6,lat_menor_que_menos35,26100,28331.11733308795,-8095.954368312484,18746.761925317358,19000.979885418816,17451.68248953376\nhybrid,H2,2010-06-01,2010,6,lat_menos35_a_menos15,20880,17324.75706552223,3512.7188914608896,12153.381949082022,15620.576664673605,18808.59412325443\nhybrid,H2,2010-06-01,2010,6,lat_maior_igual_menos15,31581,115246.39765612187,-1349.343497141198,36787.24619126372,114366.50099230089,31215.50842646724\nhybrid,H2,2010-07-01,2010,7,dominio_inteiro,78561,204658.03373986355,-14457.406651687808,73322.74727300208,192454.57724632788,67475.78503925544\nhybrid,H2,2010-07-01,2010,7,lat_menor_que_menos35,26100,32254.89426410252,-8894.12592832871,20258.615934110738,23629.36714364767,17451.68248953376\nhybrid,H2,2010-07-01,2010,7,lat_menos35_a_menos15,20880,18537.720619696775,2047.9603436071322,12787.358102299591,16106.766708512787,18808.59412325443\nhybrid,H2,2010-07-01,2010,7,lat_maior_igual_menos15,31581,153865.41885606427,-7611.2410669662295,40276.77323659176,152718.44339416744,31215.50842646724\nhybrid,H2,2010-08-01,2010,8,dominio_inteiro,78561,161151.3538490141,9429.882563499155,65044.204628322535,153426.99985301818,67475.78503925544\nhybrid,H2,2010-08-01,2010,8,lat_menor_que_menos35,26100,13947.368126534455,-1181.141956051687,14404.227305620845,9480.506482887155,17451.68248953376\nhybrid,H2,2010-08-01,2010,8,lat_menos35_a_menos15,20880,14835.483204306307,5249.835495033662,11438.156951524361,12978.754589653056,18808.59412325443\nhybrid,H2,2010-08-01,2010,8,lat_maior_igual_menos15,31581,132368.50251817334,5361.1890245171835,39201.820371177324,130967.73878047797,31215.50842646724\nhybrid,H2,2010-09-01,2010,9,dominio_inteiro,78561,191522.07247195294,-1588.9248334313024,72031.63836557034,182420.8006870866,67475.78503925544\nhybrid,H2,2010-09-01,2010,9,lat_menor_que_menos35,26100,16588.232790410028,-4025.4184067932806,14693.542003376318,12432.986434522772,17451.68248953376\nhybrid,H2,2010-09-01,2010,9,lat_menos35_a_menos15,20880,30417.586472651747,-1756.7092411053059,16063.578513406785,26887.99354396738,18808.59412325443\nhybrid,H2,2010-09-01,2010,9,lat_maior_igual_menos15,31581,144516.2532088912,4193.202814467284,41274.51784878723,143099.82070859644,31215.50842646724\nhybrid,H2,2010-10-01,2010,10,dominio_inteiro,78561,196721.89717495145,11026.79230672523,76992.35530476923,183442.35428863097,67475.78503925544\nhybrid,H2,2010-10-01,2010,10,lat_menor_que_menos35,26100,24553.722894934814,-700.6008533714529,18684.1096743811,16747.050266355312,17451.68248953376\nhybrid,H2,2010-10-01,2010,10,lat_menos35_a_menos15,20880,30474.275618559197,-2499.100763734611,16228.588960463756,26713.771001064273,18808.59412325443\nhybrid,H2,2010-10-01,2010,10,lat_maior_igual_menos15,31581,141693.89866145744,14226.493923831295,42079.65666992437,139981.5330212114,31215.50842646724\nhybrid,H2,2010-11-01,2010,11,dominio_inteiro,78561,234198.77229094034,-5404.717007337582,79744.25857126941,224044.55322044293,67475.78503925544\nhybrid,H2,2010-11-01,2010,11,lat_menor_que_menos35,26100,12211.314856046716,-1238.1019517616219,13323.959840081143,8553.205189864771,17451.68248953376\nhybrid,H2,2010-11-01,2010,11,lat_menos35_a_menos15,20880,45558.24384562098,108.83585604678342,20236.363430551326,41109.0580011666,18808.59412325443\nhybrid,H2,2010-11-01,2010,11,lat_maior_igual_menos15,31581,176429.21358927267,-4275.450911622744,46183.935300636935,174382.29002941155,31215.50842646724\nhybrid,H2,2010-12-01,2010,12,dominio_inteiro,78561,250864.4389974653,-5495.385279395812,84850.16623795609,234689.56310508045,67475.78503925544\nhybrid,H2,2010-12-01,2010,12,lat_menor_que_menos35,26100,18793.89152706669,-3229.893198085917,16676.36611608081,12784.039588435815,17451.68248953376\nhybrid,H2,2010-12-01,2010,12,lat_menos35_a_menos15,20880,81640.34199540796,-5357.542728760359,24848.560249033373,72764.50543566575,18808.59412325443\nhybrid,H2,2010-12-01,2010,12,lat_maior_igual_menos15,31581,150430.20547499065,3092.0506474504627,43325.23987284189,149141.01808097886,31215.50842646724\nclimatology,H2,2009-01-01,2009,1,dominio_inteiro,78561,283751.60518121684,-22933.452250016562,91883.85662634164,269807.7879830377,67475.78503925544\nclimatology,H2,2009-01-01,2009,1,lat_menor_que_menos35,26100,20640.309378574806,4173.14281155169,16410.92840220034,14570.919511966124,17451.68248953376\nclimatology,H2,2009-01-01,2009,1,lat_menos35_a_menos15,20880,67382.1330415634,3384.748717131384,24578.0548237139,60353.03382049089,18808.59412325443\nclimatology,H2,2009-01-01,2009,1,lat_maior_igual_menos15,31581,195729.16276107862,-30491.343778699636,50894.8734004274,194883.83465058068,31215.50842646724\nclimatology,H2,2009-02-01,2009,2,dominio_inteiro,78561,304754.12357583543,-38571.4600360333,91108.30617603194,286096.3440728677,67475.78503925544\nclimatology,H2,2009-02-01,2009,2,lat_menor_que_menos35,26100,44442.910526174855,-11209.613265678287,21531.622418805957,30271.60719759805,17451.68248953376\nclimatology,H2,2009-02-01,2009,2,lat_menos35_a_menos15,20880,35175.23135008154,3775.175827460829,18354.396223138086,31799.141370282166,18808.59412325443\nclimatology,H2,2009-02-01,2009,2,lat_maior_igual_menos15,31581,225135.981699579,-31137.02259781584,51222.287534087896,224025.59550498746,31215.50842646724\nclimatology,H2,2009-03-01,2009,3,dominio_inteiro,78561,293844.68927742215,-9801.634388090111,91992.96209486853,279718.1590009569,67475.78503925544\nclimatology,H2,2009-03-01,2009,3,lat_menor_que_menos35,26100,22575.040172382756,1127.07534173131,18659.207272559404,15384.147044167163,17451.68248953376\nclimatology,H2,2009-03-01,2009,3,lat_menos35_a_menos15,20880,63635.49841279934,1119.8356854980811,23379.66709824931,57388.03184285411,18808.59412325443\nclimatology,H2,2009-03-01,2009,3,lat_maior_igual_menos15,31581,207634.1506922401,-12048.545415319502,49954.08772405982,206945.98011393566,31215.50842646724\nclimatology,H2,2009-04-01,2009,4,dominio_inteiro,78561,569074.1648735206,-1245.5151620637625,133299.57059030235,542658.7475115544,67475.78503925544\nclimatology,H2,2009-04-01,2009,4,lat_menor_que_menos35,26100,39633.37654125397,-5247.308442175388,24279.849505364895,28086.730747527574,17451.68248953376\nclimatology,H2,2009-04-01,2009,4,lat_menos35_a_menos15,20880,124210.24622644692,8294.641919086687,35845.27801718842,111176.50190900537,18808.59412325443\nclimatology,H2,2009-04-01,2009,4,lat_maior_igual_menos15,31581,405230.54210581974,-4292.8486389750615,73174.44306774903,403395.51485502155,31215.50842646724\nclimatology,H2,2009-05-01,2009,5,dominio_inteiro,78561,677573.0316662843,3118.508750338573,131391.8796926872,660761.0108075369,67475.78503925544\nclimatology,H2,2009-05-01,2009,5,lat_menor_que_menos35,26100,35886.675206492204,-9556.274885237217,20434.59162312746,24692.632915309143,17451.68248953376\nclimatology,H2,2009-05-01,2009,5,lat_menos35_a_menos15,20880,24785.29623170768,13183.081501977053,16937.294564991724,21937.60576317385,18808.59412325443\nclimatology,H2,2009-05-01,2009,5,lat_maior_igual_menos15,31581,616901.0602280842,-508.2978664012626,94019.993504568,614130.7721290538,31215.50842646724\nclimatology,H2,2009-06-01,2009,6,dominio_inteiro,78561,285627.8148394744,-10383.860648190603,90534.20315101836,271604.6686259884,67475.78503925544\nclimatology,H2,2009-06-01,2009,6,lat_menor_que_menos35,26100,28573.913765259666,1481.1718991696835,19746.000286191702,19373.045314908748,17451.68248953376\nclimatology,H2,2009-06-01,2009,6,lat_menos35_a_menos15,20880,32203.876893481603,-355.0937641300261,17776.05743376538,28309.342614790443,18808.59412325443\nclimatology,H2,2009-06-01,2009,6,lat_maior_igual_menos15,31581,224850.0241807331,-11509.93878323026,53012.145431061275,223922.28069628915,31215.50842646724\nclimatology,H2,2009-07-01,2009,7,dominio_inteiro,78561,190112.84615422145,-4358.440530572552,70769.72263112618,174902.81639115844,67475.78503925544\nclimatology,H2,2009-07-01,2009,7,lat_menor_que_menos35,26100,30510.74133022855,1491.0740548968315,19387.489767611027,22484.457038877197,17451.68248953376\nclimatology,H2,2009-07-01,2009,7,lat_menos35_a_menos15,20880,51036.13795523527,-5247.0273255249485,18576.478837133385,44704.83791510986,18808.59412325443\nclimatology,H2,2009-07-01,2009,7,lat_maior_igual_menos15,31581,108565.9668687576,-602.4872599444352,32805.754026381765,107713.52143717141,31215.50842646724\nclimatology,H2,2009-08-01,2009,8,dominio_inteiro,78561,262544.9576026132,-16862.67465367727,76377.19985620119,245440.75380791683,67475.78503925544\nclimatology,H2,2009-08-01,2009,8,lat_menor_que_menos35,26100,49545.26817586252,-6404.145803630352,22987.731282889843,36424.947931251954,17451.68248953376\nclimatology,H2,2009-08-01,2009,8,lat_menos35_a_menos15,20880,15125.743273598779,-1296.018982835114,11506.687547372654,13302.13253932576,18808.59412325443\nclimatology,H2,2009-08-01,2009,8,lat_maior_igual_menos15,31581,197873.94615315192,-9162.509867211804,41882.78102593869,195713.67333733913,31215.50842646724\nclimatology,H2,2009-09-01,2009,9,dominio_inteiro,78561,211637.60338758034,19174.342022824334,78345.69086927664,196154.9739919349,67475.78503925544\nclimatology,H2,2009-09-01,2009,9,lat_menor_que_menos35,26100,23180.590315558482,386.7287418246269,17283.428896963596,15869.682620898278,17451.68248953376\nclimatology,H2,2009-09-01,2009,9,lat_menos35_a_menos15,20880,53455.12922856356,-7412.852780934656,19883.794589046156,46444.64265166046,18808.59412325443\nclimatology,H2,2009-09-01,2009,9,lat_maior_igual_menos15,31581,135001.88384345832,26200.466061934363,41178.46738326689,133840.6487193762,31215.50842646724\nclimatology,H2,2009-10-01,2009,10,dominio_inteiro,78561,257380.58313658845,-4844.060113686166,84913.96514676602,242390.0201649638,67475.78503925544\nclimatology,H2,2009-10-01,2009,10,lat_menor_que_menos35,26100,19596.92960086678,-6158.778152614832,16418.88377997279,13699.742401300551,17451.68248953376\nclimatology,H2,2009-10-01,2009,10,lat_menos35_a_menos15,20880,80807.27585967888,-1709.4550384516187,26313.27928532564,73540.7386082105,18808.59412325443\nclimatology,H2,2009-10-01,2009,10,lat_maior_igual_menos15,31581,156976.3776760428,3024.1730773802847,42181.802081467584,155149.53915545275,31215.50842646724\nclimatology,H2,2009-11-01,2009,11,dominio_inteiro,78561,288026.5804534099,11979.434205429454,95916.815809449,264557.2508376122,67475.78503925544\nclimatology,H2,2009-11-01,2009,11,lat_menor_que_menos35,26100,31851.126154817055,-10651.873009562492,19500.79301136732,23401.27779551915,17451.68248953376\nclimatology,H2,2009-11-01,2009,11,lat_menos35_a_menos15,20880,120031.0401643647,856.3694920109701,29937.395498303522,106291.22490998273,18808.59412325443\nclimatology,H2,2009-11-01,2009,11,lat_maior_igual_menos15,31581,136144.41413422808,21774.937722980976,46478.62729977816,134864.74813211034,31215.50842646724\nclimatology,H2,2009-12-01,2009,12,dominio_inteiro,78561,215449.3103301258,2140.5302820120705,84897.03963912732,202399.59360709836,67475.78503925544\nclimatology,H2,2009-12-01,2009,12,lat_menor_que_menos35,26100,20030.989381612802,2590.644618958235,16926.180832266808,14081.35938978458,17451.68248953376\nclimatology,H2,2009-12-01,2009,12,lat_menos35_a_menos15,20880,64183.26955132734,-8473.276429590886,23529.785511971335,58112.84961885813,18808.59412325443\nclimatology,H2,2009-12-01,2009,12,lat_maior_igual_menos15,31581,131235.05139718566,8023.162092644721,44441.07329488918,130205.38459845562,31215.50842646724\nclimatology,H2,2010-01-01,2010,1,dominio_inteiro,78561,272128.0363637828,18522.284863936075,94752.48995808192,246441.5194322757,67475.78503925544\nclimatology,H2,2010-01-01,2010,1,lat_menor_que_menos35,26100,39434.29786960606,-10398.514369234443,19971.33546705544,26312.66485405241,17451.68248953376\nclimatology,H2,2010-01-01,2010,1,lat_menos35_a_menos15,20880,113754.66450549138,7704.992552580603,31175.80616876908,102134.7228806437,18808.59412325443\nclimatology,H2,2010-01-01,2010,1,lat_maior_igual_menos15,31581,118939.07398868537,21215.806680589914,43605.3483222574,117994.1316975796,31215.50842646724\nclimatology,H2,2010-02-01,2010,2,dominio_inteiro,78561,337920.4703508384,487.11464048037305,108018.23680407228,308175.4484189277,67475.78503925544\nclimatology,H2,2010-02-01,2010,2,lat_menor_que_menos35,26100,68634.48208109086,-21759.23675505817,31791.761312022805,49505.599436988596,17451.68248953376\nclimatology,H2,2010-02-01,2010,2,lat_menos35_a_menos15,20880,81637.08843987744,2276.9792255931534,26120.89640441956,71846.73692707854,18808.59412325443\nclimatology,H2,2010-02-01,2010,2,lat_maior_igual_menos15,31581,187648.89982987015,19969.37216994539,50105.579087629914,186823.1120548605,31215.50842646724\nclimatology,H2,2010-03-01,2010,3,dominio_inteiro,78561,364030.7532861246,20322.581691621803,98985.48118135799,348691.8077973034,67475.78503925544\nclimatology,H2,2010-03-01,2010,3,lat_menor_que_menos35,26100,22461.588468236554,6324.676660090685,17374.611286252737,16331.145367776164,17451.68248953376\nclimatology,H2,2010-03-01,2010,3,lat_menos35_a_menos15,20880,90481.12260337695,-4614.3390428954735,27148.464398351498,82097.03250335038,18808.59412325443\nclimatology,H2,2010-03-01,2010,3,lat_maior_igual_menos15,31581,251088.04221451105,18612.24407442659,54462.40549675375,250263.62992617686,31215.50842646724\nclimatology,H2,2010-04-01,2010,4,dominio_inteiro,78561,381201.5319987544,-9844.91282136552,107708.15417206474,366979.2648197366,67475.78503925544\nclimatology,H2,2010-04-01,2010,4,lat_menor_que_menos35,26100,28723.860541327478,13700.645735681057,21213.722307622433,21100.137129970564,17451.68248953376\nclimatology,H2,2010-04-01,2010,4,lat_menos35_a_menos15,20880,49720.48937196025,2146.643109272234,22126.39435491804,44801.28475667917,18808.59412325443\nclimatology,H2,2010-04-01,2010,4,lat_maior_igual_menos15,31581,302757.1820854667,-25692.20166631881,64368.03750952426,301077.84293308685,31215.50842646724\nclimatology,H2,2010-05-01,2010,5,dominio_inteiro,78561,357531.7717992675,-14908.451622586232,103137.35951315379,334616.13603601063,67475.78503925544\nclimatology,H2,2010-05-01,2010,5,lat_menor_que_menos35,26100,54677.66305005249,7295.028215348721,28380.52896398306,38188.73544035929,17451.68248953376\nclimatology,H2,2010-05-01,2010,5,lat_menos35_a_menos15,20880,45051.82177888991,-5578.156596167479,19886.156391615514,40170.15275693039,18808.59412325443\nclimatology,H2,2010-05-01,2010,5,lat_maior_igual_menos15,31581,257802.28697032516,-16625.323241767474,54870.674157555215,256257.24783872094,31215.50842646724\nclimatology,H2,2010-06-01,2010,6,dominio_inteiro,78561,216587.389073577,-16569.57536987029,76254.65437368304,205096.3785496554,67475.78503925544\nclimatology,H2,2010-06-01,2010,6,lat_menor_que_menos35,26100,25928.396490889514,-2713.0000063478947,18506.279877513647,17538.31303281889,17451.68248953376\nclimatology,H2,2010-06-01,2010,6,lat_menos35_a_menos15,20880,19642.01115063223,5947.0550548397005,13575.308854777366,17738.92331154312,18808.59412325443\nclimatology,H2,2010-06-01,2010,6,lat_maior_igual_menos15,31581,171016.98143205524,-19803.630418362096,44173.06564139202,169819.1422052934,31215.50842646724\nclimatology,H2,2010-07-01,2010,7,dominio_inteiro,78561,283310.1863769715,-29494.905603204388,87905.23054959392,270728.9542233361,67475.78503925544\nclimatology,H2,2010-07-01,2010,7,lat_menor_que_menos35,26100,30528.197943646155,-5428.6036794781685,20582.531021296978,22439.099204675367,17451.68248953376\nclimatology,H2,2010-07-01,2010,7,lat_menos35_a_menos15,20880,18362.192199618657,4873.467517004348,13401.276154321618,15936.490309664043,18808.59412325443\nclimatology,H2,2010-07-01,2010,7,lat_maior_igual_menos15,31581,234419.7962337067,-28939.769440730568,53921.423373975325,232353.36470899676,31215.50842646724\nclimatology,H2,2010-08-01,2010,8,dominio_inteiro,78561,267235.3677686463,-4738.361299185082,81876.51408132352,257009.97626391688,67475.78503925544\nclimatology,H2,2010-08-01,2010,8,lat_menor_que_menos35,26100,14588.340095315794,2934.822473347187,15527.228554666042,9946.783395535815,17451.68248953376\nclimatology,H2,2010-08-01,2010,8,lat_menos35_a_menos15,20880,21932.694713456523,9935.397719435394,14504.3977044262,19332.37464674698,18808.59412325443\nclimatology,H2,2010-08-01,2010,8,lat_maior_igual_menos15,31581,230714.33295987395,-17608.581491967663,51844.88782223128,227730.81822163414,31215.50842646724\nclimatology,H2,2010-09-01,2010,9,dominio_inteiro,78561,239918.20447837427,-2656.6058531522285,79253.56998379133,229995.84881502925,67475.78503925544\nclimatology,H2,2010-09-01,2010,9,lat_menor_que_menos35,26100,16869.304091017544,132.76908606290817,14754.792209923267,12610.883700517086,17451.68248953376\nclimatology,H2,2010-09-01,2010,9,lat_menos35_a_menos15,20880,29698.129981571197,1130.7091544901486,16152.161968711996,26415.60070278192,18808.59412325443\nclimatology,H2,2010-09-01,2010,9,lat_maior_igual_menos15,31581,193350.77040578553,-3920.0840937052853,48346.61580515606,190969.3644117302,31215.50842646724\nclimatology,H2,2010-10-01,2010,10,dominio_inteiro,78561,225260.52622985886,7403.515646201529,79717.77061080557,209969.08022742334,67475.78503925544\nclimatology,H2,2010-10-01,2010,10,lat_menor_que_menos35,26100,28104.24051375795,3470.5324552953243,20900.20728006959,19075.194204725023,17451.68248953376\nclimatology,H2,2010-10-01,2010,10,lat_menos35_a_menos15,20880,37168.42733158155,-97.22092285103281,17945.708397088543,32568.202550802802,18808.59412325443\nclimatology,H2,2010-10-01,2010,10,lat_maior_igual_menos15,31581,159987.85838451935,4030.204113757238,40871.85493364744,158325.68347189552,31215.50842646724\nclimatology,H2,2010-11-01,2010,11,dominio_inteiro,78561,341720.93750028935,-21059.56344471703,89969.48727521129,329275.78391991626,67475.78503925544\nclimatology,H2,2010-11-01,2010,11,lat_menor_que_menos35,26100,11257.591589093754,2030.9730597734451,12903.100454330444,7947.774161170755,17451.68248953376\nclimatology,H2,2010-11-01,2010,11,lat_menos35_a_menos15,20880,58898.88895470637,1204.5407261418295,23049.71476789392,52834.23573260589,18808.59412325443\nclimatology,H2,2010-11-01,2010,11,lat_maior_igual_menos15,31581,271564.45695648924,-24295.077230632305,54016.67205298692,268493.7740261396,31215.50842646724\nclimatology,H2,2010-12-01,2010,12,dominio_inteiro,78561,306717.5081110491,-29862.6244726266,90884.45667215984,288947.51963604987,67475.78503925544\nclimatology,H2,2010-12-01,2010,12,lat_menor_que_menos35,26100,20647.038842883307,-4020.0407295525074,17726.4454780221,13944.768593675568,17451.68248953376\nclimatology,H2,2010-12-01,2010,12,lat_menos35_a_menos15,20880,85355.79260054784,-2695.7400373789715,26007.516430193442,76193.19488451391,18808.59412325443\nclimatology,H2,2010-12-01,2010,12,lat_maior_igual_menos15,31581,200714.67666761795,-23146.843705695122,47150.4947639443,198809.55615786038,31215.50842646724\n', 'research/spatial_unet/evidence/seven_blocks/H2/source_calendar.csv': 'time_alvo,limite_emissao,atmosfera,indices,sst,inicializacao_sazonal,ultimo_alvo_treino,disponibilidade_historica_comprovada\n2009-01-01,2008-12-31 23:59:59.999999999,2008-09-01,2008-10-01,2008-11-01,2008-12-01,2008-09-01,False\n2009-02-01,2009-01-31 23:59:59.999999999,2008-10-01,2008-11-01,2008-12-01,2009-01-01,2008-09-01,False\n2009-03-01,2009-02-28 23:59:59.999999999,2008-11-01,2008-12-01,2009-01-01,2009-02-01,2008-09-01,False\n2009-04-01,2009-03-31 23:59:59.999999999,2008-12-01,2009-01-01,2009-02-01,2009-03-01,2008-09-01,False\n2009-05-01,2009-04-30 23:59:59.999999999,2009-01-01,2009-02-01,2009-03-01,2009-04-01,2008-09-01,False\n2009-06-01,2009-05-31 23:59:59.999999999,2009-02-01,2009-03-01,2009-04-01,2009-05-01,2008-09-01,False\n2009-07-01,2009-06-30 23:59:59.999999999,2009-03-01,2009-04-01,2009-05-01,2009-06-01,2008-09-01,False\n2009-08-01,2009-07-31 23:59:59.999999999,2009-04-01,2009-05-01,2009-06-01,2009-07-01,2008-09-01,False\n2009-09-01,2009-08-31 23:59:59.999999999,2009-05-01,2009-06-01,2009-07-01,2009-08-01,2008-09-01,False\n2009-10-01,2009-09-30 23:59:59.999999999,2009-06-01,2009-07-01,2009-08-01,2009-09-01,2008-09-01,False\n2009-11-01,2009-10-31 23:59:59.999999999,2009-07-01,2009-08-01,2009-09-01,2009-10-01,2008-09-01,False\n2009-12-01,2009-11-30 23:59:59.999999999,2009-08-01,2009-09-01,2009-10-01,2009-11-01,2008-09-01,False\n2010-01-01,2009-12-31 23:59:59.999999999,2009-09-01,2009-10-01,2009-11-01,2009-12-01,2008-09-01,False\n2010-02-01,2010-01-31 23:59:59.999999999,2009-10-01,2009-11-01,2009-12-01,2010-01-01,2008-09-01,False\n2010-03-01,2010-02-28 23:59:59.999999999,2009-11-01,2009-12-01,2010-01-01,2010-02-01,2008-09-01,False\n2010-04-01,2010-03-31 23:59:59.999999999,2009-12-01,2010-01-01,2010-02-01,2010-03-01,2008-09-01,False\n2010-05-01,2010-04-30 23:59:59.999999999,2010-01-01,2010-02-01,2010-03-01,2010-04-01,2008-09-01,False\n2010-06-01,2010-05-31 23:59:59.999999999,2010-02-01,2010-03-01,2010-04-01,2010-05-01,2008-09-01,False\n2010-07-01,2010-06-30 23:59:59.999999999,2010-03-01,2010-04-01,2010-05-01,2010-06-01,2008-09-01,False\n2010-08-01,2010-07-31 23:59:59.999999999,2010-04-01,2010-05-01,2010-06-01,2010-07-01,2008-09-01,False\n2010-09-01,2010-08-31 23:59:59.999999999,2010-05-01,2010-06-01,2010-07-01,2010-08-01,2008-09-01,False\n2010-10-01,2010-09-30 23:59:59.999999999,2010-06-01,2010-07-01,2010-08-01,2010-09-01,2008-09-01,False\n2010-11-01,2010-10-31 23:59:59.999999999,2010-07-01,2010-08-01,2010-09-01,2010-10-01,2008-09-01,False\n2010-12-01,2010-11-30 23:59:59.999999999,2010-08-01,2010-09-01,2010-10-01,2010-11-01,2008-09-01,False\n', 'research/spatial_unet/evidence/seven_blocks/H2/split.json': '{\n  "historical_publication_vintages_verified": false,\n  "independent_holdout": false,\n  "inner_reference": [\n    "1978-10",\n    "1978-11",\n    "1978-12",\n    "1979-01",\n    "1979-02",\n    "1979-03",\n    "1979-04",\n    "1979-05",\n    "1979-06",\n    "1979-07",\n    "1979-08",\n    "1979-09",\n    "1979-10",\n    "1979-11",\n    "1979-12",\n    "1980-01",\n    "1980-02",\n    "1980-03",\n    "1980-04",\n    "1980-05",\n    "1980-06",\n    "1980-07",\n    "1980-08",\n    "1980-09",\n    "1980-10",\n    "1980-11",\n    "1980-12",\n    "1981-01",\n    "1981-02",\n    "1981-03",\n    "1981-04",\n    "1981-05",\n    "1981-06",\n    "1981-07",\n    "1981-08",\n    "1981-09",\n    "1981-10",\n    "1981-11",\n    "1981-12",\n    "1982-01",\n    "1982-02",\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06"\n  ],\n  "inner_train": [\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06"\n  ],\n  "inner_validation": [\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09"\n  ],\n  "outer": [\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12"\n  ],\n  "train": [\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09"\n  ]\n}\n', 'research/spatial_unet/evidence/seven_blocks/H2/unet/training.json': '{\n  "device": "cuda",\n  "inner_selection": false,\n  "parameter_count": 122945,\n  "selected_epochs": 31,\n  "train_end": "2008-09-01",\n  "train_months": 319,\n  "train_start": "1982-03-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H2/unet/training_history.json': '[\n  {\n    "epoch": 1,\n    "seconds": 18.70106918700003,\n    "training_mse": 2.9330211477967265\n  },\n  {\n    "epoch": 2,\n    "seconds": 19.21993258899988,\n    "training_mse": 2.8456199662438753\n  },\n  {\n    "epoch": 3,\n    "seconds": 18.76156028100013,\n    "training_mse": 2.8092188169963683\n  },\n  {\n    "epoch": 4,\n    "seconds": 18.521861586000114,\n    "training_mse": 2.777361699406256\n  },\n  {\n    "epoch": 5,\n    "seconds": 18.55427213400003,\n    "training_mse": 2.7479559919303487\n  },\n  {\n    "epoch": 6,\n    "seconds": 18.709088004000023,\n    "training_mse": 2.7443447255041904\n  },\n  {\n    "epoch": 7,\n    "seconds": 18.717428662999964,\n    "training_mse": 2.729371321986088\n  },\n  {\n    "epoch": 8,\n    "seconds": 18.639309293999986,\n    "training_mse": 2.7161903321556164\n  },\n  {\n    "epoch": 9,\n    "seconds": 18.615896567000163,\n    "training_mse": 2.7043456448282943\n  },\n  {\n    "epoch": 10,\n    "seconds": 18.658814979,\n    "training_mse": 2.696659838518005\n  },\n  {\n    "epoch": 11,\n    "seconds": 18.70146149900006,\n    "training_mse": 2.6871988504284228\n  },\n  {\n    "epoch": 12,\n    "seconds": 18.683510261000038,\n    "training_mse": 2.682900666443158\n  },\n  {\n    "epoch": 13,\n    "seconds": 18.657379089000187,\n    "training_mse": 2.6765302737304784\n  },\n  {\n    "epoch": 14,\n    "seconds": 18.68269011999996,\n    "training_mse": 2.672647118942117\n  },\n  {\n    "epoch": 15,\n    "seconds": 18.67280364599992,\n    "training_mse": 2.6678544869617236\n  },\n  {\n    "epoch": 16,\n    "seconds": 18.666849130999935,\n    "training_mse": 2.6571531841373743\n  },\n  {\n    "epoch": 17,\n    "seconds": 18.654651223999963,\n    "training_mse": 2.662722793118707\n  },\n  {\n    "epoch": 18,\n    "seconds": 18.62578660400004,\n    "training_mse": 2.651280811587845\n  },\n  {\n    "epoch": 19,\n    "seconds": 18.641555742000037,\n    "training_mse": 2.6520222630994073\n  },\n  {\n    "epoch": 20,\n    "seconds": 18.656444616000044,\n    "training_mse": 2.6471735272661645\n  },\n  {\n    "epoch": 21,\n    "seconds": 18.636439791000157,\n    "training_mse": 2.6387106460463663\n  },\n  {\n    "epoch": 22,\n    "seconds": 18.648480828000174,\n    "training_mse": 2.6445405266501685\n  },\n  {\n    "epoch": 23,\n    "seconds": 18.632152743000006,\n    "training_mse": 2.6337636488732126\n  },\n  {\n    "epoch": 24,\n    "seconds": 18.62761335999994,\n    "training_mse": 2.6309443977559246\n  },\n  {\n    "epoch": 25,\n    "seconds": 18.627815448999627,\n    "training_mse": 2.6326263212485\n  },\n  {\n    "epoch": 26,\n    "seconds": 18.652582904000155,\n    "training_mse": 2.621960198617654\n  },\n  {\n    "epoch": 27,\n    "seconds": 18.635934419000023,\n    "training_mse": 2.606512897814329\n  },\n  {\n    "epoch": 28,\n    "seconds": 18.625172891999682,\n    "training_mse": 2.6129000007545686\n  },\n  {\n    "epoch": 29,\n    "seconds": 18.624488647000362,\n    "training_mse": 2.61312098712383\n  },\n  {\n    "epoch": 30,\n    "seconds": 18.65610952299994,\n    "training_mse": 2.598794993188314\n  },\n  {\n    "epoch": 31,\n    "seconds": 18.668419008,\n    "training_mse": 2.599713182748298\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/H3/baseline_check.json': '{\n  "reproduced": true,\n  "rmse_archived": 1.7428560729125753,\n  "rmse_current": 1.7428560729125753,\n  "tolerance": 1e-06\n}\n', 'research/spatial_unet/evidence/seven_blocks/H3/complete.json': '{\n  "files": {\n    "baseline_check.json": "644d6cf9bd5f820da8f6d934bf6d771d9fc9a2aff2a2a0ea5f26f15aa33d7bd7",\n    "hybrid/arvores_multissistema.txt": "8a8a6edd084cf376aaa3faa978aaeccb423a4a37ff30bf9f9351536c806bdd9b",\n    "hybrid/arvores_seas5.txt": "92c0a71f41e592cad20f04fffb5ce9a5df93e7c8e5f6f5a62b020d49b8f41005",\n    "hybrid/clima_atmosfera_indices.npz": "91e09c9eb6d2638a663afb8881b1e230d364791b7eb82bb77aaa7d9aa671012c",\n    "hybrid/clima_cfsv2.nc": "60efc7c83f607ff4e9cf21dbedea8fbc08eadd2af5240f1bab8142c56f011a38",\n    "hybrid/clima_chuva.nc": "f92c79d522590249dc06a6d30d6821d335a27efcbe0fda5633e3d4aaacb6a055",\n    "hybrid/clima_seas5.nc": "7b2a4af5739c761ab18c6e5788a2e44c578af5612935656814cff51ee40f38be",\n    "hybrid/fit.json": "cb8dd4422964a56f38ad49e1c815143946cf45f2d661b885cb48dd70cf0e2a96",\n    "hybrid/ridge.npz": "d3c144e7ed9ea8fa5fab61cd695754759755c920081fb0952e60cfcd90aa8f3b",\n    "inner/context.json": "00b5fed9836a2bc3f3a255b6d0e4eae087274aaccd672a4237147d876d51c686",\n    "inner/network.pt": "c0c23e4d482d50f9be11b4911e42a6d25599b45498f3fb1c6e0cbd5706a71992",\n    "inner/rainfall_climatology.nc": "6532601fd4b976f0d96d6ab830493cbd0419189cf3ab88bb92d922495c61875e",\n    "inner/references.npz": "eb720e9771eac2973cec20739d8006edab73bb676a83219c74a6470f13b6b0d6",\n    "inner/scaling.npz": "6744a3d831e325143053d14db89dcefbee5798d3c77401f357347e0c0e7ea533",\n    "inner/training.json": "3c8c735d4b2dc0e35bbda761ac5812a6ef4cab49cb226af69b912dd577152f45",\n    "inner/training_history.json": "361bc891a20f8a41146c678505ad143e4b709db71505e259ef9dd60b0d6cc5b6",\n    "monthly_metrics.csv": "759c72686d504b1924e12dd226aaaa4c75a7ab1a0eb696a0148ad48d22e8b394",\n    "predictions.nc": "1dab0439b69cb71b1ea6b8cc273eca2aa680d12dcc3b074a1158be1cb2d7a715",\n    "source_calendar.csv": "83fd3d6e0c46b7dabe91db01412a1e79a5fa71a21795a15d0003ef3032d3fde0",\n    "split.json": "21c77b2eec0d5af702eff0fc9f274dd28fea3251aebc8daf2c8522525a627b09",\n    "unet/context.json": "c0a913ef821e411feb93a13e942a5e2ec53cfa7ad93a73641e35c8a70f41594a",\n    "unet/network.pt": "647b1360a1f6bb98831cfc8fcffad6814b0e146429e96088b33adda1dc4d67b5",\n    "unet/rainfall_climatology.nc": "e31e1277fbd4bf397797a5fb46936a7e58808de020c44ab5217e88d00a7c0d10",\n    "unet/references.npz": "af05f8badd01a5fb7a9a33aaad4bf6353a961283fedd31841be9bab5e0381c44",\n    "unet/scaling.npz": "8ea352de29c5e28f01aab35c933eeeeada1ce5c86a01b81eb2cd62d2ea5c1f8c",\n    "unet/training.json": "61e5529a342aaf14026b9a86c9446f231c7c8ca84bf0ff11d6a33b5cf93fa819",\n    "unet/training_history.json": "37db5dbca3e3073d06e4bb23044905a4ac375d82198ff097a2c6d4ca131c8720"\n  },\n  "signature": "02d1ffc0a7205f2bf372673d337cf5daa95f125d43a95836b953c10ae101244c"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H3/inner/training.json': '{\n  "device": "cuda",\n  "inner_selection": true,\n  "parameter_count": 122945,\n  "selected_epochs": 10,\n  "train_end": "2008-06-01",\n  "train_months": 316,\n  "train_start": "1982-03-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H3/inner/training_history.json': '[\n  {\n    "epoch": 1,\n    "inner_rmse": 1.8836100554208388,\n    "seconds": 18.669962811999994,\n    "training_mse": 2.955346069003962\n  },\n  {\n    "epoch": 2,\n    "inner_rmse": 1.8698197058016583,\n    "seconds": 19.00478386199984,\n    "training_mse": 2.855267025247405\n  },\n  {\n    "epoch": 3,\n    "inner_rmse": 1.8524789826487582,\n    "seconds": 18.48461781900005,\n    "training_mse": 2.8118346718293203\n  },\n  {\n    "epoch": 4,\n    "inner_rmse": 1.846200633388683,\n    "seconds": 18.336534915000357,\n    "training_mse": 2.785071606877484\n  },\n  {\n    "epoch": 5,\n    "inner_rmse": 1.842101346264837,\n    "seconds": 18.395201769000323,\n    "training_mse": 2.7700159029115605\n  },\n  {\n    "epoch": 6,\n    "inner_rmse": 1.84189758576626,\n    "seconds": 18.566954616000203,\n    "training_mse": 2.7524737617637536\n  },\n  {\n    "epoch": 7,\n    "inner_rmse": 1.8414856882473007,\n    "seconds": 18.60065754900006,\n    "training_mse": 2.7459933433351638\n  },\n  {\n    "epoch": 8,\n    "inner_rmse": 1.8366532633995298,\n    "seconds": 18.492023452000012,\n    "training_mse": 2.7378721320176425\n  },\n  {\n    "epoch": 9,\n    "inner_rmse": 1.8410004108707674,\n    "seconds": 18.4331386380004,\n    "training_mse": 2.728403979464422\n  },\n  {\n    "epoch": 10,\n    "inner_rmse": 1.8347130002709795,\n    "seconds": 18.473301569999876,\n    "training_mse": 2.7115478017662147\n  },\n  {\n    "epoch": 11,\n    "inner_rmse": 1.8580626812389445,\n    "seconds": 18.502907284999765,\n    "training_mse": 2.7037541096723534\n  },\n  {\n    "epoch": 12,\n    "inner_rmse": 1.8476704473932535,\n    "seconds": 18.491998455999692,\n    "training_mse": 2.704182572002652\n  },\n  {\n    "epoch": 13,\n    "inner_rmse": 1.836546850536641,\n    "seconds": 18.521239049000087,\n    "training_mse": 2.692599986927419\n  },\n  {\n    "epoch": 14,\n    "inner_rmse": 1.8373100824078437,\n    "seconds": 18.50945195999975,\n    "training_mse": 2.682672606238836\n  },\n  {\n    "epoch": 15,\n    "inner_rmse": 1.8400581317962168,\n    "seconds": 18.534131793000142,\n    "training_mse": 2.6774719365035433\n  },\n  {\n    "epoch": 16,\n    "inner_rmse": 1.839710813973919,\n    "seconds": 18.50047894399995,\n    "training_mse": 2.6740714975550204\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/H3/monthly_metrics.csv': 'modelo,bloco,mes_alvo,ano,mes,regiao,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area\nunet,H3,2011-01-01,2011,1,dominio_inteiro,78561,210101.56394971092,-11243.962473336607,78175.46461999789,197076.75842902818,67475.78503925544\nunet,H3,2011-01-01,2011,1,lat_menor_que_menos35,26100,25001.201390316517,-8042.293932892382,17776.843438334763,17923.224931795554,17451.68248953376\nunet,H3,2011-01-01,2011,1,lat_menos35_a_menos15,20880,48407.1290421748,723.1708765104413,20067.620371595025,43219.22093652644,18808.59412325443\nunet,H3,2011-01-01,2011,1,lat_maior_igual_menos15,31581,136693.2335172196,-3924.8394169546664,40331.0008100681,135934.31256070617,31215.50842646724\nunet,H3,2011-02-01,2011,2,dominio_inteiro,78561,247736.4288349531,-20765.316108938307,84324.59794815257,232140.93777046338,67475.78503925544\nunet,H3,2011-02-01,2011,2,lat_menor_que_menos35,26100,25413.681398027096,1337.0269866585732,17383.16835014522,18664.22853270909,17451.68248953376\nunet,H3,2011-02-01,2011,2,lat_menos35_a_menos15,20880,83791.41640673473,-9192.724778734148,26807.675294920802,75859.13102275958,18808.59412325443\nunet,H3,2011-02-01,2011,2,lat_maior_igual_menos15,31581,138531.33103019133,-12909.618316862732,40133.75430308655,137617.5782149947,31215.50842646724\nunet,H3,2011-03-01,2011,3,dominio_inteiro,78561,299228.8777782807,-22802.601384762675,91347.36093300208,275776.2210095324,67475.78503925544\nunet,H3,2011-03-01,2011,3,lat_menor_que_menos35,26100,34266.7574869045,-5861.8498275876045,19711.537663400173,23421.064963666668,17451.68248953376\nunet,H3,2011-03-01,2011,3,lat_menos35_a_menos15,20880,147970.15699171677,-10669.233523793519,32218.981202073395,135958.62921552773,18808.59412325443\nunet,H3,2011-03-01,2011,3,lat_maior_igual_menos15,31581,116991.96329965947,-6271.5180333815515,39416.842067528516,116396.52683033803,31215.50842646724\nunet,H3,2011-04-01,2011,4,dominio_inteiro,78561,231332.31184350455,-14446.949101742357,81564.96322367713,220091.76901656378,67475.78503925544\nunet,H3,2011-04-01,2011,4,lat_menor_que_menos35,26100,24051.990507868555,-3366.67379668355,18362.780506700277,16141.978094965592,17451.68248953376\nunet,H3,2011-04-01,2011,4,lat_menos35_a_menos15,20880,21353.963683413593,2161.2774409018457,14393.096587810665,19136.92927483744,18808.59412325443\nunet,H3,2011-04-01,2011,4,lat_maior_igual_menos15,31581,185926.35765222242,-13241.552745960653,48809.086129166186,184812.86164676075,31215.50842646724\nunet,H3,2011-05-01,2011,5,dominio_inteiro,78561,343815.3518999262,-6706.9319085069,94270.27944641933,329946.94969066855,67475.78503925544\nunet,H3,2011-05-01,2011,5,lat_menor_que_menos35,26100,32968.236308532694,190.5165232717991,21380.127338677645,22987.331816159156,17451.68248953376\nunet,H3,2011-05-01,2011,5,lat_menos35_a_menos15,20880,27037.59603420797,10951.65986152552,16672.43863015808,24284.055840224442,18808.59412325443\nunet,H3,2011-05-01,2011,5,lat_maior_igual_menos15,31581,283809.5195571856,-17849.10829330422,56217.7134775836,282675.5620342849,31215.50842646724\nunet,H3,2011-06-01,2011,6,dominio_inteiro,78561,188135.30527920928,-7801.939238404855,70326.65749198012,180040.16837996198,67475.78503925544\nunet,H3,2011-06-01,2011,6,lat_menor_que_menos35,26100,16687.33543643765,-3492.5245867073536,15426.460715442896,11930.07450738006,17451.68248953376\nunet,H3,2011-06-01,2011,6,lat_menos35_a_menos15,20880,19338.091354019165,-6261.624152204022,12921.523307049647,16876.143597706032,18808.59412325443\nunet,H3,2011-06-01,2011,6,lat_maior_igual_menos15,31581,152109.87848875247,1952.2095005065203,41978.67346948758,151233.95027487588,31215.50842646724\nunet,H3,2011-07-01,2011,7,dominio_inteiro,78561,165891.14657404565,-15092.194408753887,62567.10787583701,158139.7179010534,67475.78503925544\nunet,H3,2011-07-01,2011,7,lat_menor_que_menos35,26100,15888.501438436057,-3325.7380859851837,14149.475744247437,10928.87747646058,17451.68248953376\nunet,H3,2011-07-01,2011,7,lat_menos35_a_menos15,20880,13862.228102438388,311.60844196751714,11374.268693421036,12259.845062916862,18808.59412325443\nunet,H3,2011-07-01,2011,7,lat_maior_igual_menos15,31581,136140.41703317116,-12078.06476473622,37043.36343816854,134950.99536167597,31215.50842646724\nunet,H3,2011-08-01,2011,8,dominio_inteiro,78561,185348.373197553,7663.2531633721665,75260.74085974973,167594.30466555117,67475.78503925544\nunet,H3,2011-08-01,2011,8,lat_menor_que_menos35,26100,46439.02512490494,-7548.422217279673,23603.78760638833,32737.693915185086,17451.68248953376\nunet,H3,2011-08-01,2011,8,lat_menos35_a_menos15,20880,26229.33550240173,-2864.0576054695994,14581.817437300459,23193.12439781838,18808.59412325443\nunet,H3,2011-08-01,2011,8,lat_maior_igual_menos15,31581,112680.01257024634,18075.73298612144,37075.13581606094,111663.48635254768,31215.50842646724\nunet,H3,2011-09-01,2011,9,dominio_inteiro,78561,173843.6851574985,18974.596159841865,73992.70381235331,160977.8382363121,67475.78503925544\nunet,H3,2011-09-01,2011,9,lat_menor_que_menos35,26100,24732.033676469626,-5293.5206652879715,17093.65793478489,17104.238224576577,17451.68248953376\nunet,H3,2011-09-01,2011,9,lat_menos35_a_menos15,20880,39885.801302235246,18085.911930991337,21164.52984796278,35785.843626363545,18808.59412325443\nunet,H3,2011-09-01,2011,9,lat_maior_igual_menos15,31581,109225.85017879363,6182.2048941385,35734.51602960564,108087.75638537193,31215.50842646724\nunet,H3,2011-10-01,2011,10,dominio_inteiro,78561,193190.3195491138,-17948.53287612647,75595.6209368743,181589.0931784364,67475.78503925544\nunet,H3,2011-10-01,2011,10,lat_menor_que_menos35,26100,27378.18179502219,14.933345377445221,16108.476716578007,20485.912475149744,17451.68248953376\nunet,H3,2011-10-01,2011,10,lat_menos35_a_menos15,20880,30514.485218894413,-4967.34288247861,16554.771223587915,27147.985511353414,18808.59412325443\nunet,H3,2011-10-01,2011,10,lat_maior_igual_menos15,31581,135297.65253519715,-12996.123339025304,42932.37299670838,133955.19519193325,31215.50842646724\nunet,H3,2011-11-01,2011,11,dominio_inteiro,78561,328646.98928203026,-17817.880381314084,88289.97506578453,316160.92566007026,67475.78503925544\nunet,H3,2011-11-01,2011,11,lat_menor_que_menos35,26100,19174.089346053934,-3244.494446247816,15853.028346329927,13629.774832475961,17451.68248953376\nunet,H3,2011-11-01,2011,11,lat_menos35_a_menos15,20880,50340.79199262477,6316.48124015145,21088.78861664422,45579.828343451794,18808.59412325443\nunet,H3,2011-11-01,2011,11,lat_maior_igual_menos15,31581,259132.10794335158,-20889.867175217718,51348.15810281038,256951.3224841425,31215.50842646724\nunet,H3,2011-12-01,2011,12,dominio_inteiro,78561,302281.6430830257,10542.594448950142,98153.54049905576,281446.9772952415,67475.78503925544\nunet,H3,2011-12-01,2011,12,lat_menor_que_menos35,26100,36598.857707511954,13390.88942667842,20403.122641324997,25565.416431431895,17451.68248953376\nunet,H3,2011-12-01,2011,12,lat_menos35_a_menos15,20880,73977.08258742846,2733.3501702696085,25436.767530640587,65545.16018375901,18808.59412325443\nunet,H3,2011-12-01,2011,12,lat_maior_igual_menos15,31581,191705.70278808533,-5581.645147997886,52313.650327090174,190336.40068005054,31215.50842646724\nunet,H3,2012-01-01,2012,1,dominio_inteiro,78561,229150.1250349112,-627.1100556650199,83569.89914599294,214863.91668050105,67475.78503925544\nunet,H3,2012-01-01,2012,1,lat_menor_que_menos35,26100,24141.736439534398,977.5579312667251,16459.899813599885,17474.635788923242,17451.68248953376\nunet,H3,2012-01-01,2012,1,lat_menos35_a_menos15,20880,62863.66308295919,12240.194110428449,24654.94430120522,56018.672154468324,18808.59412325443\nunet,H3,2012-01-01,2012,1,lat_maior_igual_menos15,31581,142144.72551241762,-13844.862097360194,42455.05503118783,141370.60873710946,31215.50842646724\nunet,H3,2012-02-01,2012,2,dominio_inteiro,78561,268176.0124653372,-14784.279045455158,92824.13498909026,250242.98693315568,67475.78503925544\nunet,H3,2012-02-01,2012,2,lat_menor_que_menos35,26100,37780.42777336868,-10740.64666363597,20953.049251407385,28014.84822478218,17451.68248953376\nunet,H3,2012-02-01,2012,2,lat_menos35_a_menos15,20880,68387.15652616967,12133.968995586038,26898.691079750657,61200.52989895059,18808.59412325443\nunet,H3,2012-02-01,2012,2,lat_maior_igual_menos15,31581,162008.42816579886,-16177.601377405226,44972.39465793222,161027.60880942293,31215.50842646724\nunet,H3,2012-03-01,2012,3,dominio_inteiro,78561,242362.84254972567,9352.044227886945,87396.7353691943,227577.2285481826,67475.78503925544\nunet,H3,2012-03-01,2012,3,lat_menor_que_menos35,26100,38727.37018608521,-3517.8355370163918,20026.589180231094,28976.093555535525,17451.68248953376\nunet,H3,2012-03-01,2012,3,lat_menos35_a_menos15,20880,48051.21037574478,17009.885557442904,22287.081346467137,43632.40082732882,18808.59412325443\nunet,H3,2012-03-01,2012,3,lat_maior_igual_menos15,31581,155584.26198789568,-4140.005792539567,45083.06484249607,154968.73416531822,31215.50842646724\nunet,H3,2012-04-01,2012,4,dominio_inteiro,78561,275145.87475711654,-3035.831367746927,91878.08672586177,261685.7070638031,67475.78503925544\nunet,H3,2012-04-01,2012,4,lat_menor_que_menos35,26100,28273.506240808623,3115.034009218216,20427.824477672577,20000.39609041939,17451.68248953376\nunet,H3,2012-04-01,2012,4,lat_menos35_a_menos15,20880,41478.11303680498,-7558.734382790513,20044.245041181333,37109.51338012362,18808.59412325443\nunet,H3,2012-04-01,2012,4,lat_maior_igual_menos15,31581,205394.25547950293,1407.8690058253706,51406.017207007855,204575.79759326004,31215.50842646724\nunet,H3,2012-05-01,2012,5,dominio_inteiro,78561,221110.6910126564,3730.4660171456635,81295.4022844918,204114.18689032792,67475.78503925544\nunet,H3,2012-05-01,2012,5,lat_menor_que_menos35,26100,37052.90093111529,-1020.2521259784698,21529.064152240753,25974.495861035524,17451.68248953376\nunet,H3,2012-05-01,2012,5,lat_menos35_a_menos15,20880,41838.91334650792,5425.094432353973,19713.590885072947,36808.835991931956,18808.59412325443\nunet,H3,2012-05-01,2012,5,lat_maior_igual_menos15,31581,142218.8767350332,-674.37628922984,40052.74724717811,141330.85503736042,31215.50842646724\nunet,H3,2012-06-01,2012,6,dominio_inteiro,78561,197741.2604352634,6577.871312737465,78419.46309472248,179282.3570823628,67475.78503925544\nunet,H3,2012-06-01,2012,6,lat_menor_que_menos35,26100,40295.43996794407,-436.34934532642365,21977.91689634323,28073.151983966458,17451.68248953376\nunet,H3,2012-06-01,2012,6,lat_menos35_a_menos15,20880,43889.6739990125,-2983.4394605085254,19594.497408319265,38469.82700988012,18808.59412325443\nunet,H3,2012-06-01,2012,6,lat_maior_igual_menos15,31581,113556.14646830683,9997.660118572414,36847.048790059984,112739.37808851624,31215.50842646724\nunet,H3,2012-07-01,2012,7,dominio_inteiro,78561,273096.9100307312,-3619.049587108195,84393.46785297245,253951.73935405293,67475.78503925544\nunet,H3,2012-07-01,2012,7,lat_menor_que_menos35,26100,45075.253399121124,-5437.600669562817,25167.86230367422,30933.46505018221,17451.68248953376\nunet,H3,2012-07-01,2012,7,lat_menos35_a_menos15,20880,22905.83740278415,5748.1758739203215,14968.455685928464,19773.773603534828,18808.59412325443\nunet,H3,2012-07-01,2012,7,lat_maior_igual_menos15,31581,205115.81922882586,-3929.6247914656997,44257.14986336976,203244.5007003359,31215.50842646724\nunet,H3,2012-08-01,2012,8,dominio_inteiro,78561,189991.32029652796,-927.6097322925925,78891.5190127492,174061.49512407574,67475.78503925544\nunet,H3,2012-08-01,2012,8,lat_menor_que_menos35,26100,37751.55663898027,-7707.79822948575,22316.95376703143,27730.83625317656,17451.68248953376\nunet,H3,2012-08-01,2012,8,lat_menos35_a_menos15,20880,34954.90891568301,8158.762783072889,18073.437135115266,30365.198981487083,18808.59412325443\nunet,H3,2012-08-01,2012,8,lat_maior_igual_menos15,31581,117284.85474186468,-1378.5742858797312,38501.1281106025,115965.4598894121,31215.50842646724\nunet,H3,2012-09-01,2012,9,dominio_inteiro,78561,144477.71459647018,39433.01931321435,69340.63329846971,134484.94248932478,67475.78503925544\nunet,H3,2012-09-01,2012,9,lat_menor_que_menos35,26100,17904.00203318202,2179.254910171032,15462.590936005116,12647.674855236983,17451.68248953376\nunet,H3,2012-09-01,2012,9,lat_menos35_a_menos15,20880,35386.483219379435,12926.133555334061,18519.007734250277,31641.501661920793,18808.59412325443\nunet,H3,2012-09-01,2012,9,lat_maior_igual_menos15,31581,91187.22934390872,24327.630847709253,35359.034628214315,90195.765972167,31215.50842646724\nunet,H3,2012-10-01,2012,10,dominio_inteiro,78561,193263.20467380932,16331.900986296241,78814.56618554995,180540.3297709609,67475.78503925544\nunet,H3,2012-10-01,2012,10,lat_menor_que_menos35,26100,22249.98962254839,-3083.233727633953,16691.688644230366,16514.100716505734,17451.68248953376\nunet,H3,2012-10-01,2012,10,lat_menos35_a_menos15,20880,58349.679333789565,12239.754724142258,24312.563375511323,52679.80630133439,18808.59412325443\nunet,H3,2012-10-01,2012,10,lat_maior_igual_menos15,31581,112663.53571747139,7175.379989787936,37810.31416580826,111346.42275312077,31215.50842646724\nunet,H3,2012-11-01,2012,11,dominio_inteiro,78561,349802.4062403711,4516.603852888104,99907.41771436809,333141.3064511075,67475.78503925544\nunet,H3,2012-11-01,2012,11,lat_menor_que_menos35,26100,16795.055576374896,1883.571916744113,14244.218450501561,12187.00340810551,17451.68248953376\nunet,H3,2012-11-01,2012,11,lat_menos35_a_menos15,20880,102394.9138405469,2680.5281821279787,30192.325862079393,93173.60038213647,18808.59412325443\nunet,H3,2012-11-01,2012,11,lat_maior_igual_menos15,31581,230612.43682344933,-47.49624598398805,55470.87340178713,227780.70266086556,31215.50842646724\nunet,H3,2012-12-01,2012,12,dominio_inteiro,78561,331353.7736545563,-635.3642984498292,104304.2720340658,301030.30148863525,67475.78503925544\nunet,H3,2012-12-01,2012,12,lat_menor_que_menos35,26100,72635.1071313812,-18074.220325164497,30258.707886867225,53092.18117790645,17451.68248953376\nunet,H3,2012-12-01,2012,12,lat_menos35_a_menos15,20880,109503.9359993579,14719.30329610221,32306.588797392324,100251.84941093007,18808.59412325443\nunet,H3,2012-12-01,2012,12,lat_maior_igual_menos15,31581,149214.73052381712,2719.552730612457,41738.97534980625,147686.27089979875,31215.50842646724\nhybrid,H3,2011-01-01,2011,1,dominio_inteiro,78561,215374.7261837608,2319.200852535876,82457.78683113797,202889.76270161715,67475.78503925544\nhybrid,H3,2011-01-01,2011,1,lat_menor_que_menos35,26100,23657.430639287177,-6913.847068164007,17121.9680022763,17035.981258149597,17451.68248953376\nhybrid,H3,2011-01-01,2011,1,lat_menos35_a_menos15,20880,46982.54642810863,1819.3662575040253,20002.189988467093,41998.584779927165,18808.59412325443\nhybrid,H3,2011-01-01,2011,1,lat_maior_igual_menos15,31581,144734.74911636504,7413.681663195859,45333.62884039458,143855.1966635404,31215.50842646724\nhybrid,H3,2011-02-01,2011,2,dominio_inteiro,78561,229445.56096992837,-7864.851929177129,82364.63718503992,214449.56552062352,67475.78503925544\nhybrid,H3,2011-02-01,2011,2,lat_menor_que_menos35,26100,24690.819652311813,2285.170677871213,17251.813001869356,18171.473345163016,17451.68248953376\nhybrid,H3,2011-02-01,2011,2,lat_menos35_a_menos15,20880,80772.76148942471,-7407.373693321661,26292.30722054774,73176.58225753598,18808.59412325443\nhybrid,H3,2011-02-01,2011,2,lat_maior_igual_menos15,31581,123981.9798281918,-2742.6489137266776,38820.5169626228,123101.50991792452,31215.50842646724\nhybrid,H3,2011-03-01,2011,3,dominio_inteiro,78561,290351.795961164,-14153.10112345164,91301.45603138075,266486.51317541994,67475.78503925544\nhybrid,H3,2011-03-01,2011,3,lat_menor_que_menos35,26100,35357.33137090356,-4230.582738227362,19907.553760462844,24288.301372205126,17451.68248953376\nhybrid,H3,2011-03-01,2011,3,lat_menos35_a_menos15,20880,147973.02664037902,-8629.727557634165,32701.64192632847,135758.9916586982,18808.59412325443\nhybrid,H3,2011-03-01,2011,3,lat_maior_igual_menos15,31581,107021.43794988145,-1292.7908275901113,38692.26034458943,106439.22014451658,31215.50842646724\nhybrid,H3,2011-04-01,2011,4,dominio_inteiro,78561,222615.26395172568,-11523.119718106773,79937.73057519438,212334.91989725188,67475.78503925544\nhybrid,H3,2011-04-01,2011,4,lat_menor_que_menos35,26100,21578.157841219996,-2911.2077805976314,17215.29781280414,14530.767380123012,17451.68248953376\nhybrid,H3,2011-04-01,2011,4,lat_menos35_a_menos15,20880,20796.441908056167,2200.7548526504415,14013.335142401642,18649.51959579979,18808.59412325443\nhybrid,H3,2011-04-01,2011,4,lat_maior_igual_menos15,31581,180240.6642024495,-10812.666790159583,48709.097619988584,179154.6329213291,31215.50842646724\nhybrid,H3,2011-05-01,2011,5,dominio_inteiro,78561,354875.8177466388,-8293.786566849954,95396.00613985502,340991.4161716656,67475.78503925544\nhybrid,H3,2011-05-01,2011,5,lat_menor_que_menos35,26100,32322.795320132416,-1538.8058651891015,20625.13722178562,22434.314232997127,17451.68248953376\nhybrid,H3,2011-05-01,2011,5,lat_menos35_a_menos15,20880,28388.13395508076,11164.294256106365,17169.03057373717,25571.582648262734,18808.59412325443\nhybrid,H3,2011-05-01,2011,5,lat_maior_igual_menos15,31581,294164.8884714256,-17919.27495776721,57601.838344332224,292985.5192904058,31215.50842646724\nhybrid,H3,2011-06-01,2011,6,dominio_inteiro,78561,192408.00701953607,-11744.751822303711,70569.64471843539,183911.08092266286,67475.78503925544\nhybrid,H3,2011-06-01,2011,6,lat_menor_que_menos35,26100,17667.337068410467,-5689.749538845355,15419.442195655569,12680.600419689361,17451.68248953376\nhybrid,H3,2011-06-01,2011,6,lat_menos35_a_menos15,20880,20527.702890517685,-6270.349625451057,13101.734071999357,17899.524911482946,18808.59412325443\nhybrid,H3,2011-06-01,2011,6,lat_maior_igual_menos15,31581,154212.96706060792,215.34734199270315,42048.46845078047,153330.95559149055,31215.50842646724\nhybrid,H3,2011-07-01,2011,7,dominio_inteiro,78561,173294.03131864374,-17798.140189183734,64361.86174688253,165123.23866027198,67475.78503925544\nhybrid,H3,2011-07-01,2011,7,lat_menor_que_menos35,26100,17055.894664634598,-3131.308189626449,14700.837403125615,11716.559877915173,17451.68248953376\nhybrid,H3,2011-07-01,2011,7,lat_menos35_a_menos15,20880,13751.153820363834,152.24933328986936,11258.490769442085,12158.501424595135,18808.59412325443\nhybrid,H3,2011-07-01,2011,7,lat_maior_igual_menos15,31581,142486.9828336453,-14819.081332847154,38402.53357431483,141248.17735776166,31215.50842646724\nhybrid,H3,2011-08-01,2011,8,dominio_inteiro,78561,189223.55927630904,5889.914720047203,75941.93831930109,170639.73559317237,67475.78503925544\nhybrid,H3,2011-08-01,2011,8,lat_menor_que_menos35,26100,49072.69684033263,-7718.908966050875,24555.981516679563,34596.12579534368,17451.68248953376\nhybrid,H3,2011-08-01,2011,8,lat_menos35_a_menos15,20880,26593.920546885267,-3005.7823808983208,14721.202567731918,23513.964288883188,18808.59412325443\nhybrid,H3,2011-08-01,2011,8,lat_maior_igual_menos15,31581,113556.9418890911,16614.606066996403,36664.75423488962,112529.64550894551,31215.50842646724\nhybrid,H3,2011-09-01,2011,9,dominio_inteiro,78561,182894.90781649796,20123.83770693296,75127.03306051917,169935.86739003737,67475.78503925544\nhybrid,H3,2011-09-01,2011,9,lat_menor_que_menos35,26100,24656.43292303085,-5792.2497480196025,16981.64292876007,17099.97760674637,17451.68248953376\nhybrid,H3,2011-09-01,2011,9,lat_menos35_a_menos15,20880,40844.329274563366,18271.648242112162,21413.160758241745,36702.410454966775,18808.59412325443\nhybrid,H3,2011-09-01,2011,9,lat_maior_igual_menos15,31581,117394.14561890371,7644.439212840408,36732.22937351735,116133.47932832423,31215.50842646724\nhybrid,H3,2011-10-01,2011,10,dominio_inteiro,78561,188541.58295771986,-16851.859730922257,74346.00842438555,177083.2134375232,67475.78503925544\nhybrid,H3,2011-10-01,2011,10,lat_menor_que_menos35,26100,27293.72225962535,-822.408535837229,15756.505707925231,20480.702358613667,17451.68248953376\nhybrid,H3,2011-10-01,2011,10,lat_menos35_a_menos15,20880,31072.90897107968,-4213.135422571185,16660.929749558803,27680.793194199712,18808.59412325443\nhybrid,H3,2011-10-01,2011,10,lat_maior_igual_menos15,31581,130174.95172701485,-11816.31577251384,41928.57296690151,128921.7178847098,31215.50842646724\nhybrid,H3,2011-11-01,2011,11,dominio_inteiro,78561,334693.97027312714,-20031.601639285374,89890.56434074137,322346.2082108455,67475.78503925544\nhybrid,H3,2011-11-01,2011,11,lat_menor_que_menos35,26100,18893.02154863838,-4001.5214277323466,15784.59728840185,13455.465671493646,17451.68248953376\nhybrid,H3,2011-11-01,2011,11,lat_menos35_a_menos15,20880,49381.2090247564,6043.841060697637,20864.58969884819,44750.982893260625,18808.59412325443\nhybrid,H3,2011-11-01,2011,11,lat_maior_igual_menos15,31581,266419.73969973234,-22073.921272250664,53241.37735349133,264139.7596460912,31215.50842646724\nhybrid,H3,2011-12-01,2011,12,dominio_inteiro,78561,286243.4210652127,14733.24071026517,96836.59446574854,266010.49043156055,67475.78503925544\nhybrid,H3,2011-12-01,2011,12,lat_menor_que_menos35,26100,36309.09552996496,14161.885630301364,20966.911972973663,25421.02410425517,17451.68248953376\nhybrid,H3,2011-12-01,2011,12,lat_menos35_a_menos15,20880,71209.7957680823,3676.935680978255,25156.70623029631,63127.43649306784,18808.59412325443\nhybrid,H3,2011-12-01,2011,12,lat_maior_igual_menos15,31581,178724.52976716546,-3105.58060101445,50712.976262478565,177462.02983423756,31215.50842646724\nhybrid,H3,2012-01-01,2012,1,dominio_inteiro,78561,232094.08623955486,-5688.940645183943,84041.88917586286,218466.2221820897,67475.78503925544\nhybrid,H3,2012-01-01,2012,1,lat_menor_que_menos35,26100,23679.056929084745,1804.71093417429,16465.388643657196,17201.480702045727,17451.68248953376\nhybrid,H3,2012-01-01,2012,1,lat_menos35_a_menos15,20880,57717.80486153841,8742.526606447651,23151.522088093054,51456.242001892795,18808.59412325443\nhybrid,H3,2012-01-01,2012,1,lat_maior_igual_menos15,31581,150697.2244489317,-16236.178185805884,44424.97844411261,149808.49947815115,31215.50842646724\nhybrid,H3,2012-02-01,2012,2,dominio_inteiro,78561,274274.31914414006,-19542.98179401419,94029.5508817733,256025.10933465877,67475.78503925544\nhybrid,H3,2012-02-01,2012,2,lat_menor_que_menos35,26100,39102.94678145475,-10417.445256514662,21564.843447569037,28903.479862220847,17451.68248953376\nhybrid,H3,2012-02-01,2012,2,lat_menos35_a_menos15,20880,65045.77792172441,10124.201584705614,25944.257005022246,58046.619704427,18808.59412325443\nhybrid,H3,2012-02-01,2012,2,lat_maior_igual_menos15,31581,170125.59444096088,-19249.738122205144,46520.450429182034,169075.00976801087,31215.50842646724\nhybrid,H3,2012-03-01,2012,3,dominio_inteiro,78561,242404.5732015307,6254.529993995103,86404.35474270319,228322.85151714453,67475.78503925544\nhybrid,H3,2012-03-01,2012,3,lat_menor_que_menos35,26100,37634.34115816485,-3079.5911599786127,19784.758612471076,28118.740869551395,17451.68248953376\nhybrid,H3,2012-03-01,2012,3,lat_menos35_a_menos15,20880,42961.257491066994,15706.11666860725,20923.290500750954,39010.4737253948,18808.59412325443\nhybrid,H3,2012-03-01,2012,3,lat_maior_igual_menos15,31581,161808.97455229887,-6371.995514633532,45696.305629481154,161193.63692219838,31215.50842646724\nhybrid,H3,2012-04-01,2012,4,dominio_inteiro,78561,263560.9289197675,-7518.598714784541,88935.81817288301,251132.87635649912,67475.78503925544\nhybrid,H3,2012-04-01,2012,4,lat_menor_que_menos35,26100,25719.821780287395,3343.8860868993415,19536.11280720157,18161.631882363665,17451.68248953376\nhybrid,H3,2012-04-01,2012,4,lat_menos35_a_menos15,20880,40233.417855149055,-7237.360291486286,19876.794757935488,36167.42371013046,18808.59412325443\nhybrid,H3,2012-04-01,2012,4,lat_maior_igual_menos15,31581,197607.68928433105,-3625.1245101975965,49522.910607745944,196803.820764005,31215.50842646724\nhybrid,H3,2012-05-01,2012,5,dominio_inteiro,78561,228200.95040628404,2670.7707476904175,82794.25720870595,211205.15388102538,67475.78503925544\nhybrid,H3,2012-05-01,2012,5,lat_menor_que_menos35,26100,38233.56920032451,-2156.791121698438,21846.195781541257,26868.545345659688,17451.68248953376\nhybrid,H3,2012-05-01,2012,5,lat_menos35_a_menos15,20880,38997.16531685931,4582.669760672182,19323.21723736119,34282.6075343888,18808.59412325443\nhybrid,H3,2012-05-01,2012,5,lat_maior_igual_menos15,31581,150970.2158891002,244.89210871667,41624.84418980351,150054.0010009769,31215.50842646724\nhybrid,H3,2012-06-01,2012,6,dominio_inteiro,78561,196816.3362870103,3193.294833228058,78075.26077538847,178723.8399046795,67475.78503925544\nhybrid,H3,2012-06-01,2012,6,lat_menor_que_menos35,26100,38625.30504265926,-3669.488664528867,21033.897233341526,26660.068181051483,17451.68248953376\nhybrid,H3,2012-06-01,2012,6,lat_menos35_a_menos15,20880,44714.30041478726,-7503.014272652169,19532.498701767643,39383.15429105897,18808.59412325443\nhybrid,H3,2012-06-01,2012,6,lat_maior_igual_menos15,31581,113476.73082956375,14365.797770409097,37508.864840279304,112680.61743256905,31215.50842646724\nhybrid,H3,2012-07-01,2012,7,dominio_inteiro,78561,235681.915966815,-4844.403207342387,76361.03583373304,218927.79576089283,67475.78503925544\nhybrid,H3,2012-07-01,2012,7,lat_menor_que_menos35,26100,39236.91556728494,-5698.375961852256,23147.89106733754,26666.092494865014,17451.68248953376\nhybrid,H3,2012-07-01,2012,7,lat_menos35_a_menos15,20880,19374.811974603817,2523.360513654936,13277.911906870519,16758.200376176097,18808.59412325443\nhybrid,H3,2012-07-01,2012,7,lat_maior_igual_menos15,31581,177070.18842492625,-1669.3877591450664,39935.23285952498,175503.50288985172,31215.50842646724\nhybrid,H3,2012-08-01,2012,8,dominio_inteiro,78561,184952.80215690238,-5331.196013958468,76594.11232801506,169314.2867384301,67475.78503925544\nhybrid,H3,2012-08-01,2012,8,lat_menor_que_menos35,26100,38551.57446190109,-7687.055386526998,22102.17693376701,28527.196277531497,17451.68248953376\nhybrid,H3,2012-08-01,2012,8,lat_menos35_a_menos15,20880,31962.642204081145,6181.35077671623,17172.741372814784,27666.844006118048,18808.59412325443\nhybrid,H3,2012-08-01,2012,8,lat_maior_igual_menos15,31581,114438.58549092014,-3825.4914041477005,37319.194021433286,113120.24645478054,31215.50842646724\nhybrid,H3,2012-09-01,2012,9,dominio_inteiro,78561,134543.19913419458,34873.86026666547,65086.88048676966,125571.23000410222,67475.78503925544\nhybrid,H3,2012-09-01,2012,9,lat_menor_que_menos35,26100,15636.942517079631,3025.577809808119,14881.769090389736,11127.301058533712,17451.68248953376\nhybrid,H3,2012-09-01,2012,9,lat_menos35_a_menos15,20880,32777.16408897156,10794.620118987265,17249.14344146892,29211.709553036708,18808.59412325443\nhybrid,H3,2012-09-01,2012,9,lat_maior_igual_menos15,31581,86129.09252814337,21053.662337870086,32955.96795491101,85232.21939253183,31215.50842646724\nhybrid,H3,2012-10-01,2012,10,dominio_inteiro,78561,200264.93678360264,15350.53516069248,79124.58906296178,187307.51332950898,67475.78503925544\nhybrid,H3,2012-10-01,2012,10,lat_menor_que_menos35,26100,22872.621682822304,-2242.1213043589996,17055.673206762793,16934.238603394868,17451.68248953376\nhybrid,H3,2012-10-01,2012,10,lat_menos35_a_menos15,20880,56061.325728590105,11677.791977693689,23747.56011469334,50440.56934299099,18808.59412325443\nhybrid,H3,2012-10-01,2012,10,lat_maior_igual_menos15,31581,121330.98937219018,5914.864487357792,38321.355741505635,119932.70538312307,31215.50842646724\nhybrid,H3,2012-11-01,2012,11,dominio_inteiro,78561,348642.7127914126,1036.203615692426,99352.09290033225,331570.76343580557,67475.78503925544\nhybrid,H3,2012-11-01,2012,11,lat_menor_que_menos35,26100,16958.035934611813,2417.110962530238,14385.067693005647,12286.628960846745,17451.68248953376\nhybrid,H3,2012-11-01,2012,11,lat_menos35_a_menos15,20880,106997.62633262688,1578.8607300905542,30550.569711865435,97452.1408218891,18808.59412325443\nhybrid,H3,2012-11-01,2012,11,lat_maior_igual_menos15,31581,224687.05052417394,-2959.7680769283666,54416.45549546117,221831.99365306975,31215.50842646724\nhybrid,H3,2012-12-01,2012,12,dominio_inteiro,78561,325786.6596958328,2557.0692365902437,103324.61651124092,295991.82903473283,67475.78503925544\nhybrid,H3,2012-12-01,2012,12,lat_menor_que_menos35,26100,71022.95786387677,-17044.801495896005,29826.49399549979,51936.27121066213,17451.68248953376\nhybrid,H3,2012-12-01,2012,12,lat_menos35_a_menos15,20880,107560.63584310103,15395.293586330708,31961.461642131973,98360.81910773623,18808.59412325443\nhybrid,H3,2012-12-01,2012,12,lat_maior_igual_menos15,31581,147203.06598885503,4206.5771461555405,41536.66087360918,145694.73871633442,31215.50842646724\nclimatology,H3,2011-01-01,2011,1,dominio_inteiro,78561,232467.06530808096,-23905.22403547936,78830.64230779628,219061.09747131667,67475.78503925544\nclimatology,H3,2011-01-01,2011,1,lat_menor_que_menos35,26100,24782.58604087401,-6481.366297759116,17839.330540515482,17806.786412781355,17451.68248953376\nclimatology,H3,2011-01-01,2011,1,lat_menos35_a_menos15,20880,52694.87423801893,3166.970453247195,21124.61528275232,47047.18994497216,18808.59412325443\nclimatology,H3,2011-01-01,2011,1,lat_maior_igual_menos15,31581,154989.60502918804,-20590.82819096744,39866.69648452848,154207.12111356313,31215.50842646724\nclimatology,H3,2011-02-01,2011,2,dominio_inteiro,78561,303050.4678822545,-33328.90930231247,92605.13368422684,286708.06136029365,67475.78503925544\nclimatology,H3,2011-02-01,2011,2,lat_menor_que_menos35,26100,26817.835916319695,2115.4853006601334,17712.987164467573,19759.428629606577,17451.68248953376\nclimatology,H3,2011-02-01,2011,2,lat_menos35_a_menos15,20880,86385.5873648412,-7464.065402153472,27705.655618922377,78155.29766171513,18808.59412325443\nclimatology,H3,2011-02-01,2011,2,lat_maior_igual_menos15,31581,189847.0446010936,-27980.329200819135,47186.490900836885,188793.33506897202,31215.50842646724\nclimatology,H3,2011-03-01,2011,3,dominio_inteiro,78561,326649.65686781926,-31760.036831424106,95134.19704388035,303558.8582984297,67475.78503925544\nclimatology,H3,2011-03-01,2011,3,lat_menor_que_menos35,26100,32074.620160606402,-4206.98803037405,19522.8944196105,21981.46112932267,17451.68248953376\nclimatology,H3,2011-03-01,2011,3,lat_menos35_a_menos15,20880,151429.2135397625,-9309.201537498739,33326.155662106816,139100.74572794497,18808.59412325443\nclimatology,H3,2011-03-01,2011,3,lat_maior_igual_menos15,31581,143145.8231674504,-18243.847263551317,42285.14696216304,142476.65144116204,31215.50842646724\nclimatology,H3,2011-04-01,2011,4,dominio_inteiro,78561,259724.54740369308,-19293.57138485834,85005.52797283977,247680.4793094215,67475.78503925544\nclimatology,H3,2011-04-01,2011,4,lat_menor_que_menos35,26100,24921.286795128013,-3325.2972007095814,18502.389871150255,16654.780138323393,17451.68248953376\nclimatology,H3,2011-04-01,2011,4,lat_menos35_a_menos15,20880,23950.80573734385,3675.781478924677,15340.537177177146,21395.548314149004,18808.59412325443\nclimatology,H3,2011-04-01,2011,4,lat_maior_igual_menos15,31581,210852.45487122124,-19644.055663073435,51162.60092451237,209630.15085694913,31215.50842646724\nclimatology,H3,2011-05-01,2011,5,dominio_inteiro,78561,413704.1670637769,-15775.650549008045,103295.52557060914,395689.77987164736,67475.78503925544\nclimatology,H3,2011-05-01,2011,5,lat_menor_que_menos35,26100,43185.08115823801,442.5289904475212,23322.18556624651,30396.429942145198,17451.68248953376\nclimatology,H3,2011-05-01,2011,5,lat_menos35_a_menos15,20880,36749.460393057714,14280.42696755426,19405.73531716643,32867.259766725554,18808.59412325443\nclimatology,H3,2011-05-01,2011,5,lat_maior_igual_menos15,31581,333769.62551248126,-30498.606507009827,60567.6046871962,332426.0901627766,31215.50842646724\nclimatology,H3,2011-06-01,2011,6,dominio_inteiro,78561,200622.34883078374,-16271.16433446994,71250.40790294344,191739.3596415963,67475.78503925544\nclimatology,H3,2011-06-01,2011,6,lat_menor_que_menos35,26100,19022.779765478423,-3264.160418778658,16307.238669723272,13623.764367843816,17451.68248953376\nclimatology,H3,2011-06-01,2011,6,lat_menos35_a_menos15,20880,19166.296426539535,-3872.0001115039922,12862.81547237141,16728.10074518099,18808.59412325443\nclimatology,H3,2011-06-01,2011,6,lat_maior_igual_menos15,31581,162433.2726387658,-9135.00380418729,42080.35376084875,161387.49452857152,31215.50842646724\nclimatology,H3,2011-07-01,2011,7,dominio_inteiro,78561,199326.7458200061,-20675.088531263173,69468.95370418951,190545.0219569155,67475.78503925544\nclimatology,H3,2011-07-01,2011,7,lat_menor_que_menos35,26100,17978.250829934685,-2706.5521817207336,15035.926870942116,12447.93897997114,17451.68248953376\nclimatology,H3,2011-07-01,2011,7,lat_menos35_a_menos15,20880,13760.479307420486,1403.0120845343918,11461.967345064506,12111.612061350424,18808.59412325443\nclimatology,H3,2011-07-01,2011,7,lat_maior_igual_menos15,31581,167588.0156826509,-19371.54843407683,42971.05948818289,165985.47091559393,31215.50842646724\nclimatology,H3,2011-08-01,2011,8,dominio_inteiro,78561,178477.79180277855,1657.3319592177868,72837.765507821,161778.59720923565,67475.78503925544\nclimatology,H3,2011-08-01,2011,8,lat_menor_que_menos35,26100,42204.66530376464,-6224.948819041252,22606.517739355564,29587.39007850219,17451.68248953376\nclimatology,H3,2011-08-01,2011,8,lat_menos35_a_menos15,20880,26884.58801676694,-2724.4094475302845,14689.653601741418,23818.236729769873,18808.59412325443\nclimatology,H3,2011-08-01,2011,8,lat_maior_igual_menos15,31581,109388.53848224698,10606.690225789323,35541.59416672401,108372.97040096356,31215.50842646724\nclimatology,H3,2011-09-01,2011,9,dominio_inteiro,78561,165711.86059178194,12052.193342075916,71872.25803062809,152947.9319855678,67475.78503925544\nclimatology,H3,2011-09-01,2011,9,lat_menor_que_menos35,26100,24265.746288699127,-3659.0934894680977,17070.58704471588,16833.95943672807,17451.68248953376\nclimatology,H3,2011-09-01,2011,9,lat_menos35_a_menos15,20880,39855.41469339428,18431.198641109513,21380.417262879433,35607.23748084014,18808.59412325443\nclimatology,H3,2011-09-01,2011,9,lat_maior_igual_menos15,31581,101590.6996096885,-2719.9118095654994,33421.25372303277,100506.7350679996,31215.50842646724\nclimatology,H3,2011-10-01,2011,10,dominio_inteiro,78561,226409.0931820666,-28822.774337420706,80639.9902837202,214066.8157356023,67475.78503925544\nclimatology,H3,2011-10-01,2011,10,lat_menor_que_menos35,26100,28607.171640656754,1296.553024560213,16565.34194549918,21301.85611109503,17451.68248953376\nclimatology,H3,2011-10-01,2011,10,lat_menos35_a_menos15,20880,30547.606289651198,-5465.200233288575,16629.820057099685,27394.599186651736,18808.59412325443\nclimatology,H3,2011-10-01,2011,10,lat_maior_igual_menos15,31581,167254.31525175864,-24654.127128692344,47444.82828112133,165370.36043785553,31215.50842646724\nclimatology,H3,2011-11-01,2011,11,dominio_inteiro,78561,397205.9894452775,-27144.64009547967,97677.7275558688,383539.1295484533,67475.78503925544\nclimatology,H3,2011-11-01,2011,11,lat_menor_que_menos35,26100,18474.268389507666,-1685.85888504982,15760.65600591898,13116.063280235885,17451.68248953376\nclimatology,H3,2011-11-01,2011,11,lat_menos35_a_menos15,20880,56628.88676564569,8677.22845485073,22951.371233870857,51198.121316210076,18808.59412325443\nclimatology,H3,2011-11-01,2011,11,lat_maior_igual_menos15,31581,322102.8342901241,-34136.00966528058,58965.70031607896,319224.9449520074,31215.50842646724\nclimatology,H3,2011-12-01,2011,12,dominio_inteiro,78561,352671.6864662065,1440.9689082084806,105281.67958805204,330314.3034494623,67475.78503925544\nclimatology,H3,2011-12-01,2011,12,lat_menor_que_menos35,26100,38759.59740272058,14240.552057772875,21261.25231912732,27191.49265541169,17451.68248953376\nclimatology,H3,2011-12-01,2011,12,lat_menos35_a_menos15,20880,80020.68441067864,2742.0772442250163,26700.683537358942,70997.40923682033,18808.59412325443\nclimatology,H3,2011-12-01,2011,12,lat_maior_igual_menos15,31581,233891.40465280737,-15541.66039378941,57319.74373156577,232125.4015572303,31215.50842646724\nclimatology,H3,2012-01-01,2012,1,dominio_inteiro,78561,254429.2104220562,-7128.948400713736,87963.30235336511,239815.579725017,67475.78503925544\nclimatology,H3,2012-01-01,2012,1,lat_menor_que_menos35,26100,24406.970841714017,2150.7938890084624,16476.015652127564,17655.9517725528,17451.68248953376\nclimatology,H3,2012-01-01,2012,1,lat_menos35_a_menos15,20880,63332.5803181931,9942.532739624148,24512.14640217065,56339.149708084056,18808.59412325443\nclimatology,H3,2012-01-01,2012,1,lat_maior_igual_menos15,31581,166689.6592621491,-19222.275029346347,46975.1402990669,165820.47824438015,31215.50842646724\nclimatology,H3,2012-02-01,2012,2,dominio_inteiro,78561,294144.1160854059,-21081.568146306614,95686.98850303056,275536.98760397173,67475.78503925544\nclimatology,H3,2012-02-01,2012,2,lat_menor_que_menos35,26100,37165.03236694694,-9551.975140929222,20981.251413822174,27467.16443871723,17451.68248953376\nclimatology,H3,2012-02-01,2012,2,lat_menos35_a_menos15,20880,75255.42480647724,12532.787472602387,28364.39057073655,67442.1828243748,18808.59412325443\nclimatology,H3,2012-02-01,2012,2,lat_maior_igual_menos15,31581,181723.65891198174,-24062.38047797978,46341.34651847184,180627.64034087968,31215.50842646724\nclimatology,H3,2012-03-01,2012,3,dominio_inteiro,78561,270866.98969221173,4895.960444882046,92150.49166718079,255929.76259788603,67475.78503925544\nclimatology,H3,2012-03-01,2012,3,lat_menor_que_menos35,26100,39090.59544870103,-2205.4325875639915,20476.638801813126,29203.4591818873,17451.68248953376\nclimatology,H3,2012-03-01,2012,3,lat_menos35_a_menos15,20880,46452.992881545084,16666.793526283,22230.43557634065,42295.846660261595,18808.59412325443\nclimatology,H3,2012-03-01,2012,3,lat_maior_igual_menos15,31581,185323.40136196563,-9565.400493836962,49443.41728902701,184430.45675573713,31215.50842646724\nclimatology,H3,2012-04-01,2012,4,dominio_inteiro,78561,356035.85247113253,-3666.96062893793,103596.37012705207,343070.30250179616,67475.78503925544\nclimatology,H3,2012-04-01,2012,4,lat_menor_que_menos35,26100,24722.69078481513,3293.614542454481,19325.735079556704,17553.91726335741,17451.68248953376\nclimatology,H3,2012-04-01,2012,4,lat_menos35_a_menos15,20880,44748.6800086641,-5423.165092425421,21208.88153386861,40095.533495852855,18808.59412325443\nclimatology,H3,2012-04-01,2012,4,lat_maior_igual_menos15,31581,286564.48167765327,-1537.41007896699,63061.753513626754,285420.85174258595,31215.50842646724\nclimatology,H3,2012-05-01,2012,5,dominio_inteiro,78561,308958.4473928603,12167.266321109142,93288.83332616417,294604.73751676694,67475.78503925544\nclimatology,H3,2012-05-01,2012,5,lat_menor_que_menos35,26100,30687.627419085344,-1369.4331372380257,20082.024197876453,21723.540593984526,17451.68248953376\nclimatology,H3,2012-05-01,2012,5,lat_menos35_a_menos15,20880,36132.18525283311,3471.5719107915647,18334.62364673661,31795.559728019092,18808.59412325443\nclimatology,H3,2012-05-01,2012,5,lat_maior_igual_menos15,31581,242138.63472094186,10065.127547555603,54872.1854815511,241085.63719476335,31215.50842646724\nclimatology,H3,2012-06-01,2012,6,dominio_inteiro,78561,220752.1010347273,12944.234453982208,83180.73257138999,203609.9077194005,67475.78503925544\nclimatology,H3,2012-06-01,2012,6,lat_menor_que_menos35,26100,35382.51905574648,-1919.4586518108845,20352.639792650938,24582.347994986594,17451.68248953376\nclimatology,H3,2012-06-01,2012,6,lat_menos35_a_menos15,20880,46724.95594135594,-8569.172276420984,19695.316040271427,41243.00642441377,18808.59412325443\nclimatology,H3,2012-06-01,2012,6,lat_maior_igual_menos15,31581,138644.6260376249,23432.865382214077,43132.776738467626,137784.55330000012,31215.50842646724\nclimatology,H3,2012-07-01,2012,7,dominio_inteiro,78561,215828.4933791092,3832.3787386342883,75433.34449999407,199132.87279166203,67475.78503925544\nclimatology,H3,2012-07-01,2012,7,lat_menor_que_menos35,26100,40300.659201741684,-4932.4242215156555,23376.929408609867,27620.750453933404,17451.68248953376\nclimatology,H3,2012-07-01,2012,7,lat_menos35_a_menos15,20880,18896.095388951027,3245.2095027472824,13192.629874603823,16352.166900593265,18808.59412325443\nclimatology,H3,2012-07-01,2012,7,lat_maior_igual_menos15,31581,156631.7387884165,5519.593457402661,38863.78521678038,155159.95543713536,31215.50842646724\nclimatology,H3,2012-08-01,2012,8,dominio_inteiro,78561,184065.45408369097,-2080.8863482773304,77046.18004160747,167920.92351698573,67475.78503925544\nclimatology,H3,2012-08-01,2012,8,lat_menor_que_menos35,26100,40026.38976536766,-6877.109463572502,22687.134094536304,29543.665202420394,17451.68248953376\nclimatology,H3,2012-08-01,2012,8,lat_menos35_a_menos15,20880,33033.38238184747,6351.054801126942,17496.774974135682,28632.780146526024,18808.59412325443\nclimatology,H3,2012-08-01,2012,8,lat_maior_igual_menos15,31581,111005.68193647584,-1554.8316858317703,36862.27097293548,109744.47816803932,31215.50842646724\nclimatology,H3,2012-09-01,2012,9,dominio_inteiro,78561,145708.514274861,39264.67197977123,69311.27141508949,135830.04986484133,67475.78503925544\nclimatology,H3,2012-09-01,2012,9,lat_menor_que_menos35,26100,19111.232124023794,2629.970917880535,15931.486773252487,13479.674888094889,17451.68248953376\nclimatology,H3,2012-09-01,2012,9,lat_menos35_a_menos15,20880,29909.075722862523,10938.078447628068,16997.603606543737,26664.754197083377,18808.59412325443\nclimatology,H3,2012-09-01,2012,9,lat_maior_igual_menos15,31581,96688.20642797467,25696.622614262626,36382.181035293266,95685.62077966306,31215.50842646724\nclimatology,H3,2012-10-01,2012,10,dominio_inteiro,78561,179718.50175138086,14902.418686260935,76695.94719310384,167172.90533007577,67475.78503925544\nclimatology,H3,2012-10-01,2012,10,lat_menor_que_menos35,26100,22649.985653613672,-3103.8814694583416,16755.49507138133,16843.0883729558,17451.68248953376\nclimatology,H3,2012-10-01,2012,10,lat_menos35_a_menos15,20880,54378.5438472335,9690.649223498534,23216.22079587821,48771.68555441228,18808.59412325443\nclimatology,H3,2012-10-01,2012,10,lat_maior_igual_menos15,31581,102689.97225053371,8315.650932220742,36724.2313258443,101558.13140270769,31215.50842646724\nclimatology,H3,2012-11-01,2012,11,dominio_inteiro,78561,315998.6144415178,3881.198756449041,95220.23440485925,301585.3166385627,67475.78503925544\nclimatology,H3,2012-11-01,2012,11,lat_menor_que_menos35,26100,15376.050431732236,1082.4611985683441,13764.393158137798,11106.791057455643,17451.68248953376\nclimatology,H3,2012-11-01,2012,11,lat_menos35_a_menos15,20880,85450.97791055446,1973.7963717208477,27761.89354722679,77804.64272593892,18808.59412325443\nclimatology,H3,2012-11-01,2012,11,lat_maior_igual_menos15,31581,215171.5860992311,824.9411861598492,53693.94769949466,212673.8828551681,31215.50842646724\nclimatology,H3,2012-12-01,2012,12,dominio_inteiro,78561,357498.54067982023,-3372.3903523506015,107824.59404591477,326574.6558903295,67475.78503925544\nclimatology,H3,2012-12-01,2012,12,lat_menor_que_menos35,26100,73224.69559501467,-17524.440656155348,30194.545720010996,53458.50236733988,17451.68248953376\nclimatology,H3,2012-12-01,2012,12,lat_menos35_a_menos15,20880,113418.32583480855,15069.3467983632,32618.37602536386,103942.86629711976,18808.59412325443\nclimatology,H3,2012-12-01,2012,12,lat_maior_igual_menos15,31581,170855.51924999696,-917.2964945584536,45011.67230053991,169173.28722586983,31215.50842646724\n', 'research/spatial_unet/evidence/seven_blocks/H3/source_calendar.csv': 'time_alvo,limite_emissao,atmosfera,indices,sst,inicializacao_sazonal,ultimo_alvo_treino,disponibilidade_historica_comprovada\n2011-01-01,2010-12-31 23:59:59.999999999,2010-09-01,2010-10-01,2010-11-01,2010-12-01,2010-09-01,False\n2011-02-01,2011-01-31 23:59:59.999999999,2010-10-01,2010-11-01,2010-12-01,2011-01-01,2010-09-01,False\n2011-03-01,2011-02-28 23:59:59.999999999,2010-11-01,2010-12-01,2011-01-01,2011-02-01,2010-09-01,False\n2011-04-01,2011-03-31 23:59:59.999999999,2010-12-01,2011-01-01,2011-02-01,2011-03-01,2010-09-01,False\n2011-05-01,2011-04-30 23:59:59.999999999,2011-01-01,2011-02-01,2011-03-01,2011-04-01,2010-09-01,False\n2011-06-01,2011-05-31 23:59:59.999999999,2011-02-01,2011-03-01,2011-04-01,2011-05-01,2010-09-01,False\n2011-07-01,2011-06-30 23:59:59.999999999,2011-03-01,2011-04-01,2011-05-01,2011-06-01,2010-09-01,False\n2011-08-01,2011-07-31 23:59:59.999999999,2011-04-01,2011-05-01,2011-06-01,2011-07-01,2010-09-01,False\n2011-09-01,2011-08-31 23:59:59.999999999,2011-05-01,2011-06-01,2011-07-01,2011-08-01,2010-09-01,False\n2011-10-01,2011-09-30 23:59:59.999999999,2011-06-01,2011-07-01,2011-08-01,2011-09-01,2010-09-01,False\n2011-11-01,2011-10-31 23:59:59.999999999,2011-07-01,2011-08-01,2011-09-01,2011-10-01,2010-09-01,False\n2011-12-01,2011-11-30 23:59:59.999999999,2011-08-01,2011-09-01,2011-10-01,2011-11-01,2010-09-01,False\n2012-01-01,2011-12-31 23:59:59.999999999,2011-09-01,2011-10-01,2011-11-01,2011-12-01,2010-09-01,False\n2012-02-01,2012-01-31 23:59:59.999999999,2011-10-01,2011-11-01,2011-12-01,2012-01-01,2010-09-01,False\n2012-03-01,2012-02-29 23:59:59.999999999,2011-11-01,2011-12-01,2012-01-01,2012-02-01,2010-09-01,False\n2012-04-01,2012-03-31 23:59:59.999999999,2011-12-01,2012-01-01,2012-02-01,2012-03-01,2010-09-01,False\n2012-05-01,2012-04-30 23:59:59.999999999,2012-01-01,2012-02-01,2012-03-01,2012-04-01,2010-09-01,False\n2012-06-01,2012-05-31 23:59:59.999999999,2012-02-01,2012-03-01,2012-04-01,2012-05-01,2010-09-01,False\n2012-07-01,2012-06-30 23:59:59.999999999,2012-03-01,2012-04-01,2012-05-01,2012-06-01,2010-09-01,False\n2012-08-01,2012-07-31 23:59:59.999999999,2012-04-01,2012-05-01,2012-06-01,2012-07-01,2010-09-01,False\n2012-09-01,2012-08-31 23:59:59.999999999,2012-05-01,2012-06-01,2012-07-01,2012-08-01,2010-09-01,False\n2012-10-01,2012-09-30 23:59:59.999999999,2012-06-01,2012-07-01,2012-08-01,2012-09-01,2010-09-01,False\n2012-11-01,2012-10-31 23:59:59.999999999,2012-07-01,2012-08-01,2012-09-01,2012-10-01,2010-09-01,False\n2012-12-01,2012-11-30 23:59:59.999999999,2012-08-01,2012-09-01,2012-10-01,2012-11-01,2010-09-01,False\n', 'research/spatial_unet/evidence/seven_blocks/H3/split.json': '{\n  "historical_publication_vintages_verified": false,\n  "independent_holdout": false,\n  "inner_reference": [\n    "1980-10",\n    "1980-11",\n    "1980-12",\n    "1981-01",\n    "1981-02",\n    "1981-03",\n    "1981-04",\n    "1981-05",\n    "1981-06",\n    "1981-07",\n    "1981-08",\n    "1981-09",\n    "1981-10",\n    "1981-11",\n    "1981-12",\n    "1982-01",\n    "1982-02",\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06"\n  ],\n  "inner_train": [\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06"\n  ],\n  "inner_validation": [\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09"\n  ],\n  "outer": [\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09",\n    "2012-10",\n    "2012-11",\n    "2012-12"\n  ],\n  "train": [\n    "1982-03",\n    "1982-04",\n    "1982-05",\n    "1982-06",\n    "1982-07",\n    "1982-08",\n    "1982-09",\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09"\n  ]\n}\n', 'research/spatial_unet/evidence/seven_blocks/H3/unet/training.json': '{\n  "device": "cuda",\n  "inner_selection": false,\n  "parameter_count": 122945,\n  "selected_epochs": 10,\n  "train_end": "2010-09-01",\n  "train_months": 343,\n  "train_start": "1982-03-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H3/unet/training_history.json': '[\n  {\n    "epoch": 1,\n    "seconds": 20.12191340599975,\n    "training_mse": 2.985191215231537\n  },\n  {\n    "epoch": 2,\n    "seconds": 20.583162018000166,\n    "training_mse": 2.883534359167338\n  },\n  {\n    "epoch": 3,\n    "seconds": 20.074586129999716,\n    "training_mse": 2.850056513405402\n  },\n  {\n    "epoch": 4,\n    "seconds": 19.88966725299997,\n    "training_mse": 2.8119370798327834\n  },\n  {\n    "epoch": 5,\n    "seconds": 20.013366795000366,\n    "training_mse": 2.801445203689375\n  },\n  {\n    "epoch": 6,\n    "seconds": 20.15941438400023,\n    "training_mse": 2.78350458339769\n  },\n  {\n    "epoch": 7,\n    "seconds": 20.062814570999763,\n    "training_mse": 2.766660253786137\n  },\n  {\n    "epoch": 8,\n    "seconds": 20.02729846200009,\n    "training_mse": 2.7680179836451138\n  },\n  {\n    "epoch": 9,\n    "seconds": 20.041623291000178,\n    "training_mse": 2.7489530206074173\n  },\n  {\n    "epoch": 10,\n    "seconds": 20.055742260999978,\n    "training_mse": 2.7380906131455225\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/H4/baseline_check.json': '{\n  "reproduced": true,\n  "rmse_archived": 1.7036435332466895,\n  "rmse_current": 1.7036435332466897,\n  "tolerance": 1e-06\n}\n', 'research/spatial_unet/evidence/seven_blocks/H4/complete.json': '{\n  "files": {\n    "baseline_check.json": "f63e083fc255c449e3d6ded37cda31783f7fce53bc5e77da09f32f1faeeba638",\n    "hybrid/arvores_multissistema.txt": "9a08a8a8448d39a7110d413cd4a77ad622087004977a217ff14acb369d30897b",\n    "hybrid/arvores_seas5.txt": "af632271ebc8a9b9ab09c1f457b843b73f1edf8ac2691a9f34bf012566246122",\n    "hybrid/clima_atmosfera_indices.npz": "de26e66fc981f843a166cfe4545e6aa237e99da7f9626d096059e068019a4de9",\n    "hybrid/clima_cfsv2.nc": "50573683678bef9837c240b4effd7b346659cd8133c5ef0b9e42f98584567240",\n    "hybrid/clima_chuva.nc": "513f5745fd14248fe260758e19efbb8838a8e618ef12555c18c5f76c7e6a4c97",\n    "hybrid/clima_seas5.nc": "20de4786a3a4b6fec1b45908c8be8b381b372d417d627c0893dae6f071e9c715",\n    "hybrid/fit.json": "71dc8f55d121d8e4596a16bf5af7ab7f2d855268226e5b7ed431156cd8ca8089",\n    "hybrid/ridge.npz": "05da40482cee30f993682736db383f6d2243f87c3bb37b1b72eb3e525757b3c2",\n    "inner/context.json": "d13ae70d1b8044789f4f264a89aea0d82e95f5df71615c6d1b5821288fbf1bce",\n    "inner/network.pt": "02625e6d0eb50670656705710a87d7113ab87321ab76abbd9f5a6ec95ff8acc6",\n    "inner/rainfall_climatology.nc": "44b2548cf58f73a24b7c2de1150249911ad1d1b47250b1f4f97a34ab024d2196",\n    "inner/references.npz": "b5fc4a18e4228546beb7ad9948c842529e94f6660c2dcccbfa323187da9bf5c5",\n    "inner/scaling.npz": "81174ba6243f4a981c88fc7669741ce1e5513a30f8e957e99f8bff3548194d75",\n    "inner/training.json": "91b3dce3ef9877051688692dfa9139e818f42b88f59ade24909e68eeed0b9ae7",\n    "inner/training_history.json": "53787e9943551bcaf186863383f31ef8ba92329e4f7623d49c86106fef56a273",\n    "monthly_metrics.csv": "280d7197b163b5de0104078f25dd0438bcf4e9705e3b0681054e33c3c1d6e603",\n    "predictions.nc": "2f192714206bc3379202f32ffb690cbd1df83a53134bfa5701d4c7291f948f06",\n    "source_calendar.csv": "db9860b421d3190bc58f469badfad1a2375b9e9d2d14a235b3a07b32810739ef",\n    "split.json": "231802dd65c0e6995c5177860d036abd0e0f7292dade92dce64749a0717f1494",\n    "unet/context.json": "2133a3e7f1d1ff69b4ce0b25115561c2116b5557d05d22e5d859a315faaf1929",\n    "unet/network.pt": "f93a472e93dab9b26f0d47ea7862123dda34b2ad1359d36211221977ceea49e9",\n    "unet/rainfall_climatology.nc": "196b33ab9653326fae33592bb07ffb884bca89d75694a31e1dec295b57ecc1d3",\n    "unet/references.npz": "9a8890e4f803802c9adc1afed1678dbcf0b276186abcec26cf2ccb9d1af0280c",\n    "unet/scaling.npz": "cc6587ea3e7198b05bb00cec345072a30d379dc7dfc63cfec7d8d5e9f580b835",\n    "unet/training.json": "31c17e8568e8f5b159cf40ca194bd2def932192be5923fdad5e4622ead6d8981",\n    "unet/training_history.json": "6ae88f922f6f1fd73f1f5ecc91b4c64cfb16f8f91bd1e22b0771362bbdadbdd9"\n  },\n  "signature": "02d1ffc0a7205f2bf372673d337cf5daa95f125d43a95836b953c10ae101244c"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H4/inner/training.json': '{\n  "device": "cuda",\n  "inner_selection": true,\n  "parameter_count": 122945,\n  "selected_epochs": 18,\n  "train_end": "2010-06-01",\n  "train_months": 333,\n  "train_start": "1982-10-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H4/inner/training_history.json': '[\n  {\n    "epoch": 1,\n    "inner_rmse": 1.7490146955380073,\n    "seconds": 19.81208703099992,\n    "training_mse": 2.996751690054083\n  },\n  {\n    "epoch": 2,\n    "inner_rmse": 1.7595197185540004,\n    "seconds": 20.014099175999945,\n    "training_mse": 2.909498519009656\n  },\n  {\n    "epoch": 3,\n    "inner_rmse": 1.7468157757720137,\n    "seconds": 19.430350074000216,\n    "training_mse": 2.8935402449186856\n  },\n  {\n    "epoch": 4,\n    "inner_rmse": 1.7534443314161714,\n    "seconds": 19.29965264900011,\n    "training_mse": 2.8597320536593416\n  },\n  {\n    "epoch": 5,\n    "inner_rmse": 1.7377623211598492,\n    "seconds": 19.509813274999942,\n    "training_mse": 2.848800112177302\n  },\n  {\n    "epoch": 6,\n    "inner_rmse": 1.7361563600639036,\n    "seconds": 19.578731738999977,\n    "training_mse": 2.8170095041349485\n  },\n  {\n    "epoch": 7,\n    "inner_rmse": 1.7374756128082869,\n    "seconds": 19.56187898799999,\n    "training_mse": 2.795554685163068\n  },\n  {\n    "epoch": 8,\n    "inner_rmse": 1.7299105239388146,\n    "seconds": 19.480439353999827,\n    "training_mse": 2.8012261569678962\n  },\n  {\n    "epoch": 9,\n    "inner_rmse": 1.7374640188855446,\n    "seconds": 19.485702475999915,\n    "training_mse": 2.779120666486723\n  },\n  {\n    "epoch": 10,\n    "inner_rmse": 1.7337072121407286,\n    "seconds": 19.473797276999903,\n    "training_mse": 2.7720021721837043\n  },\n  {\n    "epoch": 11,\n    "inner_rmse": 1.7326224373484633,\n    "seconds": 19.539318147999893,\n    "training_mse": 2.7588013195776724\n  },\n  {\n    "epoch": 12,\n    "inner_rmse": 1.7304579235553204,\n    "seconds": 19.516260916000192,\n    "training_mse": 2.7702192780491828\n  },\n  {\n    "epoch": 13,\n    "inner_rmse": 1.7353327272395698,\n    "seconds": 19.545388059999823,\n    "training_mse": 2.752079733141192\n  },\n  {\n    "epoch": 14,\n    "inner_rmse": 1.7288197539608332,\n    "seconds": 19.50389575700001,\n    "training_mse": 2.742974375103329\n  },\n  {\n    "epoch": 15,\n    "inner_rmse": 1.7386919851060152,\n    "seconds": 19.516955295000116,\n    "training_mse": 2.7304000983367094\n  },\n  {\n    "epoch": 16,\n    "inner_rmse": 1.732080352477341,\n    "seconds": 19.48671203199956,\n    "training_mse": 2.738516561977856\n  },\n  {\n    "epoch": 17,\n    "inner_rmse": 1.7353239019515259,\n    "seconds": 19.49603174599997,\n    "training_mse": 2.725977260429222\n  },\n  {\n    "epoch": 18,\n    "inner_rmse": 1.727768488057825,\n    "seconds": 19.4756265609999,\n    "training_mse": 2.7180227710678055\n  },\n  {\n    "epoch": 19,\n    "inner_rmse": 1.7357650571725176,\n    "seconds": 19.480516613999953,\n    "training_mse": 2.718853008997691\n  },\n  {\n    "epoch": 20,\n    "inner_rmse": 1.7289354809636894,\n    "seconds": 19.473445904000073,\n    "training_mse": 2.7115278222539403\n  },\n  {\n    "epoch": 21,\n    "inner_rmse": 1.7446608833271564,\n    "seconds": 19.47673279999981,\n    "training_mse": 2.7120436254564346\n  },\n  {\n    "epoch": 22,\n    "inner_rmse": 1.7293010386861034,\n    "seconds": 19.447649328000352,\n    "training_mse": 2.696984212319772\n  },\n  {\n    "epoch": 23,\n    "inner_rmse": 1.7325140861786328,\n    "seconds": 19.50434379699982,\n    "training_mse": 2.691075595649513\n  },\n  {\n    "epoch": 24,\n    "inner_rmse": 1.735316197500947,\n    "seconds": 19.45289299099977,\n    "training_mse": 2.693974123344765\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/H4/monthly_metrics.csv': 'modelo,bloco,mes_alvo,ano,mes,regiao,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area\nunet,H4,2013-01-01,2013,1,dominio_inteiro,78561,329715.99027020053,24300.92296315357,106159.09799201041,311201.9899248957,67475.78503925544\nunet,H4,2013-01-01,2013,1,lat_menor_que_menos35,26100,29453.414419481916,7852.257387422025,20426.272332139313,20728.242173686183,17451.68248953376\nunet,H4,2013-01-01,2013,1,lat_menos35_a_menos15,20880,93923.64480850604,-225.83381685055792,30419.10267884098,85748.42250227949,18808.59412325443\nunet,H4,2013-01-01,2013,1,lat_maior_igual_menos15,31581,206338.93104221256,16674.499392582104,55313.72298103012,204725.32524893002,31215.50842646724\nunet,H4,2013-02-01,2013,2,dominio_inteiro,78561,351483.40060206613,-1483.3305123597383,103263.5665136762,334546.5077257196,67475.78503925544\nunet,H4,2013-02-01,2013,2,lat_menor_que_menos35,26100,34084.89289254513,2912.8227239847183,20702.03802496195,24978.717634332523,17451.68248953376\nunet,H4,2013-02-01,2013,2,lat_menos35_a_menos15,20880,59130.32085564825,3821.7804311923683,24225.09629381448,52598.88745596976,18808.59412325443\nunet,H4,2013-02-01,2013,2,lat_maior_igual_menos15,31581,258268.18685387273,-8217.933667536825,58336.43219489977,256968.90263541733,31215.50842646724\nunet,H4,2013-03-01,2013,3,dominio_inteiro,78561,278571.2445095714,10982.730685655028,88728.99121541902,266305.44575052324,67475.78503925544\nunet,H4,2013-03-01,2013,3,lat_menor_que_menos35,26100,22329.4254089292,4852.539670675993,18001.6599252522,15411.100515792727,17451.68248953376\nunet,H4,2013-03-01,2013,3,lat_menos35_a_menos15,20880,46419.5938210691,1587.3617957569659,21128.48972330615,41808.1282336067,18808.59412325443\nunet,H4,2013-03-01,2013,3,lat_maior_igual_menos15,31581,209822.22527957312,4542.829219222069,49598.841566860676,209086.21700112376,31215.50842646724\nunet,H4,2013-04-01,2013,4,dominio_inteiro,78561,260398.87966637113,33696.75156941172,93133.16343022045,247227.5466610007,67475.78503925544\nunet,H4,2013-04-01,2013,4,lat_menor_que_menos35,26100,32309.53752710709,6190.721424818039,22907.124144256115,22970.631636117923,17451.68248953376\nunet,H4,2013-04-01,2013,4,lat_menos35_a_menos15,20880,28975.266913543514,3689.168817489408,17889.300761724822,25883.817018637572,18808.59412325443\nunet,H4,2013-04-01,2013,4,lat_maior_igual_menos15,31581,199114.07522572053,23816.86132710427,52336.73852423951,198373.09800624524,31215.50842646724\nunet,H4,2013-05-01,2013,5,dominio_inteiro,78561,177681.78074834903,-675.6594926211983,70468.28793719597,168369.8634576149,67475.78503925544\nunet,H4,2013-05-01,2013,5,lat_menor_que_menos35,26100,19068.297701230484,-4121.561390399933,16970.971467077732,13220.6810748637,17451.68248953376\nunet,H4,2013-05-01,2013,5,lat_menos35_a_menos15,20880,24953.042404611737,6119.3602738678455,16325.76706071198,22116.965023482175,18808.59412325443\nunet,H4,2013-05-01,2013,5,lat_maior_igual_menos15,31581,133660.44064250684,-2673.458376089111,37171.54940940626,133032.21735926898,31215.50842646724\nunet,H4,2013-06-01,2013,6,dominio_inteiro,78561,266700.253202949,-7466.080322826281,84919.70908660628,245402.92535553826,67475.78503925544\nunet,H4,2013-06-01,2013,6,lat_menor_que_menos35,26100,45408.89715235415,-7988.220734000206,22108.989433288574,29826.552312993455,17451.68248953376\nunet,H4,2013-06-01,2013,6,lat_menos35_a_menos15,20880,44811.364234436325,-7596.493366518989,17480.727274643257,40290.380228775786,18808.59412325443\nunet,H4,2013-06-01,2013,6,lat_maior_igual_menos15,31581,176479.99181615858,8118.633777692914,45329.99237867445,175285.99281376903,31215.50842646724\nunet,H4,2013-07-01,2013,7,dominio_inteiro,78561,184778.67887379485,12919.450862653553,75773.48977049626,172163.26709021284,67475.78503925544\nunet,H4,2013-07-01,2013,7,lat_menor_que_menos35,26100,25232.797228504955,7173.353467404842,20387.90427595377,17448.84997573915,17451.68248953376\nunet,H4,2013-07-01,2013,7,lat_menos35_a_menos15,20880,28356.94214352012,-5564.270378122106,15862.541762674227,24759.613006897693,18808.59412325443\nunet,H4,2013-07-01,2013,7,lat_maior_igual_menos15,31581,131188.93950176978,11310.367773370817,39523.04373186827,129954.80410757603,31215.50842646724\nunet,H4,2013-08-01,2013,8,dominio_inteiro,78561,153709.12444920966,-3705.433652306907,66448.26329297852,139690.66687445162,67475.78503925544\nunet,H4,2013-08-01,2013,8,lat_menor_que_menos35,26100,33898.2522310276,-8321.047220468521,19555.15100324154,24000.32854960215,17451.68248953376\nunet,H4,2013-08-01,2013,8,lat_menos35_a_menos15,20880,25346.0392310878,2508.690848113969,14200.364330915734,22117.789725872128,18808.59412325443\nunet,H4,2013-08-01,2013,8,lat_maior_igual_menos15,31581,94464.83298709424,2106.9227200476453,32692.747958821245,93572.54859897737,31215.50842646724\nunet,H4,2013-09-01,2013,9,dominio_inteiro,78561,125912.94775127203,10671.965339314193,62548.46839134395,113863.13522463263,67475.78503925544\nunet,H4,2013-09-01,2013,9,lat_menor_que_menos35,26100,26489.46254972702,1610.5226119756699,18503.81363749504,18545.91503858959,17451.68248953376\nunet,H4,2013-09-01,2013,9,lat_menos35_a_menos15,20880,25453.11455869431,-2523.2641215734184,14617.587819423527,22332.66802437247,18808.59412325443\nunet,H4,2013-09-01,2013,9,lat_maior_igual_menos15,31581,73970.3706428507,11584.706848911941,29427.066934425384,72984.55216167057,31215.50842646724\nunet,H4,2013-10-01,2013,10,dominio_inteiro,78561,141846.72651126928,6424.550025369972,66475.97879519314,132524.2405175179,67475.78503925544\nunet,H4,2013-10-01,2013,10,lat_menor_que_menos35,26100,16658.76263614658,3894.0650039315224,14810.953351974487,12237.873362530423,17451.68248953376\nunet,H4,2013-10-01,2013,10,lat_menos35_a_menos15,20880,35464.129108750916,-2454.974140541628,17875.888524720445,31513.507842766416,18808.59412325443\nunet,H4,2013-10-01,2013,10,lat_maior_igual_menos15,31581,89723.8347663718,4985.459161980078,33789.1369184982,88772.85931222109,31215.50842646724\nunet,H4,2013-11-01,2013,11,dominio_inteiro,78561,226841.18798534607,-6898.839588105679,84529.77839209512,212032.95872409796,67475.78503925544\nunet,H4,2013-11-01,2013,11,lat_menor_que_menos35,26100,25374.433062997996,-8803.871677607298,16830.79589268565,18459.621235285365,17451.68248953376\nunet,H4,2013-11-01,2013,11,lat_menos35_a_menos15,20880,53778.17101358999,3775.574256900698,21576.956606362015,47354.27050996163,18808.59412325443\nunet,H4,2013-11-01,2013,11,lat_maior_igual_menos15,31581,147688.58390875807,-1870.5421673990786,46122.02589304745,146219.06697885095,31215.50842646724\nunet,H4,2013-12-01,2013,12,dominio_inteiro,78561,354003.3680525279,3964.4026959680486,95622.16529418039,331761.2224686893,67475.78503925544\nunet,H4,2013-12-01,2013,12,lat_menor_que_menos35,26100,26057.938317816974,4744.286928519607,16846.7772539109,18129.253291526158,17451.68248953376\nunet,H4,2013-12-01,2013,12,lat_menos35_a_menos15,20880,151586.18287943228,481.4591330580879,33134.32839628565,138740.26889949053,18808.59412325443\nunet,H4,2013-12-01,2013,12,lat_maior_igual_menos15,31581,176359.24685527864,-1261.3433656096458,45641.05964398384,174891.7002776727,31215.50842646724\nunet,H4,2014-01-01,2014,1,dominio_inteiro,78561,401151.3949291951,13475.358418410644,106187.9944362808,374476.2057488132,67475.78503925544\nunet,H4,2014-01-01,2014,1,lat_menor_que_menos35,26100,33647.253177597304,-8297.732433125377,19230.4416860193,24361.466767321755,17451.68248953376\nunet,H4,2014-01-01,2014,1,lat_menos35_a_menos15,20880,176376.22133131232,22206.65265340358,40070.10090140998,160780.6803058025,18808.59412325443\nunet,H4,2014-01-01,2014,1,lat_maior_igual_menos15,31581,191127.92042028555,-433.56180186755955,46887.45184885152,189334.05867568892,31215.50842646724\nunet,H4,2014-02-01,2014,2,dominio_inteiro,78561,327061.15720445407,-1437.3957700505853,93167.78354882449,311670.67218664475,67475.78503925544\nunet,H4,2014-02-01,2014,2,lat_menor_que_menos35,26100,23061.267457693248,3504.375179409981,18750.20123296976,16608.427964316896,17451.68248953376\nunet,H4,2014-02-01,2014,2,lat_menos35_a_menos15,20880,79016.43831803642,6841.52421259135,26071.080655805767,70947.56805147039,18808.59412325443\nunet,H4,2014-02-01,2014,2,lat_maior_igual_menos15,31581,224983.4514287244,-11783.295162051916,48346.50166004896,224114.6761708575,31215.50842646724\nunet,H4,2014-03-01,2014,3,dominio_inteiro,78561,225819.2782157232,-3664.0708493813872,87738.33913628012,209303.29146777772,67475.78503925544\nunet,H4,2014-03-01,2014,3,lat_menor_que_menos35,26100,34351.72739054927,-9080.281681329012,21866.263254851103,23645.891833417,17451.68248953376\nunet,H4,2014-03-01,2014,3,lat_menos35_a_menos15,20880,48415.51350236474,4440.530015952885,22523.20521376282,43238.45731308792,18808.59412325443\nunet,H4,2014-03-01,2014,3,lat_maior_igual_menos15,31581,143052.03732280916,975.6808159947395,43348.8706676662,142418.94232127283,31215.50842646724\nunet,H4,2014-04-01,2014,4,dominio_inteiro,78561,194342.2726303358,15403.152091616765,76502.59285944141,179411.7746196322,67475.78503925544\nunet,H4,2014-04-01,2014,4,lat_menor_que_menos35,26100,36110.44180094305,1350.8811186254025,20765.38450112939,26229.166427958222,17451.68248953376\nunet,H4,2014-04-01,2014,4,lat_menos35_a_menos15,20880,40262.04519165117,10463.678827628493,20072.830976083875,35526.77955800808,18808.59412325443\nunet,H4,2014-04-01,2014,4,lat_maior_igual_menos15,31581,117969.78563774159,3588.592145362869,35664.37738222815,117655.82863366588,31215.50842646724\nunet,H4,2014-05-01,2014,5,dominio_inteiro,78561,237234.73797014015,-5232.457558621652,78519.53399154451,227651.7912020821,67475.78503925544\nunet,H4,2014-05-01,2014,5,lat_menor_que_menos35,26100,20821.40602325483,-2217.2181912064552,17380.936038434505,14388.864070259893,17451.68248953376\nunet,H4,2014-05-01,2014,5,lat_menos35_a_menos15,20880,24693.50703216983,5767.171794886701,16025.712774607353,22292.756952496726,18808.59412325443\nunet,H4,2014-05-01,2014,5,lat_maior_igual_menos15,31581,191719.82491471546,-8782.411162301898,45112.88517850265,190970.17017932545,31215.50842646724\nunet,H4,2014-06-01,2014,6,dominio_inteiro,78561,243163.3959721738,-12609.34612554498,76550.06302454136,229573.23033215545,67475.78503925544\nunet,H4,2014-06-01,2014,6,lat_menor_que_menos35,26100,21091.25996021,-632.191687554121,16501.042889803648,14565.882759245898,17451.68248953376\nunet,H4,2014-06-01,2014,6,lat_menos35_a_menos15,20880,52061.680121574595,-4910.224042329937,17285.71249135211,45889.15467482066,18808.59412325443\nunet,H4,2014-06-01,2014,6,lat_maior_igual_menos15,31581,170010.4558903892,-7066.930395660922,42763.307643385604,169118.19289808892,31215.50842646724\nunet,H4,2014-07-01,2014,7,dominio_inteiro,78561,218777.78974508855,-5632.018300930038,72506.87137070857,207079.7650245517,67475.78503925544\nunet,H4,2014-07-01,2014,7,lat_menor_que_menos35,26100,28565.209208652588,-7024.162094533443,18991.173944413662,21008.666456301507,17451.68248953376\nunet,H4,2014-07-01,2014,7,lat_menos35_a_menos15,20880,23501.178897708654,518.2195114716887,15141.116438880563,20849.41434971363,18808.59412325443\nunet,H4,2014-07-01,2014,7,lat_maior_igual_menos15,31581,166711.4016387273,873.9242821317166,38374.580987414345,165221.68421853657,31215.50842646724\nunet,H4,2014-08-01,2014,8,dominio_inteiro,78561,172106.40700126058,2478.551856311038,66875.05049431138,162539.8232925554,67475.78503925544\nunet,H4,2014-08-01,2014,8,lat_menor_que_menos35,26100,20519.917145309395,-6909.411708295345,16595.68445044756,14587.529127896049,17451.68248953376\nunet,H4,2014-08-01,2014,8,lat_menos35_a_menos15,20880,18567.80386033744,12095.379267225042,14294.615529349074,16447.157840542524,18808.59412325443\nunet,H4,2014-08-01,2014,8,lat_maior_igual_menos15,31581,133018.68599561375,-2707.4157026186585,35984.750514514744,131505.13632411684,31215.50842646724\nunet,H4,2014-09-01,2014,9,dominio_inteiro,78561,131739.01030189075,348.5073148375377,63698.602953732945,121691.7592532277,67475.78503925544\nunet,H4,2014-09-01,2014,9,lat_menor_que_menos35,26100,21951.02548777681,-6711.566809296608,17559.197731494904,15551.939211951883,17451.68248953376\nunet,H4,2014-09-01,2014,9,lat_menos35_a_menos15,20880,25352.73494545548,-425.1551321465522,14719.825969377533,22565.037453758057,18808.59412325443\nunet,H4,2014-09-01,2014,9,lat_maior_igual_menos15,31581,84435.24986865846,7485.229256280698,31419.57925286051,83574.78258751777,31215.50842646724\nunet,H4,2014-10-01,2014,10,dominio_inteiro,78561,190891.18831693617,7778.558321330696,77237.62950326875,179108.08861113203,67475.78503925544\nunet,H4,2014-10-01,2014,10,lat_menor_que_menos35,26100,15159.912111810427,3199.682290136814,14444.86045128107,10525.192397513276,17451.68248953376\nunet,H4,2014-10-01,2014,10,lat_menos35_a_menos15,20880,59190.57728687515,18900.89840534143,24881.95665802248,53371.42578460186,18808.59412325443\nunet,H4,2014-10-01,2014,10,lat_maior_igual_menos15,31581,116540.69891825061,-14322.02237414755,37910.8123939652,115211.4704290169,31215.50842646724\nunet,H4,2014-11-01,2014,11,dominio_inteiro,78561,272373.0949742179,-25493.74758801423,81845.20278623886,256869.50639928892,67475.78503925544\nunet,H4,2014-11-01,2014,11,lat_menor_que_menos35,26100,31651.729076587035,-7390.745381653309,17725.626600027084,21444.632358210063,17451.68248953376\nunet,H4,2014-11-01,2014,11,lat_menos35_a_menos15,20880,35396.51704212623,-715.8679564539343,18032.87471810542,31693.267949069705,18808.59412325443\nunet,H4,2014-11-01,2014,11,lat_maior_igual_menos15,31581,205324.84885550456,-17387.134249906987,46086.70146810636,203731.60609200917,31215.50842646724\nunet,H4,2014-12-01,2014,12,dominio_inteiro,78561,213993.95215006275,11157.632977148518,80604.3269792106,198275.4628996431,67475.78503925544\nunet,H4,2014-12-01,2014,12,lat_menor_que_menos35,26100,27127.52551135938,1373.0526645332575,18579.573987826705,19464.828824607164,17451.68248953376\nunet,H4,2014-12-01,2014,12,lat_menos35_a_menos15,20880,72678.43130422175,6934.211668604985,25302.980876373127,65501.5934515282,18808.59412325443\nunet,H4,2014-12-01,2014,12,lat_maior_igual_menos15,31581,114187.99533448162,2850.3686440102756,36721.77211501077,113309.04062350774,31215.50842646724\nhybrid,H4,2013-01-01,2013,1,dominio_inteiro,78561,333162.87457953545,17780.235138169126,104156.73505868828,313691.3449350728,67475.78503925544\nhybrid,H4,2013-01-01,2013,1,lat_menor_que_menos35,26100,32122.785880783435,6948.82822372214,20572.269470366515,22530.025652426026,17451.68248953376\nhybrid,H4,2013-01-01,2013,1,lat_menos35_a_menos15,20880,93402.6537722495,-111.68603315896871,30293.336575856127,85148.5386827647,18808.59412325443\nhybrid,H4,2013-01-01,2013,1,lat_maior_igual_menos15,31581,207637.43492650252,10943.092947605955,53291.12901246565,206012.7805998821,31215.50842646724\nhybrid,H4,2013-02-01,2013,2,dominio_inteiro,78561,345514.71361860813,-7767.133183284182,98531.70170011229,329661.97594467417,67475.78503925544\nhybrid,H4,2013-02-01,2013,2,lat_menor_que_menos35,26100,32723.15595981688,2098.6877438850643,19794.6605402899,23917.57693900705,17451.68248953376\nhybrid,H4,2013-02-01,2013,2,lat_menos35_a_menos15,20880,52471.28499864979,4115.188685321571,22190.349946520706,46672.839026807305,18808.59412325443\nhybrid,H4,2013-02-01,2013,2,lat_maior_igual_menos15,31581,260320.27266014143,-13981.009612490818,56546.691213301696,259071.5599788598,31215.50842646724\nhybrid,H4,2013-03-01,2013,3,dominio_inteiro,78561,264248.7700412648,617.3582005159064,84297.57648713415,251750.63563391424,67475.78503925544\nhybrid,H4,2013-03-01,2013,3,lat_menor_que_menos35,26100,22439.943568237875,2030.4252462106951,17386.3659359182,15607.165601606626,17451.68248953376\nhybrid,H4,2013-03-01,2013,3,lat_menos35_a_menos15,20880,49629.48118416323,1567.3660826278829,21590.70841219993,44622.23128727752,18808.59412325443\nhybrid,H4,2013-03-01,2013,3,lat_maior_igual_menos15,31581,192179.3452888637,-2980.4331283226716,45320.50213901603,191521.23874503007,31215.50842646724\nhybrid,H4,2013-04-01,2013,4,dominio_inteiro,78561,228083.08308427775,22854.25155594201,87329.70716410683,215444.31273596056,67475.78503925544\nhybrid,H4,2013-04-01,2013,4,lat_menor_que_menos35,26100,30185.91130519279,5735.381162410549,22060.25993398327,21557.433394967564,17451.68248953376\nhybrid,H4,2013-04-01,2013,4,lat_menos35_a_menos15,20880,30468.37044268141,4721.522883058238,18279.2068199707,27199.298065725223,18808.59412325443\nhybrid,H4,2013-04-01,2013,4,lat_maior_igual_menos15,31581,167428.80133640356,12397.347510473224,46990.24041015287,166687.58127526776,31215.50842646724\nhybrid,H4,2013-05-01,2013,5,dominio_inteiro,78561,185019.87964517256,-5466.85512498732,71065.88461561859,175362.88524717197,67475.78503925544\nhybrid,H4,2013-05-01,2013,5,lat_menor_que_menos35,26100,19510.79452977306,-5592.17119233074,16740.759223373745,13422.69097750102,17451.68248953376\nhybrid,H4,2013-05-01,2013,5,lat_menos35_a_menos15,20880,25485.01911601679,7544.313423646411,16504.196756633675,22586.175005782246,18808.59412325443\nhybrid,H4,2013-05-01,2013,5,lat_maior_igual_menos15,31581,140024.06599938273,-7418.997356302991,37820.92863561118,139354.0192638887,31215.50842646724\nhybrid,H4,2013-06-01,2013,6,dominio_inteiro,78561,270337.12906639295,-12877.879123356615,84253.39756341619,247139.29922934223,67475.78503925544\nhybrid,H4,2013-06-01,2013,6,lat_menor_que_menos35,26100,51623.683962739306,-9463.468138410755,23996.24628940171,33969.38084432424,17451.68248953376\nhybrid,H4,2013-06-01,2013,6,lat_menos35_a_menos15,20880,42254.30281487162,-5576.637758111592,17095.625062212683,37934.59365444942,18808.59412325443\nhybrid,H4,2013-06-01,2013,6,lat_maior_igual_menos15,31581,176459.14228878205,2162.226773165731,43161.5262118018,175235.32473056857,31215.50842646724\nhybrid,H4,2013-07-01,2013,7,dominio_inteiro,78561,163704.78624712635,11962.663709378521,71191.06726341491,151695.1595115366,67475.78503925544\nhybrid,H4,2013-07-01,2013,7,lat_menor_que_menos35,26100,24737.358007466853,7103.275627552021,19975.77181005729,17097.682866664345,17451.68248953376\nhybrid,H4,2013-07-01,2013,7,lat_menos35_a_menos15,20880,26268.628758607814,-655.5444132202456,15158.127994772236,22865.866315389376,18808.59412325443\nhybrid,H4,2013-07-01,2013,7,lat_maior_igual_menos15,31581,112698.79948105165,5514.932495046743,36057.16745858538,111731.61032948286,31215.50842646724\nhybrid,H4,2013-08-01,2013,8,dominio_inteiro,78561,148610.7500810013,357.5737207308075,66032.35249336985,135079.08117997722,67475.78503925544\nhybrid,H4,2013-08-01,2013,8,lat_menor_que_menos35,26100,32449.256149136036,-7038.78081700014,19468.987553519102,23118.23581473013,17451.68248953376\nhybrid,H4,2013-08-01,2013,8,lat_menos35_a_menos15,20880,25986.635637643398,5431.582305743142,15857.863981657616,22637.056717664258,18808.59412325443\nhybrid,H4,2013-08-01,2013,8,lat_maior_igual_menos15,31581,90174.85829422185,1964.7722319878048,30705.500958193137,89323.78864758281,31215.50842646724\nhybrid,H4,2013-09-01,2013,9,dominio_inteiro,78561,116085.24219680601,9592.885196968895,59012.690749024856,104548.4546820948,67475.78503925544\nhybrid,H4,2013-09-01,2013,9,lat_menor_que_menos35,26100,26111.658411583958,2011.1472665014915,17426.022696482578,18249.701594952196,17451.68248953376\nhybrid,H4,2013-09-01,2013,9,lat_menos35_a_menos15,20880,22364.647468485036,1210.132873308286,14039.51901177757,19568.966059132486,18808.59412325443\nhybrid,H4,2013-09-01,2013,9,lat_maior_igual_menos15,31581,67608.936316737,6371.605057159118,27547.149040764707,66729.78702801012,31215.50842646724\nhybrid,H4,2013-10-01,2013,10,dominio_inteiro,78561,133670.75296971764,6077.488153062702,64558.31814412274,124005.6410249483,67475.78503925544\nhybrid,H4,2013-10-01,2013,10,lat_menor_que_menos35,26100,18781.66969583517,5189.806470349462,15903.82097058068,13636.565924827797,17451.68248953376\nhybrid,H4,2013-10-01,2013,10,lat_menos35_a_menos15,20880,32281.4711196078,-31.223798393635036,16956.82829066021,28590.06803728469,18808.59412325443\nhybrid,H4,2013-10-01,2013,10,lat_maior_igual_menos15,31581,82607.61215427467,918.9054811068745,31697.668882881848,81779.0070628358,31215.50842646724\nhybrid,H4,2013-11-01,2013,11,dominio_inteiro,78561,213057.20630684297,-5680.9709759325815,82405.6079846049,198959.72687093186,67475.78503925544\nhybrid,H4,2013-11-01,2013,11,lat_menor_que_menos35,26100,24840.966109052435,-8499.776175502788,17169.11242259279,18022.603246729053,17451.68248953376\nhybrid,H4,2013-11-01,2013,11,lat_menos35_a_menos15,20880,51714.30754558352,6535.074469749816,21415.113444842853,45753.67966555913,18808.59412325443\nhybrid,H4,2013-11-01,2013,11,lat_maior_igual_menos15,31581,136501.93265220703,-3716.2692701796095,43821.38211716924,135183.44395864365,31215.50842646724\nhybrid,H4,2013-12-01,2013,12,dominio_inteiro,78561,354519.1633811805,2513.062488063768,94837.60831191331,332226.19895518594,67475.78503925544\nhybrid,H4,2013-12-01,2013,12,lat_menor_que_menos35,26100,26563.34682270842,5227.907167539368,17541.544258090253,18765.282073138325,17451.68248953376\nhybrid,H4,2013-12-01,2013,12,lat_menos35_a_menos15,20880,153721.8570000079,90.02843908037767,32843.58117534881,140631.6095288358,18808.59412325443\nhybrid,H4,2013-12-01,2013,12,lat_maior_igual_menos15,31581,174233.9595584642,-2804.8731185559773,44452.482878474235,172829.30735321183,31215.50842646724\nhybrid,H4,2014-01-01,2014,1,dominio_inteiro,78561,381497.2905536173,6317.788357747351,101281.76045021317,357563.9189678075,67475.78503925544\nhybrid,H4,2014-01-01,2014,1,lat_menor_que_menos35,26100,32358.67971499721,-9690.510068672804,18626.093941386778,23578.64000324275,17451.68248953376\nhybrid,H4,2014-01-01,2014,1,lat_menos35_a_menos15,20880,156747.6566882155,20720.0317259059,36573.93536832211,143265.2583289288,18808.59412325443\nhybrid,H4,2014-01-01,2014,1,lat_maior_igual_menos15,31581,192390.95415040452,-4711.733299485746,46081.73114050428,190720.02063563594,31215.50842646724\nhybrid,H4,2014-02-01,2014,2,dominio_inteiro,78561,320758.20331073983,-4038.5249917037872,90649.70495505615,304888.8853260093,67475.78503925544\nhybrid,H4,2014-02-01,2014,2,lat_menor_que_menos35,26100,22799.612642800566,2634.524793092031,18298.937084129335,16447.78706641319,17451.68248953376\nhybrid,H4,2014-02-01,2014,2,lat_menos35_a_menos15,20880,83224.74672624198,6488.792377748437,26774.413277512453,74527.93218992061,18808.59412325443\nhybrid,H4,2014-02-01,2014,2,lat_maior_igual_menos15,31581,214733.84394169727,-13161.842162544253,45576.35459341438,213913.16606967547,31215.50842646724\nhybrid,H4,2014-03-01,2014,3,dominio_inteiro,78561,185009.0552732232,-8206.403707627356,79920.85495744906,168410.16552201723,67475.78503925544\nhybrid,H4,2014-03-01,2014,3,lat_menor_que_menos35,26100,34555.650844359545,-10824.468523104595,22105.043482552537,23606.109536826054,17451.68248953376\nhybrid,H4,2014-03-01,2014,3,lat_menos35_a_menos15,20880,47236.14782334087,2031.9425389413873,21654.062251985866,42106.04073002278,18808.59412325443\nhybrid,H4,2014-03-01,2014,3,lat_maior_igual_menos15,31581,103217.2566055228,586.122276535853,36161.749222910665,102698.01525516839,31215.50842646724\nhybrid,H4,2014-04-01,2014,4,dominio_inteiro,78561,165604.45094599645,9068.174693675888,71471.08712949629,151549.3217905717,67475.78503925544\nhybrid,H4,2014-04-01,2014,4,lat_menor_que_menos35,26100,38501.54340466688,-704.7000591617519,21012.887069317367,28049.998393822934,17451.68248953376\nhybrid,H4,2014-04-01,2014,4,lat_menos35_a_menos15,20880,28176.054459319843,4687.792763778913,16865.08597048327,24882.39687691975,18808.59412325443\nhybrid,H4,2014-04-01,2014,4,lat_maior_igual_menos15,31581,98926.85308200972,5085.081989058728,33593.11408969565,98616.92651982902,31215.50842646724\nhybrid,H4,2014-05-01,2014,5,dominio_inteiro,78561,265821.41797133023,-1168.1855879954478,81069.09977374486,257053.86140438935,67475.78503925544\nhybrid,H4,2014-05-01,2014,5,lat_menor_que_menos35,26100,18235.609039773077,-1722.569480045126,16641.725805927697,12463.770597117034,17451.68248953376\nhybrid,H4,2014-05-01,2014,5,lat_menos35_a_menos15,20880,21472.713559012645,5255.954567967025,15048.392880998803,19326.47292620481,18808.59412325443\nhybrid,H4,2014-05-01,2014,5,lat_maior_igual_menos15,31581,226113.0953725446,-4701.570675917349,49378.98108681836,225263.61788106748,31215.50842646724\nhybrid,H4,2014-06-01,2014,6,dominio_inteiro,78561,230742.57084970194,-5932.20628996017,75728.2132661936,216975.8107469278,67475.78503925544\nhybrid,H4,2014-06-01,2014,6,lat_menor_que_menos35,26100,20361.33499803394,-964.7214916563954,16394.629746948656,13989.055892868308,17451.68248953376\nhybrid,H4,2014-06-01,2014,6,lat_menos35_a_menos15,20880,54534.60623756274,-5259.361486092849,17727.327537314228,48052.68695875113,18808.59412325443\nhybrid,H4,2014-06-01,2014,6,lat_maior_igual_menos15,31581,155846.62961410524,291.8766877890748,41606.25598193073,154934.06789530837,31215.50842646724\nhybrid,H4,2014-07-01,2014,7,dominio_inteiro,78561,208710.41245618655,-6946.046603995072,71514.9634095262,196978.6065613247,67475.78503925544\nhybrid,H4,2014-07-01,2014,7,lat_menor_que_menos35,26100,29426.504312414603,-9901.723795471129,19389.672200028625,21606.877875120696,17451.68248953376\nhybrid,H4,2014-07-01,2014,7,lat_menos35_a_menos15,20880,22909.127163864618,-1295.2921068421015,14510.43524492283,20386.506233589476,18808.59412325443\nhybrid,H4,2014-07-01,2014,7,lat_maior_igual_menos15,31581,156374.78097990731,4250.969298318159,37614.855964574745,154985.22245261457,31215.50842646724\nhybrid,H4,2014-08-01,2014,8,dominio_inteiro,78561,162806.15520381686,-1252.4885590695321,63685.947398558215,153238.45855543046,67475.78503925544\nhybrid,H4,2014-08-01,2014,8,lat_menor_que_menos35,26100,21272.387009732858,-7596.651091249396,16684.128482025422,15075.0342397327,17451.68248953376\nhybrid,H4,2014-08-01,2014,8,lat_menos35_a_menos15,20880,16955.02702322258,10935.130135341842,13399.169902535872,14976.647428484599,18808.59412325443\nhybrid,H4,2014-08-01,2014,8,lat_maior_igual_menos15,31581,124578.74117086141,-4590.96760316198,33602.64901399692,123186.77688721316,31215.50842646724\nhybrid,H4,2014-09-01,2014,9,dominio_inteiro,78561,130597.41512168724,-1192.0635783322218,62349.053435074435,120079.61083108449,67475.78503925544\nhybrid,H4,2014-09-01,2014,9,lat_menor_que_menos35,26100,22754.36708168187,-7325.021610641817,17776.394249859713,16134.778199042892,17451.68248953376\nhybrid,H4,2014-09-01,2014,9,lat_menos35_a_menos15,20880,28399.965968005603,-1912.5449238200447,14793.794409230715,25257.41080010365,18808.59412325443\nhybrid,H4,2014-09-01,2014,9,lat_maior_igual_menos15,31581,79443.08207199976,8045.502956129638,29778.864775983995,78687.42183193793,31215.50842646724\nhybrid,H4,2014-10-01,2014,10,dominio_inteiro,78561,183124.84218580672,5116.026069660853,75198.02706451603,171919.0949061027,67475.78503925544\nhybrid,H4,2014-10-01,2014,10,lat_menor_que_menos35,26100,14396.841143426262,601.6548329667351,13905.213731150476,10066.672062199346,17451.68248953376\nhybrid,H4,2014-10-01,2014,10,lat_menos35_a_menos15,20880,56393.67109246942,17759.66075132659,24086.333378536594,50796.55776273484,18808.59412325443\nhybrid,H4,2014-10-01,2014,10,lat_maior_igual_menos15,31581,112334.32994991103,-13245.289514632472,37206.47995482897,111055.86508116852,31215.50842646724\nhybrid,H4,2014-11-01,2014,11,dominio_inteiro,78561,275649.85770529916,-27979.783862176857,81627.80948469676,259973.04985245923,67475.78503925544\nhybrid,H4,2014-11-01,2014,11,lat_menor_que_menos35,26100,31588.74705553095,-8250.832917158508,18019.2285652092,21246.506922020413,17451.68248953376\nhybrid,H4,2014-11-01,2014,11,lat_menos35_a_menos15,20880,36096.08449494935,-1851.8074841281777,18144.049869237053,32419.27199632095,18808.59412325443\nhybrid,H4,2014-11-01,2014,11,lat_maior_igual_menos15,31581,207965.02615481883,-17877.14346089017,45464.531050250516,206307.2709341179,31215.50842646724\nhybrid,H4,2014-12-01,2014,12,dominio_inteiro,78561,206037.11998607716,13388.415358003984,79962.63606362925,190854.95702880583,67475.78503925544\nhybrid,H4,2014-12-01,2014,12,lat_menor_que_menos35,26100,26540.670918495747,592.6265582563157,18195.989398214904,19066.79318069512,17451.68248953376\nhybrid,H4,2014-12-01,2014,12,lat_menos35_a_menos15,20880,70011.93024076076,7220.824241643093,24934.000859608495,63161.69548354202,18808.59412325443\nhybrid,H4,2014-12-01,2014,12,lat_maior_igual_menos15,31581,109484.51882682065,5574.964558104575,36832.64580580585,108626.46836456867,31215.50842646724\nclimatology,H4,2013-01-01,2013,1,dominio_inteiro,78561,304141.45466963237,17506.646335562516,99357.57460702228,284837.5830283441,67475.78503925544\nclimatology,H4,2013-01-01,2013,1,lat_menor_que_menos35,26100,33472.79833149901,7067.121479995549,20944.152294702828,23446.59095246029,17451.68248953376\nclimatology,H4,2013-01-01,2013,1,lat_menos35_a_menos15,20880,86942.84265805282,3718.5954878645134,29258.02157942223,78977.56693297529,18808.59412325443\nclimatology,H4,2013-01-01,2013,1,lat_maior_igual_menos15,31581,183725.81368008052,6720.929367702454,49155.40073289722,182413.4251429085,31215.50842646724\nclimatology,H4,2013-02-01,2013,2,dominio_inteiro,78561,363240.8827020171,-10462.95591140457,100956.74505577551,347477.16188578366,67475.78503925544\nclimatology,H4,2013-02-01,2013,2,lat_menor_que_menos35,26100,31719.863643645982,2038.4366336762905,19719.88609328866,23292.83547959914,17451.68248953376\nclimatology,H4,2013-02-01,2013,2,lat_menos35_a_menos15,20880,55787.61303110859,5886.785147285904,23287.095524900942,49720.17940237186,18808.59412325443\nclimatology,H4,2013-02-01,2013,2,lat_maior_igual_menos15,31581,275733.40602726256,-18388.177692366764,57949.763437585905,274464.14700381266,31215.50842646724\nclimatology,H4,2013-03-01,2013,3,dominio_inteiro,78561,288171.8182819378,3297.2373160030693,87563.15897816792,275166.270085811,67475.78503925544\nclimatology,H4,2013-03-01,2013,3,lat_menor_que_menos35,26100,22980.192406110757,3420.08923342824,17759.947432905436,16006.366735815653,17451.68248953376\nclimatology,H4,2013-03-01,2013,3,lat_menos35_a_menos15,20880,51669.97954317616,2565.7053221426904,22428.051966646686,46342.07075319973,18808.59412325443\nclimatology,H4,2013-03-01,2013,3,lat_maior_igual_menos15,31581,213521.64633265088,-2688.557239567861,47375.1595786158,212817.8325967956,31215.50842646724\nclimatology,H4,2013-04-01,2013,4,dominio_inteiro,78561,226603.4029143954,22550.85652502766,88400.27785360208,213780.36490712705,67475.78503925544\nclimatology,H4,2013-04-01,2013,4,lat_menor_que_menos35,26100,30837.207231136694,6680.938105434179,22376.7017570436,21955.04076907664,17451.68248953376\nclimatology,H4,2013-04-01,2013,4,lat_menos35_a_menos15,20880,29970.066236827355,4709.074975795578,18130.961478031706,26802.348197490992,18808.59412325443\nclimatology,H4,2013-04-01,2013,4,lat_maior_igual_menos15,31581,165796.12944643135,11160.843443797901,47892.61461852677,165022.97594055944,31215.50842646724\nclimatology,H4,2013-05-01,2013,5,dominio_inteiro,78561,198016.33609057902,-10753.070905953413,73083.13062870665,187120.61828011143,67475.78503925544\nclimatology,H4,2013-05-01,2013,5,lat_menor_que_menos35,26100,24173.681312183562,-6110.074359118938,17841.790646374226,16523.589877390776,17451.68248953376\nclimatology,H4,2013-05-01,2013,5,lat_menos35_a_menos15,20880,22979.71172340816,7313.083525515394,15658.98069563019,20366.282870669602,18808.59412325443\nclimatology,H4,2013-05-01,2013,5,lat_maior_igual_menos15,31581,150862.9430549873,-11956.080072349869,39582.35928670224,150230.74553205105,31215.50842646724\nclimatology,H4,2013-06-01,2013,6,dominio_inteiro,78561,288135.6064243315,-12642.567350076046,87871.02066756552,261572.09922559513,67475.78503925544\nclimatology,H4,2013-06-01,2013,6,lat_menor_que_menos35,26100,60749.76445890012,-8331.884546101093,25767.69334369898,40425.9485294021,17451.68248953376\nclimatology,H4,2013-06-01,2013,6,lat_menos35_a_menos15,20880,48363.71517350165,-5018.2451295531355,18856.654936560895,43274.88341882694,18808.59412325443\nclimatology,H4,2013-06-01,2013,6,lat_maior_igual_menos15,31581,179022.1267919297,707.5623255781829,43246.67238730565,177871.2672773661,31215.50842646724\nclimatology,H4,2013-07-01,2013,7,dominio_inteiro,78561,170143.1671837245,11867.34025798738,73136.52741583809,157126.33945078024,67475.78503925544\nclimatology,H4,2013-07-01,2013,7,lat_menor_que_menos35,26100,26987.68168696812,6740.587789148092,20577.27830722928,18581.332856181863,17451.68248953376\nclimatology,H4,2013-07-01,2013,7,lat_menos35_a_menos15,20880,27873.136861328057,277.69323494751006,15283.716389757581,24234.629584573842,18808.59412325443\nclimatology,H4,2013-07-01,2013,7,lat_maior_igual_menos15,31581,115282.34863542832,4849.059233891778,37275.53271885123,114310.37701002455,31215.50842646724\nclimatology,H4,2013-08-01,2013,8,dominio_inteiro,78561,159855.78333139938,-6981.757800351363,68156.03405794548,144794.72385752067,67475.78503925544\nclimatology,H4,2013-08-01,2013,8,lat_menor_que_menos35,26100,37351.17911658365,-8020.7585009634495,20546.25518706441,26485.744154550874,17451.68248953376\nclimatology,H4,2013-08-01,2013,8,lat_menos35_a_menos15,20880,25956.77129240693,5967.9341525961645,16113.963068142068,22640.478269286086,18808.59412325443\nclimatology,H4,2013-08-01,2013,8,lat_maior_igual_menos15,31581,96547.8329224088,-4928.933451984078,31495.815802739,95668.50143368372,31215.50842646724\nclimatology,H4,2013-09-01,2013,9,dominio_inteiro,78561,114762.97701018918,9199.281829436542,58464.07365422952,103569.9210656395,67475.78503925544\nclimatology,H4,2013-09-01,2013,9,lat_menor_que_menos35,26100,26090.45574541743,342.11183539032936,17267.11248704791,18426.464409455228,17451.68248953376\nclimatology,H4,2013-09-01,2013,9,lat_menos35_a_menos15,20880,21691.26224706193,2612.712794650113,14088.363857799442,19010.063478267748,18808.59412325443\nclimatology,H4,2013-09-01,2013,9,lat_maior_igual_menos15,31581,66981.25901770982,6244.4571993961,27108.597309382167,66133.39317791653,31215.50842646724\nclimatology,H4,2013-10-01,2013,10,dominio_inteiro,78561,132999.96145883173,4094.259833627322,65457.930777580186,123240.63323319273,67475.78503925544\nclimatology,H4,2013-10-01,2013,10,lat_menor_que_menos35,26100,19018.764619273512,3877.0685925483704,16131.404757618904,13795.842384081416,17451.68248953376\nclimatology,H4,2013-10-01,2013,10,lat_menos35_a_menos15,20880,32561.71635529598,153.69059894880047,17107.641341249167,28840.563304977524,18808.59412325443\nclimatology,H4,2013-10-01,2013,10,lat_maior_igual_menos15,31581,81419.48048426225,63.50064213015139,32218.884678712115,80604.2275441338,31215.50842646724\nclimatology,H4,2013-11-01,2013,11,dominio_inteiro,78561,199302.113457707,-11481.33281257705,79379.82244408198,185205.74310078306,67475.78503925544\nclimatology,H4,2013-11-01,2013,11,lat_menor_que_menos35,26100,25343.142846143463,-9931.95775860548,17169.863916873932,18377.061563039773,17451.68248953376\nclimatology,H4,2013-11-01,2013,11,lat_menos35_a_menos15,20880,52328.45594430716,5930.187958084483,21371.77025131203,46336.56269482753,18808.59412325443\nclimatology,H4,2013-11-01,2013,11,lat_maior_igual_menos15,31581,121630.51466725633,-7479.563012056053,40838.18827589601,120492.11884291578,31215.50842646724\nclimatology,H4,2013-12-01,2013,12,dominio_inteiro,78561,346571.9282864966,3066.1784090044093,93638.89723303594,325944.46114616434,67475.78503925544\nclimatology,H4,2013-12-01,2013,12,lat_menor_que_menos35,26100,26062.691565706282,4982.246356368065,17368.401477634907,18314.06165868628,17451.68248953376\nclimatology,H4,2013-12-01,2013,12,lat_menos35_a_menos15,20880,136414.67517491707,852.9219128208351,30740.097740542667,124854.70765502643,18808.59412325443\nclimatology,H4,2013-12-01,2013,12,lat_maior_igual_menos15,31581,184094.56154587324,-2768.9898601844907,45530.398014858365,182775.69183245162,31215.50842646724\nclimatology,H4,2014-01-01,2014,1,dominio_inteiro,78561,342571.0164310141,9276.503742178727,97247.89137917518,322470.80279355554,67475.78503925544\nclimatology,H4,2014-01-01,2014,1,lat_menor_que_menos35,26100,28699.79691324605,-7379.657771103084,17631.137937150896,21104.42657268739,17451.68248953376\nclimatology,H4,2014-01-01,2014,1,lat_menos35_a_menos15,20880,126503.88875483326,19223.57748249342,32889.13350730523,115572.96394298965,18808.59412325443\nclimatology,H4,2014-01-01,2014,1,lat_maior_igual_menos15,31581,187367.33076293484,-2567.415969211608,46727.619934719056,185793.41227787847,31215.50842646724\nclimatology,H4,2014-02-01,2014,2,dominio_inteiro,78561,326318.98140476376,-1692.3241212434368,93779.22102703375,309417.90714332816,67475.78503925544\nclimatology,H4,2014-02-01,2014,2,lat_menor_que_menos35,26100,23875.617226621405,2117.8738895356655,18519.65286991,17361.671038707096,17451.68248953376\nclimatology,H4,2014-02-01,2014,2,lat_menos35_a_menos15,20880,91315.69690696069,8571.565268135513,28747.903094247333,81817.03397488839,18808.59412325443\nclimatology,H4,2014-02-01,2014,2,lat_maior_igual_menos15,31581,211127.66727118165,-12381.763278914616,46511.66506287642,210239.2021297327,31215.50842646724\nclimatology,H4,2014-03-01,2014,3,dominio_inteiro,78561,185783.38175093255,-405.5062142703682,81331.54611158185,168943.42618156562,67475.78503925544\nclimatology,H4,2014-03-01,2014,3,lat_menor_que_menos35,26100,35398.31413058688,-9919.7335357368,22473.143300265074,24037.31903659657,17451.68248953376\nclimatology,H4,2014-03-01,2014,3,lat_menos35_a_menos15,20880,45046.17786278741,1073.7219084464014,21114.096246898174,40105.65657407154,18808.59412325443\nclimatology,H4,2014-03-01,2014,3,lat_maior_igual_menos15,31581,105338.88975755827,8440.50541302003,37744.3065644186,104800.45057089753,31215.50842646724\nclimatology,H4,2014-04-01,2014,4,dominio_inteiro,78561,195318.36044645976,20583.386355960276,78409.44752051448,180439.63323996175,67475.78503925544\nclimatology,H4,2014-04-01,2014,4,lat_menor_que_menos35,26100,40708.02147722754,1698.3331249654293,21903.225090414286,29593.678244568768,17451.68248953376\nclimatology,H4,2014-04-01,2014,4,lat_menos35_a_menos15,20880,28541.268775408018,2902.8978365133516,16714.713664765935,25197.760422378535,18808.59412325443\nclimatology,H4,2014-04-01,2014,4,lat_maior_igual_menos15,31581,126069.0701938242,15982.155394481495,39791.50876533426,125648.19457301447,31215.50842646724\nclimatology,H4,2014-05-01,2014,5,dominio_inteiro,78561,317055.52773401263,-793.4723646936473,86933.19875975535,308349.71670937043,67475.78503925544\nclimatology,H4,2014-05-01,2014,5,lat_menor_que_menos35,26100,17650.39614491681,-1139.8709594607353,16327.576309859753,12015.865087855733,17451.68248953376\nclimatology,H4,2014-05-01,2014,5,lat_menos35_a_menos15,20880,20235.589970962857,2666.030760622816,14453.435515870573,18337.03557023711,18808.59412325443\nclimatology,H4,2014-05-01,2014,5,lat_maior_igual_menos15,31581,279169.54161813296,-2319.632165855728,56152.18693402503,277996.8160512776,31215.50842646724\nclimatology,H4,2014-06-01,2014,6,dominio_inteiro,78561,246089.88152102963,-4302.862836526241,78113.0509362123,231571.96852133883,67475.78503925544\nclimatology,H4,2014-06-01,2014,6,lat_menor_que_menos35,26100,21652.66184480765,102.82691019773483,16813.406352877617,14921.549088302865,17451.68248953376\nclimatology,H4,2014-06-01,2014,6,lat_menos35_a_menos15,20880,56507.26258739234,-6862.471966711339,17637.94817797141,49800.61014342719,18808.59412325443\nclimatology,H4,2014-06-01,2014,6,lat_maior_igual_menos15,31581,167929.95708882966,2456.7822199873626,43661.69640536327,166849.8092896088,31215.50842646724\nclimatology,H4,2014-07-01,2014,7,dominio_inteiro,78561,225886.7632616058,-139.5471779257059,76234.25938838907,213877.0567282548,67475.78503925544\nclimatology,H4,2014-07-01,2014,7,lat_menor_que_menos35,26100,29170.49386641622,-8123.0005439817905,19378.579033643007,21461.14968531795,17451.68248953376\nclimatology,H4,2014-07-01,2014,7,lat_menos35_a_menos15,20880,24604.803321409338,-2690.101183387451,15119.937323206104,21924.538801611052,18808.59412325443\nclimatology,H4,2014-07-01,2014,7,lat_maior_igual_menos15,31581,172111.46607378026,10673.554549443536,41735.743031539954,170491.3682413258,31215.50842646724\nclimatology,H4,2014-08-01,2014,8,dominio_inteiro,78561,156823.05221257347,891.4302184474654,60559.04155688314,148135.77194237753,67475.78503925544\nclimatology,H4,2014-08-01,2014,8,lat_menor_que_menos35,26100,20003.999854796763,-7294.6537646353245,16265.380910426378,14218.894083255169,17451.68248953376\nclimatology,H4,2014-08-01,2014,8,lat_menos35_a_menos15,20880,14240.92681733019,9963.772317879368,12242.243696013931,12550.674877354568,18808.59412325443\nclimatology,H4,2014-08-01,2014,8,lat_maior_igual_menos15,31581,122578.12554044652,-1777.6883347965777,32051.41695044283,121366.2029817678,31215.50842646724\nclimatology,H4,2014-09-01,2014,9,dominio_inteiro,78561,137437.3549838978,4928.566115935566,64185.950258932775,127097.81452889171,67475.78503925544\nclimatology,H4,2014-09-01,2014,9,lat_menor_que_menos35,26100,20939.258461948368,-8063.058605283499,16813.740304082632,14956.511235652923,17451.68248953376\nclimatology,H4,2014-09-01,2014,9,lat_menos35_a_menos15,20880,32179.15293613613,-2260.912006985629,15622.884925551945,28563.29650547405,18808.59412325443\nclimatology,H4,2014-09-01,2014,9,lat_maior_igual_menos15,31581,84318.94358581332,15252.536728204694,31749.325029298197,83578.00678776472,31215.50842646724\nclimatology,H4,2014-10-01,2014,10,dominio_inteiro,78561,178614.40296342608,9126.440619760135,74583.6372271668,167641.12546864984,67475.78503925544\nclimatology,H4,2014-10-01,2014,10,lat_menor_que_menos35,26100,13717.237600782155,231.43489503860474,13642.667382717133,9658.400520857558,17451.68248953376\nclimatology,H4,2014-10-01,2014,10,lat_menos35_a_menos15,20880,58254.64299109097,17884.46452473005,24124.26629203034,52567.37133961417,18808.59412325443\nclimatology,H4,2014-10-01,2014,10,lat_maior_igual_menos15,31581,106642.52237155296,-8989.45880000852,36816.70355241932,105415.3536081781,31215.50842646724\nclimatology,H4,2014-11-01,2014,11,dominio_inteiro,78561,273148.7984930296,-25624.575648575585,81952.48759651298,257583.1464364914,67475.78503925544\nclimatology,H4,2014-11-01,2014,11,lat_menor_que_menos35,26100,31504.222699342918,-8624.955691039562,18108.135396778584,21262.070436493646,17451.68248953376\nclimatology,H4,2014-11-01,2014,11,lat_menos35_a_menos15,20880,35685.440229061496,-3317.8474728237197,17924.186791137996,32037.68946095019,18808.59412325443\nclimatology,H4,2014-11-01,2014,11,lat_maior_igual_menos15,31581,205959.1355646252,-13681.772484712303,45920.1654085964,204283.38653904753,31215.50842646724\nclimatology,H4,2014-12-01,2014,12,dominio_inteiro,78561,225151.2389383847,15111.905185127456,84262.32065035665,209145.98294504994,67475.78503925544\nclimatology,H4,2014-12-01,2014,12,lat_menor_que_menos35,26100,27244.927381611695,-1223.0720945596695,18103.221080094576,19514.053595574787,17451.68248953376\nclimatology,H4,2014-12-01,2014,12,lat_menos35_a_menos15,20880,75827.56262422854,7151.784370096226,25963.350793863472,68377.81128662916,18808.59412325443\nclimatology,H4,2014-12-01,2014,12,lat_maior_igual_menos15,31581,122078.74893254446,9183.1929095909,40195.7487763986,121254.11806284601,31215.50842646724\n', 'research/spatial_unet/evidence/seven_blocks/H4/source_calendar.csv': 'time_alvo,limite_emissao,atmosfera,indices,sst,inicializacao_sazonal,ultimo_alvo_treino,disponibilidade_historica_comprovada\n2013-01-01,2012-12-31 23:59:59.999999999,2012-09-01,2012-10-01,2012-11-01,2012-12-01,2012-09-01,False\n2013-02-01,2013-01-31 23:59:59.999999999,2012-10-01,2012-11-01,2012-12-01,2013-01-01,2012-09-01,False\n2013-03-01,2013-02-28 23:59:59.999999999,2012-11-01,2012-12-01,2013-01-01,2013-02-01,2012-09-01,False\n2013-04-01,2013-03-31 23:59:59.999999999,2012-12-01,2013-01-01,2013-02-01,2013-03-01,2012-09-01,False\n2013-05-01,2013-04-30 23:59:59.999999999,2013-01-01,2013-02-01,2013-03-01,2013-04-01,2012-09-01,False\n2013-06-01,2013-05-31 23:59:59.999999999,2013-02-01,2013-03-01,2013-04-01,2013-05-01,2012-09-01,False\n2013-07-01,2013-06-30 23:59:59.999999999,2013-03-01,2013-04-01,2013-05-01,2013-06-01,2012-09-01,False\n2013-08-01,2013-07-31 23:59:59.999999999,2013-04-01,2013-05-01,2013-06-01,2013-07-01,2012-09-01,False\n2013-09-01,2013-08-31 23:59:59.999999999,2013-05-01,2013-06-01,2013-07-01,2013-08-01,2012-09-01,False\n2013-10-01,2013-09-30 23:59:59.999999999,2013-06-01,2013-07-01,2013-08-01,2013-09-01,2012-09-01,False\n2013-11-01,2013-10-31 23:59:59.999999999,2013-07-01,2013-08-01,2013-09-01,2013-10-01,2012-09-01,False\n2013-12-01,2013-11-30 23:59:59.999999999,2013-08-01,2013-09-01,2013-10-01,2013-11-01,2012-09-01,False\n2014-01-01,2013-12-31 23:59:59.999999999,2013-09-01,2013-10-01,2013-11-01,2013-12-01,2012-09-01,False\n2014-02-01,2014-01-31 23:59:59.999999999,2013-10-01,2013-11-01,2013-12-01,2014-01-01,2012-09-01,False\n2014-03-01,2014-02-28 23:59:59.999999999,2013-11-01,2013-12-01,2014-01-01,2014-02-01,2012-09-01,False\n2014-04-01,2014-03-31 23:59:59.999999999,2013-12-01,2014-01-01,2014-02-01,2014-03-01,2012-09-01,False\n2014-05-01,2014-04-30 23:59:59.999999999,2014-01-01,2014-02-01,2014-03-01,2014-04-01,2012-09-01,False\n2014-06-01,2014-05-31 23:59:59.999999999,2014-02-01,2014-03-01,2014-04-01,2014-05-01,2012-09-01,False\n2014-07-01,2014-06-30 23:59:59.999999999,2014-03-01,2014-04-01,2014-05-01,2014-06-01,2012-09-01,False\n2014-08-01,2014-07-31 23:59:59.999999999,2014-04-01,2014-05-01,2014-06-01,2014-07-01,2012-09-01,False\n2014-09-01,2014-08-31 23:59:59.999999999,2014-05-01,2014-06-01,2014-07-01,2014-08-01,2012-09-01,False\n2014-10-01,2014-09-30 23:59:59.999999999,2014-06-01,2014-07-01,2014-08-01,2014-09-01,2012-09-01,False\n2014-11-01,2014-10-31 23:59:59.999999999,2014-07-01,2014-08-01,2014-09-01,2014-10-01,2012-09-01,False\n2014-12-01,2014-11-30 23:59:59.999999999,2014-08-01,2014-09-01,2014-10-01,2014-11-01,2012-09-01,False\n', 'research/spatial_unet/evidence/seven_blocks/H4/split.json': '{\n  "historical_publication_vintages_verified": false,\n  "independent_holdout": false,\n  "inner_reference": [\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06"\n  ],\n  "inner_train": [\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06"\n  ],\n  "inner_validation": [\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09"\n  ],\n  "outer": [\n    "2013-01",\n    "2013-02",\n    "2013-03",\n    "2013-04",\n    "2013-05",\n    "2013-06",\n    "2013-07",\n    "2013-08",\n    "2013-09",\n    "2013-10",\n    "2013-11",\n    "2013-12",\n    "2014-01",\n    "2014-02",\n    "2014-03",\n    "2014-04",\n    "2014-05",\n    "2014-06",\n    "2014-07",\n    "2014-08",\n    "2014-09",\n    "2014-10",\n    "2014-11",\n    "2014-12"\n  ],\n  "train": [\n    "1982-10",\n    "1982-11",\n    "1982-12",\n    "1983-01",\n    "1983-02",\n    "1983-03",\n    "1983-04",\n    "1983-05",\n    "1983-06",\n    "1983-07",\n    "1983-08",\n    "1983-09",\n    "1983-10",\n    "1983-11",\n    "1983-12",\n    "1984-01",\n    "1984-02",\n    "1984-03",\n    "1984-04",\n    "1984-05",\n    "1984-06",\n    "1984-07",\n    "1984-08",\n    "1984-09",\n    "1984-10",\n    "1984-11",\n    "1984-12",\n    "1985-01",\n    "1985-02",\n    "1985-03",\n    "1985-04",\n    "1985-05",\n    "1985-06",\n    "1985-07",\n    "1985-08",\n    "1985-09",\n    "1985-10",\n    "1985-11",\n    "1985-12",\n    "1986-01",\n    "1986-02",\n    "1986-03",\n    "1986-04",\n    "1986-05",\n    "1986-06",\n    "1986-07",\n    "1986-08",\n    "1986-09",\n    "1986-10",\n    "1986-11",\n    "1986-12",\n    "1987-01",\n    "1987-02",\n    "1987-03",\n    "1987-04",\n    "1987-05",\n    "1987-06",\n    "1987-07",\n    "1987-08",\n    "1987-09",\n    "1987-10",\n    "1987-11",\n    "1987-12",\n    "1988-01",\n    "1988-02",\n    "1988-03",\n    "1988-04",\n    "1988-05",\n    "1988-06",\n    "1988-07",\n    "1988-08",\n    "1988-09",\n    "1988-10",\n    "1988-11",\n    "1988-12",\n    "1989-01",\n    "1989-02",\n    "1989-03",\n    "1989-04",\n    "1989-05",\n    "1989-06",\n    "1989-07",\n    "1989-08",\n    "1989-09",\n    "1989-10",\n    "1989-11",\n    "1989-12",\n    "1990-01",\n    "1990-02",\n    "1990-03",\n    "1990-04",\n    "1990-05",\n    "1990-06",\n    "1990-07",\n    "1990-08",\n    "1990-09",\n    "1990-10",\n    "1990-11",\n    "1990-12",\n    "1991-01",\n    "1991-02",\n    "1991-03",\n    "1991-04",\n    "1991-05",\n    "1991-06",\n    "1991-07",\n    "1991-08",\n    "1991-09",\n    "1991-10",\n    "1991-11",\n    "1991-12",\n    "1992-01",\n    "1992-02",\n    "1992-03",\n    "1992-04",\n    "1992-05",\n    "1992-06",\n    "1992-07",\n    "1992-08",\n    "1992-09",\n    "1992-10",\n    "1992-11",\n    "1992-12",\n    "1993-01",\n    "1993-02",\n    "1993-03",\n    "1993-04",\n    "1993-05",\n    "1993-06",\n    "1993-07",\n    "1993-08",\n    "1993-09",\n    "1993-10",\n    "1993-11",\n    "1993-12",\n    "1994-01",\n    "1994-02",\n    "1994-03",\n    "1994-04",\n    "1994-05",\n    "1994-06",\n    "1994-07",\n    "1994-08",\n    "1994-09",\n    "1994-10",\n    "1994-11",\n    "1994-12",\n    "1995-01",\n    "1995-02",\n    "1995-03",\n    "1995-04",\n    "1995-05",\n    "1995-06",\n    "1995-07",\n    "1995-08",\n    "1995-09",\n    "1995-10",\n    "1995-11",\n    "1995-12",\n    "1996-01",\n    "1996-02",\n    "1996-03",\n    "1996-04",\n    "1996-05",\n    "1996-06",\n    "1996-07",\n    "1996-08",\n    "1996-09",\n    "1996-10",\n    "1996-11",\n    "1996-12",\n    "1997-01",\n    "1997-02",\n    "1997-03",\n    "1997-04",\n    "1997-05",\n    "1997-06",\n    "1997-07",\n    "1997-08",\n    "1997-09",\n    "1997-10",\n    "1997-11",\n    "1997-12",\n    "1998-01",\n    "1998-02",\n    "1998-03",\n    "1998-04",\n    "1998-05",\n    "1998-06",\n    "1998-07",\n    "1998-08",\n    "1998-09",\n    "1998-10",\n    "1998-11",\n    "1998-12",\n    "1999-01",\n    "1999-02",\n    "1999-03",\n    "1999-04",\n    "1999-05",\n    "1999-06",\n    "1999-07",\n    "1999-08",\n    "1999-09",\n    "1999-10",\n    "1999-11",\n    "1999-12",\n    "2000-01",\n    "2000-02",\n    "2000-03",\n    "2000-04",\n    "2000-05",\n    "2000-06",\n    "2000-07",\n    "2000-08",\n    "2000-09",\n    "2000-10",\n    "2000-11",\n    "2000-12",\n    "2001-01",\n    "2001-02",\n    "2001-03",\n    "2001-04",\n    "2001-05",\n    "2001-06",\n    "2001-07",\n    "2001-08",\n    "2001-09",\n    "2001-10",\n    "2001-11",\n    "2001-12",\n    "2002-01",\n    "2002-02",\n    "2002-03",\n    "2002-04",\n    "2002-05",\n    "2002-06",\n    "2002-07",\n    "2002-08",\n    "2002-09",\n    "2002-10",\n    "2002-11",\n    "2002-12",\n    "2003-01",\n    "2003-02",\n    "2003-03",\n    "2003-04",\n    "2003-05",\n    "2003-06",\n    "2003-07",\n    "2003-08",\n    "2003-09",\n    "2003-10",\n    "2003-11",\n    "2003-12",\n    "2004-01",\n    "2004-02",\n    "2004-03",\n    "2004-04",\n    "2004-05",\n    "2004-06",\n    "2004-07",\n    "2004-08",\n    "2004-09",\n    "2004-10",\n    "2004-11",\n    "2004-12",\n    "2005-01",\n    "2005-02",\n    "2005-03",\n    "2005-04",\n    "2005-05",\n    "2005-06",\n    "2005-07",\n    "2005-08",\n    "2005-09",\n    "2005-10",\n    "2005-11",\n    "2005-12",\n    "2006-01",\n    "2006-02",\n    "2006-03",\n    "2006-04",\n    "2006-05",\n    "2006-06",\n    "2006-07",\n    "2006-08",\n    "2006-09",\n    "2006-10",\n    "2006-11",\n    "2006-12",\n    "2007-01",\n    "2007-02",\n    "2007-03",\n    "2007-04",\n    "2007-05",\n    "2007-06",\n    "2007-07",\n    "2007-08",\n    "2007-09",\n    "2007-10",\n    "2007-11",\n    "2007-12",\n    "2008-01",\n    "2008-02",\n    "2008-03",\n    "2008-04",\n    "2008-05",\n    "2008-06",\n    "2008-07",\n    "2008-08",\n    "2008-09",\n    "2008-10",\n    "2008-11",\n    "2008-12",\n    "2009-01",\n    "2009-02",\n    "2009-03",\n    "2009-04",\n    "2009-05",\n    "2009-06",\n    "2009-07",\n    "2009-08",\n    "2009-09",\n    "2009-10",\n    "2009-11",\n    "2009-12",\n    "2010-01",\n    "2010-02",\n    "2010-03",\n    "2010-04",\n    "2010-05",\n    "2010-06",\n    "2010-07",\n    "2010-08",\n    "2010-09",\n    "2010-10",\n    "2010-11",\n    "2010-12",\n    "2011-01",\n    "2011-02",\n    "2011-03",\n    "2011-04",\n    "2011-05",\n    "2011-06",\n    "2011-07",\n    "2011-08",\n    "2011-09",\n    "2011-10",\n    "2011-11",\n    "2011-12",\n    "2012-01",\n    "2012-02",\n    "2012-03",\n    "2012-04",\n    "2012-05",\n    "2012-06",\n    "2012-07",\n    "2012-08",\n    "2012-09"\n  ]\n}\n', 'research/spatial_unet/evidence/seven_blocks/H4/unet/training.json': '{\n  "device": "cuda",\n  "inner_selection": false,\n  "parameter_count": 122945,\n  "selected_epochs": 18,\n  "train_end": "2012-09-01",\n  "train_months": 360,\n  "train_start": "1982-10-01"\n}\n', 'research/spatial_unet/evidence/seven_blocks/H4/unet/training_history.json': '[\n  {\n    "epoch": 1,\n    "seconds": 21.171626597999875,\n    "training_mse": 3.0082419567637975\n  },\n  {\n    "epoch": 2,\n    "seconds": 21.623359331999836,\n    "training_mse": 2.8982314613130358\n  },\n  {\n    "epoch": 3,\n    "seconds": 21.061453625000013,\n    "training_mse": 2.8559495402706996\n  },\n  {\n    "epoch": 4,\n    "seconds": 20.929081594999843,\n    "training_mse": 2.8317435397042168\n  },\n  {\n    "epoch": 5,\n    "seconds": 21.066754058000242,\n    "training_mse": 2.807327257262336\n  },\n  {\n    "epoch": 6,\n    "seconds": 21.145039903999987,\n    "training_mse": 2.798392551475101\n  },\n  {\n    "epoch": 7,\n    "seconds": 21.059018397999807,\n    "training_mse": 2.7862269586986965\n  },\n  {\n    "epoch": 8,\n    "seconds": 21.017326183000023,\n    "training_mse": 2.768914735979504\n  },\n  {\n    "epoch": 9,\n    "seconds": 21.060424656000123,\n    "training_mse": 2.7595142834716375\n  },\n  {\n    "epoch": 10,\n    "seconds": 21.095600543000273,\n    "training_mse": 2.7534091591835024\n  },\n  {\n    "epoch": 11,\n    "seconds": 21.09601885100028,\n    "training_mse": 2.7433984405464598\n  },\n  {\n    "epoch": 12,\n    "seconds": 21.11318236399984,\n    "training_mse": 2.7344369649887086\n  },\n  {\n    "epoch": 13,\n    "seconds": 21.076305877999857,\n    "training_mse": 2.7290671666463218\n  },\n  {\n    "epoch": 14,\n    "seconds": 21.082653727000434,\n    "training_mse": 2.7223758565055\n  },\n  {\n    "epoch": 15,\n    "seconds": 21.095046078999985,\n    "training_mse": 2.724016895559099\n  },\n  {\n    "epoch": 16,\n    "seconds": 21.101597160999972,\n    "training_mse": 2.7142679538991716\n  },\n  {\n    "epoch": 17,\n    "seconds": 21.078469306999978,\n    "training_mse": 2.714252007007599\n  },\n  {\n    "epoch": 18,\n    "seconds": 21.069799074000002,\n    "training_mse": 2.7085363063547345\n  }\n]\n', 'research/spatial_unet/evidence/seven_blocks/global.csv': 'modelo,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area,rmse,mae,vies,rmse_area\nclimatology,13198248,45202524.14564265,-69688.76594280757,14525151.589091651,42592905.18072296,11335931.886594914,1.8506453532404494,1.100536343088238,-0.005280152785643031,1.9383848994707493\nhybrid,13198248,40576786.85674469,64885.73412620918,13821373.636109674,38050340.972427554,11335931.886594914,1.7533987173382521,1.0472127540041432,0.004916238437572107,1.8321062766271015\nunet,13198248,41102735.369748324,280034.8668725812,13994274.326886788,38543317.508053556,11335931.886594914,1.764725739441376,1.0603130299481256,0.021217578793229313,1.8439363783418774\n', 'research/spatial_unet/evidence/seven_blocks/signature.json': '{\n  "code": {\n    "research/common/__init__.py": "4f6bfaae6ba45669c7d41e1b087df0582970932f06cdea87a4ff3b352f4135f0",\n    "research/common/baseline.py": "653594bf49fa338304129d7d22e7dc31a4e32647b26ee87af7bea4bad94a4dfa",\n    "research/common/inputs.py": "d22140d4ad3db4ac3fecdca26c7ebdd1d780486b51622a1f333fce2a637e6851",\n    "research/common/synthetic_fixture.py": "da4de7a662c2c8c25f324c5b543ab04914d490ffde5e316c39187eb8f1640bff",\n    "research/spatial_unet/__init__.py": "ee933fd666f8d33250d76a02fcd8b898d3a2dc7b341dde8165411387cfa27c19",\n    "research/spatial_unet/network.py": "f1d10bbe1634018857b4fda8ae2753c1cf43d553f221e338bd1d6dcd89f40654",\n    "research/spatial_unet/run_experiment.py": "4484f2a6c0b29bee18e003ecfab880f0e16e192f1d1b9c71e90bc486f1cd6412",\n    "research/spatial_unet/test_spatial.py": "65215314cb5ebd1229ded058c4aab91ab71b9ceccc6af8a83de25668301a8516",\n    "research/spatial_unet/training.py": "698d27f397a312c3a4a5f832a15fa09fe431095431b74ca328a8034c3f232275"\n  },\n  "cuda": "12.8",\n  "cudnn": 91002,\n  "device": "cuda",\n  "gpu": "Tesla T4",\n  "inputs": {\n    "files": {\n      "cfsv2_desenvolvimento.nc": "1c20df191e347268d3844bff855904a963d5769544929dfcc03363a5f5720163",\n      "cfsv2_manifesto.json": "02423b1016bae9c90314d7aea1cab6b374d09c010c226f635576d7f79f42d196",\n      "seas5_51_desenvolvimento.nc": "bb4ba9085b32f008172ba1987b5c158263d0019a63e52046ad4c99dc20e02fa8",\n      "seas5_51_manifesto.json": "c7ce3a54f9a6f16189f5732ce12898ad42e1128de1a9395170c50b97718a45a9",\n      "treino_cloud_cover.nc": "607ee1210615a0ca8a5c954737617e668a5f016eb881554cb1468fda74b4fde1",\n      "treino_geopotential_850.nc": "ca3baca0177e2e90b42901803f348a4b4affe88bee202b76af3f293fa7035c17",\n      "treino_rel_hum_850.nc": "90f62b44b1b461259e7539249914e26cf09993f3bcd572ba4871931e45264b19",\n      "treino_shum_850.nc": "35cf8071b68d1e924ad87e0f8bd801d0a66d1d78ae75fc3cb623bde85c3f47bb",\n      "treino_surface_pressure.nc": "f00a5e44a03ea923ac0f693b975ef71cd347f8ad3d07692c602e3ac9a25c10fb",\n      "treino_t2.nc": "7d8c53b438289cddf87a11414ad10c8167f52ff33c6eff459a3288975079ef23",\n      "treino_temperature_850.nc": "1db7b4fdeba87b6a25f7fb6f1e40abec50964234dd9cc918cd34f902b8e41bac",\n      "treino_tp.nc": "012bebbe0e38aefdf4f9e2432c3ca22f71d94ea1a1b5421da9d9e5048632cab0",\n      "treino_tp_alvo.nc": "915f67f2627e37399830d1dd92b209ce1160082b0645662675eab17df957abe4",\n      "treino_u_850.nc": "73c898f27bea25cf2ab0914bbe5c29acc9794a93f8b6b3affa39854bebccf761",\n      "treino_v_850.nc": "7bab583972e5718bb0e0fe0fe6a056bd56c968af1f1497205182f7621e26c241"\n    },\n    "final_training": false,\n    "forecast_partitions": [\n      "desenvolvimento"\n    ],\n    "indices_sha256": "69226b2e65f783b4d4971a254d1e325d0908447c76ec1df3bf532ecef1f0cf5b",\n    "library_sha256": "4b92dd77d868d7806fd0ae37d2837abfcde36e173df75b1bd6d88760593ee9c3",\n    "rainfall_read": [\n      "1976-10-01",\n      "2020-12-01"\n    ],\n    "sst_used": false,\n    "test_partition_read": false,\n    "versions": {\n      "lightgbm": "4.6.0",\n      "numpy": "2.0.2",\n      "pandas": "2.3.3",\n      "xarray": "2025.12.0"\n    }\n  },\n  "protocol": {\n    "architecture": {\n      "activation": "SiLU",\n      "channels": [\n        16,\n        32,\n        64\n      ],\n      "convolutions_per_block": 2,\n      "kernel": 3,\n      "normalization": "GroupNorm_4",\n      "output": "one_unbounded_rainfall_anomaly",\n      "padding": "replicate",\n      "upsampling": "bilinear_align_corners_false"\n    },\n    "augmentation": "none: do not flip geography or mix months across temporal boundaries",\n    "baseline": {\n      "architecture": "two_LightGBM_and_local_ridge",\n      "residual_scales": [\n        0.9,\n        0.875\n      ],\n      "weights": [\n        0.375,\n        0.375,\n        0.25\n      ]\n    },\n    "batch_size": 2,\n    "blend": "none",\n    "comparison_limit": "same source information and target months; U-Net uses all pixels in monthly maps, baseline trees use 5000 sampled pixels per month",\n    "evaluation": [\n      "2007-01",\n      "2020-12"\n    ],\n    "id": "spatial_unet_v1",\n    "inner_label_lag": 4,\n    "inner_validation_months": 24,\n    "lags": {\n      "atmosphere": 4,\n      "indices": 3,\n      "seasonal_initialization": 1,\n      "training_rainfall": 4\n    },\n    "loss": "unweighted pixel MSE in mm/day anomalies; mean over complete monthly maps",\n    "max_epochs": 40,\n    "max_training_months": 360,\n    "optimizer": {\n      "gradient_clip_norm": 1.0,\n      "learning_rate": 0.001,\n      "name": "AdamW",\n      "weight_decay": 0.0001\n    },\n    "outer_metrics": [\n      "RMSE",\n      "MAE",\n      "bias",\n      "area_weighted_RMSE",\n      "year",\n      "block",\n      "latitude_band"\n    ],\n    "patience": 6,\n    "prospective_registration": "separate future protocol after historical development; never alter the existing hybrid registry",\n    "reconstruction": "maximum(0, training_rainfall_climatology + predicted_anomaly); no tuned residual scale",\n    "scaling": "per-channel mean and standard deviation fitted on training maps only; standard deviation floor 1e-6",\n    "seed": 42,\n    "selection": "lowest reconstructed inner RMSE; earliest epoch on ties; refit from seed on full outer training window for that epoch count",\n    "sources": "same 31 available features as the lagged multisystem component; no SST or new source",\n    "status": "development_experiment_not_independent_holdout"\n  },\n  "python": "3.12.13",\n  "torch": "2.10.0+cu128"\n}\n', 'research/spatial_unet/network.py': '"""A compact U-Net for continuous rainfall anomalies, not image classification."""\nimport random\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef seed_everything(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    # Some CUDA interpolation backward kernels have no deterministic implementation.\n    # Keep the warning visible and archive the environment; do not promise bit equality.\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\nclass Block(nn.Sequential):\n    def __init__(self, inputs, outputs):\n        super().__init__(nn.Conv2d(inputs, outputs, 3, padding=1, padding_mode=\'replicate\'),\n                         nn.GroupNorm(4, outputs), nn.SiLU(),\n                         nn.Conv2d(outputs, outputs, 3, padding=1, padding_mode=\'replicate\'),\n                         nn.GroupNorm(4, outputs), nn.SiLU())\n\n\nclass RainfallUNet(nn.Module):\n    def __init__(self, inputs=31, widths=(16, 32, 64)):\n        super().__init__()\n        a, b, c = widths\n        self.encoder1 = Block(inputs, a)\n        self.encoder2 = Block(a, b)\n        self.bottleneck = Block(b, c)\n        self.decoder2 = Block(c + b, b)\n        self.decoder1 = Block(b + a, a)\n        self.output = nn.Conv2d(a, 1, 1)\n        # Start at the climatology. Negative anomalies remain learnable.\n        nn.init.zeros_(self.output.weight)\n        nn.init.zeros_(self.output.bias)\n\n    def forward(self, x):\n        e1 = self.encoder1(x)\n        e2 = self.encoder2(F.avg_pool2d(e1, 2))\n        b = self.bottleneck(F.avg_pool2d(e2, 2))\n        d2 = self.decoder2(torch.cat([F.interpolate(b, size=e2.shape[-2:], mode=\'bilinear\', align_corners=False), e2], 1))\n        d1 = self.decoder1(torch.cat([F.interpolate(d2, size=e1.shape[-2:], mode=\'bilinear\', align_corners=False), e1], 1))\n        return self.output(d1)[:, 0]\n\n\ndef restore(path, device=\'cpu\'):\n    checkpoint = torch.load(path, map_location=device, weights_only=True)\n    model = RainfallUNet(widths=tuple(checkpoint[\'widths\'])).to(device)\n    model.load_state_dict(checkpoint[\'state_dict\'])\n    model.eval()\n    return model, checkpoint\n', 'research/spatial_unet/protocol.json': '{\n  "id": "spatial_unet_v1",\n  "status": "development_experiment_not_independent_holdout",\n  "sources": "same 31 available features as the lagged multisystem component; no SST or new source",\n  "lags": {"atmosphere": 4, "indices": 3, "seasonal_initialization": 1, "training_rainfall": 4},\n  "evaluation": ["2007-01", "2020-12"],\n  "max_training_months": 360,\n  "seed": 42,\n  "architecture": {"channels": [16, 32, 64], "convolutions_per_block": 2, "kernel": 3, "padding": "replicate", "activation": "SiLU", "normalization": "GroupNorm_4", "upsampling": "bilinear_align_corners_false", "output": "one_unbounded_rainfall_anomaly"},\n  "optimizer": {"name": "AdamW", "learning_rate": 0.001, "weight_decay": 0.0001, "gradient_clip_norm": 1.0},\n  "batch_size": 2,\n  "max_epochs": 40,\n  "patience": 6,\n  "inner_validation_months": 24,\n  "inner_label_lag": 4,\n  "selection": "lowest reconstructed inner RMSE; earliest epoch on ties; refit from seed on full outer training window for that epoch count",\n  "loss": "unweighted pixel MSE in mm/day anomalies; mean over complete monthly maps",\n  "scaling": "per-channel mean and standard deviation fitted on training maps only; standard deviation floor 1e-6",\n  "augmentation": "none: do not flip geography or mix months across temporal boundaries",\n  "reconstruction": "maximum(0, training_rainfall_climatology + predicted_anomaly); no tuned residual scale",\n  "baseline": {"architecture": "two_LightGBM_and_local_ridge", "weights": [0.375, 0.375, 0.25], "residual_scales": [0.9, 0.875]},\n  "blend": "none",\n  "outer_metrics": ["RMSE", "MAE", "bias", "area_weighted_RMSE", "year", "block", "latitude_band"],\n  "comparison_limit": "same source information and target months; U-Net uses all pixels in monthly maps, baseline trees use 5000 sampled pixels per month",\n  "prospective_registration": "separate future protocol after historical development; never alter the existing hybrid registry"\n}\n', 'research/spatial_unet/run_experiment.py': '"""Compare a monthly-map U-Net to the unchanged reference model with longer input lags on development years."""\nfrom pathlib import Path\nimport argparse\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport sys\nimport tempfile\nimport numpy as np\nimport pandas as pd\nimport torch\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\nfrom research.common.inputs import Inputs, Context, require, write_json, sha256\nfrom research.common import baseline\nfrom research.spatial_unet.training import MonthlyMaps, fit_network, inference_maps, split_inner\n\n\ndef signature(data, protocol, device):\n    code = {}\n    for folder in [\'common\', \'spatial_unet\']:\n        for p in sorted((ROOT / \'research\' / folder).glob(\'*.py\')):\n            code[p.relative_to(ROOT).as_posix()] = sha256(p)\n    return dict(inputs=data.audit, protocol=protocol, code=code,\n        python=platform.python_version(), torch=torch.__version__, device=str(device),\n        gpu=torch.cuda.get_device_name(device) if device.type == \'cuda\' else None,\n        cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version())\n\n\ndef run_block(data, block, config, directory, device):\n    m = data.lib\n    start = pd.Timestamp(block[\'inicio\'])\n    cutoff = m.origem_mensal([start], 4)[0]\n    dates = pd.date_range(start, periods=24, freq=\'MS\')\n    training = m.calendario_pareado(cutoff)\n    reference = m.janela_referencia(cutoff)\n    inner_train, inner_reference, inner_dates = split_inner(training, reference, config[\'inner_validation_months\'])\n    directory.mkdir(parents=True, exist_ok=True)\n    m.calendario_emissoes(dates, cutoff).to_csv(directory / \'source_calendar.csv\', index=False)\n    write_json(directory / \'split.json\', dict(train=training.strftime(\'%Y-%m\').tolist(),\n        inner_train=inner_train.strftime(\'%Y-%m\').tolist(), inner_reference=inner_reference.strftime(\'%Y-%m\').tolist(),\n        inner_validation=inner_dates.strftime(\'%Y-%m\').tolist(), outer=dates.strftime(\'%Y-%m\').tolist(),\n        independent_holdout=False, historical_publication_vintages_verified=False))\n    # Verify the inexpensive control before spending GPU time on the neural fit.\n    state = baseline.fit(data, cutoff, directory / \'hybrid\')\n    hybrid = baseline.predict(data, state, dates)\n    control_metrics = m.pesquisa_diagnosticos(data.rain.sel(time=dates), {\'hybrid\': hybrid}, block[\'nome\'])\n    reference_metrics = pd.read_csv(ROOT / \'research/lagged_sources/evidence/metricas_blocos.csv\')\n    old = reference_metrics[(reference_metrics.bloco == block[\'nome\']) & (reference_metrics.modelo == \'controle_defasado\')].iloc[0]\n    now = m.pesquisa_resumir(control_metrics[control_metrics.regiao == \'dominio_inteiro\'], [\'modelo\']).iloc[0]\n    for key in [\'rmse\', \'mae\', \'vies\']:\n        require(abs(float(now[key]) - float(old[key])) < 1e-6, f\'Control did not reproduce archived {key}. Stop comparison.\')\n    write_json(directory / \'baseline_check.json\', dict(reproduced=True, tolerance=1e-6,\n        rmse_archived=float(old.rmse), rmse_current=float(now.rmse)))\n    print(block[\'nome\'], \'fixed reference model reproduced; starting neural fit.\', flush=True)\n    baseline_climate = state[\'climate\']\n    del state\n    gc.collect()\n    scratch = directory / \'temporary_maps\'\n    scratch.mkdir(exist_ok=True)\n    with tempfile.TemporaryDirectory(dir=scratch) as temp:\n        temp = Path(temp)\n        inner = Context(data, inner_reference, inner_train)\n        inner.save(directory / \'inner\')\n        train = MonthlyMaps(inner, inner_train, temp / \'inner_train\', training=True)\n        valid = MonthlyMaps(inner, inner_dates, temp / \'inner_valid\', scaling=train.scaling)\n        try:\n            net, epochs = fit_network(train, config, directory / \'inner\', device, validation=valid)\n        finally:\n            train.close()\n            valid.close()\n        del train, valid, inner, net\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n\n        # Final preprocessing is refitted after epoch selection, using only outer training.\n        context = Context(data, reference, training)\n        context.save(directory / \'unet\')\n        train = MonthlyMaps(context, training, temp / \'outer_train\', training=True)\n        try:\n            net, _ = fit_network(train, config, directory / \'unet\', device, epochs=epochs)\n            scaling = train.scaling\n        finally:\n            train.close()\n        valid = MonthlyMaps(context, dates, temp / \'outer_eval\', scaling=scaling)\n        try:\n            prediction = inference_maps(net, valid, device, config[\'batch_size\'])\n        finally:\n            valid.close()\n        coords = dict(time=dates, lat=data.rain.lat, lon=data.rain.lon)\n        predictions = dict(unet=xr.DataArray(prediction, dims=(\'time\', \'lat\', \'lon\'), coords=coords, attrs=dict(units=\'mm/day\')))\n        del net, train, valid\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n\n    require(np.array_equal(baseline_climate.values, context.climate.values), \'Baseline and U-Net rainfall references differ.\')\n    predictions[\'hybrid\'] = hybrid\n    predictions[\'climatology\'] = xr.DataArray(context.climate.values[dates.month - 1], dims=(\'time\', \'lat\', \'lon\'), coords=coords, attrs=dict(units=\'mm/day\'))\n    metrics = m.pesquisa_diagnosticos(data.rain.sel(time=dates), predictions, block[\'nome\'])\n    xr.Dataset(predictions, attrs=dict(training_end=str(cutoff.date()), independent_holdout=\'false\', chosen_epochs=epochs)).to_netcdf(\n        directory / \'predictions.nc\', engine=\'h5netcdf\', encoding={k: dict(zlib=True, complevel=4) for k in predictions})\n    metrics.to_csv(directory / \'monthly_metrics.csv\', index=False)\n    return metrics\n\n\ndef summarize(data, directory, frames):\n    all_metrics = pd.concat(frames, ignore_index=True)\n    require(not all_metrics.duplicated([\'modelo\', \'mes_alvo\', \'regiao\']).any(), \'Duplicate evaluation rows.\')\n    all_metrics.to_csv(directory / \'monthly_metrics.csv\', index=False)\n    domain = all_metrics[all_metrics.regiao == \'dominio_inteiro\']\n    for name, frame, groups in [(\'global\', domain, [\'modelo\']), (\'blocks\', domain, [\'modelo\', \'bloco\']),\n                                (\'years\', domain, [\'modelo\', \'ano\']), (\'regions\', all_metrics, [\'modelo\', \'regiao\'])]:\n        summary = data.lib.pesquisa_resumir(frame, groups)\n        summary.to_csv(directory / (name + \'.csv\'), index=False)\n    totals = pd.read_csv(directory / \'global.csv\').set_index(\'modelo\')\n    complete = len(domain.bloco.unique()) == 7\n    result = dict(complete_seven_blocks=complete, blocks=sorted(domain.bloco.unique().tolist()),\n        unet_rmse=float(totals.loc[\'unet\', \'rmse\']), hybrid_rmse=float(totals.loc[\'hybrid\', \'rmse\']),\n        delta_rmse=float(totals.loc[\'unet\', \'rmse\'] - totals.loc[\'hybrid\', \'rmse\']),\n        model_promoted=False, forecast_issued=False, independent_holdout=False,\n        interpretation=\'Development comparison on previously consulted years. A pilot is not the seven-block result.\')\n    write_json(directory / \'summary.json\', result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef execute(output, blocks=(\'H1\',), official=None, seas5=None, cfsv2=None, device=None):\n    config = json.loads(Path(__file__).with_name(\'protocol.json\').read_text())\n    require(config[\'id\'] == \'spatial_unet_v1\' and config[\'lags\'] == dict(atmosphere=4, indices=3, seasonal_initialization=1, training_rainfall=4), \'Different temporal protocol.\')\n    require(config[\'inner_label_lag\'] == 4 and config[\'max_training_months\'] == 360, \'Different training protocol.\')\n    device = torch.device(device or (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n    if device.type == \'cpu\':\n        torch.set_num_threads(4)\n        print(\'CPU selected: full-grid training may be slow. Prefer a Kaggle GPU.\', flush=True)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    data = Inputs(official, seas5, cfsv2, output=output)\n    run_signature = signature(data, config, device)\n    signature_file = output / \'signature.json\'\n    if signature_file.exists():\n        require(json.loads(signature_file.read_text()) == run_signature, \'Changed inputs, code or device. Preserve this run and choose another output directory.\')\n    else:\n        write_json(signature_file, run_signature)\n    selected = [b for b in data.lib.BLOCOS if b[\'nome\'] in blocks]\n    require(len(selected) == len(set(blocks)) and selected, \'Unknown or empty block selection.\')\n    frames = []\n    for block in selected:\n        directory = output / block[\'nome\']\n        complete = directory / \'complete.json\'\n        if complete.exists():\n            record = json.loads(complete.read_text())\n            require(record[\'signature\'] == sha256(signature_file), \'Changed completed-block signature.\')\n            for name, digest in record[\'files\'].items():\n                require(sha256(directory / name) == digest, f\'Changed block artifact: {name}\')\n            metrics = pd.read_csv(directory / \'monthly_metrics.csv\')\n            print(block[\'nome\'], \'verified cached result\', flush=True)\n        else:\n            require(not directory.exists(), f\'Incomplete run in {directory}. Preserve it and use a new output directory.\')\n            metrics = run_block(data, block, config, directory, device)\n            files = {p.relative_to(directory).as_posix(): sha256(p) for p in directory.rglob(\'*\') if p.is_file()}\n            write_json(complete, dict(signature=sha256(signature_file), files=files))\n        frames.append(metrics)\n    return summarize(data, output, frames)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--output\', required=True)\n    parser.add_argument(\'--blocks\', nargs=\'+\', default=[\'H1\'])\n    parser.add_argument(\'--official\')\n    parser.add_argument(\'--seas5\')\n    parser.add_argument(\'--cfsv2\')\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda\'])\n    args = parser.parse_args()\n    execute(**vars(args))\n', 'research/spatial_unet/test_spatial.py': '"""Synthetic temporal, spatial and training checks; never a rainfall skill benchmark."""\nfrom pathlib import Path\nimport copy\nimport json\nimport sys\nimport tempfile\nfrom types import SimpleNamespace\nimport unittest\nimport numpy as np\nimport pandas as pd\nimport torch\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT))\nfrom research.common.inputs import Context, library\nfrom research.spatial_unet.network import RainfallUNet, seed_everything, restore\nfrom research.spatial_unet.training import MonthlyMaps, split_inner, fit_network, inference_maps\n\n\nfrom research.common.synthetic_fixture import fixture\n\n\nclass SpatialTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(2)\n\n    def setUp(self):\n        self.data = fixture()\n        self.reference = pd.date_range(\'1976-10-01\', \'2006-09-01\', freq=\'MS\')\n        self.training = self.reference[self.reference >= \'1982-03-01\']\n\n    def test_inner_selection_has_label_gap_and_no_new_history(self):\n        train, reference, valid = split_inner(self.training, self.reference)\n        self.assertEqual(len(valid), 24)\n        self.assertEqual(train[-1], valid[0] - pd.DateOffset(months=4))\n        self.assertEqual(reference[0], self.reference[0])\n        self.assertTrue(reference.max() < valid.min())\n\n    def test_future_observations_do_not_change_context_or_features(self):\n        a = Context(self.data, self.reference, self.training)\n        expected = a.features(\'2007-01-01\')\n        self.data.rain.loc[dict(time=slice(\'2006-10-01\', None))] = 9000\n        for field in self.data.atmosphere.values():\n            field.loc[dict(time=slice(\'2006-10-01\', None))] = 9000\n        self.data.lib.INDICES_OC.loc[\'2006-11-01\':] = 9000\n        self.data.seas.loc[dict(time=slice(\'2007-02-01\', None))] = 9000\n        self.data.cfs.loc[dict(time=slice(\'2007-02-01\', None))] = 9000\n        b = Context(self.data, self.reference, self.training)\n        np.testing.assert_array_equal(a.climate, b.climate)\n        np.testing.assert_array_equal(expected, b.features(\'2007-01-01\'))\n\n    def test_training_climatology_excludes_entire_target_map(self):\n        t = self.training[36]\n        a = Context(self.data, self.reference, self.training).features(t, training=True)[22]\n        self.data.rain.loc[dict(time=t)] += np.arange(117).reshape(9, 13)\n        b = Context(self.data, self.reference, self.training).features(t, training=True)[22]\n        np.testing.assert_allclose(a, b, rtol=0, atol=1e-6)\n\n    def test_context_matches_existing_baseline_transform(self):\n        m = self.data.lib\n        ctx = Context(self.data, self.reference, self.training)\n        climate, _, _ = m.estatisticas_climatologia(self.data.rain, self.reference[-1])\n        ca = m.ajustar_clima_atmosfera(self.data.atmosphere, m.origem_mensal(self.reference, 4))\n        ca[\'_clima_indices\'] = m.ajustar_clima_indices(m.INDICES_OC, m.origem_mensal(self.reference, 4), self.reference[-1])\n        cs = m.ajustar_clima_seas5(self.data.seas, self.reference[-1])\n        cc = m.ajustar_clima_previsao(self.data.cfs, self.reference[-1], \'CFSv2\')\n        t = pd.Timestamp(\'2007-01-01\')\n        x = np.column_stack([m.matriz_mes(self.data.atmosphere, ca, climate, t),\n            m.valores_seas5(self.data.seas, t), m.valores_anomalia_seas5(self.data.seas, cs, t),\n            m.valores_cfsv2(self.data.cfs, t), m.valores_anomalia_cfsv2(self.data.cfs, cc, t)])\n        np.testing.assert_array_equal(ctx.features(t).reshape(31, -1).T, x)\n\n    def test_odd_grid_shape_and_initial_climatology(self):\n        seed_everything()\n        net = RainfallUNet()\n        with torch.inference_mode():\n            result = net(torch.zeros(1, 31, 301, 261))\n        self.assertEqual(tuple(result.shape), (1, 301, 261))\n        self.assertEqual(float(result.abs().max()), 0.)\n\n    def test_gradients_and_checkpoint_roundtrip(self):\n        seed_everything()\n        net = RainfallUNet(widths=(4, 8, 16))\n        x = torch.randn(2, 31, 9, 13)\n        optimizer = torch.optim.AdamW(net.parameters(), lr=.001)\n        loss = ((net(x) - 1) ** 2).mean()\n        loss.backward()\n        self.assertTrue(torch.isfinite(net.output.weight.grad).all())\n        optimizer.step()\n        self.assertFalse(torch.equal(net(x), torch.zeros(2, 9, 13)))\n        with tempfile.TemporaryDirectory() as folder:\n            path = Path(folder) / \'weights.pt\'\n            torch.save(dict(state_dict=net.state_dict(), widths=[4, 8, 16]), path)\n            loaded, _ = restore(path)\n            torch.testing.assert_close(net(x), loaded(x), rtol=0, atol=0)\n\n    def test_training_loop_and_scaler_use_only_training_maps(self):\n        config = json.loads(Path(__file__).with_name(\'protocol.json\').read_text())\n        config.update(max_epochs=2, patience=2)\n        config[\'architecture\'][\'channels\'] = [4, 8, 16]\n        ctx = Context(self.data, self.reference, self.training)\n        with tempfile.TemporaryDirectory() as folder:\n            folder = Path(folder)\n            train = MonthlyMaps(ctx, self.training[-4:], folder / \'train\', training=True)\n            valid = MonthlyMaps(ctx, pd.date_range(\'2007-01-01\', periods=2, freq=\'MS\'), folder / \'valid\', scaling=train.scaling)\n            try:\n                np.testing.assert_array_equal(valid.scaling[\'mean\'], train.scaling[\'mean\'])\n                model, epochs = fit_network(train, config, folder / \'model\', torch.device(\'cpu\'), validation=valid)\n                self.assertIn(epochs, [1, 2])\n                pred = inference_maps(model, valid, torch.device(\'cpu\'))\n                self.assertTrue(np.isfinite(pred).all() and (pred >= 0).all())\n                self.assertEqual(pred.shape, (2, 9, 13))\n            finally:\n                train.close()\n                valid.close()\n\n    def test_invalid_month_cannot_be_used_for_training(self):\n        ctx = Context(self.data, self.reference, self.training)\n        with self.assertRaises(ValueError):\n            ctx.features(\'2007-01-01\', training=True)\n        with self.assertRaises(ValueError):\n            ctx.features(\'2006-09-01\', training=False)\n\n\nif __name__ == \'__main__\':\n    unittest.main(argv=[__file__])\n', 'research/spatial_unet/training.py': '"""Monthly map datasets and chronological epoch selection."""\nfrom pathlib import Path\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom .network import RainfallUNet, seed_everything\nfrom research.common.inputs import require, write_json\n\n\ndef split_inner(training, reference, validation_months=24):\n    training, reference = pd.DatetimeIndex(training), pd.DatetimeIndex(reference)\n    validation = training[-validation_months:]\n    cutoff = validation[0] - pd.DateOffset(months=4)\n    fit = training[training <= cutoff]\n    refs = reference[reference <= cutoff]\n    require(len(fit) >= 24 and fit[-1] < validation[0], \'Insufficient chronological inner training.\')\n    return fit, refs, validation\n\n\nclass MonthlyMaps(Dataset):\n    def __init__(self, context, dates, directory, training=False, scaling=None):\n        self.dates = pd.DatetimeIndex(dates)\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=False)\n        d = context.data\n        n, h, w = len(self.dates), d.rain.sizes[\'lat\'], d.rain.sizes[\'lon\']\n        self.x = np.lib.format.open_memmap(self.directory / \'features.npy\', mode=\'w+\', dtype=\'float32\', shape=(n, 31, h, w))\n        self.y = np.lib.format.open_memmap(self.directory / \'target.npy\', mode=\'w+\', dtype=\'float32\', shape=(n, h, w))\n        self.base = np.lib.format.open_memmap(self.directory / \'climate.npy\', mode=\'w+\', dtype=\'float32\', shape=(n, h, w))\n        totals, squares, pixels = np.zeros(31), np.zeros(31), 0\n        for i, t in enumerate(self.dates):\n            x = context.features(t, training)\n            self.x[i] = x\n            self.base[i] = x[22]\n            self.y[i] = d.rain.sel(time=t).values\n            if scaling is None:\n                require(training, \'Only training maps may fit scaling.\')\n                flat = x.reshape(31, -1).astype(\'float64\')\n                totals += flat.sum(1)\n                squares += np.square(flat).sum(1)\n                pixels += flat.shape[1]\n        if scaling is None:\n            mean = totals / pixels\n            scale = np.sqrt(np.maximum(squares / pixels - mean ** 2, 0))\n            self.scaling = dict(mean=mean.astype(\'float32\'), scale=np.maximum(scale, 1e-6).astype(\'float32\'))\n        else:\n            self.scaling = {k: np.array(v, copy=True) for k, v in scaling.items()}\n        for arr in [self.x, self.y, self.base]:\n            require(all(np.isfinite(arr[i]).all() for i in range(n)), \'Invalid map cache.\')\n            arr.flush()\n\n    def __len__(self):\n        return len(self.dates)\n\n    def __getitem__(self, i):\n        x = (self.x[i] - self.scaling[\'mean\'][:, None, None]) / self.scaling[\'scale\'][:, None, None]\n        return torch.from_numpy(x), torch.from_numpy(self.y[i] - self.base[i]), torch.from_numpy(self.base[i].copy())\n\n    def close(self):\n        # Only files created by this instance; no recursive or input-directory deletion.\n        for arr in [self.x, self.y, self.base]:\n            arr.flush()\n            arr._mmap.close()\n        for name in [\'features.npy\', \'target.npy\', \'climate.npy\']:\n            (self.directory / name).unlink()\n        self.directory.rmdir()\n\n\ndef inference_maps(model, dataset, device, batch_size=2):\n    model.eval()\n    out = []\n    with torch.inference_mode():\n        for x, _, base in DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0):\n            anomaly = model(x.to(device)).cpu()\n            out.append(torch.clamp(base + anomaly, min=0).numpy())\n    result = np.concatenate(out)\n    require(np.isfinite(result).all(), \'Nonfinite network output.\')\n    return result\n\n\ndef fit_network(train, config, folder, device, validation=None, epochs=None):\n    seed_everything(config[\'seed\'])\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    model = RainfallUNet(widths=tuple(config[\'architecture\'][\'channels\'])).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config[\'optimizer\'][\'learning_rate\'], weight_decay=config[\'optimizer\'][\'weight_decay\'])\n    generator = torch.Generator().manual_seed(config[\'seed\'])\n    loader = DataLoader(train, batch_size=config[\'batch_size\'], shuffle=True, generator=generator, num_workers=0)\n    limit = epochs if epochs is not None else config[\'max_epochs\']\n    history, best, best_epoch, stale = [], float(\'inf\'), 0, 0\n    for epoch in range(1, limit + 1):\n        t0, total, count = time.perf_counter(), 0., 0\n        model.train()\n        for x, y, _ in loader:\n            x, y = x.to(device), y.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            prediction = model(x)\n            loss = torch.mean((prediction - y) ** 2)\n            require(torch.isfinite(loss).item(), \'Training loss diverged.\')\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), config[\'optimizer\'][\'gradient_clip_norm\'], error_if_nonfinite=True)\n            optimizer.step()\n            total += loss.item() * y.numel()\n            count += y.numel()\n        row = dict(epoch=epoch, training_mse=total / count, seconds=time.perf_counter() - t0)\n        if validation is not None:\n            pred = inference_maps(model, validation, device, config[\'batch_size\'])\n            truth = validation.y.astype(\'float64\')\n            rmse = float(np.sqrt(np.mean((pred.astype(\'float64\') - truth) ** 2)))\n            row[\'inner_rmse\'] = rmse\n            if rmse < best:\n                best, best_epoch, stale = rmse, epoch, 0\n            else:\n                stale += 1\n        history.append(row)\n        write_json(folder / \'training_history.json\', history)\n        print(folder.name, row, flush=True)\n        if validation is not None and stale >= config[\'patience\']:\n            break\n    selected = best_epoch if validation is not None else limit\n    # Inner weights are never evaluated on the outer block. Refit starts from seed.\n    torch.save(dict(state_dict=model.state_dict(), widths=config[\'architecture\'][\'channels\'],\n        epochs_run=epoch, selected_epochs=selected, seed=config[\'seed\']), folder / \'network.pt\')\n    np.savez_compressed(folder / \'scaling.npz\', **train.scaling)\n    write_json(folder / \'training.json\', dict(selected_epochs=selected,\n        parameter_count=sum(p.numel() for p in model.parameters()), train_months=len(train),\n        train_start=str(train.dates[0].date()), train_end=str(train.dates[-1].date()),\n        inner_selection=validation is not None, device=str(device)))\n    return model, selected\n', 'research/workflow/__init__.py': '"""One entry point for configured, traceable development comparisons."""\n', 'research/workflow/__main__.py': '"""Usage: python -m research.workflow check|run|verify|list ..."""\nimport argparse\nimport json\nimport sys\nfrom .config import load, WorkflowError\nfrom .runner import preflight, run, verify_run, list_runs\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=\'Configured rainfall development comparisons. No automatic model promotion or forecast issuance.\')\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    for command in [\'check\', \'run\']:\n        commands.add_parser(command).add_argument(\'--config\', required=True, help=\'JSON file; relative paths resolve from its directory\')\n    commands.add_parser(\'verify\').add_argument(\'--run\', required=True, help=\'Completed run directory\')\n    commands.add_parser(\'list\').add_argument(\'--root\', required=True, help=\'Output root containing workflow runs\')\n    args = parser.parse_args(argv)\n    try:\n        if args.command == \'check\':\n            audit = preflight(load(args.config))\n            value = dict(status=\'ready\', mode=audit[\'mode\'], fingerprint=audit[\'fingerprint\'],\n                         input_artifacts=len(audit[\'inputs\']), independent_holdout=False,\n                         message=\'Inputs and environment checked. No model was fitted and no output run was created.\')\n        elif args.command == \'run\':\n            value = dict(status=\'completed\', directory=str(run(args.config)))\n        elif args.command == \'verify\':\n            value = verify_run(args.run)\n        else:\n            value = list_runs(args.root)\n        print(json.dumps(value, indent=2))\n        return 0\n    except (WorkflowError, ValueError, OSError, ImportError, KeyError, TypeError) as error:\n        print(\'Workflow stopped:\', error, file=sys.stderr)\n        return 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n', 'research/workflow/config.py': '"""Validate user settings before reading large arrays or fitting models."""\nfrom pathlib import Path\nimport json\nimport re\n\nROOT = Path(__file__).resolve().parents[2]\nBLOCKS = (\'H1\', \'H2\', \'H3\', \'H4\', \'A\', \'B\', \'C\')\nDEFAULT_EVIDENCE = ROOT / \'research/spatial_unet/evidence/seven_blocks\'\nPROTOCOL = ROOT / \'research/spatial_unet/protocol.json\'\n\n\nclass WorkflowError(ValueError):\n    """An actionable configuration, provenance or execution error."""\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise WorkflowError(message)\n\n\ndef _unique_keys(pairs):\n    result = {}\n    for key, value in pairs:\n        require(key not in result, f\'Duplicate JSON key: {key}. Keep one value.\')\n        result[key] = value\n    return result\n\n\ndef read_json(path):\n    try:\n        return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'), object_pairs_hook=_unique_keys)\n    except json.JSONDecodeError as error:\n        raise WorkflowError(f\'Invalid JSON in {path}, line {error.lineno}: {error.msg}\') from error\n\n\ndef resolve_path(value, base, label):\n    require(isinstance(value, str) and value.strip(), f\'Set {label} to a nonempty file or directory path.\')\n    path = Path(value).expanduser()\n    return (path if path.is_absolute() else base / path).resolve()\n\n\ndef normalize(raw, base):\n    require(isinstance(raw, dict), \'Configuration must be a JSON object.\')\n    require(set(raw) <= {\'schema\', \'name\', \'mode\', \'output_root\', \'inputs\', \'device\', \'experiment\'},\n            \'Unknown configuration field. Use saved.example.json or train.example.json; scientific settings live in protocol.json.\')\n    require(raw.get(\'schema\') == \'rainfall_workflow_v1\', \'Set schema to rainfall_workflow_v1.\')\n    require(isinstance(raw.get(\'name\'), str) and re.fullmatch(r\'[a-z][a-z0-9_-]{0,63}\', raw[\'name\']),\n            \'Name must start with a lowercase letter and use up to 64 letters, digits, underscores or hyphens.\')\n    mode = raw.get(\'mode\')\n    require(mode in {\'saved\', \'train\'}, \'Mode must be saved (no fitting) or train (all seven blocks).\')\n    experiment = raw.get(\'experiment\', \'hybrid_unet_v1\')\n    require(experiment in {\'hybrid_unet_v1\', \'fixed_blend_v1\'}, \'Experiment must be hybrid_unet_v1 or fixed_blend_v1.\')\n    require(experiment != \'fixed_blend_v1\' or mode == \'saved\', \'fixed_blend_v1 evaluates saved maps only; it does not train models.\')\n    supplied = raw.get(\'inputs\')\n    require(isinstance(supplied, dict), \'inputs must be an object containing the dataset paths.\')\n    keys = {\'observations\', \'predictions\', \'evidence\'} if mode == \'saved\' else {\'official\', \'seas5\', \'cfsv2\'}\n    require(set(supplied) <= keys, f\'Unexpected inputs for {mode}. Expected: {sorted(keys)}.\')\n    required = keys - ({\'evidence\'} if mode == \'saved\' else set())\n    require(required <= set(supplied), f\'Missing inputs: {sorted(required - set(supplied))}.\')\n    paths = {k: str(resolve_path(supplied[k], base, \'inputs.\' + k)) for k in required}\n    if mode == \'saved\':\n        paths[\'evidence\'] = str(resolve_path(supplied[\'evidence\'], base, \'inputs.evidence\') if supplied.get(\'evidence\') else DEFAULT_EVIDENCE)\n        require(\'device\' not in raw, \'saved mode does not use a GPU; remove device.\')\n    else:\n        require(raw.get(\'device\', \'cuda\') in {\'cpu\', \'cuda\'}, \'Device must be cpu or cuda.\')\n    output = resolve_path(raw.get(\'output_root\'), base, \'output_root\')\n    # Never create an output inside a source dataset or code bundle.\n    for key, value in paths.items():\n        parent = Path(value).parent if key == \'observations\' else Path(value)\n        require(not output.is_relative_to(parent), f\'output_root is inside inputs.{key}. Choose a separate output folder.\')\n    require(not output.is_relative_to(ROOT / \'research\') and not output.is_relative_to(ROOT / \'competition\'),\n            \'Use an outputs directory outside the source folders.\')\n    result = dict(schema=raw[\'schema\'], name=raw[\'name\'], mode=mode, output_root=str(output), inputs=paths)\n    result[\'experiment\'] = experiment\n    if mode == \'train\':\n        result[\'device\'] = raw.get(\'device\', \'cuda\')\n    return result\n\n\ndef load(path):\n    path = Path(path).resolve()\n    require(path.is_file(), f\'Configuration not found: {path}. Copy an example and edit its paths first.\')\n    return normalize(read_json(path), path.parent)\n\n\ndef discover_saved(root=\'/kaggle/input\'):\n    """Locate one complete saved run; never choose silently among duplicates."""\n    root = Path(root)\n    observations = sorted(root.rglob(\'treino_tp.nc\'))\n    runs = sorted({p.parent for p in root.rglob(\'signature.json\')\n                   if all((p.parent / b / \'predictions.nc\').is_file() for b in BLOCKS)})\n    require(len(observations) == 1, f\'Found {len(observations)} treino_tp.nc files under {root}. Attach one official dataset or set its path manually.\')\n    require(len(runs) == 1, f\'Found {len(runs)} complete seven-block outputs under {root}. Attach the saved U-Net notebook OUTPUT (including predictions.nc), or set paths manually.\')\n    return dict(observations=str(observations[0]), predictions=str(runs[0]), evidence=str(runs[0]))\n', 'research/workflow/report.py': '"""Generate a portable HTML report from verified diagnostic tables."""\nfrom html import escape\nfrom pathlib import Path\nimport base64\nimport json\nimport pandas as pd\nfrom research.common.presentation import display_frame\n\n\ndef table_html(frame):\n    frame = display_frame(frame)\n    if \'n\' in frame:\n        frame[\'n\'] = frame.n.map(lambda n: f\'{int(n):,}\')\n    labels = {\'model\': \'Model\', \'n\': \'Samples\', \'rmse\': \'RMSE\', \'mae\': \'MAE\', \'bias\': \'Bias\',\n              \'delta_rmse_vs_hybrid\': \'Δ RMSE vs reference model\', \'calendar_month\': \'Month\', \'hybrid_rmse\': \'Reference model RMSE\',\n              \'unet_rmse\': \'U-Net RMSE\', \'delta_rmse\': \'Δ RMSE\', \'region\': \'Latitude band\',\n              \'intensity\': \'Observed mm/day\', \'block\': \'Block\', \'year\': \'Year\', \'selected_epochs\': \'Selected epoch\',\n              \'inner_epochs_run\': \'Epochs run\', \'first_inner_rmse\': \'First inner RMSE\',\n              \'best_inner_rmse\': \'Best inner RMSE\', \'last_inner_rmse\': \'Last inner RMSE\'}\n    return frame.rename(columns=labels).to_html(index=False, float_format=lambda x: f\'{x:.6f}\', border=0)\n\n\ndef render(run, record):\n    run = Path(run)\n    report = run / \'diagnostics\'\n    summary = json.loads((report / \'summary.json\').read_text())\n    scores = pd.read_csv(report / \'global.csv\')\n    baseline = scores.set_index(\'model\').loc[\'hybrid\', \'rmse\']\n    scores[\'delta_rmse_vs_hybrid\'] = scores.rmse - baseline\n    scores = scores[[\'model\', \'rmse\', \'mae\', \'bias\', \'delta_rmse_vs_hybrid\', \'n\']]\n    signed = summary[\'unet_rmse\'] - summary[\'hybrid_rmse\']\n    outcome = (\'The reference model has lower pooled RMSE.\' if signed > 0 else\n               \'The U-Net has lower pooled RMSE on these development years.\' if signed < 0 else \'Pooled RMSE is tied.\')\n    metadata = {\n        \'Run\': record[\'run_id\'], \'Mode\': record[\'mode\'], \'Comparison\': \'Fixed reference model / compact U-Net / monthly climatology\',\n        \'Period\': \'January 2007–December 2020 · seven chronological blocks\',\n        \'Started (UTC)\': record[\'started_at\'], \'Run fingerprint\': record[\'fingerprint\'],\n        \'Source signature\': record[\'source_signature_sha256\'], \'Scope\': \'Development years already consulted; not an independent holdout\',\n    }\n    parts = [f\'<h1>Rainfall experiment report</h1><p class="lead">{escape(outcome)} U-Net − reference model RMSE: <strong>{signed:+.6f} mm/day</strong>.</p>\',\n             \'<p class="note">No model is automatically promoted. No prospective forecast is issued. This report does not establish future skill.</p>\',\n             \'<p>January 2007–December 2020 · Seven chronological blocks · Full 0.25° grid</p>\',\n             \'<details><summary>Run details and provenance</summary><dl>\' + \'\'.join(f\'<dt>{escape(k)}</dt><dd>{escape(str(v))}</dd>\' for k, v in metadata.items()) + \'</dl></details>\',\n             \'<h2>Overall comparison</h2><p>Monthly mean daily precipitation, in mm/day. RMSE pools squared errors and counts; it does not average block RMSEs. The full supplied grid includes ocean cells.</p>\',\n             table_html(scores)]\n    sections = [\n        (\'Calendar month\', \'calendar_month.png\', \'calendar_comparison.csv\', [\'calendar_month\', \'hybrid_rmse\', \'unet_rmse\', \'delta_rmse\'],\n         \'Each month pools fourteen years. Positive differences favor the reference model.\'),\n        (\'Location\', \'spatial_errors.png\', \'region_comparison.csv\', [\'region\', \'hybrid_rmse\', \'unet_rmse\', \'delta_rmse\'],\n         f"U-Net RMSE is lower in {summary[\'unet_lower_rmse_grid_fraction\']:.2%} of grid cells. This is not an area-weighted or land-only percentage."),\n        (\'Observed rainfall intensity\', \'rainfall_intensity.png\', \'intensity_comparison.csv\', [\'intensity\', \'n\', \'hybrid_rmse\', \'unet_rmse\', \'delta_rmse\'],\n         \'Bins describe the observed monthly mean, not daily extremes. The future observation is unavailable at issue time: these bins cannot select an operational model. Conditional bias is affected by regression to the mean.\'),\n        (\'Original training histories\', \'training_curves.png\', \'training.csv\', [\'block\', \'selected_epochs\', \'inner_epochs_run\', \'first_inner_rmse\', \'best_inner_rmse\', \'last_inner_rmse\'],\n         \'Epoch selection uses inner validation only. Training losses are online anomaly losses; validation uses fixed end-of-epoch weights and clipped rainfall. Their gap is not an exact generalization-gap estimate.\'),\n    ]\n    for title, image, table, columns, explanation in sections:\n        encoded = base64.b64encode((report / image).read_bytes()).decode(\'ascii\')\n        parts += [f\'<h2>{escape(title)}</h2><p>{escape(explanation)}</p>\',\n                  f\'<img alt="{escape(title)}" src="data:image/png;base64,{encoded}">\',\n                  \'<details><summary>View numerical results</summary>\' + table_html(pd.read_csv(report / table)[columns]) + \'</details>\']\n    for name, title in [(\'blocks\', \'Two-year blocks\'), (\'years\', \'Individual years\')]:\n        table = pd.read_csv(run / \'comparison\' / (name + \'.csv\'))\n        parts += [f\'<details><summary>{title}</summary>\', table_html(table), \'</details>\']\n    parts += [\'<h2>Reproduction and limits</h2><p>config.json records resolved paths. identity.json records code, data and environment fingerprints. source_evidence preserves the source run metadata. manifest.json records the output hashes. Use the workflow verify command to check the saved package.</p>\',\n              f\'<p>{summary["monthly_region_scores_reproduced"]:,} monthly model × region error records were reproduced from the maps. Dates, grid, units, prediction hashes and selected epochs were checked.</p>\',\n              \'<p>Historical source publication vintages are not established by this check. The tree models sample training grid points; the U-Net trains on complete maps. Spatial samples are correlated. Observed improvements are descriptive, without a claim of statistical significance.</p>\']\n    html = \'<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Rainfall experiment report</title><style>body{font:16px/1.55 system-ui,sans-serif;color:#172b36;max-width:1100px;margin:40px auto;padding:0 24px;background:#f8fafb}h1,h2{line-height:1.2}h2{margin-top:44px}.lead{font-size:20px}.note{border-left:4px solid #087e8b;padding:12px 16px;background:#e9f4f5}details{margin:18px 0}summary{cursor:pointer;font-weight:650;margin-bottom:12px}dt{font-weight:650}dd{margin:0 0 12px;overflow-wrap:anywhere}table{border-collapse:collapse;width:100%;font-size:14px;display:block;overflow:auto;background:white}th,td{padding:9px 12px;border-bottom:1px solid #dae2e6;text-align:right}th:first-child,td:first-child{text-align:left}img{max-width:100%;height:auto;background:white;margin:12px 0}</style><body>\' + \'\\n\'.join(parts) + \'</body></html>\'\n    (run / \'report.html\').write_text(html, encoding=\'utf-8\', newline=\'\\n\')\n    (run / \'REPORT.md\').write_text(\n        \'# Rainfall experiment report\\n\\n\' + outcome + f\' U-Net minus reference model RMSE: {signed:+.6f} mm/day.\\n\\n\'\n        + \'Open report.html for the complete tables and embedded figures.\\n\\n\'\n        + f\'Run: `{record["run_id"]}`. Mode: `{record["mode"]}`.\\n\\n\'\n        + \'Development comparison, 2007–2020. No automatic promotion and no prospective forecast.\\n\', encoding=\'utf-8\', newline=\'\\n\')\n', 'research/workflow/runner.py': '"""Orchestrate the existing fixed experiment and diagnostic without retuning it."""\nfrom contextlib import redirect_stdout\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport platform\nimport shutil\nimport sys\n\nfrom .config import ROOT, BLOCKS, PROTOCOL, WorkflowError, require, read_json, load\n\nREPORT_FILES = (\'monthly_metrics.csv\', \'split.json\', \'source_calendar.csv\', \'baseline_check.json\',\n                \'inner/training_history.json\', \'inner/training.json\', \'unet/training_history.json\', \'unet/training.json\')\nPACKAGES = (\'numpy\', \'pandas\', \'xarray\', \'matplotlib\', \'h5netcdf\', \'lightgbm\', \'torch\')\n\n\ndef utcnow():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef identity_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + \'\\n\', encoding=\'utf-8\', newline=\'\\n\')\n\n\ndef code_files():\n    names = []\n    for folder in [\'common\', \'spatial_unet\', \'diagnostics\', \'workflow\', \'fixed_blend\']:\n        names.extend(p for p in (ROOT / \'research\' / folder).glob(\'*.py\'))\n    names += [PROTOCOL, ROOT / \'research/fixed_blend/protocol.json\', ROOT / \'research/lagged_sources/library.py\', ROOT / \'research/lagged_sources/official_hashes.json\',\n              ROOT / \'research/lagged_sources/ocean_indices.csv\', ROOT / \'research/lagged_sources/evidence/metricas_blocos.csv\']\n    return sorted(set(names))\n\n\ndef verify_hash(path, expected):\n    path = Path(path)\n    require(path.is_file(), f\'Missing file: {path}. Attach or restore the saved output; reports alone do not contain forecast maps.\')\n    actual = sha256(path)\n    require(actual == expected, f\'File hash differs: {path}. Use the matching snapshot; do not edit the expected hash to bypass this check.\')\n    return actual\n\n\ndef inspect_saved(inputs):\n    evidence, predictions = Path(inputs[\'evidence\']), Path(inputs[\'predictions\'])\n    require((evidence / \'signature.json\').is_file(), f\'No signature.json in {evidence}. Set inputs.evidence to the complete saved-run reports.\')\n    signature = read_json(evidence / \'signature.json\')\n    require(signature.get(\'protocol\') == read_json(PROTOCOL), \'The saved run uses a different scientific protocol. This adapter supports spatial_unet_v1 only.\')\n    signature_hash = sha256(evidence / \'signature.json\')\n    expected_rain = next(row[\'sha256\'] for row in read_json(ROOT / \'research/lagged_sources/official_hashes.json\') if row[\'nome\'] == \'treino_tp.nc\')\n    require(signature.get(\'inputs\', {}).get(\'files\', {}).get(\'treino_tp.nc\') == expected_rain, \'Source signature names a different observation snapshot.\')\n    inventory = {\'signature.json\': signature_hash,\n                 \'observations\': verify_hash(inputs[\'observations\'], expected_rain)}\n    for block in BLOCKS:\n        path = evidence / block / \'complete.json\'\n        require(path.is_file(), f\'Missing completed block {block}: {path}. This workflow requires all seven blocks.\')\n        completion = read_json(path)\n        require(completion.get(\'signature\') == signature_hash, f\'{block}: source signature does not match the completed block.\')\n        inventory[block + \'/complete.json\'] = sha256(path)\n        for name in (\'predictions.nc\',) + REPORT_FILES:\n            require(name in completion.get(\'files\', {}), f\'{block}: completion record has no {name}.\')\n            source = predictions / block / name if name == \'predictions.nc\' else evidence / block / name\n            inventory[block + \'/\' + name] = verify_hash(source, completion[\'files\'][name])\n    # The diagnostics also cross-check this aggregate against the verified maps.\n    require((evidence / \'global.csv\').is_file(), \'Missing global.csv in source evidence.\')\n    inventory[\'global.csv\'] = sha256(evidence / \'global.csv\')\n    return inventory, signature_hash\n\n\ndef preflight(config):\n    for name in (\'numpy\', \'pandas\', \'xarray\', \'matplotlib\', \'h5netcdf\'):\n        require(importlib.util.find_spec(name) is not None, f\'Missing {name}. Install research/workflow/requirements_evaluation.txt in an evaluation environment.\')\n    require(sys.version_info >= (3, 12), \'Use Python 3.12 or later for this workflow.\')\n    if config[\'mode\'] == \'saved\':\n        inventory, source_signature = inspect_saved(config[\'inputs\'])\n    else:\n        require(importlib.util.find_spec(\'torch\') is not None, \'Training needs PyTorch. Use the separate spatial_unet training environment, preferably on a Kaggle GPU.\')\n        from research.common.inputs import preflight as training_preflight\n        _, audit = training_preflight(**config[\'inputs\'])\n        import torch\n        require(config[\'device\'] != \'cuda\' or torch.cuda.is_available(), \'CUDA was requested but is unavailable. Enable a Kaggle GPU, or explicitly choose cpu for a slower run.\')\n        inventory, source_signature = audit[\'files\'], None\n    versions = {}\n    for package in PACKAGES:\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            versions[package] = None\n    code = {p.relative_to(ROOT).as_posix(): sha256(p) for p in code_files()}\n    facts = dict(mode=config[\'mode\'], device=config.get(\'device\'), scientific_protocol=read_json(PROTOCOL),\n                 inputs=inventory, code=code, environment=dict(python=platform.python_version(), packages=versions))\n    facts[\'experiment\'] = config[\'experiment\']\n    if config[\'experiment\'] == \'fixed_blend_v1\':\n        from research.fixed_blend.evaluate import validate_protocol\n        facts[\'comparison_protocol\'] = validate_protocol()\n    return dict(**facts, source_signature_sha256=source_signature, fingerprint=identity_hash(facts),\n                scope=\'development_2007_2020\', independent_holdout=False, model_promoted=False, forecast_issued=False)\n\n\ndef copy_evidence(source, destination):\n    source, destination = Path(source), Path(destination)\n    names = [\'signature.json\', \'global.csv\'] + [f\'{b}/{n}\' for b in BLOCKS for n in (\'complete.json\',) + REPORT_FILES]\n    for name in names:\n        target = destination / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(source / name, target)\n\n\ndef comparisons(evidence, destination):\n    import numpy as np\n    import pandas as pd\n    destination.mkdir()\n    months = pd.concat([pd.read_csv(evidence / b / \'monthly_metrics.csv\') for b in BLOCKS], ignore_index=True)\n    months = months[months.regiao == \'dominio_inteiro\']\n    require(not months.duplicated([\'modelo\', \'mes_alvo\']).any(), \'Duplicate months in paired comparison.\')\n    for filename, key in [(\'blocks\', \'bloco\'), (\'years\', \'ano\')]:\n        values = months.groupby([key, \'modelo\'], as_index=False)[[\'n\', \'sse\', \'soma_erro_absoluto\', \'soma_erro\']].sum()\n        values[\'rmse\'] = np.sqrt(values.sse / values.n)\n        values[\'mae\'] = values.soma_erro_absoluto / values.n\n        values[\'bias\'] = values.soma_erro / values.n\n        baseline = values[values.modelo == \'hybrid\'].set_index(key)\n        values[\'delta_rmse_vs_hybrid\'] = values.rmse - values[key].map(baseline.rmse)\n        values[\'_period\'] = values[key].map({b: i for i, b in enumerate(BLOCKS)}) if key == \'bloco\' else values[key]\n        values[\'_model\'] = values.modelo.map({\'hybrid\': 0, \'unet\': 1, \'climatology\': 2})\n        values = values.sort_values([\'_period\', \'_model\'])\n        values.rename(columns={\'modelo\': \'model\', \'bloco\': \'block\', \'ano\': \'year\'})[[\n            \'block\' if key == \'bloco\' else \'year\', \'model\', \'n\', \'rmse\', \'mae\', \'bias\', \'delta_rmse_vs_hybrid\'\n        ]].to_csv(destination / (filename + \'.csv\'), index=False, lineterminator=\'\\n\')\n\n\ndef write_manifest(run):\n    files = {p.relative_to(run).as_posix(): dict(sha256=sha256(p), bytes=p.stat().st_size)\n             for p in sorted(run.rglob(\'*\')) if p.is_file() and p != run / \'manifest.json\'}\n    write_json(run / \'manifest.json\', dict(schema=\'rainfall_run_manifest_v1\', files=files))\n\n\ndef verify_run(run):\n    run = Path(run).resolve()\n    require((run / \'state.json\').is_file(), f\'No workflow state in {run}.\')\n    require(read_json(run / \'state.json\').get(\'status\') == \'completed\', \'This run is incomplete or failed. Inspect state.json and execution.log; it is not a valid comparison.\')\n    require((run / \'manifest.json\').is_file(), \'Completed run has no manifest.json.\')\n    manifest = read_json(run / \'manifest.json\')\n    require(manifest.get(\'schema\') == \'rainfall_run_manifest_v1\' and isinstance(manifest.get(\'files\'), dict), \'Invalid run manifest.\')\n    expected = manifest[\'files\']\n    require({\'state.json\', \'config.json\', \'identity.json\', \'report.html\', \'diagnostics/summary.json\'} <= set(expected), \'Run manifest omits required artifacts.\')\n    for name, item in expected.items():\n        p = (run / name).resolve()\n        require(p.is_relative_to(run) and p != run and name != \'manifest.json\', \'Manifest path leaves the run or references itself.\')\n        verify_hash(p, item[\'sha256\'])\n        require(p.stat().st_size == item[\'bytes\'], f\'Size differs: {name}.\')\n    actual = {p.relative_to(run).as_posix() for p in run.rglob(\'*\') if p.is_file() and p != run / \'manifest.json\'}\n    require(actual == set(expected), \'Files were added or removed after completion. Keep extra notes outside the frozen run directory.\')\n    return dict(status=\'verified\', run_id=read_json(run / \'identity.json\')[\'run_id\'], files=len(expected))\n\n\nclass Tee:\n    def __init__(self, stream, log):\n        self.stream, self.log = stream, log\n    def write(self, text):\n        self.stream.write(text); self.log.write(text)\n        return len(text)\n    def flush(self):\n        self.stream.flush(); self.log.flush()\n\n\ndef run(config_path):\n    config = load(config_path)\n    audit = preflight(config)\n    timestamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n    run_id = f\'{config["name"]}-{timestamp}-{audit["fingerprint"][:10]}\'\n    folder = Path(config[\'output_root\']) / run_id\n    folder.mkdir(parents=True, exist_ok=False)\n    record = dict(**audit, run_id=run_id, started_at=utcnow())\n    stage = \'snapshot\'\n    write_json(folder / \'config.json\', config)\n    write_json(folder / \'identity.json\', record)\n    write_json(folder / \'state.json\', dict(status=\'running\', stage=stage, started_at=record[\'started_at\']))\n    try:\n        # Code and configuration are captured before any fitting takes place.\n        for name, expected in audit[\'code\'].items():\n            target = folder / \'code\' / name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(ROOT / name, target)\n            verify_hash(target, expected)\n        with (folder / \'execution.log\').open(\'w\', encoding=\'utf-8\', newline=\'\\n\') as log, redirect_stdout(Tee(sys.stdout, log)):\n            print(\'Run:\', run_id, \'| mode:\', config[\'mode\'], \'| development years 2007–2020\', flush=True)\n            if config[\'mode\'] == \'train\':\n                stage = \'training\'\n                write_json(folder / \'state.json\', dict(status=\'running\', stage=stage, started_at=record[\'started_at\']))\n                from research.spatial_unet.run_experiment import execute as fit\n                fit(folder / \'training\', blocks=BLOCKS, **config[\'inputs\'], device=config[\'device\'])\n                source = dict(predictions=str(folder / \'training\'), evidence=str(folder / \'training\'),\n                              observations=str(Path(config[\'inputs\'][\'official\']) / \'treino_tp.nc\'))\n                record[\'source_artifacts\'], record[\'source_signature_sha256\'] = inspect_saved(source)\n            else:\n                source = config[\'inputs\']\n            stage = \'diagnostics\'\n            write_json(folder / \'state.json\', dict(status=\'running\', stage=stage, started_at=record[\'started_at\']))\n            copy_evidence(source[\'evidence\'], folder / \'source_evidence\')\n            from research.diagnostics.analyze_errors import execute as diagnose\n            summary = diagnose(source[\'predictions\'], source[\'observations\'], folder / \'diagnostics\', evidence=folder / \'source_evidence\')\n            if config[\'experiment\'] == \'fixed_blend_v1\':\n                stage = \'fixed_blend\'\n                write_json(folder / \'state.json\', dict(status=\'running\', stage=stage, started_at=record[\'started_at\']))\n                from research.fixed_blend.evaluate import execute as evaluate_blend\n                summary = evaluate_blend(source[\'predictions\'], source[\'observations\'], folder / \'fixed_blend\',\n                                         folder / \'diagnostics\', folder / \'source_evidence\')\n            record[\'models_fitted_in_this_run\'] = config[\'mode\'] == \'train\'\n            record[\'evaluation_summary\'] = summary\n            write_json(folder / \'identity.json\', record)\n            stage = \'report\'\n            write_json(folder / \'state.json\', dict(status=\'running\', stage=stage, started_at=record[\'started_at\']))\n            comparisons(folder / \'source_evidence\', folder / \'comparison\')\n            if config[\'experiment\'] == \'fixed_blend_v1\':\n                from research.fixed_blend.report import render\n            else:\n                from .report import render\n            render(folder, record)\n            print(\'Report:\', folder / \'report.html\', flush=True)\n        for name, expected in audit[\'code\'].items():\n            verify_hash(ROOT / name, expected)\n        if config[\'mode\'] == \'saved\':\n            current, _ = inspect_saved(source)\n            require(current == audit[\'inputs\'], \'Inputs changed during evaluation. This run cannot be marked complete.\')\n        write_json(folder / \'state.json\', dict(status=\'completed\', started_at=record[\'started_at\'], finished_at=utcnow(),\n                                             model_promoted=False, forecast_issued=False))\n        write_manifest(folder)\n        verify_run(folder)\n    except (Exception, KeyboardInterrupt) as error:\n        write_json(folder / \'state.json\', dict(status=\'interrupted\' if isinstance(error, KeyboardInterrupt) else \'failed\', stage=stage, started_at=record[\'started_at\'], failed_at=utcnow(),\n                                             error_type=type(error).__name__, error=str(error), model_promoted=False, forecast_issued=False))\n        raise WorkflowError(f\'Run failed during {stage}: {error}\\nPartial outputs preserved in {folder}. Fix the cause and start a new run; no model was promoted.\') from error\n    return folder\n\n\ndef list_runs(root):\n    root = Path(root)\n    require(root.is_dir(), f\'Output directory not found: {root}. Complete a run first or correct --root.\')\n    rows = []\n    for path in sorted(root.glob(\'*/state.json\')):\n        state = read_json(path)\n        record = read_json(path.parent / \'identity.json\')\n        summary = record.get(\'evaluation_summary\', {})\n        rows.append(dict(run_id=record[\'run_id\'], status=state[\'status\'], mode=record[\'mode\'],\n                         experiment=record.get(\'experiment\', \'hybrid_unet_v1\'),\n                         hybrid_rmse=summary.get(\'hybrid_rmse\'), unet_rmse=summary.get(\'unet_rmse\'),\n                         blend_rmse=summary.get(\'blend_rmse\')))\n    return rows\n', 'research/workflow/test_workflow.py': '"""Check configuration boundaries, artifact integrity and failed-run behavior."""\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\n\nfrom research.workflow import config, runner\n\n\ndef example(base):\n    return dict(schema=\'rainfall_workflow_v1\', name=\'example\', mode=\'saved\', output_root=str(base / \'outputs\'),\n                inputs=dict(observations=str(base / \'official/treino_tp.nc\'), predictions=str(base / \'maps\')))\n\n\nclass WorkflowChecks(unittest.TestCase):\n    def test_relative_paths_follow_configuration_not_cwd(self):\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            raw = example(root)\n            raw[\'inputs\'][\'observations\'] = \'official/treino_tp.nc\'\n            value = config.normalize(raw, root)\n            self.assertEqual(Path(value[\'inputs\'][\'observations\']), root / \'official/treino_tp.nc\')\n\n    def test_scientific_or_unknown_settings_are_rejected(self):\n        raw = example(Path(\'/example\'))\n        raw[\'learning_rate\'] = .1\n        with self.assertRaisesRegex(config.WorkflowError, \'Unknown configuration\'):\n            config.normalize(raw, Path.cwd())\n\n    def test_no_output_inside_input_or_gpu_setting_in_saved_mode(self):\n        with tempfile.TemporaryDirectory() as directory:\n            raw = example(Path(directory))\n            raw[\'output_root\'] = str(Path(raw[\'inputs\'][\'predictions\']) / \'overwrite\')\n            with self.assertRaisesRegex(config.WorkflowError, \'inside inputs\'):\n                config.normalize(raw, Path(directory))\n            raw = example(Path(directory)); raw[\'device\'] = \'cuda\'\n            with self.assertRaisesRegex(config.WorkflowError, \'does not use a GPU\'):\n                config.normalize(raw, Path(directory))\n\n    def test_duplicate_json_key_is_rejected(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory) / \'config.json\'\n            path.write_text(\'{"mode":"saved","mode":"train"}\')\n            with self.assertRaisesRegex(config.WorkflowError, \'Duplicate JSON key\'):\n                config.load(path)\n\n    def test_discovery_rejects_missing_and_ambiguous_maps(self):\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            (root / \'treino_tp.nc\').touch()\n            with self.assertRaisesRegex(config.WorkflowError, \'0 complete\'):\n                config.discover_saved(root)\n            for run in [\'one\', \'two\']:\n                base = root / run; base.mkdir(); (base / \'signature.json\').touch()\n                for block in config.BLOCKS:\n                    (base / block).mkdir(); (base / block / \'predictions.nc\').touch()\n            with self.assertRaisesRegex(config.WorkflowError, \'2 complete\'):\n                config.discover_saved(root)\n\n    def completed_fixture(self, root):\n        for name, value in [(\'state.json\', dict(status=\'completed\')), (\'identity.json\', dict(run_id=\'test\')),\n                            (\'config.json\', {}), (\'diagnostics/summary.json\', {})]:\n            path = root / name; path.parent.mkdir(exist_ok=True, parents=True); runner.write_json(path, value)\n        (root / \'report.html\').write_text(\'original\')\n        runner.write_manifest(root)\n\n    def test_changed_and_extra_artifacts_invalidate_completed_run(self):\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            self.completed_fixture(root)\n            self.assertEqual(runner.verify_run(root)[\'status\'], \'verified\')\n            (root / \'report.html\').write_text(\'changed\')\n            with self.assertRaisesRegex(config.WorkflowError, \'hash differs\'):\n                runner.verify_run(root)\n            (root / \'report.html\').write_text(\'original\')\n            (root / \'extra.txt\').write_text(\'not frozen\')\n            with self.assertRaisesRegex(config.WorkflowError, \'added or removed\'):\n                runner.verify_run(root)\n\n    def test_manifest_cannot_read_outside_run(self):\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            self.completed_fixture(root)\n            manifest = config.read_json(root / \'manifest.json\')\n            manifest[\'files\'][\'../outside\'] = dict(sha256=\'0\' * 64, bytes=0)\n            runner.write_json(root / \'manifest.json\', manifest)\n            with self.assertRaisesRegex(config.WorkflowError, \'leaves the run\'):\n                runner.verify_run(root)\n\n    def test_failure_preserves_inputs_and_never_marks_run_complete(self):\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            path = root / \'settings.json\'\n            runner.write_json(path, example(root))\n            original = path.read_bytes()\n            audit = dict(mode=\'saved\', code={}, inputs={}, fingerprint=\'a\' * 64, source_signature_sha256=\'b\' * 64)\n            with patch.object(runner, \'preflight\', return_value=audit), patch.object(runner, \'copy_evidence\', side_effect=ValueError(\'injected missing input\')):\n                with self.assertRaisesRegex(config.WorkflowError, \'Partial outputs preserved\'):\n                    runner.run(path)\n            states = list((root / \'outputs\').glob(\'*/state.json\'))\n            self.assertEqual(len(states), 1)\n            state = config.read_json(states[0])\n            self.assertEqual(state[\'status\'], \'failed\')\n            self.assertFalse(state[\'model_promoted\'])\n            self.assertEqual(path.read_bytes(), original)\n            with self.assertRaisesRegex(config.WorkflowError, \'incomplete or failed\'):\n                runner.verify_run(states[0].parent)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n'}
BUNDLE_SHA256 = {'research/common/__init__.py': '4f6bfaae6ba45669c7d41e1b087df0582970932f06cdea87a4ff3b352f4135f0', 'research/common/baseline.py': 'e2d5e9b633338f2223ee0183587f4fc2b297e71ec75f96e58704a99c32065476', 'research/common/inputs.py': 'd22140d4ad3db4ac3fecdca26c7ebdd1d780486b51622a1f333fce2a637e6851', 'research/common/presentation.py': '869ff90ebd68655e72452aea103841f4e209f4f2eb6b3d32172a77d4cda228a9', 'research/common/synthetic_fixture.py': 'da4de7a662c2c8c25f324c5b543ab04914d490ffde5e316c39187eb8f1640bff', 'research/diagnostics/__init__.py': 'b1af9579ed030dcf6d331837fa659e3e3ea9f5b2441e232b4edbf8e53217954f', 'research/diagnostics/analyze_errors.py': '7ea1920f89dd50d0eab7d83411a0914a819ec0f24f36f538a1ada52534159827', 'research/diagnostics/test_diagnostics.py': '019ce6c5c39cf1dc9a39e57e8ad5f0e80088e89d81ad2a493d03c31e663ad92a', 'research/fixed_blend/__init__.py': '32dbd815c95b70d943619e32716a1f9956df438d993a4f55c925c9c0d390ed81', 'research/fixed_blend/evaluate.py': '6c90f6cb96eaa2841fcec4ad57921b3bf9382f8143d3d2a4b8b04996e1cb44a9', 'research/fixed_blend/protocol.json': '73298bd32c3503532a7068a326c24df5a89163065c97214556c507bdf7a65e6f', 'research/fixed_blend/report.py': '83008093470e80e15488f5ad0e433f91acdeba7af560ae27d5fee9cd55560d03', 'research/fixed_blend/test_blend.py': '263e1efa8944c679ff8db3be4afc25f1fec9a1619a1994130d6c20b7d30c4209', 'research/lagged_sources/evidence/metricas_blocos.csv': 'da2c2af92556c0cb64606e4a8dfd6d5d00af215ecf408cbe326d776df60cb457', 'research/lagged_sources/library.py': '4b92dd77d868d7806fd0ae37d2837abfcde36e173df75b1bd6d88760593ee9c3', 'research/lagged_sources/ocean_indices.csv': '69226b2e65f783b4d4971a254d1e325d0908447c76ec1df3bf532ecef1f0cf5b', 'research/lagged_sources/official_hashes.json': 'bd5648197c4cf25f85d89a32c701b422f1e0b788421766cc7b8aca99ea42223d', 'research/spatial_unet/__init__.py': 'ee933fd666f8d33250d76a02fcd8b898d3a2dc7b341dde8165411387cfa27c19', 'research/spatial_unet/evidence/seven_blocks/A/baseline_check.json': 'e14a23353cd8d4a73ed10f51337c8ca9aac882a12e654dc4979efc3108d062b0', 'research/spatial_unet/evidence/seven_blocks/A/complete.json': '0bb726d929c72c999e89bb43653b70dcf2033f352f76bcba1932eaf0ec023f3a', 'research/spatial_unet/evidence/seven_blocks/A/inner/training.json': '72363f855838e9d4185559d95a00ccc80aa4fe055ebc71763b8c3da5cca09ab2', 'research/spatial_unet/evidence/seven_blocks/A/inner/training_history.json': '3bf2693365a121adaba2828a1e941cd30d298886bf95ec27f96db262e720c950', 'research/spatial_unet/evidence/seven_blocks/A/monthly_metrics.csv': 'e94feeaa24c019b6843a190806ca108dc5ff32556c9a82e5abc996e0e7a279b2', 'research/spatial_unet/evidence/seven_blocks/A/source_calendar.csv': 'd128b8d7afa6626f8bef46d11c0db715d22a6d0b791554fe858b052c250ec991', 'research/spatial_unet/evidence/seven_blocks/A/split.json': '7b13475e8dd96144a7c4f91724bb81cc0c68d9ad10be33f475089fe284b4cb97', 'research/spatial_unet/evidence/seven_blocks/A/unet/training.json': '90d834c5f3c9894531f5cd9829c2059e0e288b63f89928f0495ff19b397c1bbc', 'research/spatial_unet/evidence/seven_blocks/A/unet/training_history.json': '7ebb09bab80a6e31d2e63c1073cc9c6438575321d050f86ad9f39a297e57fec0', 'research/spatial_unet/evidence/seven_blocks/B/baseline_check.json': 'f1d3970e916b9b4a2ccdec3cc2f7f1476d1d4b6fec65a8ae6c7bae72194f2a37', 'research/spatial_unet/evidence/seven_blocks/B/complete.json': '109976d9d35e6ccb343daeb34139f5ddf03ccff552eb3f73803773277fdf2a74', 'research/spatial_unet/evidence/seven_blocks/B/inner/training.json': '86fb5a91dfbcd0ee802b032691767bbe20429316f8ffd55fcc0111626f197ee8', 'research/spatial_unet/evidence/seven_blocks/B/inner/training_history.json': 'a57824d77f62ed7657c61a93057c187b997ec0fb6340c5240ee1be6135e5f2c1', 'research/spatial_unet/evidence/seven_blocks/B/monthly_metrics.csv': '8553057907dcccc2740682c99771b9d010737cd32145493c88f68847a9ba156f', 'research/spatial_unet/evidence/seven_blocks/B/source_calendar.csv': 'fca88bee094820b02b12ad18bc65e9b1090125af011a407174e89400fb2f6e70', 'research/spatial_unet/evidence/seven_blocks/B/split.json': '9f91c64097ede8c47da2404ca89cee79f38e14e508b5bce309925187967a0953', 'research/spatial_unet/evidence/seven_blocks/B/unet/training.json': 'df8c77187b46c19cb97ce24af3e2f11dc87c69c0993d812b1c9abad0c8f5ced7', 'research/spatial_unet/evidence/seven_blocks/B/unet/training_history.json': '73cfbf2afb218971a24bbfe5ebe960a1e9a0087bbc538d26bc64a2b87d70b01e', 'research/spatial_unet/evidence/seven_blocks/C/baseline_check.json': '7f122d971f5a90098d901fe995d1a7c33700c97be5ebea93d3668d9e584207ad', 'research/spatial_unet/evidence/seven_blocks/C/complete.json': '2da0f3ca0c924c11de9721e6735b5174539e6e34151f548b27d602be83a184af', 'research/spatial_unet/evidence/seven_blocks/C/inner/training.json': '9d8fb4da86c2dcc78793842be18bd8e5a4f7ff5f3e30d8fc3511c54bae7e1a31', 'research/spatial_unet/evidence/seven_blocks/C/inner/training_history.json': '588792d87882e50dba74e53f279ff5006c71e4b3a29ebb47a1ee9ec0272b42f9', 'research/spatial_unet/evidence/seven_blocks/C/monthly_metrics.csv': '7e1e251ba0121ca2f2f0e610aa1ed25b964cd0b82cbdb1c01947218063ee4eff', 'research/spatial_unet/evidence/seven_blocks/C/source_calendar.csv': '546f288ede5e7bb85086c835b6f94bde96f77bf14f4fe2259a1933d1cec609a4', 'research/spatial_unet/evidence/seven_blocks/C/split.json': '913001edf8a679143de0b4a95bf96f57d359eb22a2183a0a7e4f4d2d25e02aba', 'research/spatial_unet/evidence/seven_blocks/C/unet/training.json': '9f08d5f86d56687d2fc626545a93d151f602216d68bcf9a9919bf12ed7c47e52', 'research/spatial_unet/evidence/seven_blocks/C/unet/training_history.json': 'b5f1f29b8884f21e05eec676a5ac873c8916c46ab1f71412b32613217aa128e1', 'research/spatial_unet/evidence/seven_blocks/H1/baseline_check.json': 'eaf22b2e51fc3a65d5f9777c76871e1d678a67e77c35f37d4e28fb928b9a228f', 'research/spatial_unet/evidence/seven_blocks/H1/complete.json': '64478bbd8173af1be6c81b842f7c4519d4f86db289e20fac826e0c25a3b893cf', 'research/spatial_unet/evidence/seven_blocks/H1/inner/training.json': '6c2f05c250aa4c694bd9b96bbdc76d497e9bd18b7ec2c814f5141a7f3712082c', 'research/spatial_unet/evidence/seven_blocks/H1/inner/training_history.json': '03d6e27f74a03baf79da6b6cf37428bea999c084f454ed4874b89a8b8dddc77c', 'research/spatial_unet/evidence/seven_blocks/H1/monthly_metrics.csv': '73eaea553f587b1b664ced3760bb1e47c33b25d271ce595c6fc7abf986efb862', 'research/spatial_unet/evidence/seven_blocks/H1/source_calendar.csv': '321b876fcfb1e4c8ef4a4e0e51c0c45b4319c47f09160ad6c08c157c396c402d', 'research/spatial_unet/evidence/seven_blocks/H1/split.json': '2339fc58697d911f5631261d50aa0f2242ac54eabdce38a0d2d9a47a31d2fea7', 'research/spatial_unet/evidence/seven_blocks/H1/unet/training.json': '6c856cbb90035b6e4bbf6a1db33f7acea11edeec23c9787a8ad54f7d39af81f2', 'research/spatial_unet/evidence/seven_blocks/H1/unet/training_history.json': '3013155d59efc48bb61ae1eb8e5924c18ca7f4c366a6a6eee7ffd8245e53e6b4', 'research/spatial_unet/evidence/seven_blocks/H2/baseline_check.json': '1a086678d5bd3a8347fac1a42cb69dcd3e5adae04290c9dad83715f94a05367e', 'research/spatial_unet/evidence/seven_blocks/H2/complete.json': 'f5ea5ff51bdceff56c69b38ee41b78366494f59747b6eb97013730b570920ef3', 'research/spatial_unet/evidence/seven_blocks/H2/inner/training.json': '5ee6d6965db0b3886a2f2bf0391fc9dacd1d3a58c19f9f26af13720dac29a248', 'research/spatial_unet/evidence/seven_blocks/H2/inner/training_history.json': 'dfe2400a2c50455d20669d19208df4aea35130d25ff16c12476291ba8cbb9489', 'research/spatial_unet/evidence/seven_blocks/H2/monthly_metrics.csv': '0fcd88f5c2c791aa09776186eae38f9fd06c54e3c7c13d9486a3c672ce755f9a', 'research/spatial_unet/evidence/seven_blocks/H2/source_calendar.csv': '40a890485099c431c3418b091adafd8d103cc20192bce2d24d18e280a7068507', 'research/spatial_unet/evidence/seven_blocks/H2/split.json': '585fff2facadac324229427d0c7b3ed990b20051cf7dc3f1632f1b99b6daffee', 'research/spatial_unet/evidence/seven_blocks/H2/unet/training.json': '72c64d5d55baa6dcc719dfc38c5d65df70e8f3f052f525cfde085703284d3535', 'research/spatial_unet/evidence/seven_blocks/H2/unet/training_history.json': '79910009d5bd562eb266ae0c58fe33930d990af9b96a9fea35c5338c8b071d58', 'research/spatial_unet/evidence/seven_blocks/H3/baseline_check.json': '644d6cf9bd5f820da8f6d934bf6d771d9fc9a2aff2a2a0ea5f26f15aa33d7bd7', 'research/spatial_unet/evidence/seven_blocks/H3/complete.json': '133b63c716b60e9083e270616af4bf877e25ba4549e229fa38af6d2ac35fa264', 'research/spatial_unet/evidence/seven_blocks/H3/inner/training.json': '3c8c735d4b2dc0e35bbda761ac5812a6ef4cab49cb226af69b912dd577152f45', 'research/spatial_unet/evidence/seven_blocks/H3/inner/training_history.json': '361bc891a20f8a41146c678505ad143e4b709db71505e259ef9dd60b0d6cc5b6', 'research/spatial_unet/evidence/seven_blocks/H3/monthly_metrics.csv': '759c72686d504b1924e12dd226aaaa4c75a7ab1a0eb696a0148ad48d22e8b394', 'research/spatial_unet/evidence/seven_blocks/H3/source_calendar.csv': '83fd3d6e0c46b7dabe91db01412a1e79a5fa71a21795a15d0003ef3032d3fde0', 'research/spatial_unet/evidence/seven_blocks/H3/split.json': '21c77b2eec0d5af702eff0fc9f274dd28fea3251aebc8daf2c8522525a627b09', 'research/spatial_unet/evidence/seven_blocks/H3/unet/training.json': '61e5529a342aaf14026b9a86c9446f231c7c8ca84bf0ff11d6a33b5cf93fa819', 'research/spatial_unet/evidence/seven_blocks/H3/unet/training_history.json': '37db5dbca3e3073d06e4bb23044905a4ac375d82198ff097a2c6d4ca131c8720', 'research/spatial_unet/evidence/seven_blocks/H4/baseline_check.json': 'f63e083fc255c449e3d6ded37cda31783f7fce53bc5e77da09f32f1faeeba638', 'research/spatial_unet/evidence/seven_blocks/H4/complete.json': 'c96c02069fa58efdd016535b73703749b7b88c910a5f073f66ef03b132a4d490', 'research/spatial_unet/evidence/seven_blocks/H4/inner/training.json': '91b3dce3ef9877051688692dfa9139e818f42b88f59ade24909e68eeed0b9ae7', 'research/spatial_unet/evidence/seven_blocks/H4/inner/training_history.json': '53787e9943551bcaf186863383f31ef8ba92329e4f7623d49c86106fef56a273', 'research/spatial_unet/evidence/seven_blocks/H4/monthly_metrics.csv': '280d7197b163b5de0104078f25dd0438bcf4e9705e3b0681054e33c3c1d6e603', 'research/spatial_unet/evidence/seven_blocks/H4/source_calendar.csv': 'db9860b421d3190bc58f469badfad1a2375b9e9d2d14a235b3a07b32810739ef', 'research/spatial_unet/evidence/seven_blocks/H4/split.json': '231802dd65c0e6995c5177860d036abd0e0f7292dade92dce64749a0717f1494', 'research/spatial_unet/evidence/seven_blocks/H4/unet/training.json': '31c17e8568e8f5b159cf40ca194bd2def932192be5923fdad5e4622ead6d8981', 'research/spatial_unet/evidence/seven_blocks/H4/unet/training_history.json': '6ae88f922f6f1fd73f1f5ecc91b4c64cfb16f8f91bd1e22b0771362bbdadbdd9', 'research/spatial_unet/evidence/seven_blocks/global.csv': '81876a63390371ba3f3a4eb757222c54e487a8df7e13571d54ac8ffeb314308b', 'research/spatial_unet/evidence/seven_blocks/signature.json': '02d1ffc0a7205f2bf372673d337cf5daa95f125d43a95836b953c10ae101244c', 'research/spatial_unet/network.py': 'f1d10bbe1634018857b4fda8ae2753c1cf43d553f221e338bd1d6dcd89f40654', 'research/spatial_unet/protocol.json': 'b5819403ce177b1c69cfd698160c0e2339f040d5d9ab0a4ad9608a469872c694', 'research/spatial_unet/run_experiment.py': 'ba96e4040cf2f168ad9b1ca7100257be816ec6190473d65355bc57720c3394f4', 'research/spatial_unet/test_spatial.py': '65215314cb5ebd1229ded058c4aab91ab71b9ceccc6af8a83de25668301a8516', 'research/spatial_unet/training.py': '698d27f397a312c3a4a5f832a15fa09fe431095431b74ca328a8034c3f232275', 'research/workflow/__init__.py': '4a5871494e07fb5dc9bed6efc910d5d8aadb4d876c103e0562acb899824ee459', 'research/workflow/__main__.py': '1ae45af70bb0cea703fced2626c4a9120799f9ac5e26191057c250bcab8c25be', 'research/workflow/config.py': 'c590c9e30b6cb196a765abbee24ccfbac14127b5aefd1571f5a1e78ff55e4ef4', 'research/workflow/report.py': '98ea73206d7da9e9c3d129f5f8105cdbc22d40eea9308e63e22ebe4f5fb1bbcb', 'research/workflow/runner.py': '267f6383c70f535ff58c0a41d75b5e977e001215e5887affa12ee52624c94482', 'research/workflow/test_workflow.py': 'cff78ab0e3a64f0ccc0676bc9152401066a12351bc37b265b62972dea457ed4b'}
for name, content in BUNDLE.items():
    assert hashlib.sha256(content.encode('utf-8')).hexdigest() == BUNDLE_SHA256[name]
    path = CODE_ROOT/name
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise RuntimeError('Different source code already exists. Preserve this run and choose a new RUN_ROOT.')
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8', newline='\n')
sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT/'research/prospective'))
print('Reviewed source bundle:', CODE_ROOT)


## 1. Configure paths and mode
Saved mode evaluates the previous seven-block output and uses no GPU. Keep the default unless you intend to refit all models. The lookup requires the forecast maps, not only the report archive.

In [ ]:
from research.workflow.config import discover_saved, WorkflowError
from research.workflow.runner import write_json
MODE = "saved"  # "train" reruns the complete fixed seven-block comparison.
EXPERIMENT = "hybrid_unet_v1"  # "fixed_blend_v1" evaluates only 75% reference model + 25% U-Net, in saved mode.
MANUAL_INPUTS = None
# Saved example:
# MANUAL_INPUTS = {"observations": "/kaggle/input/.../treino_tp.nc",
#                  "predictions": "/kaggle/input/.../spatial_unet/results",
#                  "evidence": "/kaggle/input/.../spatial_unet/results"}
if MODE == "saved":
    INPUTS = MANUAL_INPUTS if MANUAL_INPUTS is not None else discover_saved()
elif MODE == "train":
    def unique_parent(filename):
        found = sorted(Path('/kaggle/input').rglob(filename))
        if len(found) != 1:
            raise WorkflowError(f"Found {len(found)} {filename} files. Set MANUAL_INPUTS to the intended directories.")
        return str(found[0].parent)
    INPUTS = MANUAL_INPUTS if MANUAL_INPUTS is not None else {
        "official": unique_parent("treino_tp.nc"),
        "seas5": unique_parent("seas5_51_manifesto.json"),
        "cfsv2": unique_parent("cfsv2_manifesto.json"),
    }
else:
    raise WorkflowError("MODE must be saved or train.")
CONFIG = {"schema": "rainfall_workflow_v1", "name": "hybrid-unet-comparison",
          "mode": MODE, "experiment": EXPERIMENT, "output_root": str(BASE/'experiments'), "inputs": INPUTS}
if MODE == "train":
    CONFIG["device"] = "cuda"
CONFIG_FILE = RUN_ROOT/'workflow_config.json'
write_json(CONFIG_FILE, CONFIG)
print(json.dumps(CONFIG, indent=2))


## 2. Check before execution
These tests use small synthetic fixtures. Preflight verifies the configured real files without fitting a model. Missing data, changed hashes or an incompatible training environment stop here.

In [ ]:
import unittest
from research.workflow.__main__ import main
suite = unittest.defaultTestLoader.loadTestsFromNames(['research.workflow.test_workflow', 'research.diagnostics.test_diagnostics', 'research.fixed_blend.test_blend'])
assert unittest.TextTestRunner(verbosity=2).run(suite).wasSuccessful(), 'Checks failed.'
assert main(['check', '--config', str(CONFIG_FILE)]) == 0, 'Preflight stopped; read the message above.'


## 3. Execute the configured comparison
Each execution creates a new run directory and preserves its configuration, code, evidence, logs and outputs. No result changes the current forecasting model or issues a forecast.

In [ ]:
from research.workflow.runner import run, verify_run
RUN = run(CONFIG_FILE)
print(verify_run(RUN))


## 4. Read and preserve the report
The HTML embeds its figures and can be downloaded on its own. Save the complete notebook output version for the audit files and, in training mode, the fitted models and predictions.

In [ ]:
from IPython.display import display, HTML, FileLink
display(FileLink(str((RUN/'report.html').relative_to(BASE))))
display(HTML((RUN/'report.html').read_text(encoding='utf-8')))
print('Complete run:', RUN)
print('Save Version with outputs to preserve this run.')
